In [ ]:
!pip install dukascopy_python

In [ ]:
# ════════════════════════════════════════════════════════════════════
# VERİ ÇEKİMİ + FEATURE ENGINEERING
# Blok 00 → feature_df ve df (teknik indikatörler) hazır
# Sıra: install → FX çek → macro çek → TZ-fix → 4H align → features
# ════════════════════════════════════════════════════════════════════

# !pip install dukascopy_python hmmlearn

import numpy as np
import pandas as pd
import yfinance as yf
import pandas_datareader.data as web
import warnings
import logging

from sklearn.feature_selection import mutual_info_regression

warnings.filterwarnings("ignore")
logging.getLogger("yfinance").setLevel(logging.ERROR)

# ════════════════════════════════════════════════════════════════════
# BLOK 00 — GLOBAL PARAMETRELER
# ════════════════════════════════════════════════════════════════════

START = "2004-01-01"

start_dk = pd.Timestamp("2000-01-01").to_pydatetime()
end_dk   = pd.Timestamp("2026-01-01").to_pydatetime()

BAR_PER_DAY = 6

W_5D   = 5   * BAR_PER_DAY    #  30 bar
W_14D  = 14  * BAR_PER_DAY    #  84 bar
W_20D  = 20  * BAR_PER_DAY    # 120 bar
W_21D  = 21  * BAR_PER_DAY    # 126 bar
W_50D  = 50  * BAR_PER_DAY    # 300 bar
W_63D  = 63  * BAR_PER_DAY    # 378 bar
W_252D = 252 * BAR_PER_DAY    # 1512 bar


# ════════════════════════════════════════════════════════════════════
# BLOK 01 — YARDIMCI FONKSİYONLAR
# ════════════════════════════════════════════════════════════════════

def make_index_tz_naive(idx):
    idx = pd.to_datetime(idx)
    try:
        if getattr(idx, "tz", None) is not None:
            idx = idx.tz_convert(None)
    except Exception:
        pass
    try:
        if getattr(idx, "tz", None) is not None:
            idx = idx.tz_localize(None)
    except Exception:
        pass
    return idx


def force_tz_naive_series(s):
    s = s.copy()
    s.index = pd.to_datetime(s.index)
    try:
        if s.index.tz is not None:
            s.index = s.index.tz_convert(None)
    except Exception:
        pass
    try:
        if getattr(s.index, "tz", None) is not None:
            s.index = s.index.tz_localize(None)
    except Exception:
        pass
    s = s.sort_index()
    s = s[~s.index.duplicated(keep="last")]
    return s


def prepare_ohlc(df, name="pair"):
    df = df.copy()
    df.index = make_index_tz_naive(df.index)
    df = df.sort_index()
    needed = ["open", "high", "low", "close"]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"{name} içinde eksik kolon(lar): {missing}")
    if "volume" not in df.columns:
        df["volume"] = np.nan
    return df


def align_market_to_4h(s, idx, interpolate=False):
    s = pd.Series(s).copy()
    s.index = make_index_tz_naive(s.index)
    s = s.sort_index()
    s = s[~s.index.duplicated(keep="last")]
    idx = make_index_tz_naive(idx)
    out = s.reindex(s.index.union(idx)).sort_index()
    if interpolate:
        out = out.interpolate(method="time")
    return out.ffill().reindex(idx)


def align_macro_to_4h(s, idx, lag_periods=0):
    s = pd.Series(s).copy()
    s.index = make_index_tz_naive(s.index)
    s = s.sort_index()
    s = s[~s.index.duplicated(keep="last")]
    idx = make_index_tz_naive(idx)
    out = s.reindex(s.index.union(idx)).sort_index().ffill().reindex(idx)
    if lag_periods > 0:
        out = out.shift(lag_periods)
    return out


def rolling_zscore(s, w=252):
    return (s - s.rolling(w).mean()) / (s.rolling(w).std() + 1e-10)


def safe_logret(s):
    return np.log(s / s.shift(1))


def calc_rsi(series, period=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).ewm(alpha=1/period, adjust=False).mean()
    loss  = (-delta.clip(upper=0)).ewm(alpha=1/period, adjust=False).mean()
    rs    = gain / (loss + 1e-10)
    return 100 - (100 / (1 + rs))

In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK 02 — DUKASCOPY FX ÇEKİMİ
# ════════════════════════════════════════════════════════════════════

import dukascopy_python as dk
from dukascopy_python.instruments import (
    INSTRUMENT_FX_MAJORS_EUR_USD,
    INSTRUMENT_FX_MAJORS_GBP_USD,
    INSTRUMENT_FX_MAJORS_USD_CHF,
    INSTRUMENT_FX_MAJORS_USD_JPY,
    INSTRUMENT_FX_CROSSES_EUR_CHF,
    INSTRUMENT_FX_CROSSES_EUR_GBP,
    INSTRUMENT_FX_CROSSES_EUR_JPY,
)

def fetch_pair_dk(instrument):
    bid = dk.fetch(
        instrument=instrument,
        interval=dk.INTERVAL_HOUR_4,
        offer_side=dk.OFFER_SIDE_BID,
        start=start_dk,
        end=end_dk,
    ).copy().sort_index()

    ask = dk.fetch(
        instrument=instrument,
        interval=dk.INTERVAL_HOUR_4,
        offer_side=dk.OFFER_SIDE_ASK,
        start=start_dk,
        end=end_dk,
    ).copy().sort_index()

    bid = bid.rename(columns={
        "open": "open_bid", "high": "high_bid",
        "low": "low_bid",   "close": "close_bid", "volume": "volume_bid"
    })
    ask = ask.rename(columns={
        "open": "open_ask", "high": "high_ask",
        "low": "low_ask",   "close": "close_ask", "volume": "volume_ask"
    })

    df = bid.join(ask)
    df["open"]   = (df["open_bid"]   + df["open_ask"])   / 2
    df["high"]   = (df["high_bid"]   + df["high_ask"])   / 2
    df["low"]    = (df["low_bid"]    + df["low_ask"])    / 2
    df["close"]  = (df["close_bid"]  + df["close_ask"])  / 2
    df["volume"] = (df["volume_bid"] + df["volume_ask"]) / 2
    return df

print("Dukascopy FX pair çekimi başlıyor...")
eurusd_df = fetch_pair_dk(INSTRUMENT_FX_MAJORS_EUR_USD)
gbpusd_df = fetch_pair_dk(INSTRUMENT_FX_MAJORS_GBP_USD)
usdchf_df = fetch_pair_dk(INSTRUMENT_FX_MAJORS_USD_CHF)
usdjpy_df = fetch_pair_dk(INSTRUMENT_FX_MAJORS_USD_JPY)
eurchf_df = fetch_pair_dk(INSTRUMENT_FX_CROSSES_EUR_CHF)
eurgbp_df = fetch_pair_dk(INSTRUMENT_FX_CROSSES_EUR_GBP)
eurjpy_df = fetch_pair_dk(INSTRUMENT_FX_CROSSES_EUR_JPY)
print("✓ FX pair verisi tamam")

# ── Gold (Dukascopy XAU/USD) ──
gold_raw  = dk.fetch(
    instrument="XAU/USD",
    interval=dk.INTERVAL_HOUR_4,
    offer_side=dk.OFFER_SIDE_BID,
    start=start_dk,
    end=end_dk,
).copy().sort_index()
gold_yf = gold_raw["close"].copy()   # Series olarak sakla — align_market_to_4h için
print("✓ Gold verisi tamam")


# ════════════════════════════════════════════════════════════════════
# BLOK 03 — OHLC HAZIRLIK + 4H BASE INDEX
# ════════════════════════════════════════════════════════════════════

eurusd_df = prepare_ohlc(eurusd_df, "eurusd_df")
gbpusd_df = prepare_ohlc(gbpusd_df, "gbpusd_df")
usdchf_df = prepare_ohlc(usdchf_df, "usdchf_df")
usdjpy_df = prepare_ohlc(usdjpy_df, "usdjpy_df")
eurchf_df = prepare_ohlc(eurchf_df, "eurchf_df")
eurgbp_df = prepare_ohlc(eurgbp_df, "eurgbp_df")
eurjpy_df = prepare_ohlc(eurjpy_df, "eurjpy_df")

base_index = eurusd_df.loc[eurusd_df.index >= pd.Timestamp(START)].index.copy()
base_index = make_index_tz_naive(base_index)

# Ana OHLC serileri (EUR/USD mid)
O = eurusd_df["open"].reindex(base_index)
H = eurusd_df["high"].reindex(base_index)
L = eurusd_df["low"].reindex(base_index)
C = eurusd_df["close"].reindex(base_index)
V = eurusd_df["volume"].reindex(base_index)

log_ret = np.log(C / C.shift(1))

print(f"EUR/USD 4H bar sayısı : {len(C)}")
print(f"İlk tarih             : {base_index.min().date()}")
print(f"Son tarih             : {base_index.max().date()}")
print(f"log_ret mean={log_ret.mean():.6f}, std={log_ret.std():.6f}")


Dukascopy FX pair çekimi başlıyor...


INFO:DUKASCRIPT:current timestamp :2008-01-23T00:00:00
INFO:DUKASCRIPT:current timestamp :2012-03-20T00:00:00
INFO:DUKASCRIPT:current timestamp :2016-03-10T04:00:00
INFO:DUKASCRIPT:current timestamp :2020-12-31T00:00:00
INFO:DUKASCRIPT:current timestamp :2025-10-22T00:00:00
INFO:DUKASCRIPT:current timestamp :2008-01-23T00:00:00
INFO:DUKASCRIPT:current timestamp :2012-03-20T00:00:00
INFO:DUKASCRIPT:current timestamp :2016-03-10T04:00:00
INFO:DUKASCRIPT:current timestamp :2020-12-31T00:00:00
INFO:DUKASCRIPT:current timestamp :2025-10-22T00:00:00
INFO:DUKASCRIPT:current timestamp :2008-01-17T08:00:00
INFO:DUKASCRIPT:current timestamp :2012-02-24T12:00:00
INFO:DUKASCRIPT:current timestamp :2016-02-05T08:00:00
INFO:DUKASCRIPT:current timestamp :2020-11-26T20:00:00
INFO:DUKASCRIPT:current timestamp :2025-09-18T12:00:00
INFO:DUKASCRIPT:current timestamp :2008-01-17T08:00:00
INFO:DUKASCRIPT:current timestamp :2012-02-24T12:00:00
INFO:DUKASCRIPT:current timestamp :2016-02-05T08:00:00
INFO:DUKAS

✓ FX pair verisi tamam


INFO:DUKASCRIPT:current timestamp :2008-01-24T20:00:00
INFO:DUKASCRIPT:current timestamp :2012-07-23T16:00:00
INFO:DUKASCRIPT:current timestamp :2017-07-09T20:00:00
INFO:DUKASCRIPT:current timestamp :2022-08-04T20:00:00


✓ Gold verisi tamam
EUR/USD 4H bar sayısı : 37614
İlk tarih             : 2004-01-01
Son tarih             : 2025-12-31
log_ret mean=-0.000002, std=0.002214


In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK 04 — PAIR CLOSE + LOG RETURN SERİLERİ
# ════════════════════════════════════════════════════════════════════

gbpusd = align_market_to_4h(gbpusd_df["close"], base_index)
usdchf = align_market_to_4h(usdchf_df["close"], base_index)
usdjpy = align_market_to_4h(usdjpy_df["close"], base_index)
eurchf = align_market_to_4h(eurchf_df["close"], base_index)
eurgbp = align_market_to_4h(eurgbp_df["close"], base_index)
eurjpy = align_market_to_4h(eurjpy_df["close"], base_index)

gbp_ret = safe_logret(gbpusd)
chf_ret = safe_logret(usdchf)
jpy_ret = safe_logret(usdjpy)
ech_ret = safe_logret(eurchf)
egb_ret = safe_logret(eurgbp)
ejp_ret = safe_logret(eurjpy)

print("✓ Pair close ve log return serileri hazır")

✓ Pair close ve log return serileri hazır


In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK 05 — YFINANCE + FRED MAKRO VERİ ÇEKİMİ
# ════════════════════════════════════════════════════════════════════

print("\nYFinance macro çekiliyor...")
vix_yf  = yf.download("^VIX",     start=START, auto_adjust=True, progress=False)["Close"].squeeze()
dxy_yf  = yf.download("DX-Y.NYB", start=START, auto_adjust=True, progress=False)["Close"].squeeze()
oil_yf  = yf.download("BZ=F",     start=START, auto_adjust=True, progress=False)["Close"].squeeze()
hyg_yf  = yf.download("HYG",      start=START, auto_adjust=True, progress=False)["Close"].squeeze()
lqd_yf  = yf.download("LQD",      start=START, auto_adjust=True, progress=False)["Close"].squeeze()
spx_yf  = yf.download("^GSPC",    start=START, auto_adjust=True, progress=False)["Close"].squeeze()
print("✓ YFinance tamam")

print("FRED çekiliyor...")
us10y  = web.DataReader("DGS10",         "fred", START).squeeze().dropna()
us2y   = web.DataReader("DGS2",          "fred", START).squeeze().dropna()
cfnai  = web.DataReader("CFNAI",         "fred", START).squeeze().dropna()
cpi    = web.DataReader("CPIAUCSL",      "fred", START).squeeze().dropna()
m2     = web.DataReader("M2SL",          "fred", START).squeeze().dropna()
fed    = web.DataReader("WALCL",         "fred", START).squeeze().dropna()
tga    = web.DataReader("WDTGAL",        "fred", START).squeeze().dropna()
rrp    = web.DataReader("RRPONTSYD",     "fred", START).squeeze().dropna()
hy_oas = web.DataReader("BAMLH0A0HYM2",  "fred", START).squeeze().dropna()
unemp  = web.DataReader("UNRATE",        "fred", START).squeeze().dropna()
mfg_o  = web.DataReader("AMTMNO",        "fred", START).squeeze().dropna()
print("✓ FRED tamam")

# ── TZ-naive düzelt ──
for s in [vix_yf, dxy_yf, oil_yf, hyg_yf, lqd_yf, spx_yf,
          us10y, us2y, cfnai, cpi, m2, fed, tga, rrp, hy_oas, unemp, mfg_o]:
    s.index = make_index_tz_naive(s.index)

gold_yf.index = make_index_tz_naive(gold_yf.index)



YFinance macro çekiliyor...
✓ YFinance tamam
FRED çekiliyor...


ReadTimeout: HTTPSConnectionPool(host='fred.stlouisfed.org', port=443): Read timed out. (read timeout=30)

In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK 05 — YFINANCE + FRED MAKRO VERİ ÇEKİMİ
# ════════════════════════════════════════════════════════════════════

import requests, time, io
import pandas as pd

# ── FRED API KEY ──────────────────────────────────────────────────
FRED_API_KEY = "e96810a58bb2f1bf181099530c4b26c0"   # ← tek değişiklik

# ── FRED çekim fonksiyonu ─────────────────────────────────────────
def fetch_fred(series_id, start, api_key, retries=4, backoff=3):
    url = (
        f"https://api.stlouisfed.org/fred/series/observations"
        f"?series_id={series_id}"
        f"&observation_start={start}"
        f"&api_key={api_key}"
        f"&file_type=json"
    )
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=60)
            r.raise_for_status()
            obs = r.json()["observations"]
            s = pd.Series(
                {d["date"]: float(d["value"]) for d in obs if d["value"] != "."},
                name=series_id
            )
            s.index = pd.to_datetime(s.index)
            return s.dropna()
        except Exception as e:
            if attempt < retries - 1:
                wait = backoff * (2 ** attempt)   # 3 → 6 → 12 → 24s
                print(f"  ⚠ {series_id} attempt {attempt+1} başarısız, {wait}s bekleniyor...")
                time.sleep(wait)
            else:
                raise RuntimeError(f"❌ {series_id} çekilemedi: {e}")

# ── YFinance ─────────────────────────────────────────────────────
print("\nYFinance macro çekiliyor...")
vix_yf  = yf.download("^VIX",     start=START, auto_adjust=True, progress=False)["Close"].squeeze()
dxy_yf  = yf.download("DX-Y.NYB", start=START, auto_adjust=True, progress=False)["Close"].squeeze()
oil_yf  = yf.download("BZ=F",     start=START, auto_adjust=True, progress=False)["Close"].squeeze()
hyg_yf  = yf.download("HYG",      start=START, auto_adjust=True, progress=False)["Close"].squeeze()
lqd_yf  = yf.download("LQD",      start=START, auto_adjust=True, progress=False)["Close"].squeeze()
spx_yf  = yf.download("^GSPC",    start=START, auto_adjust=True, progress=False)["Close"].squeeze()
print("✓ YFinance tamam")

# ── FRED ─────────────────────────────────────────────────────────
FRED_SERIES = {
    "us10y"  : "DGS10",
    "us2y"   : "DGS2",
    "cfnai"  : "CFNAI",
    "cpi"    : "CPIAUCSL",
    "m2"     : "M2SL",
    "fed"    : "WALCL",
    "tga"    : "WDTGAL",
    "rrp"    : "RRPONTSYD",
    "hy_oas" : "BAMLH0A0HYM2",
    "unemp"  : "UNRATE",
    "mfg_o"  : "AMTMNO",
}

print("FRED çekiliyor...")
fred_data = {}
for name, sid in FRED_SERIES.items():
    fred_data[name] = fetch_fred(sid, START, api_key=FRED_API_KEY)
    time.sleep(0.3)
    print(f"  ✓ {name} — {len(fred_data[name])} obs")

us10y  = fred_data["us10y"];   us2y   = fred_data["us2y"]
cfnai  = fred_data["cfnai"];   cpi    = fred_data["cpi"]
m2     = fred_data["m2"];      fed    = fred_data["fed"]
tga    = fred_data["tga"];     rrp    = fred_data["rrp"]
hy_oas = fred_data["hy_oas"];  unemp  = fred_data["unemp"]
mfg_o  = fred_data["mfg_o"]
print("✓ FRED tamam")

# ── TZ-naive düzelt ──────────────────────────────────────────────
for s in [vix_yf, dxy_yf, oil_yf, hyg_yf, lqd_yf, spx_yf,
          us10y, us2y, cfnai, cpi, m2, fed, tga, rrp, hy_oas, unemp, mfg_o]:
    s.index = make_index_tz_naive(s.index)

gold_yf.index = make_index_tz_naive(gold_yf.index)


YFinance macro çekiliyor...
✓ YFinance tamam
FRED çekiliyor...
  ✓ us10y — 5609 obs
  ✓ us2y — 5609 obs
  ✓ cfnai — 268 obs
  ✓ cpi — 267 obs
  ✓ m2 — 268 obs
  ✓ fed — 1170 obs
  ✓ tga — 1170 obs
  ✓ rrp — 3246 obs
  ✓ hy_oas — 787 obs
  ✓ unemp — 267 obs
  ✓ mfg_o — 268 obs
✓ FRED tamam


In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK 05B — DIRECT LAG APPLICATION
# Amaç:
# - Mevcut değişken isimlerini bozmadan release-lag uygulamak
# - Sonraki kodlarda değişiklik yapmadan leakage riskini azaltmak
# ════════════════════════════════════════════════════════════════════

def lag_series_by_days(s, days):
    """
    Calendar-day lag.
    Makro seriler için kullanılır.
    Örn: CPI 30 gün sonra biliniyor varsayımı.
    """
    s = s.copy().dropna()
    s.index = make_index_tz_naive(s.index)
    s.index = s.index + pd.Timedelta(days=int(days))
    return s.sort_index()


def lag_series_by_bdays(s, bdays):
    """
    Business-day lag.
    Günlük piyasa serileri için kullanılır.
    """
    s = s.copy().dropna()
    s.index = make_index_tz_naive(s.index)
    s.index = s.index + pd.offsets.BDay(int(bdays))
    return s.sort_index()


print("\nRelease-lag uygulanıyor...")

# ------------------------------------------------------------
# 1) YFinance market-based günlük seriler
# ------------------------------------------------------------
# Günlük kapanış datası aynı günün intraday barlarında bilinmez.
# Bu yüzden 1 business day lag.

vix_yf = lag_series_by_bdays(vix_yf, 1)
dxy_yf = lag_series_by_bdays(dxy_yf, 1)
oil_yf = lag_series_by_bdays(oil_yf, 1)
hyg_yf = lag_series_by_bdays(hyg_yf, 1)
lqd_yf = lag_series_by_bdays(lqd_yf, 1)
spx_yf = lag_series_by_bdays(spx_yf, 1)


# ------------------------------------------------------------
# 2) FRED daily / market-like seriler
# ------------------------------------------------------------
# Günlük faiz ve credit spread verileri için 1 business day güvenli.
# TGA için 3 business day daha konservatif.

us10y  = lag_series_by_bdays(us10y, 1)
us2y   = lag_series_by_bdays(us2y, 1)
hy_oas = lag_series_by_bdays(hy_oas, 1)
rrp    = lag_series_by_bdays(rrp, 1)
tga    = lag_series_by_bdays(tga, 3)


# ------------------------------------------------------------
# 3) FRED weekly seriler
# ------------------------------------------------------------
# WALCL haftalık Fed bilançosu.
# 7 calendar day lag daha savunulabilir.

fed = lag_series_by_days(fed, 7)


# ------------------------------------------------------------
# 4) FRED monthly macro seriler
# ------------------------------------------------------------
# Aylık makrolar açıklanma gecikmesiyle kullanılır.
# CPI, M2, CFNAI için 30 gün.
# Unemployment için 30 gün.
# Manufacturing orders için 45 gün.

cpi   = lag_series_by_days(cpi, 30)
m2    = lag_series_by_days(m2, 30)
cfnai = lag_series_by_days(cfnai, 30)
unemp = lag_series_by_days(unemp, 30)
mfg_o = lag_series_by_days(mfg_o, 45)

print("✓ Release-lag mevcut değişkenlerin üzerine uygulandı")


Release-lag uygulanıyor...
✓ Release-lag mevcut değişkenlerin üzerine uygulandı


In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK 05A — RAW DATA HEALTH CHECK
# Amaç: YFinance + FRED serileri boş mu, tarih aralığı ne, NaN oranı ne?
# ════════════════════════════════════════════════════════════════════

raw_series = {
    # YFinance / market
    "vix_yf": vix_yf,
    "dxy_yf": dxy_yf,
    "oil_yf": oil_yf,
    "hyg_yf": hyg_yf,
    "lqd_yf": lqd_yf,
    "spx_yf": spx_yf,
    "gold_yf": gold_yf,

    # FRED
    "us10y": us10y,
    "us2y": us2y,
    "cfnai": cfnai,
    "cpi": cpi,
    "m2": m2,
    "fed": fed,
    "tga": tga,
    "rrp": rrp,
    "hy_oas": hy_oas,
    "unemp": unemp,
    "mfg_o": mfg_o,
}

health_rows = []

for name, s in raw_series.items():
    s = pd.Series(s).copy()
    s.index = make_index_tz_naive(s.index)
    s = s.sort_index()
    s = s[~s.index.duplicated(keep="last")]

    n_total = len(s)
    n_nan = int(s.isna().sum())
    n_valid = int(s.notna().sum())

    first_valid = s.dropna().index.min() if n_valid > 0 else pd.NaT
    last_valid = s.dropna().index.max() if n_valid > 0 else pd.NaT

    health_rows.append({
        "series": name,
        "n_total": n_total,
        "n_valid": n_valid,
        "n_nan": n_nan,
        "nan_pct": 100 * n_nan / max(n_total, 1),
        "first_valid": first_valid,
        "last_valid": last_valid,
        "is_empty": n_valid == 0,
    })

raw_health = pd.DataFrame(health_rows).sort_values(["is_empty", "n_valid"], ascending=[False, True])

print("=" * 120)
print("RAW DATA HEALTH CHECK — YFinance + FRED + Gold")
print("=" * 120)
print(raw_health.to_string(index=False))

RAW DATA HEALTH CHECK — YFinance + FRED + Gold
 series  n_total  n_valid  n_nan  nan_pct first_valid          last_valid  is_empty
    cpi      267      267      0      0.0  2004-01-31 2026-05-01 00:00:00     False
  unemp      267      267      0      0.0  2004-01-15 2026-04-15 00:00:00     False
  cfnai      268      268      0      0.0  2004-01-31 2026-05-01 00:00:00     False
     m2      268      268      0      0.0  2004-01-31 2026-05-01 00:00:00     False
  mfg_o      268      268      0      0.0  2004-02-15 2026-05-16 00:00:00     False
 hy_oas      776      776      0      0.0  2023-06-06 2026-06-04 00:00:00     False
    fed     1170     1170      0      0.0  2004-01-14 2026-06-10 00:00:00     False
    tga     1170     1170      0      0.0  2004-01-12 2026-06-08 00:00:00     False
    rrp     3246     3246      0      0.0  2004-01-07 2026-06-05 00:00:00     False
 oil_yf     4692     4692      0      0.0  2007-07-31 2026-06-08 00:00:00     False
 hyg_yf     4819     4819    

In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK 06 — MAKRO SERİLERİ 4H INDEX'E HİZALA
# ════════════════════════════════════════════════════════════════════

# yfinance (piyasa — günlük, time-interpolate edilir)
vix_4h  = align_market_to_4h(vix_yf,  base_index)
dxy_4h  = align_market_to_4h(dxy_yf,  base_index)
gold_4h = align_market_to_4h(gold_yf, base_index)
oil_4h  = align_market_to_4h(oil_yf,  base_index)
hyg_4h  = align_market_to_4h(hyg_yf,  base_index)
lqd_4h  = align_market_to_4h(lqd_yf,  base_index)
spx_4h  = align_market_to_4h(spx_yf,  base_index)

# FRED (düşük frekans — ffill + lag_periods=1 bar lookahead önlemi)
us10y_4h  = align_macro_to_4h(us10y,  base_index, lag_periods=1)
us2y_4h   = align_macro_to_4h(us2y,   base_index, lag_periods=1)
cfnai_4h  = align_macro_to_4h(cfnai,  base_index, lag_periods=1)
cpi_4h    = align_macro_to_4h(cpi,    base_index, lag_periods=1)
m2_4h     = align_macro_to_4h(m2,     base_index, lag_periods=1)
fed_4h    = align_macro_to_4h(fed,    base_index, lag_periods=1)
tga_4h    = align_macro_to_4h(tga,    base_index, lag_periods=1)
rrp_4h    = align_macro_to_4h(rrp,    base_index, lag_periods=1)
hy_oas_4h = align_macro_to_4h(hy_oas, base_index, lag_periods=1)
unemp_4h  = align_macro_to_4h(unemp,  base_index, lag_periods=1)
mfg_o_4h  = align_macro_to_4h(mfg_o,  base_index, lag_periods=1)

print("✓ Tüm macro seriler 4H index'e hizalandı")

✓ Tüm macro seriler 4H index'e hizalandı


In [ ]:
aligned_macro_check = pd.DataFrame({
    "vix_4h": vix_4h,
    "dxy_4h": dxy_4h,
    "gold_4h": gold_4h,
    "oil_4h": oil_4h,
    "hyg_4h": hyg_4h,
    "lqd_4h": lqd_4h,
    "spx_4h": spx_4h,
    "us10y_4h": us10y_4h,
    "us2y_4h": us2y_4h,
    "cfnai_4h": cfnai_4h,
    "cpi_4h": cpi_4h,
    "m2_4h": m2_4h,
    "fed_4h": fed_4h,
    "tga_4h": tga_4h,
    "rrp_4h": rrp_4h,
    "hy_oas_4h": hy_oas_4h,
    "unemp_4h": unemp_4h,
    "mfg_o_4h": mfg_o_4h,
})

check = pd.DataFrame({
    "n_total": aligned_macro_check.shape[0],
    "n_valid": aligned_macro_check.notna().sum(),
    "n_nan": aligned_macro_check.isna().sum(),
    "nan_pct": aligned_macro_check.isna().mean() * 100,
    "first_valid": aligned_macro_check.apply(lambda x: x.dropna().index.min() if x.notna().any() else pd.NaT),
    "last_valid": aligned_macro_check.apply(lambda x: x.dropna().index.max() if x.notna().any() else pd.NaT),
}).sort_values("nan_pct", ascending=False)

print("=" * 120)
print("ALIGNED 4H MACRO CHECK")
print("=" * 120)
print(check.to_string())

ALIGNED 4H MACRO CHECK
           n_total  n_valid  n_nan    nan_pct         first_valid          last_valid
hy_oas_4h    37614     4145  33469  88.980167 2023-06-06 04:00:00 2025-12-31 20:00:00
oil_4h       37614    31804   5810  15.446376 2007-07-31 00:00:00 2025-12-31 20:00:00
hyg_4h       37614    32302   5312  14.122401 2007-04-12 00:00:00 2025-12-31 20:00:00
mfg_o_4h     37614    37415    199   0.529058 2004-02-16 00:00:00 2025-12-31 20:00:00
m2_4h        37614    37477    137   0.364226 2004-02-02 00:00:00 2025-12-31 20:00:00
cpi_4h       37614    37477    137   0.364226 2004-02-02 00:00:00 2025-12-31 20:00:00
cfnai_4h     37614    37477    137   0.364226 2004-02-02 00:00:00 2025-12-31 20:00:00
unemp_4h     37614    37551     63   0.167491 2004-01-15 04:00:00 2025-12-31 20:00:00
fed_4h       37614    37557     57   0.151539 2004-01-14 04:00:00 2025-12-31 20:00:00
tga_4h       37614    37569     45   0.119636 2004-01-12 04:00:00 2025-12-31 20:00:00
rrp_4h       37614    37588    

In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK 06B — FEATURE START IMPACT CHECK
# Hangi seri modeli hangi tarihten başlatıyor?
# ════════════════════════════════════════════════════════════════════

macro_start_impact = check[["first_valid", "n_nan", "nan_pct"]].copy()
macro_start_impact = macro_start_impact.sort_values("first_valid")

print("=" * 120)
print("MACRO START IMPACT CHECK")
print("=" * 120)
print(macro_start_impact.to_string())

MACRO START IMPACT CHECK
                  first_valid  n_nan    nan_pct
gold_4h   2004-01-01 00:00:00      0   0.000000
lqd_4h    2004-01-05 00:00:00     13   0.034562
dxy_4h    2004-01-05 00:00:00     13   0.034562
vix_4h    2004-01-05 00:00:00     13   0.034562
spx_4h    2004-01-05 00:00:00     13   0.034562
us2y_4h   2004-01-05 04:00:00     14   0.037220
us10y_4h  2004-01-05 04:00:00     14   0.037220
rrp_4h    2004-01-07 04:00:00     26   0.069123
tga_4h    2004-01-12 04:00:00     45   0.119636
fed_4h    2004-01-14 04:00:00     57   0.151539
unemp_4h  2004-01-15 04:00:00     63   0.167491
cfnai_4h  2004-02-02 00:00:00    137   0.364226
cpi_4h    2004-02-02 00:00:00    137   0.364226
m2_4h     2004-02-02 00:00:00    137   0.364226
mfg_o_4h  2004-02-16 00:00:00    199   0.529058
hyg_4h    2007-04-12 00:00:00   5312  14.122401
oil_4h    2007-07-31 00:00:00   5810  15.446376
hy_oas_4h 2023-06-06 04:00:00  33469  88.980167


In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK 05C — EK MAKRO / POLICY / TERM STRUCTURE / ENERGY ADAYLARI
# Amaç: EUR/USD için central bank divergence + rates + energy + risk proxy
# ════════════════════════════════════════════════════════════════════

print("Ek FRED makro serileri çekiliyor...")

extra_fred_codes = {
    # US policy / short rate
    "fedfunds": "FEDFUNDS",

    # ECB policy
    "ecb_dfr": "ECBDFR",
    "ecb_mrr": "ECBMRRFR",
    "ecb_mlf": "ECBMLFR",

    # US nominal / real rates / breakeven
    "us10y_daily": "DGS10",
    "us2y_daily": "DGS2",
    "us10y_real": "DFII10",
    "us10y_be": "T10YIE",

    # Germany / Euro Area 10Y yields
    "de10y": "IRLTLT01DEM156N",
    "ea10y": "IRLTLT01EZM156N",
    "it10y": "IRLTLT01ITM156N",

    # Euro Area inflation
    "ea_hicp": "CP0000EZ19M086NEST",

    # US growth / liquidity
    "us_indpro": "INDPRO",
    "us_retail": "RSAFS",
    "m1": "M1SL",

    # Credit / stress alternatives
    "bbb_oas": "BAMLC0A4CBBB",

    # Energy
    "brent_fred": "DCOILBRENTEU",
    "eu_gas": "PNGASEUUSDM",
}

extra_raw = {}

for name, code in extra_fred_codes.items():
    try:
        s = web.DataReader(code, "fred", START).squeeze().dropna()
        s.index = make_index_tz_naive(s.index)
        s = s.sort_index()
        s = s[~s.index.duplicated(keep="last")]
        extra_raw[name] = s

        print(
            f"[✓] {name:<14} | {code:<18} | "
            f"n={len(s):>6} | {s.index.min().date()} → {s.index.max().date()}"
        )

    except Exception as e:
        print(f"[✗] {name:<14} | {code:<18} | ERROR: {e}")

print("✓ Ek FRED çekimi tamam")

Ek FRED makro serileri çekiliyor...
[✓] fedfunds       | FEDFUNDS           | n=   269 | 2004-01-01 → 2026-05-01
[✗] ecb_dfr        | ECBDFR             | ERROR: HTTPSConnectionPool(host='fred.stlouisfed.org', port=443): Read timed out. (read timeout=30)


KeyboardInterrupt: 

In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK 05C — EK MAKRO / POLICY / TERM STRUCTURE / ENERGY ADAYLARI
# Amaç: EUR/USD için central bank divergence + rates + energy + risk proxy
# ════════════════════════════════════════════════════════════════════

print("Ek FRED makro serileri çekiliyor...")

extra_fred_codes = {
    # US policy / short rate
    "fedfunds"   : "FEDFUNDS",

    # ECB policy
    "ecb_dfr"    : "ECBDFR",
    "ecb_mrr"    : "ECBMRRFR",
    "ecb_mlf"    : "ECBMLFR",

    # US nominal / real rates / breakeven
    "us10y_daily": "DGS10",
    "us2y_daily" : "DGS2",
    "us10y_real" : "DFII10",
    "us10y_be"   : "T10YIE",

    # Germany / Euro Area 10Y yields
    "de10y"      : "IRLTLT01DEM156N",
    "ea10y"      : "IRLTLT01EZM156N",
    "it10y"      : "IRLTLT01ITM156N",

    # Euro Area inflation
    "ea_hicp"    : "CP0000EZ19M086NEST",

    # US growth / liquidity
    "us_indpro"  : "INDPRO",
    "us_retail"  : "RSAFS",
    "m1"         : "M1SL",

    # Credit / stress alternatives
    "bbb_oas"    : "BAMLC0A4CBBB",

    # Energy
    "brent_fred" : "DCOILBRENTEU",
    "eu_gas"     : "PNGASEUUSDM",
}

extra_raw = {}

for name, code in extra_fred_codes.items():
    try:
        s = fetch_fred(code, START, api_key=FRED_API_KEY)   # ← aynı fetch_fred fonksiyonu
        s = s.sort_index()
        s = s[~s.index.duplicated(keep="last")]
        s.index = make_index_tz_naive(s.index)
        extra_raw[name] = s

        print(
            f"[✓] {name:<14} | {code:<18} | "
            f"n={len(s):>6} | {s.index.min().date()} → {s.index.max().date()}"
        )
        time.sleep(0.3)   # rate-limit

    except Exception as e:
        print(f"[✗] {name:<14} | {code:<18} | ERROR: {e}")

print("✓ Ek FRED çekimi tamam")

Ek FRED makro serileri çekiliyor...
[✓] fedfunds       | FEDFUNDS           | n=   269 | 2004-01-01 → 2026-05-01
[✓] ecb_dfr        | ECBDFR             | n=  8190 | 2004-01-01 → 2026-06-03
[✓] ecb_mrr        | ECBMRRFR           | n=  6441 | 2008-10-15 → 2026-06-03
[✓] ecb_mlf        | ECBMLFR            | n=  8190 | 2004-01-01 → 2026-06-03
[✓] us10y_daily    | DGS10              | n=  5609 | 2004-01-02 → 2026-06-03
[✓] us2y_daily     | DGS2               | n=  5609 | 2004-01-02 → 2026-06-03
[✓] us10y_real     | DFII10             | n=  5609 | 2004-01-02 → 2026-06-03
[✓] us10y_be       | T10YIE             | n=  5610 | 2004-01-02 → 2026-06-04
[✓] de10y          | IRLTLT01DEM156N    | n=   268 | 2004-01-01 → 2026-04-01
[✓] ea10y          | IRLTLT01EZM156N    | n=   265 | 2004-01-01 → 2026-01-01
[✓] it10y          | IRLTLT01ITM156N    | n=   267 | 2004-01-01 → 2026-03-01
[✓] ea_hicp        | CP0000EZ19M086NEST | n=   268 | 2004-01-01 → 2026-04-01
[✓] us_indpro      | INDPRO             

In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK 05D — EK MAKRO SERİLERE DIRECT RELEASE-LAG APPLICATION
# Amaç:
# - BLOK 05C ile çekilen extra_raw serilerini aynı isimlerle lag'lemek
# - Sonraki kodlarda extra_raw["fedfunds"], extra_raw["ecb_dfr"] vb.
#   kullanımı aynen devam etsin
# ════════════════════════════════════════════════════════════════════

def lag_series_by_days(s, days):
    """
    Calendar-day lag.
    Aylık/haftalık makro seriler için kullanılır.
    """
    s = pd.Series(s).copy().dropna()
    s.index = make_index_tz_naive(s.index)
    s = s.sort_index()
    s = s[~s.index.duplicated(keep="last")]
    s.index = s.index + pd.Timedelta(days=int(days))
    return s.sort_index()


def lag_series_by_bdays(s, bdays):
    """
    Business-day lag.
    Günlük finansal/piyasa serileri için kullanılır.
    """
    s = pd.Series(s).copy().dropna()
    s.index = make_index_tz_naive(s.index)
    s = s.sort_index()
    s = s[~s.index.duplicated(keep="last")]
    s.index = s.index + pd.offsets.BDay(int(bdays))
    return s.sort_index()


print("\nEk makro serilere release-lag uygulanıyor...")

# ------------------------------------------------------------
# Lag şeması
# ------------------------------------------------------------
# Not:
# - Policy rate serileri genelde karar günü bilinir ama modelde
#   timestamp/release saat karmaşasını azaltmak için konservatif lag uygulanır.
# - Günlük market/rate serileri 1 business day.
# - Aylık makro seriler 30-45 calendar day.
# - Energy spot günlükse 1 business day, aylık gas serisi 30 day.

EXTRA_RELEASE_LAGS = {
    # US policy / short rate
    # FEDFUNDS aylık effective fed funds rate.
    "fedfunds": ("days", 30),

    # ECB policy rates
    # ECBDFR / ECBMRRFR / ECBMLFR karar serisi gibi hareket eder.
    # Günlük indexli olabilir; 1 business day güvenli.
    "ecb_dfr": ("bdays", 1),
    "ecb_mrr": ("bdays", 1),
    "ecb_mlf": ("bdays", 1),

    # US nominal / real rates / breakeven
    "us10y_daily": ("bdays", 1),
    "us2y_daily":  ("bdays", 1),
    "us10y_real":  ("bdays", 1),
    "us10y_be":    ("bdays", 1),

    # Germany / Euro Area / Italy 10Y yields
    # Bu FRED serileri aylık OECD long-term rate gibi gelebilir.
    # Aylık seri olduğu için 30 gün daha güvenli.
    "de10y": ("days", 30),
    "ea10y": ("days", 30),
    "it10y": ("days", 30),

    # Euro Area inflation
    "ea_hicp": ("days", 30),

    # US growth / liquidity
    "us_indpro": ("days", 30),
    "us_retail": ("days", 30),
    "m1":        ("days", 30),

    # Credit / stress alternatives
    # BBB OAS günlük/market-like seri.
    "bbb_oas": ("bdays", 1),

    # Energy
    # Brent daily spot: 1 business day.
    # EU gas FRED serisi genelde aylık commodity price gibi gelir: 30 day.
    "brent_fred": ("bdays", 1),
    "eu_gas":     ("days", 30),
}


extra_raw_unlagged = {k: v.copy() for k, v in extra_raw.items()}

for name, s in list(extra_raw.items()):
    if name not in EXTRA_RELEASE_LAGS:
        print(f"[!] {name:<14} için lag tanımı yok, dokunulmadı.")
        continue

    lag_type, lag_value = EXTRA_RELEASE_LAGS[name]

    try:
        old_min = s.index.min()
        old_max = s.index.max()

        if lag_type == "days":
            extra_raw[name] = lag_series_by_days(s, lag_value)
        elif lag_type == "bdays":
            extra_raw[name] = lag_series_by_bdays(s, lag_value)
        else:
            raise ValueError(f"Bilinmeyen lag_type: {lag_type}")

        new_s = extra_raw[name]

        print(
            f"[✓] {name:<14} | {lag_type:<5} +{lag_value:<3} | "
            f"{old_min.date()} → {old_max.date()}  ==>  "
            f"{new_s.index.min().date()} → {new_s.index.max().date()}"
        )

    except Exception as e:
        print(f"[✗] {name:<14} lag uygulanamadı | ERROR: {e}")

print("✓ Ek makro seriler mevcut isimleriyle lag'li hale getirildi")


Ek makro serilere release-lag uygulanıyor...
[✓] fedfunds       | days  +30  | 2004-01-01 → 2026-05-01  ==>  2004-01-31 → 2026-05-31
[✓] ecb_dfr        | bdays +1   | 2004-01-01 → 2026-06-03  ==>  2004-01-02 → 2026-06-04
[✓] ecb_mrr        | bdays +1   | 2008-10-15 → 2026-06-03  ==>  2008-10-16 → 2026-06-04
[✓] ecb_mlf        | bdays +1   | 2004-01-01 → 2026-06-03  ==>  2004-01-02 → 2026-06-04
[✓] us10y_daily    | bdays +1   | 2004-01-02 → 2026-06-03  ==>  2004-01-05 → 2026-06-04
[✓] us2y_daily     | bdays +1   | 2004-01-02 → 2026-06-03  ==>  2004-01-05 → 2026-06-04
[✓] us10y_real     | bdays +1   | 2004-01-02 → 2026-06-03  ==>  2004-01-05 → 2026-06-04
[✓] us10y_be       | bdays +1   | 2004-01-02 → 2026-06-04  ==>  2004-01-05 → 2026-06-05
[✓] de10y          | days  +30  | 2004-01-01 → 2026-04-01  ==>  2004-01-31 → 2026-05-01
[✓] ea10y          | days  +30  | 2004-01-01 → 2026-01-01  ==>  2004-01-31 → 2026-01-31
[✓] it10y          | days  +30  | 2004-01-01 → 2026-03-01  ==>  2004-01-31

In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK 05D — EK MAKROLAR 4H ALIGN + HEALTH CHECK
# ════════════════════════════════════════════════════════════════════

extra_4h = {}

for name, s in extra_raw.items():
    extra_4h[name + "_4h"] = align_macro_to_4h(
        s,
        base_index,
        lag_periods=1
    )

extra_4h_df = pd.DataFrame(extra_4h, index=base_index)

extra_check = pd.DataFrame({
    "n_total": extra_4h_df.shape[0],
    "n_valid": extra_4h_df.notna().sum(),
    "n_nan": extra_4h_df.isna().sum(),
    "nan_pct": extra_4h_df.isna().mean() * 100,
    "first_valid": extra_4h_df.apply(
        lambda x: x.dropna().index.min() if x.notna().any() else pd.NaT
    ),
    "last_valid": extra_4h_df.apply(
        lambda x: x.dropna().index.max() if x.notna().any() else pd.NaT
    ),
}).sort_values("nan_pct", ascending=False)

print("=" * 120)
print("EXTRA FRED 4H HEALTH CHECK")
print("=" * 120)
print(extra_check.to_string())

EXTRA FRED 4H HEALTH CHECK
                n_total  n_valid  n_nan    nan_pct         first_valid          last_valid
bbb_oas_4h        37614     4145  33469  88.980167 2023-06-06 04:00:00 2025-12-31 20:00:00
ecb_mrr_4h        37614    29836   7778  20.678471 2008-10-16 04:00:00 2025-12-31 20:00:00
eu_gas_4h         37614    37477    137   0.364226 2004-02-02 00:00:00 2025-12-31 20:00:00
fedfunds_4h       37614    37477    137   0.364226 2004-02-02 00:00:00 2025-12-31 20:00:00
de10y_4h          37614    37477    137   0.364226 2004-02-02 00:00:00 2025-12-31 20:00:00
ea10y_4h          37614    37477    137   0.364226 2004-02-02 00:00:00 2025-12-31 20:00:00
ea_hicp_4h        37614    37477    137   0.364226 2004-02-02 00:00:00 2025-12-31 20:00:00
us_indpro_4h      37614    37477    137   0.364226 2004-02-02 00:00:00 2025-12-31 20:00:00
it10y_4h          37614    37477    137   0.364226 2004-02-02 00:00:00 2025-12-31 20:00:00
us_retail_4h      37614    37477    137   0.364226 2004-02-02 0

In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK 05E — DERIVED MACRO FEATURES
# Policy divergence, term spread, sovereign stress, real-rate, energy shock
# ════════════════════════════════════════════════════════════════════

def get_extra_col(name):
    col = name + "_4h"
    if col in extra_4h_df.columns:
        return extra_4h_df[col]
    return pd.Series(index=base_index, dtype=float)

# Policy divergence
fedfunds_4h = get_extra_col("fedfunds")
ecb_dfr_4h  = get_extra_col("ecb_dfr")
ecb_mrr_4h  = get_extra_col("ecb_mrr")
ecb_mlf_4h  = get_extra_col("ecb_mlf")

policy_diff_fed_ecb_dfr = fedfunds_4h - ecb_dfr_4h
policy_diff_fed_ecb_mrr = fedfunds_4h - ecb_mrr_4h

# Term structure / rates
us10y_daily_4h = get_extra_col("us10y_daily")
us2y_daily_4h  = get_extra_col("us2y_daily")
de10y_4h       = get_extra_col("de10y")
ea10y_4h       = get_extra_col("ea10y")
it10y_4h       = get_extra_col("it10y")

us_curve_10y2y = us10y_daily_4h - us2y_daily_4h
us_de_10y_spread = us10y_daily_4h - de10y_4h
us_ea_10y_spread = us10y_daily_4h - ea10y_4h

# Sovereign / periphery stress
it_de_10y_spread = it10y_4h - de10y_4h
it_ea_10y_spread = it10y_4h - ea10y_4h

# Real yield / breakeven
us10y_real_4h = get_extra_col("us10y_real")
us10y_be_4h   = get_extra_col("us10y_be")

real_nominal_gap = us10y_daily_4h - us10y_real_4h

# Euro Area inflation
ea_hicp_4h = get_extra_col("ea_hicp")
ea_hicp_yoy = ea_hicp_4h.pct_change(6 * 252) * 100

# US CPI YoY zaten cpi_4h üzerinden üretilebilir
us_cpi_yoy = cpi_4h.pct_change(6 * 252) * 100
inflation_gap_us_ea = us_cpi_yoy - ea_hicp_yoy

# Energy macro
brent_fred_4h = get_extra_col("brent_fred")
eu_gas_4h     = get_extra_col("eu_gas")

brent_ret_21d = safe_logret(brent_fred_4h).rolling(W_21D).sum()
eu_gas_ret_21d = safe_logret(eu_gas_4h).rolling(W_21D).sum()

energy_pressure_eu = rolling_zscore(eu_gas_ret_21d, W_252D) - rolling_zscore(brent_ret_21d, W_252D)

# Credit stress alternative
bbb_oas_4h = get_extra_col("bbb_oas")
bbb_oas_z63 = rolling_zscore(bbb_oas_4h, W_63D)

# BTC risk-on proxy — yfinance kısmında btc_yf çekildiyse kullanacağız
# btc_4h = align_market_to_4h(btc_yf, base_index)
# btc_ret_21d = safe_logret(btc_4h).rolling(W_21D).sum()
# btc_riskon_z = rolling_zscore(btc_ret_21d, W_252D)

derived_macro_df = pd.DataFrame({
    # Policy divergence
    "policy_diff_fed_ecb_dfr": policy_diff_fed_ecb_dfr,
    "policy_diff_fed_ecb_mrr": policy_diff_fed_ecb_mrr,

    # US and transatlantic rates
    "us_curve_10y2y": us_curve_10y2y,
    "us_de_10y_spread": us_de_10y_spread,
    "us_ea_10y_spread": us_ea_10y_spread,

    # Sovereign stress
    "it_de_10y_spread": it_de_10y_spread,
    "it_ea_10y_spread": it_ea_10y_spread,

    # Real yield / inflation
    "us10y_real": us10y_real_4h,
    "us10y_be": us10y_be_4h,
    "real_nominal_gap": real_nominal_gap,
    "ea_hicp_yoy": ea_hicp_yoy,
    "us_cpi_yoy": us_cpi_yoy,
    "inflation_gap_us_ea": inflation_gap_us_ea,

    # Energy
    "brent_ret_21d": brent_ret_21d,
    "eu_gas_ret_21d": eu_gas_ret_21d,
    "energy_pressure_eu": energy_pressure_eu,

    # Credit
    "bbb_oas": bbb_oas_4h,
    "bbb_oas_z63": bbb_oas_z63,
}, index=base_index)

derived_check = pd.DataFrame({
    "n_total": derived_macro_df.shape[0],
    "n_valid": derived_macro_df.notna().sum(),
    "n_nan": derived_macro_df.isna().sum(),
    "nan_pct": derived_macro_df.isna().mean() * 100,
    "first_valid": derived_macro_df.apply(
        lambda x: x.dropna().index.min() if x.notna().any() else pd.NaT
    ),
    "last_valid": derived_macro_df.apply(
        lambda x: x.dropna().index.max() if x.notna().any() else pd.NaT
    ),
}).sort_values("nan_pct", ascending=False)

print("=" * 120)
print("DERIVED MACRO FEATURE CHECK")
print("=" * 120)
print(derived_check.to_string())

DERIVED MACRO FEATURE CHECK
                         n_total  n_valid  n_nan    nan_pct         first_valid          last_valid
bbb_oas_z63                37614     3768  33846  89.982453 2023-08-30 00:00:00 2025-12-31 20:00:00
bbb_oas                    37614     4145  33469  88.980167 2023-06-06 04:00:00 2025-12-31 20:00:00
policy_diff_fed_ecb_mrr    37614    29836   7778  20.678471 2008-10-16 04:00:00 2025-12-31 20:00:00
energy_pressure_eu         37614    35840   1774   4.716329 2005-02-03 16:00:00 2025-12-31 20:00:00
inflation_gap_us_ea        37614    35965   1649   4.384006 2005-01-06 20:00:00 2025-12-31 20:00:00
ea_hicp_yoy                37614    35965   1649   4.384006 2005-01-06 20:00:00 2025-12-31 20:00:00
us_cpi_yoy                 37614    35965   1649   4.384006 2005-01-06 20:00:00 2025-12-31 20:00:00
eu_gas_ret_21d             37614    37351    263   0.699208 2004-03-01 08:00:00 2025-12-31 20:00:00
brent_ret_21d              37614    37474    140   0.372202 2004-02-02 1

In [ ]:
# YFinance BTC risk-on proxy
btc_yf = yf.download("BTC-USD", start=START, auto_adjust=True, progress=False)["Close"].squeeze()
btc_yf.index = make_index_tz_naive(btc_yf.index)

btc_4h = align_market_to_4h(btc_yf, base_index)

btc_ret_21d = safe_logret(btc_4h).rolling(W_21D).sum()
btc_riskon_z = rolling_zscore(btc_ret_21d, W_252D)

In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK 05F — CALENDAR RISK FEATURES
# Event calendar olmadan leak-free takvim proxy'leri
# ════════════════════════════════════════════════════════════════════

calendar_df = pd.DataFrame(index=base_index)

calendar_df["hour"] = calendar_df.index.hour
calendar_df["dayofweek"] = calendar_df.index.dayofweek
calendar_df["month"] = calendar_df.index.month

calendar_df["is_friday"] = (calendar_df.index.dayofweek == 4).astype(int)
calendar_df["is_monday"] = (calendar_df.index.dayofweek == 0).astype(int)
calendar_df["is_month_start"] = calendar_df.index.is_month_start.astype(int)
calendar_df["is_month_end"] = calendar_df.index.is_month_end.astype(int)

# 4H FX için session proxy
calendar_df["is_london_us_overlap"] = calendar_df["hour"].between(12, 16).astype(int)
calendar_df["is_asia_session"] = calendar_df["hour"].between(0, 8).astype(int)
calendar_df["is_us_session"] = calendar_df["hour"].between(13, 21).astype(int)

# NFP proxy: ayın ilk cuma günü
calendar_df["week_of_month"] = ((calendar_df.index.day - 1) // 7 + 1)
calendar_df["is_first_friday"] = (
    (calendar_df.index.dayofweek == 4) &
    (calendar_df["week_of_month"] == 1)
).astype(int)

print("✓ Calendar risk features hazır")

✓ Calendar risk features hazır


In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK 05G — MAIN / SPECIAL MACRO FEATURE SELECTION
# ════════════════════════════════════════════════════════════════════

MAIN_DERIVED_MACRO_COLS = [
    # Policy divergence
    "policy_diff_fed_ecb_dfr",

    # Term structure / transatlantic rates
    "us_curve_10y2y",
    "us_de_10y_spread",
    "us_ea_10y_spread",

    # Euro sovereign stress
    "it_de_10y_spread",
    "it_ea_10y_spread",

    # Real yield / breakeven
    "us10y_real",
    "us10y_be",
    "real_nominal_gap",

    # Inflation divergence
    "ea_hicp_yoy",
    "us_cpi_yoy",
    "inflation_gap_us_ea",

    # Energy macro
    "brent_ret_21d",
    "eu_gas_ret_21d",
    "energy_pressure_eu",
]

SPECIAL_DERIVED_MACRO_COLS = [
    # Late sample / stress only
    "bbb_oas",
    "bbb_oas_z63",

    # Starts 2008, use only if sample accepts 2008+
    "policy_diff_fed_ecb_mrr",
]

main_derived_macro_df = derived_macro_df[
    [c for c in MAIN_DERIVED_MACRO_COLS if c in derived_macro_df.columns]
].copy()

special_derived_macro_df = derived_macro_df[
    [c for c in SPECIAL_DERIVED_MACRO_COLS if c in derived_macro_df.columns]
].copy()

print("=" * 120)
print("MAIN DERIVED MACRO FEATURES")
print("=" * 120)
print(main_derived_macro_df.isna().mean().mul(100).sort_values(ascending=False).round(3))

print("\n" + "=" * 120)
print("SPECIAL / LATE-SAMPLE DERIVED MACRO FEATURES")
print("=" * 120)
print(special_derived_macro_df.isna().mean().mul(100).sort_values(ascending=False).round(3))

MAIN DERIVED MACRO FEATURES
energy_pressure_eu         4.716
us_cpi_yoy                 4.384
inflation_gap_us_ea        4.384
ea_hicp_yoy                4.384
eu_gas_ret_21d             0.699
brent_ret_21d              0.372
policy_diff_fed_ecb_dfr    0.364
us_ea_10y_spread           0.364
us_de_10y_spread           0.364
it_de_10y_spread           0.364
it_ea_10y_spread           0.364
us10y_real                 0.037
us_curve_10y2y             0.037
us10y_be                   0.037
real_nominal_gap           0.037
dtype: float64

SPECIAL / LATE-SAMPLE DERIVED MACRO FEATURES
bbb_oas_z63                89.982
bbb_oas                    88.980
policy_diff_fed_ecb_mrr    20.678
dtype: float64


In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK 07 — 4H FEATURE ENGINEERING
# L1 (return) → L2 (vol) → L3 (momentum) → L4 (macro + CC + MI)
# ════════════════════════════════════════════════════════════════════

lr = log_ret.copy()

# ── L2: Volatility ───────────────────────────────────────────────
hist_vol_20 = lr.rolling(W_20D).std()
ewma_vol    = lr.pow(2).ewm(alpha=0.06, adjust=False).mean().pow(0.5)

# Simple intrabar range ratio
atr_ratio   = (H - L) / C

# True ATR ratio
tr1 = H - L
tr2 = (H - C.shift(1)).abs()
tr3 = (L - C.shift(1)).abs()
true_range = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
atr_14 = true_range.rolling(W_14D).mean()
atr_ratio_true = atr_14 / C

log_hl      = np.log(H / L)
log_co_prev = np.log(C / C.shift(1))
log_oc_prev = np.log(O / C.shift(1))

rs_inner = (
    np.log(H / C) * np.log(H / O) +
    np.log(L / C) * np.log(L / O)
)

parkinson_vol = (log_hl.pow(2) / (4 * np.log(2))).rolling(W_20D).mean().pow(0.5)

garman_klass_vol = (
    (0.5 * log_hl.pow(2) - (2 * np.log(2) - 1) * log_co_prev.pow(2))
    .clip(lower=0)
    .rolling(W_20D)
    .mean()
    .pow(0.5)
)

rogers_satchell_vol = rs_inner.clip(lower=0).rolling(W_20D).mean().pow(0.5)


# ── L3: Momentum / mean reversion ────────────────────────────────
ma_50     = C.rolling(W_50D).mean()
ma_spread = (C / ma_50) - 1

rsi_raw  = calc_rsi(C, period=W_14D)
rsi_norm = (rsi_raw - 50) / 50

z_score_ret = (lr - lr.rolling(W_21D).mean()) / (lr.rolling(W_21D).std() + 1e-10)
mom_5       = lr.rolling(W_5D).sum()
mom_21      = lr.rolling(W_21D).sum()


# ── L4: Market macro ─────────────────────────────────────────────
vix_level    = vix_4h
dxy_ret      = safe_logret(dxy_4h)
gold_ret     = safe_logret(gold_4h)
oil_ret      = safe_logret(oil_4h)
hyg_lqd_spr  = np.log(hyg_4h / lqd_4h)
spread_2y10y = us10y_4h - us2y_4h
us10y_chg    = us10y_4h.diff()


# ── Cross-currency ────────────────────────────────────────────────
usd_factor   = ((-lr) + (-gbp_ret) + chf_ret + jpy_ret) / 4
usd_factor_z = rolling_zscore(usd_factor, W_63D)
eur_dom      = lr - gbp_ret
chf_haven    = -ech_ret

corr_eur_gbp = lr.rolling(W_21D).corr(gbp_ret)
corr_eur_chf = lr.rolling(W_21D).corr(-chf_ret)
corr_eur_jpy = lr.rolling(W_21D).corr(-jpy_ret)


# ── Rolling Mutual Information ────────────────────────────────────
print("MI hesaplanıyor... (yavaş adım)")

def rolling_mi(x, y, window=W_63D):
    result = pd.Series(index=x.index, dtype=float)
    xa, ya = x.values, y.values
    for i in range(window, len(xa)):
        xi = xa[i-window:i].reshape(-1, 1)
        yi = ya[i-window:i]
        if np.isnan(xi).any() or np.isnan(yi).any():
            continue
        try:
            result.iloc[i] = mutual_info_regression(xi, yi, random_state=42)[0]
        except Exception:
            continue
    return result

mi_eur_gbp = rolling_mi(lr, gbp_ret,  window=W_63D)
mi_eur_chf = rolling_mi(lr, -chf_ret, window=W_63D)
mi_eur_jpy = rolling_mi(lr, -jpy_ret, window=W_63D)
print("✓ MI tamam")


# ════════════════════════════════════════════════════════════════════
# BLOK 08 — FRED TABANLI MACRO FEATURES
# ════════════════════════════════════════════════════════════════════

cfnai_a = cfnai_4h
cpi_a   = cpi_4h

cpi_yoy   = cpi_a.pct_change(W_252D) * 100
cpi_yoy_z = rolling_zscore(cpi_yoy, W_252D)

# CPI-normalized real M2
cpi_base = cpi_a.dropna().iloc[0]
real_m2 = m2_4h / (cpi_a / cpi_base)
real_m2_z = rolling_zscore(real_m2.pct_change(W_252D) * 100, W_252D)

net_liq   = fed_4h - tga_4h - rrp_4h
net_liq_z = rolling_zscore(net_liq.pct_change(W_252D) * 100, W_252D)

dxy_trend_z = rolling_zscore((dxy_4h / dxy_4h.rolling(W_50D).mean()) - 1, W_252D)
vix_z       = rolling_zscore(vix_level, W_252D)

# Late-sample only: hy_oas_4h starts very late, do not use hy_z in main expert sets.
hy_z        = rolling_zscore(hy_oas_4h, W_252D)

spx_trend_z = rolling_zscore((spx_4h / spx_4h.rolling(W_50D).mean()) - 1, W_252D)
unemp_z     = -rolling_zscore(unemp_4h, W_252D)
mfg_z       = rolling_zscore(mfg_o_4h.pct_change(W_252D) * 100, W_252D)

MI hesaplanıyor... (yavaş adım)
✓ MI tamam


In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK 08B — YENİ EK MAKRO FEATURE DEVŞİRME
# Policy divergence → term structure → sovereign stress → inflation gap → energy
# Not: Bu blok, extra_4h_df oluşturulduktan sonra çalıştırılmalıdır.
# ════════════════════════════════════════════════════════════════════

def get_extra_4h(name):
    col = name + "_4h"
    if "extra_4h_df" in globals() and col in extra_4h_df.columns:
        return extra_4h_df[col]
    return pd.Series(index=base_index, dtype=float)


# ─────────────────────────────────────────────────────────────────
# 1) Policy divergence: Fed - ECB
# ─────────────────────────────────────────────────────────────────

fedfunds_4h_new = get_extra_4h("fedfunds")
ecb_dfr_4h_new  = get_extra_4h("ecb_dfr")
ecb_mlf_4h_new  = get_extra_4h("ecb_mlf")

policy_diff_fed_ecb_dfr = fedfunds_4h_new - ecb_dfr_4h_new
policy_diff_fed_ecb_mlf = fedfunds_4h_new - ecb_mlf_4h_new

policy_diff_z63  = rolling_zscore(policy_diff_fed_ecb_dfr, W_63D)
policy_diff_z252 = rolling_zscore(policy_diff_fed_ecb_dfr, W_252D)
policy_diff_chg5 = policy_diff_fed_ecb_dfr.diff(W_5D)
policy_diff_chg21 = policy_diff_fed_ecb_dfr.diff(W_21D)


# ─────────────────────────────────────────────────────────────────
# 2) Term structure / transatlantic yield spreads
# ─────────────────────────────────────────────────────────────────

us10y_daily_4h_new = get_extra_4h("us10y_daily")
us2y_daily_4h_new  = get_extra_4h("us2y_daily")
us10y_real_4h_new  = get_extra_4h("us10y_real")
us10y_be_4h_new    = get_extra_4h("us10y_be")

de10y_4h_new = get_extra_4h("de10y")
ea10y_4h_new = get_extra_4h("ea10y")
it10y_4h_new = get_extra_4h("it10y")

us_curve_10y2y = us10y_daily_4h_new - us2y_daily_4h_new
us_de_10y_spread = us10y_daily_4h_new - de10y_4h_new
us_ea_10y_spread = us10y_daily_4h_new - ea10y_4h_new

us_curve_z63 = rolling_zscore(us_curve_10y2y, W_63D)
us_curve_z252 = rolling_zscore(us_curve_10y2y, W_252D)

us_de_10y_spread_z63 = rolling_zscore(us_de_10y_spread, W_63D)
us_de_10y_spread_z252 = rolling_zscore(us_de_10y_spread, W_252D)
us_de_10y_spread_chg21 = us_de_10y_spread.diff(W_21D)

us_ea_10y_spread_z63 = rolling_zscore(us_ea_10y_spread, W_63D)
us_ea_10y_spread_z252 = rolling_zscore(us_ea_10y_spread, W_252D)


# ─────────────────────────────────────────────────────────────────
# 3) Sovereign credit / safe asset rotation
# ─────────────────────────────────────────────────────────────────

it_de_10y_spread = it10y_4h_new - de10y_4h_new
it_ea_10y_spread = it10y_4h_new - ea10y_4h_new

it_de_10y_spread_z63 = rolling_zscore(it_de_10y_spread, W_63D)
it_de_10y_spread_z252 = rolling_zscore(it_de_10y_spread, W_252D)
it_de_10y_spread_chg21 = it_de_10y_spread.diff(W_21D)

it_ea_10y_spread_z63 = rolling_zscore(it_ea_10y_spread, W_63D)
it_ea_10y_spread_z252 = rolling_zscore(it_ea_10y_spread, W_252D)


# ─────────────────────────────────────────────────────────────────
# 4) Real yield / breakeven / inflation expectations
# ─────────────────────────────────────────────────────────────────

us10y_real = us10y_real_4h_new
us10y_be = us10y_be_4h_new

us10y_real_z63 = rolling_zscore(us10y_real, W_63D)
us10y_real_z252 = rolling_zscore(us10y_real, W_252D)
us10y_real_chg5 = us10y_real.diff(W_5D)
us10y_real_chg21 = us10y_real.diff(W_21D)

us10y_be_z63 = rolling_zscore(us10y_be, W_63D)
us10y_be_z252 = rolling_zscore(us10y_be, W_252D)

real_nominal_gap = us10y_daily_4h_new - us10y_real
real_nominal_gap_z63 = rolling_zscore(real_nominal_gap, W_63D)


# ─────────────────────────────────────────────────────────────────
# 5) Inflation divergence: US CPI YoY - Euro Area HICP YoY
# ─────────────────────────────────────────────────────────────────

ea_hicp_4h_new = get_extra_4h("ea_hicp")

ea_hicp_yoy = ea_hicp_4h_new.pct_change(W_252D) * 100
us_cpi_yoy = cpi_4h.pct_change(W_252D) * 100

inflation_gap_us_ea = us_cpi_yoy - ea_hicp_yoy

ea_hicp_yoy_z = rolling_zscore(ea_hicp_yoy, W_252D)
us_cpi_yoy_z2 = rolling_zscore(us_cpi_yoy, W_252D)
inflation_gap_us_ea_z63 = rolling_zscore(inflation_gap_us_ea, W_63D)
inflation_gap_us_ea_z252 = rolling_zscore(inflation_gap_us_ea, W_252D)


# ─────────────────────────────────────────────────────────────────
# 6) Growth divergence: US growth proxies
# Şimdilik Euro Area growth gelmediği için US growth momentum üretiyoruz.
# ─────────────────────────────────────────────────────────────────

us_indpro_4h_new = get_extra_4h("us_indpro")
us_retail_4h_new = get_extra_4h("us_retail")

us_indpro_yoy = us_indpro_4h_new.pct_change(W_252D) * 100
us_retail_yoy = us_retail_4h_new.pct_change(W_252D) * 100

us_indpro_yoy_z = rolling_zscore(us_indpro_yoy, W_252D)
us_retail_yoy_z = rolling_zscore(us_retail_yoy, W_252D)

us_growth_proxy = 0.5 * us_indpro_yoy_z + 0.5 * us_retail_yoy_z


# ─────────────────────────────────────────────────────────────────
# 7) Liquidity / money
# ─────────────────────────────────────────────────────────────────

m1_4h_new = get_extra_4h("m1")

m1_real = m1_4h_new / (cpi_4h / cpi_4h.dropna().iloc[0])
m1_real_yoy = m1_real.pct_change(W_252D) * 100
m1_real_yoy_z = rolling_zscore(m1_real_yoy, W_252D)

money_liquidity_proxy = 0.5 * real_m2_z + 0.5 * m1_real_yoy_z


# ─────────────────────────────────────────────────────────────────
# 8) Energy macro: Brent + EU gas pressure
# ─────────────────────────────────────────────────────────────────

brent_fred_4h_new = get_extra_4h("brent_fred")
eu_gas_4h_new = get_extra_4h("eu_gas")

brent_ret_21d = safe_logret(brent_fred_4h_new).rolling(W_21D).sum()
eu_gas_ret_21d = safe_logret(eu_gas_4h_new).rolling(W_21D).sum()

brent_ret_21d_z = rolling_zscore(brent_ret_21d, W_252D)
eu_gas_ret_21d_z = rolling_zscore(eu_gas_ret_21d, W_252D)

energy_pressure_eu = eu_gas_ret_21d_z - brent_ret_21d_z
energy_pressure_eu_z63 = rolling_zscore(energy_pressure_eu, W_63D)


# ─────────────────────────────────────────────────────────────────
# 9) Credit stress alternative — late sample only
# ─────────────────────────────────────────────────────────────────

bbb_oas_4h_new = get_extra_4h("bbb_oas")

bbb_oas = bbb_oas_4h_new
bbb_oas_z63 = rolling_zscore(bbb_oas, W_63D)

# Not: bbb_oas ve bbb_oas_z63 ana expert setlerde kullanılmamalı.
# Sadece 2023+ özel stress sample için saklanmalı.


# ─────────────────────────────────────────────────────────────────
# 10) New macro dataframe
# ─────────────────────────────────────────────────────────────────

new_macro_features_df = pd.DataFrame({
    # Policy divergence
    "policy_diff_fed_ecb_dfr": policy_diff_fed_ecb_dfr,
    "policy_diff_fed_ecb_mlf": policy_diff_fed_ecb_mlf,
    "policy_diff_z63": policy_diff_z63,
    "policy_diff_z252": policy_diff_z252,
    "policy_diff_chg5": policy_diff_chg5,
    "policy_diff_chg21": policy_diff_chg21,

    # Term structure / transatlantic spreads
    "us_curve_10y2y": us_curve_10y2y,
    "us_curve_z63": us_curve_z63,
    "us_curve_z252": us_curve_z252,
    "us_de_10y_spread": us_de_10y_spread,
    "us_de_10y_spread_z63": us_de_10y_spread_z63,
    "us_de_10y_spread_z252": us_de_10y_spread_z252,
    "us_de_10y_spread_chg21": us_de_10y_spread_chg21,
    "us_ea_10y_spread": us_ea_10y_spread,
    "us_ea_10y_spread_z63": us_ea_10y_spread_z63,
    "us_ea_10y_spread_z252": us_ea_10y_spread_z252,

    # Sovereign stress
    "it_de_10y_spread": it_de_10y_spread,
    "it_de_10y_spread_z63": it_de_10y_spread_z63,
    "it_de_10y_spread_z252": it_de_10y_spread_z252,
    "it_de_10y_spread_chg21": it_de_10y_spread_chg21,
    "it_ea_10y_spread": it_ea_10y_spread,
    "it_ea_10y_spread_z63": it_ea_10y_spread_z63,
    "it_ea_10y_spread_z252": it_ea_10y_spread_z252,

    # Real yield / breakeven
    "us10y_real": us10y_real,
    "us10y_real_z63": us10y_real_z63,
    "us10y_real_z252": us10y_real_z252,
    "us10y_real_chg5": us10y_real_chg5,
    "us10y_real_chg21": us10y_real_chg21,
    "us10y_be": us10y_be,
    "us10y_be_z63": us10y_be_z63,
    "us10y_be_z252": us10y_be_z252,
    "real_nominal_gap": real_nominal_gap,
    "real_nominal_gap_z63": real_nominal_gap_z63,

    # Inflation divergence
    "ea_hicp_yoy": ea_hicp_yoy,
    "ea_hicp_yoy_z": ea_hicp_yoy_z,
    "us_cpi_yoy": us_cpi_yoy,
    "us_cpi_yoy_z2": us_cpi_yoy_z2,
    "inflation_gap_us_ea": inflation_gap_us_ea,
    "inflation_gap_us_ea_z63": inflation_gap_us_ea_z63,
    "inflation_gap_us_ea_z252": inflation_gap_us_ea_z252,

    # Growth
    "us_indpro_yoy": us_indpro_yoy,
    "us_retail_yoy": us_retail_yoy,
    "us_indpro_yoy_z": us_indpro_yoy_z,
    "us_retail_yoy_z": us_retail_yoy_z,
    "us_growth_proxy": us_growth_proxy,

    # Liquidity / money
    "m1_real_yoy": m1_real_yoy,
    "m1_real_yoy_z": m1_real_yoy_z,
    "money_liquidity_proxy": money_liquidity_proxy,

    # Energy
    "brent_ret_21d": brent_ret_21d,
    "brent_ret_21d_z": brent_ret_21d_z,
    "eu_gas_ret_21d": eu_gas_ret_21d,
    "eu_gas_ret_21d_z": eu_gas_ret_21d_z,
    "energy_pressure_eu": energy_pressure_eu,
    "energy_pressure_eu_z63": energy_pressure_eu_z63,

    # Credit stress — late sample only
    "bbb_oas": bbb_oas,
    "bbb_oas_z63": bbb_oas_z63,
}, index=base_index)

new_macro_check = pd.DataFrame({
    "n_total": new_macro_features_df.shape[0],
    "n_valid": new_macro_features_df.notna().sum(),
    "n_nan": new_macro_features_df.isna().sum(),
    "nan_pct": new_macro_features_df.isna().mean() * 100,
    "first_valid": new_macro_features_df.apply(
        lambda x: x.dropna().index.min() if x.notna().any() else pd.NaT
    ),
    "last_valid": new_macro_features_df.apply(
        lambda x: x.dropna().index.max() if x.notna().any() else pd.NaT
    ),
}).sort_values("nan_pct", ascending=False)

print("=" * 120)
print("NEW MACRO DERIVED FEATURE CHECK")
print("=" * 120)
print(new_macro_check.to_string())

NEW MACRO DERIVED FEATURE CHECK
                          n_total  n_valid  n_nan    nan_pct         first_valid          last_valid
bbb_oas_z63                 37614     3768  33846  89.982453 2023-08-30 00:00:00 2025-12-31 20:00:00
bbb_oas                     37614     4145  33469  88.980167 2023-06-06 04:00:00 2025-12-31 20:00:00
money_liquidity_proxy       37614    34454   3160   8.401127 2005-12-13 12:00:00 2025-12-31 20:00:00
ea_hicp_yoy_z               37614    34454   3160   8.401127 2005-12-13 12:00:00 2025-12-31 20:00:00
inflation_gap_us_ea_z252    37614    34454   3160   8.401127 2005-12-13 12:00:00 2025-12-31 20:00:00
us_cpi_yoy_z2               37614    34454   3160   8.401127 2005-12-13 12:00:00 2025-12-31 20:00:00
us_retail_yoy_z             37614    34454   3160   8.401127 2005-12-13 12:00:00 2025-12-31 20:00:00
us_indpro_yoy_z             37614    34454   3160   8.401127 2005-12-13 12:00:00 2025-12-31 20:00:00
m1_real_yoy_z               37614    34454   3160   8.40112

In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK 08C — GÜÇLENDİRİLMİŞ MACRO COMPOSITE SCORES
# growth_score → liq_score → policy/rates/stress/energy → risk_score
# ════════════════════════════════════════════════════════════════════

def mean_score(cols):
    """
    Aynı indexteki feature'ları row-wise ortalar.
    Tüm kolonlar NaN ise sonuç NaN olur.
    """
    valid_cols = []

    for x in cols:
        if isinstance(x, pd.Series):
            valid_cols.append(x)

    if len(valid_cols) == 0:
        return pd.Series(index=base_index, dtype=float)

    return pd.concat(valid_cols, axis=1).mean(axis=1)


# ─────────────────────────────────────────────────────────────────
# 1) Growth score
# Eski yapı korunur, yeni US growth proxy varsa eklenir.
# ─────────────────────────────────────────────────────────────────

growth_components = [
    rolling_zscore(cfnai_a, W_252D),
    mfg_z,
    unemp_z,
]

if "us_growth_proxy" in globals():
    growth_components.append(us_growth_proxy)

if "us_indpro_yoy_z" in globals():
    growth_components.append(us_indpro_yoy_z)

if "us_retail_yoy_z" in globals():
    growth_components.append(us_retail_yoy_z)

growth_score = mean_score(growth_components)


# ─────────────────────────────────────────────────────────────────
# 2) Liquidity score
# Pozitif değer: daha destekleyici likidite / daha gevşek koşullar.
# DXY trendi güçlü ise USD likidite baskısı varsayımıyla negatif alınır.
# ─────────────────────────────────────────────────────────────────

liq_components = [
    real_m2_z,
    net_liq_z,
    -dxy_trend_z,
]

if "m1_real_yoy_z" in globals():
    liq_components.append(m1_real_yoy_z)

if "money_liquidity_proxy" in globals():
    liq_components.append(money_liquidity_proxy)

liq_score = mean_score(liq_components)


# ─────────────────────────────────────────────────────────────────
# 3) Policy divergence score
# Pozitif değer: Fed-ECB farkı yükseliyor / USD lehine politika baskısı.
# Bu EUR/USD için genelde negatif EUR baskısı olarak yorumlanabilir.
# ─────────────────────────────────────────────────────────────────

policy_components = []

if "policy_diff_z63" in globals():
    policy_components.append(policy_diff_z63)

if "policy_diff_z252" in globals():
    policy_components.append(policy_diff_z252)

if "policy_diff_chg21" in globals():
    policy_components.append(rolling_zscore(policy_diff_chg21, W_252D))

policy_score = mean_score(policy_components)


# ─────────────────────────────────────────────────────────────────
# 4) Rates / real yield score
# Pozitif değer: ABD faiz / reel getiri / spread baskısı yüksek.
# USD lehine baskı olarak okunur.
# ─────────────────────────────────────────────────────────────────

rates_components = []

if "us10y_real_z63" in globals():
    rates_components.append(us10y_real_z63)

if "us10y_real_z252" in globals():
    rates_components.append(us10y_real_z252)

if "us_de_10y_spread_z63" in globals():
    rates_components.append(us_de_10y_spread_z63)

if "us_ea_10y_spread_z63" in globals():
    rates_components.append(us_ea_10y_spread_z63)

if "us_curve_z63" in globals():
    rates_components.append(us_curve_z63)

if "real_nominal_gap_z63" in globals():
    rates_components.append(real_nominal_gap_z63)

rates_score = mean_score(rates_components)


# ─────────────────────────────────────────────────────────────────
# 5) Sovereign stress score
# Pozitif değer: Euro periphery stress artıyor.
# EUR için negatif risk primi olarak yorumlanabilir.
# ─────────────────────────────────────────────────────────────────

sovereign_components = []

if "it_de_10y_spread_z63" in globals():
    sovereign_components.append(it_de_10y_spread_z63)

if "it_de_10y_spread_z252" in globals():
    sovereign_components.append(it_de_10y_spread_z252)

if "it_de_10y_spread_chg21" in globals():
    sovereign_components.append(rolling_zscore(it_de_10y_spread_chg21, W_252D))

if "it_ea_10y_spread_z63" in globals():
    sovereign_components.append(it_ea_10y_spread_z63)

sovereign_stress_score = mean_score(sovereign_components)


# ─────────────────────────────────────────────────────────────────
# 6) Inflation / breakeven score
# Pozitif değer: US inflation / breakeven tarafı EA'ya göre daha güçlü.
# Bu tek başına çift yönlü yorumlanabilir; rates ile beraber daha anlamlıdır.
# ─────────────────────────────────────────────────────────────────

inflation_components = []

if "inflation_gap_us_ea_z63" in globals():
    inflation_components.append(inflation_gap_us_ea_z63)

if "inflation_gap_us_ea_z252" in globals():
    inflation_components.append(inflation_gap_us_ea_z252)

if "us10y_be_z63" in globals():
    inflation_components.append(us10y_be_z63)

if "us_cpi_yoy_z2" in globals():
    inflation_components.append(us_cpi_yoy_z2)

inflation_score = mean_score(inflation_components)


# ─────────────────────────────────────────────────────────────────
# 7) Energy stress score
# Pozitif değer: Avrupa enerji baskısı yüksek.
# EUR için negatif baskı olarak yorumlanabilir.
# ─────────────────────────────────────────────────────────────────

energy_components = []

if "energy_pressure_eu" in globals():
    energy_components.append(energy_pressure_eu)

if "energy_pressure_eu_z63" in globals():
    energy_components.append(energy_pressure_eu_z63)

if "eu_gas_ret_21d_z" in globals():
    energy_components.append(eu_gas_ret_21d_z)

energy_score = mean_score(energy_components)


# ─────────────────────────────────────────────────────────────────
# 8) Risk score
# Eski hy_z ana score'dan çıkarıldı.
# Çünkü hy_oas_4h çok geç başlıyor ve ana sample'ı bozuyor.
# Pozitif değer: risk-off / stress yüksek.
# ─────────────────────────────────────────────────────────────────

risk_components = [
    vix_z,
    -spx_trend_z,
]

# hy_z late sample olduğu için ana risk score'a otomatik eklenmiyor.
# Eğer 2023+ özel stress testi yapılacaksa ayrıca eklenebilir.
USE_HY_IN_MAIN_RISK = False

if USE_HY_IN_MAIN_RISK and "hy_z" in globals():
    risk_components.append(hy_z)

if "sovereign_stress_score" in globals():
    risk_components.append(sovereign_stress_score)

if "energy_score" in globals():
    risk_components.append(energy_score)

risk_score = mean_score(risk_components)


# ─────────────────────────────────────────────────────────────────
# 9) Macro pressure score
# Pozitif değer: genel olarak USD lehine / EUR üzerinde baskı yaratabilecek makro ortam.
# Bu skor yön modeli için compact macro-state feature olarak kullanılabilir.
# ─────────────────────────────────────────────────────────────────

macro_pressure_components = []

# USD lehine / EUR aleyhine baskılar
for x_name in [
    "policy_score",
    "rates_score",
    "sovereign_stress_score",
    "energy_score",
    "risk_score",
]:
    if x_name in globals():
        macro_pressure_components.append(globals()[x_name])

# EUR/USD için destekleyici olabilecek genel likidite/growth kanalı ters işaretlenebilir.
# Burada -liq_score kullanıyoruz: sıkı/dar likidite USD baskısı gibi okunur.
if "liq_score" in globals():
    macro_pressure_components.append(-liq_score)

macro_pressure_score = mean_score(macro_pressure_components)


# ─────────────────────────────────────────────────────────────────
# 10) Health check
# ─────────────────────────────────────────────────────────────────

macro_scores_df = pd.DataFrame({
    "growth_score": growth_score,
    "liq_score": liq_score,
    "policy_score": policy_score,
    "rates_score": rates_score,
    "sovereign_stress_score": sovereign_stress_score,
    "inflation_score": inflation_score,
    "energy_score": energy_score,
    "risk_score": risk_score,
    "macro_pressure_score": macro_pressure_score,
}, index=base_index)

macro_scores_check = pd.DataFrame({
    "n_total": macro_scores_df.shape[0],
    "n_valid": macro_scores_df.notna().sum(),
    "n_nan": macro_scores_df.isna().sum(),
    "nan_pct": macro_scores_df.isna().mean() * 100,
    "first_valid": macro_scores_df.apply(
        lambda x: x.dropna().index.min() if x.notna().any() else pd.NaT
    ),
    "last_valid": macro_scores_df.apply(
        lambda x: x.dropna().index.max() if x.notna().any() else pd.NaT
    ),
}).sort_values("nan_pct", ascending=False)

print("=" * 120)
print("MACRO COMPOSITE SCORE CHECK")
print("=" * 120)
print(macro_scores_check.to_string())

print("✓ Güçlendirilmiş FRED / macro composite features hazır")

MACRO COMPOSITE SCORE CHECK
                        n_total  n_valid  n_nan   nan_pct         first_valid          last_valid
liq_score                 37614    35791   1823  4.846600 2005-02-15 08:00:00 2025-12-31 20:00:00
energy_score              37614    35840   1774  4.716329 2005-02-03 16:00:00 2025-12-31 20:00:00
growth_score              37614    36040   1574  4.184612 2004-12-21 16:00:00 2025-12-31 20:00:00
sovereign_stress_score    37614    37100    514  1.366512 2004-04-26 20:00:00 2025-12-31 20:00:00
policy_score              37614    37100    514  1.366512 2004-04-26 20:00:00 2025-12-31 20:00:00
risk_score                37614    37100    514  1.366512 2004-04-26 20:00:00 2025-12-31 20:00:00
rates_score               37614    37223    391  1.039507 2004-03-30 00:00:00 2025-12-31 20:00:00
inflation_score           37614    37223    391  1.039507 2004-03-30 00:00:00 2025-12-31 20:00:00
macro_pressure_score      37614    37223    391  1.039507 2004-03-30 00:00:00 2025-12-31 2

In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK 09 — TZ-NAIVE PATCH (tüm serileri normalize et)
# ════════════════════════════════════════════════════════════════════

TZ_NAIVE_SERIES_VARS = [
    # ── Core / returns
    'lr',
    'log_ret',

    # ── L2 volatility
    'hist_vol_20',
    'ewma_vol',
    'atr_ratio',
    'atr_14',
    'atr_ratio_true',
    'parkinson_vol',
    'garman_klass_vol',
    'rogers_satchell_vol',

    # ── L3 momentum / mean reversion
    'ma_spread',
    'rsi_raw',
    'rsi_norm',
    'z_score_ret',
    'mom_5',
    'mom_21',

    # ── Cross-pair returns
    'gbp_ret',
    'chf_ret',
    'jpy_ret',
    'ejp_ret',
    'ech_ret',
    'egb_ret',

    # ── Market macro
    'vix_level',
    'dxy_ret',
    'gold_ret',
    'oil_ret',
    'hyg_lqd_spr',
    'spread_2y10y',
    'us10y_chg',

    # ── Cross-currency factors
    'usd_factor',
    'usd_factor_z',
    'eur_dom',
    'chf_haven',
    'corr_eur_gbp',
    'corr_eur_chf',
    'corr_eur_jpy',
    'mi_eur_gbp',
    'mi_eur_chf',
    'mi_eur_jpy',

    # ── Original FRED macro features
    'cfnai_a',
    'cpi_a',
    'cpi_yoy',
    'cpi_yoy_z',
    'real_m2',
    'real_m2_z',
    'net_liq',
    'net_liq_z',
    'dxy_trend_z',
    'vix_z',
    'hy_z',
    'spx_trend_z',
    'unemp_z',
    'mfg_z',

    # ── Macro composite scores
    'growth_score',
    'liq_score',
    'policy_score',
    'rates_score',
    'sovereign_stress_score',
    'inflation_score',
    'energy_score',
    'risk_score',
    'macro_pressure_score',

    # ── Policy divergence
    'fedfunds_4h_new',
    'ecb_dfr_4h_new',
    'ecb_mlf_4h_new',
    'policy_diff_fed_ecb_dfr',
    'policy_diff_fed_ecb_mlf',
    'policy_diff_z63',
    'policy_diff_z252',
    'policy_diff_chg5',
    'policy_diff_chg21',

    # ── Term structure / transatlantic spreads
    'us10y_daily_4h_new',
    'us2y_daily_4h_new',
    'us10y_real_4h_new',
    'us10y_be_4h_new',
    'de10y_4h_new',
    'ea10y_4h_new',
    'it10y_4h_new',
    'us_curve_10y2y',
    'us_curve_z63',
    'us_curve_z252',
    'us_de_10y_spread',
    'us_de_10y_spread_z63',
    'us_de_10y_spread_z252',
    'us_de_10y_spread_chg21',
    'us_ea_10y_spread',
    'us_ea_10y_spread_z63',
    'us_ea_10y_spread_z252',

    # ── Sovereign credit / safe asset rotation
    'it_de_10y_spread',
    'it_de_10y_spread_z63',
    'it_de_10y_spread_z252',
    'it_de_10y_spread_chg21',
    'it_ea_10y_spread',
    'it_ea_10y_spread_z63',
    'it_ea_10y_spread_z252',

    # ── Real yield / breakeven
    'us10y_real',
    'us10y_real_z63',
    'us10y_real_z252',
    'us10y_real_chg5',
    'us10y_real_chg21',
    'us10y_be',
    'us10y_be_z63',
    'us10y_be_z252',
    'real_nominal_gap',
    'real_nominal_gap_z63',

    # ── Inflation divergence
    'ea_hicp_4h_new',
    'ea_hicp_yoy',
    'ea_hicp_yoy_z',
    'us_cpi_yoy',
    'us_cpi_yoy_z2',
    'inflation_gap_us_ea',
    'inflation_gap_us_ea_z63',
    'inflation_gap_us_ea_z252',

    # ── Growth
    'us_indpro_4h_new',
    'us_retail_4h_new',
    'us_indpro_yoy',
    'us_retail_yoy',
    'us_indpro_yoy_z',
    'us_retail_yoy_z',
    'us_growth_proxy',

    # ── Liquidity / money
    'm1_4h_new',
    'm1_real',
    'm1_real_yoy',
    'm1_real_yoy_z',
    'money_liquidity_proxy',

    # ── Energy macro
    'brent_fred_4h_new',
    'eu_gas_4h_new',
    'brent_ret_21d',
    'brent_ret_21d_z',
    'eu_gas_ret_21d',
    'eu_gas_ret_21d_z',
    'energy_pressure_eu',
    'energy_pressure_eu_z63',

    # ── Credit stress / late sample only
    'bbb_oas_4h_new',
    'bbb_oas',
    'bbb_oas_z63',
]

glb = globals()

for var_name in TZ_NAIVE_SERIES_VARS:
    if var_name in glb:
        if isinstance(glb[var_name], pd.Series):
            glb[var_name] = force_tz_naive_series(glb[var_name])

# DataFrame objeleri varsa onların indexlerini de normalize et
for df_name in [
    'extra_4h_df',
    'derived_macro_df',
    'main_derived_macro_df',
    'special_derived_macro_df',
    'new_macro_features_df',
    'new_macro_main_df',
    'new_macro_special_df',
    'macro_scores_df',
    'calendar_df',
]:
    if df_name in glb:
        if isinstance(glb[df_name], pd.DataFrame):
            glb[df_name] = glb[df_name].copy()
            glb[df_name].index = make_index_tz_naive(glb[df_name].index)
            glb[df_name] = glb[df_name].sort_index()
            glb[df_name] = glb[df_name][~glb[df_name].index.duplicated(keep="last")]

print("✓ TZ-naive patch tamam")
print(f"Normalize edilen Series sayısı   : {sum([(v in glb and isinstance(glb[v], pd.Series)) for v in TZ_NAIVE_SERIES_VARS])}")
print(f"Normalize edilen DataFrame sayısı: {sum([(v in glb and isinstance(glb[v], pd.DataFrame)) for v in ['extra_4h_df','derived_macro_df','main_derived_macro_df','special_derived_macro_df','new_macro_features_df','new_macro_main_df','new_macro_special_df','macro_scores_df','calendar_df']])}")

✓ TZ-naive patch tamam
Normalize edilen Series sayısı   : 136
Normalize edilen DataFrame sayısı: 7


In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK 10 — VOLUME FEATURES (FX pair + Gold)
# ════════════════════════════════════════════════════════════════════

# ── FX pair volume 4H align ──────────────────────────────────────
vol_eurusd = align_market_to_4h(eurusd_df["volume"], base_index)
vol_gbpusd = align_market_to_4h(gbpusd_df["volume"], base_index)
vol_usdchf = align_market_to_4h(usdchf_df["volume"], base_index)
vol_usdjpy = align_market_to_4h(usdjpy_df["volume"], base_index)
vol_eurchf = align_market_to_4h(eurchf_df["volume"], base_index)
vol_eurgbp = align_market_to_4h(eurgbp_df["volume"], base_index)
vol_eurjpy = align_market_to_4h(eurjpy_df["volume"], base_index)

# ── EUR/USD volume features ──────────────────────────────────────
eur_vol_ratio = vol_eurusd / (vol_eurusd.rolling(W_20D).mean() + 1e-10)

eur_vol_imbal = (
    (eurusd_df["volume_bid"] - eurusd_df["volume_ask"]) /
    (eurusd_df["volume_bid"] + eurusd_df["volume_ask"] + 1e-10)
)
eur_vol_imbal = align_market_to_4h(eur_vol_imbal, base_index)

# OBV-like flow momentum
# pct_change yerine normalize edilmiş diff kullanıyoruz.
# OBV negatif/pozitif geçişlerde pct_change aşırı değer üretebilir.
obv = (lr.apply(np.sign) * vol_eurusd).cumsum()
obv_mom = obv.diff(W_21D) / (vol_eurusd.rolling(W_21D).sum() + 1e-10)

# ── Cross-pair volume ────────────────────────────────────────────
chf_vol_ratio    = vol_usdchf / (vol_usdchf.rolling(W_20D).mean() + 1e-10)
jpy_vol_ratio    = vol_usdjpy / (vol_usdjpy.rolling(W_20D).mean() + 1e-10)
eurchf_vol_ratio = vol_eurchf / (vol_eurchf.rolling(W_20D).mean() + 1e-10)
eurjpy_vol_ratio = vol_eurjpy / (vol_eurjpy.rolling(W_20D).mean() + 1e-10)

# ── Composite volume ─────────────────────────────────────────────
risk_off_vol = (
    rolling_zscore(chf_vol_ratio, W_252D) +
    rolling_zscore(jpy_vol_ratio, W_252D)
) / 2

eur_activity = (
    rolling_zscore(eur_vol_ratio,    W_252D) +
    rolling_zscore(eurchf_vol_ratio, W_252D) +
    rolling_zscore(eurjpy_vol_ratio, W_252D)
) / 3

vol_regime = (hist_vol_20 / (hist_vol_20.rolling(W_63D).mean() + 1e-10)) - 1

# ── Gold volume ──────────────────────────────────────────────────
# gold_raw: Dukascopy'den çekilen DataFrame — volume kolonu varsa kullanılır.
if "volume" in gold_raw.columns:
    vol_gold = align_market_to_4h(gold_raw["volume"], base_index)
else:
    # fallback: NaN hacim daha güvenlidir; sıfır hacim yapay feature üretir.
    vol_gold = pd.Series(np.nan, index=base_index)

gold_vol_ratio = vol_gold / (vol_gold.rolling(W_20D).mean() + 1e-10)
gold_vol_z     = rolling_zscore(gold_vol_ratio, W_252D)

gold_obv = (gold_ret.apply(np.sign) * vol_gold).cumsum()
gold_obv_mom = gold_obv.diff(W_21D) / (vol_gold.rolling(W_21D).sum() + 1e-10)
gold_vol_trend = vol_gold.diff(W_21D) / (vol_gold.rolling(W_21D).mean() + 1e-10)

gold_activity = (
    rolling_zscore(gold_vol_ratio, W_252D) +
    rolling_zscore(gold_obv_mom,   W_252D) +
    rolling_zscore(gold_vol_trend, W_252D)
) / 3

gold_pressure = (
    rolling_zscore(gold_vol_ratio, W_252D) +
    rolling_zscore(gold_ret,       W_252D)
) / 2

print("── Volume Feature Kontrol ──")
for name, s in [
    ("eur_vol_ratio",    eur_vol_ratio),
    ("eur_vol_imbal",    eur_vol_imbal),
    ("obv_mom",          obv_mom),
    ("chf_vol_ratio",    chf_vol_ratio),
    ("jpy_vol_ratio",    jpy_vol_ratio),
    ("eurchf_vol_ratio", eurchf_vol_ratio),
    ("eurjpy_vol_ratio", eurjpy_vol_ratio),
    ("risk_off_vol",     risk_off_vol),
    ("eur_activity",     eur_activity),
    ("vol_regime",       vol_regime),
    ("gold_vol_ratio",   gold_vol_ratio),
    ("gold_vol_z",       gold_vol_z),
    ("gold_obv_mom",     gold_obv_mom),
    ("gold_activity",    gold_activity),
    ("gold_pressure",    gold_pressure),
]:
    nan_pct = s.isna().mean() * 100
    valid = s.dropna()

    if len(valid) == 0:
        print(f"  ⚠ {name:<22}: NaN={nan_pct:.1f}%  mean=nan  std=nan")
    else:
        print(
            f"  {'✓' if nan_pct < 20 else '⚠'} {name:<22}: "
            f"NaN={nan_pct:.1f}%  mean={valid.mean():.4f}  "
            f"std={valid.std():.4f}"
        )

── Volume Feature Kontrol ──
  ✓ eur_vol_ratio         : NaN=0.3%  mean=1.0042  std=0.7453
  ✓ eur_vol_imbal         : NaN=0.0%  mean=0.0183  std=0.0769
  ✓ obv_mom               : NaN=0.3%  mean=0.0039  std=0.1073
  ✓ chf_vol_ratio         : NaN=0.3%  mean=1.0134  std=1.2258
  ✓ jpy_vol_ratio         : NaN=0.3%  mean=1.0149  std=1.0156
  ✓ eurchf_vol_ratio      : NaN=0.3%  mean=1.0157  std=1.2505
  ✓ eurjpy_vol_ratio      : NaN=0.3%  mean=1.0113  std=0.7141
  ✓ risk_off_vol          : NaN=4.3%  mean=0.0022  std=0.9783
  ✓ eur_activity          : NaN=4.3%  mean=0.0007  std=0.9482
  ✓ vol_regime            : NaN=1.3%  mean=0.0030  std=0.2154
  ✓ gold_vol_ratio        : NaN=0.3%  mean=1.0062  std=3.1140
  ✓ gold_vol_z            : NaN=4.3%  mean=-0.0006  std=1.0341
  ✓ gold_obv_mom          : NaN=0.3%  mean=0.0693  std=0.3082
  ✓ gold_activity         : NaN=4.4%  mean=0.0022  std=0.7441
  ✓ gold_pressure         : NaN=4.3%  mean=0.0010  std=0.7295


In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK 11 — FEATURE_DF OLUŞTUR (tüm ham/hesaplanan seriler)
# ════════════════════════════════════════════════════════════════════

feature_base_dict = {
    # ── core
    "log_ret"        : lr,

    # ── vol
    "hist_vol_20"    : hist_vol_20,
    "ewma_vol"       : ewma_vol,
    "atr_ratio"      : atr_ratio,
    "atr_ratio_true" : atr_ratio_true,
    "park_vol"       : parkinson_vol,
    "gk_vol"         : garman_klass_vol,
    "rs_vol"         : rogers_satchell_vol,

    # ── momentum
    "ma_spread"      : ma_spread,
    "rsi_norm"       : rsi_norm,
    "z_score_ret"    : z_score_ret,
    "mom_5"          : mom_5,
    "mom_21"         : mom_21,

    # ── pair returns
    "gbp_ret"        : gbp_ret,
    "chf_ret"        : chf_ret,
    "jpy_ret"        : jpy_ret,
    "ech_ret"        : ech_ret,
    "egb_ret"        : egb_ret,
    "ejp_ret"        : ejp_ret,

    # ── market macro
    "vix_level"      : vix_level,
    "dxy_ret"        : dxy_ret,
    "gold_ret"       : gold_ret,
    "oil_ret"        : oil_ret,
    "hyg_lqd_spr"    : hyg_lqd_spr,
    "spread_2y10y"   : spread_2y10y,
    "us10y_chg"      : us10y_chg,

    # ── cross currency
    "usd_factor"     : usd_factor,
    "usd_factor_z"   : usd_factor_z,
    "eur_dom"        : eur_dom,
    "chf_haven"      : chf_haven,
    "corr_eur_gbp"   : corr_eur_gbp,
    "corr_eur_chf"   : corr_eur_chf,
    "corr_eur_jpy"   : corr_eur_jpy,
    "mi_eur_gbp"     : mi_eur_gbp,
    "mi_eur_chf"     : mi_eur_chf,
    "mi_eur_jpy"     : mi_eur_jpy,

    # ── original FRED macro
    "cfnai_a"        : cfnai_a,
    "cpi_yoy_z"      : cpi_yoy_z,
    "real_m2_z"      : real_m2_z,
    "net_liq_z"      : net_liq_z,
    "dxy_trend_z"    : dxy_trend_z,
    "vix_z"          : vix_z,
    # "hy_z"         : hy_z,  # late-sample only; ana feature_df içine varsayılan olarak koymuyoruz
    "spx_trend_z"    : spx_trend_z,
    "unemp_z"        : unemp_z,
    "mfg_z"          : mfg_z,

    # ── strengthened composite scores
    "growth_score"   : growth_score,
    "liq_score"      : liq_score,
    "risk_score"     : risk_score,
    "policy_score"   : policy_score,
    "rates_score"    : rates_score,
    "sovereign_stress_score": sovereign_stress_score,
    "inflation_score": inflation_score,
    "energy_score"   : energy_score,
    "macro_pressure_score": macro_pressure_score,

    # ── volume
    "eur_vol_ratio"  : eur_vol_ratio,
    "eur_vol_imbal"  : eur_vol_imbal,
    "obv_mom"        : obv_mom,
    "chf_vol_ratio"  : chf_vol_ratio,
    "jpy_vol_ratio"  : jpy_vol_ratio,
    "eurchf_vol_ratio": eurchf_vol_ratio,
    "eurjpy_vol_ratio": eurjpy_vol_ratio,
    "risk_off_vol"   : risk_off_vol,
    "eur_activity"   : eur_activity,
    "vol_regime"     : vol_regime,
    "gold_vol_ratio" : gold_vol_ratio,
    "gold_vol_z"     : gold_vol_z,
    "gold_obv_mom"   : gold_obv_mom,
    "gold_activity"  : gold_activity,
    "gold_pressure"  : gold_pressure,
}

# Sadece var olan ve Series olanları al
feature_base_dict = {
    k: v for k, v in feature_base_dict.items()
    if isinstance(v, pd.Series)
}

feature_df = pd.DataFrame(feature_base_dict, index=base_index)


# ════════════════════════════════════════════════════════════════════
# Yeni macro derived feature'ları feature_df içine ekle
# Late-sample kolonları varsayılan olarak dışarıda bırak
# ════════════════════════════════════════════════════════════════════

LATE_SAMPLE_COLS = {
    "hy_z",
    "bbb_oas",
    "bbb_oas_z63",
}

# new_macro_main_df oluşturulduysa onu tercih et
if "new_macro_main_df" in globals() and isinstance(new_macro_main_df, pd.DataFrame):
    tmp = new_macro_main_df.copy()
    tmp.index = make_index_tz_naive(tmp.index)

    tmp = tmp[[c for c in tmp.columns if c not in LATE_SAMPLE_COLS]]
    feature_df = pd.concat([feature_df, tmp], axis=1)

# Eğer new_macro_main_df yoksa ama new_macro_features_df varsa, güvenli kolonları elle seç
elif "new_macro_features_df" in globals() and isinstance(new_macro_features_df, pd.DataFrame):
    tmp = new_macro_features_df.copy()
    tmp.index = make_index_tz_naive(tmp.index)

    SAFE_NEW_MACRO_COLS = [
        # Policy
        "policy_diff_fed_ecb_dfr",
        "policy_diff_fed_ecb_mlf",
        "policy_diff_z63",
        "policy_diff_z252",
        "policy_diff_chg5",
        "policy_diff_chg21",

        # Rates / term structure
        "us_curve_10y2y",
        "us_curve_z63",
        "us_curve_z252",
        "us_de_10y_spread",
        "us_de_10y_spread_z63",
        "us_de_10y_spread_z252",
        "us_de_10y_spread_chg21",
        "us_ea_10y_spread",
        "us_ea_10y_spread_z63",
        "us_ea_10y_spread_z252",

        # Sovereign stress
        "it_de_10y_spread",
        "it_de_10y_spread_z63",
        "it_de_10y_spread_z252",
        "it_de_10y_spread_chg21",
        "it_ea_10y_spread",
        "it_ea_10y_spread_z63",
        "it_ea_10y_spread_z252",

        # Real yield / breakeven
        "us10y_real",
        "us10y_real_z63",
        "us10y_real_z252",
        "us10y_real_chg5",
        "us10y_real_chg21",
        "us10y_be",
        "us10y_be_z63",
        "us10y_be_z252",
        "real_nominal_gap",
        "real_nominal_gap_z63",

        # Inflation
        "ea_hicp_yoy",
        "ea_hicp_yoy_z",
        "us_cpi_yoy",
        "us_cpi_yoy_z2",
        "inflation_gap_us_ea",
        "inflation_gap_us_ea_z63",
        "inflation_gap_us_ea_z252",

        # Growth / liquidity
        "us_indpro_yoy",
        "us_retail_yoy",
        "us_indpro_yoy_z",
        "us_retail_yoy_z",
        "us_growth_proxy",
        "m1_real_yoy",
        "m1_real_yoy_z",
        "money_liquidity_proxy",

        # Energy
        "brent_ret_21d",
        "brent_ret_21d_z",
        "eu_gas_ret_21d",
        "eu_gas_ret_21d_z",
        "energy_pressure_eu",
        "energy_pressure_eu_z63",
    ]

    cols = [c for c in SAFE_NEW_MACRO_COLS if c in tmp.columns and c not in LATE_SAMPLE_COLS]
    feature_df = pd.concat([feature_df, tmp[cols]], axis=1)


# Eğer macro_scores_df oluşturulduysa composite skorları overwrite / garanti et
if "macro_scores_df" in globals() and isinstance(macro_scores_df, pd.DataFrame):
    score_tmp = macro_scores_df.copy()
    score_tmp.index = make_index_tz_naive(score_tmp.index)

    for col in score_tmp.columns:
        feature_df[col] = score_tmp[col]


# Calendar features varsa ekle
if "calendar_df" in globals() and isinstance(calendar_df, pd.DataFrame):
    cal_tmp = calendar_df.copy()
    cal_tmp.index = make_index_tz_naive(cal_tmp.index)
    feature_df = pd.concat([feature_df, cal_tmp], axis=1)


# Duplicate kolonları temizle
feature_df = feature_df.loc[:, ~feature_df.columns.duplicated()].copy()

# Index temizliği
feature_df.index = make_index_tz_naive(feature_df.index)
feature_df = feature_df.sort_index()
feature_df = feature_df[~feature_df.index.duplicated(keep="last")]

# Sonsuz değerleri temizle
feature_df = feature_df.replace([np.inf, -np.inf], np.nan)


# ════════════════════════════════════════════════════════════════════
# Kontrol
# ════════════════════════════════════════════════════════════════════

print(f"\nfeature_df shape: {feature_df.shape}")
print(f"NaN oranı (ortalama): {feature_df.isna().mean().mean():.3f}")

nan_report = (
    feature_df.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .to_frame("nan_pct")
)

print("\nTop 25 NaN oranı yüksek feature:")
print(nan_report.head(25).round(2).to_string())

print("\nfeature_df head:")
print(feature_df.head(3));


feature_df shape: (37614, 135)
NaN oranı (ortalama): 0.023

Top 25 NaN oranı yüksek feature:
                          nan_pct
oil_ret                     15.45
hyg_lqd_spr                 14.12
mfg_z                        8.57
cpi_yoy_z                    8.40
us_retail_yoy_z              8.40
us_growth_proxy              8.40
real_m2_z                    8.40
ea_hicp_yoy_z                8.40
us_cpi_yoy_z2                8.40
inflation_gap_us_ea_z252     8.40
money_liquidity_proxy        8.40
m1_real_yoy_z                8.40
us_indpro_yoy_z              8.40
net_liq_z                    8.19
energy_pressure_eu_z63       5.72
inflation_gap_us_ea_z63      5.39
liq_score                    4.85
dxy_trend_z                  4.85
spx_trend_z                  4.85
eu_gas_ret_21d_z             4.72
energy_pressure_eu           4.72
energy_score                 4.72
brent_ret_21d_z              4.39
us_cpi_yoy                   4.38
us_retail_yoy                4.38

feature_df head:
    

In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK 12 — TEKNİK İNDİKATÖRLER
# df — EUR/USD mid OHLCV üzerinde
# master_df join için hazır
# ════════════════════════════════════════════════════════════════════

# Not:
# Bu blok tekrar Dukascopy fetch yapmaz.
# Önceki bloklarda hazırlanmış eurusd_df üzerinden çalışır.
# Böylece veri tutarlılığı korunur ve gereksiz tekrar download önlenir.

df_mid = eurusd_df.copy()
df_mid.index = make_index_tz_naive(df_mid.index)
df_mid = df_mid.sort_index()
df_mid = df_mid[~df_mid.index.duplicated(keep="last")]

# base_index ile hizala
df_mid = df_mid.reindex(base_index)

# Backtesting.py ve teknik indikatörler için büyük harf OHLCV
df_mid["Open"]   = df_mid["open"]
df_mid["High"]   = df_mid["high"]
df_mid["Low"]    = df_mid["low"]
df_mid["Close"]  = df_mid["close"]
df_mid["Volume"] = df_mid["volume"]

# Spread varsa hesapla
if {"close_ask", "close_bid"}.issubset(df_mid.columns):
    df_mid["spread_close"] = df_mid["close_ask"] - df_mid["close_bid"]
    df_mid["spread_pct"] = df_mid["spread_close"] / (df_mid["Close"] + 1e-10)
else:
    df_mid["spread_close"] = np.nan
    df_mid["spread_pct"] = np.nan

df = df_mid.copy()


# ── MA ───────────────────────────────────────────────────────────
for w in [10, 20, 30, 50, 100, 200]:
    df[f"SMA_{w}"] = df["Close"].rolling(w).mean()
    df[f"EMA_{w}"] = df["Close"].ewm(span=w, adjust=False).mean()


def wma(series, period):
    weights = np.arange(1, period + 1)
    return series.rolling(period).apply(
        lambda x: np.dot(x, weights) / weights.sum(),
        raw=True
    )


n = 20
df["HMA"] = wma(
    2 * wma(df["Close"], n // 2) - wma(df["Close"], n),
    int(np.sqrt(n))
)


# ── Bollinger Bands ───────────────────────────────────────────────
_rm = df["Close"].rolling(20).mean()
_rs = df["Close"].rolling(20).std()

df["BB_Upper"] = _rm + 2 * _rs
df["BB_Lower"] = _rm - 2 * _rs
df["BB_Mid"]   = _rm

df["Bollinger_Bandwidth"] = (
    (df["BB_Upper"] - df["BB_Lower"]) /
    (df["BB_Mid"] + 1e-10)
)

df["BB_PctB"] = (
    (df["Close"] - df["BB_Lower"]) /
    ((df["BB_Upper"] - df["BB_Lower"]) + 1e-10)
)


# ── RSI ───────────────────────────────────────────────────────────
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()

    rs = avg_gain / (avg_loss + 1e-10)
    return 100 - (100 / (1 + rs))


df["RSI"] = compute_rsi(df["Close"], 14)


# ── Stochastic ────────────────────────────────────────────────────
_low_min  = df["Low"].rolling(14).min()
_high_max = df["High"].rolling(14).max()

df["Stoch_K"] = 100 * (
    (df["Close"] - _low_min) /
    ((_high_max - _low_min) + 1e-10)
)

df["Stoch_D"] = df["Stoch_K"].rolling(3).mean()

_rsi_min = df["RSI"].rolling(14).min()
_rsi_max = df["RSI"].rolling(14).max()

df["Stoch_RSI_K"] = (
    ((df["RSI"] - _rsi_min) / ((_rsi_max - _rsi_min) + 1e-10))
    .rolling(3)
    .mean()
    * 100
)

df["Stoch_RSI_D"] = df["Stoch_RSI_K"].rolling(3).mean()


# ── MACD ──────────────────────────────────────────────────────────
_ema12 = df["Close"].ewm(span=12, adjust=False).mean()
_ema26 = df["Close"].ewm(span=26, adjust=False).mean()

df["MACD"]        = _ema12 - _ema26
df["MACD_Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
df["MACD_Hist"]   = df["MACD"] - df["MACD_Signal"]


# ── CCI ───────────────────────────────────────────────────────────
_tp = (df["High"] + df["Low"] + df["Close"]) / 3

_mad = _tp.rolling(20).apply(
    lambda x: np.mean(np.abs(x - x.mean())),
    raw=True
)

df["CCI_20"] = (
    (_tp - _tp.rolling(20).mean()) /
    ((0.015 * _mad) + 1e-10)
)


# ── ADX ───────────────────────────────────────────────────────────
_hd = df["High"].diff()
_ld = df["Low"].diff()

_pdm = np.where((_hd > _ld.abs()) & (_hd > 0), _hd, 0.0)
_ndm = np.where((_ld.abs() > _hd) & (_ld < 0), _ld.abs(), 0.0)

_tr = pd.concat([
    df["High"] - df["Low"],
    (df["High"] - df["Close"].shift()).abs(),
    (df["Low"]  - df["Close"].shift()).abs()
], axis=1).max(axis=1)

_atr = _tr.ewm(com=13, min_periods=14).mean()

_pdi = 100 * pd.Series(_pdm, index=df.index).ewm(com=13).mean() / (_atr + 1e-10)
_ndi = 100 * pd.Series(_ndm, index=df.index).ewm(com=13).mean() / (_atr + 1e-10)

_dx = 100 * (_pdi - _ndi).abs() / ((_pdi + _ndi) + 1e-10)

df["ADX_14"] = _dx.ewm(com=13, min_periods=14).mean()
df["ADX"]    = df["ADX_14"]


# ── Williams %R ───────────────────────────────────────────────────
_w_high = df["High"].rolling(14).max()
_w_low  = df["Low"].rolling(14).min()

df["Williams_%R"] = -100 * (
    (_w_high - df["Close"]) /
    ((_w_high - _w_low) + 1e-10)
)


# ── Ichimoku ─────────────────────────────────────────────────────
df["Tenkan_Sen"] = (
    df["High"].rolling(9).max() +
    df["Low"].rolling(9).min()
) / 2

df["Kijun_Sen"] = (
    df["High"].rolling(26).max() +
    df["Low"].rolling(26).min()
) / 2

# Senkou_A/B teknik çizimde ileriye projekte edilir.
# shift(26) burada current row'a geçmişten hesaplanmış değeri getirir; future leakage değildir.
df["Senkou_A"] = ((df["Tenkan_Sen"] + df["Kijun_Sen"]) / 2).shift(26)

df["Senkou_B"] = (
    (
        df["High"].rolling(52).max() +
        df["Low"].rolling(52).min()
    ) / 2
).shift(26)

# DİKKAT:
# Chikou = Close.shift(-26) future close içerir.
# Model feature'ı olarak kullanılmamalıdır.
# Bu yüzden ana df içine koymuyoruz.
# Görsel analiz için gerekirse ayrı değişkende tutulabilir:
chikou_visual_only = df["Close"].shift(-26)


# ── Return / Volatility / Momentum ───────────────────────────────
df["log_return"] = np.log(df["Close"] / df["Close"].shift(1))

df["log_return_lag1"] = df["log_return"].shift(1)
df["log_return_lag2"] = df["log_return"].shift(2)
df["log_return_lag3"] = df["log_return"].shift(3)

df["Exponential_Weighted_Log_Return"] = (
    df["log_return"]
    .ewm(span=10, adjust=False)
    .mean()
)

# 4H annualized volatility
df["Historical_Volatility"] = (
    df["log_return"].rolling(20).std() *
    np.sqrt(BAR_PER_DAY * 252)
)

df["HV"] = df["Historical_Volatility"]

df["Momentum_10"] = df["log_return"].rolling(10).sum()
df["ROC"] = df["Close"].pct_change(10) * 100


# ── Takvim ────────────────────────────────────────────────────────
df["Hour"]       = df.index.hour
df["DayOfWeek"]  = df.index.dayofweek
df["Month"]      = df.index.month
df["DayOfMonth"] = df.index.day
df["Quarter"]    = df.index.quarter
df["WeekOfYear"] = df.index.isocalendar().week.astype(int)

# Ek leak-free calendar proxies
df["Is_Friday"] = (df.index.dayofweek == 4).astype(int)
df["Is_Monday"] = (df.index.dayofweek == 0).astype(int)
df["Is_Month_Start"] = df.index.is_month_start.astype(int)
df["Is_Month_End"] = df.index.is_month_end.astype(int)

df["Week_Of_Month"] = ((df.index.day - 1) // 7 + 1)

df["Is_First_Friday"] = (
    (df.index.dayofweek == 4) &
    (df["Week_Of_Month"] == 1)
).astype(int)

df["Is_London_US_Overlap"] = df["Hour"].between(12, 16).astype(int)
df["Is_Asia_Session"] = df["Hour"].between(0, 8).astype(int)
df["Is_US_Session"] = df["Hour"].between(13, 21).astype(int)


# ── Final cleanup ────────────────────────────────────────────────
df.index = make_index_tz_naive(df.index)
df = df.sort_index()
df = df[~df.index.duplicated(keep="last")]
df = df.replace([np.inf, -np.inf], np.nan)

# Future-leak kolon güvenlik kontrolü
FORBIDDEN_TECH_COLS = [
    "Chikou",
]

for col in FORBIDDEN_TECH_COLS:
    if col in df.columns:
        raise ValueError(f"Leakage riski: {col} df içinde bulunuyor. Model feature olarak kullanılamaz.")


print(f"\ndf shape: {df.shape}")
print(f"df kolonu sayısı: {len(df.columns)}")
print(f"df NaN ortalama oranı: {df.isna().mean().mean():.4f}")

print("\nTop 20 NaN oranı yüksek teknik kolon:")
print(
    df.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .head(20)
    .round(2)
    .to_string()
)

print("✓ Teknik indikatörler tamam")

print("\n" + "═" * 60)
print("VERİ ÇEKİMİ + FEATURE ENGINEERING TAMAMLANDI")
print("Hazır değişkenler:")
print("  feature_df  — ham/hesaplanan seriler")
print("  df          — EUR/USD teknik indikatörler")
print("  C, H, L, O, V  — EUR/USD 4H OHLCV serileri")
print("  base_index  — 4H zaman index")
print("  Tüm seri değişkenleri")
print("═" * 60)


df shape: (37614, 80)
df kolonu sayısı: 80
df NaN ortalama oranı: 0.0003

Top 20 NaN oranı yüksek teknik kolon:
SMA_200                  0.53
SMA_100                  0.26
Senkou_B                 0.20
Senkou_A                 0.14
SMA_50                   0.13
Stoch_RSI_D              0.08
Stoch_RSI_K              0.08
SMA_30                   0.08
ADX_14                   0.07
ADX                      0.07
Kijun_Sen                0.07
HMA                      0.06
Historical_Volatility    0.05
HV                       0.05
SMA_20                   0.05
BB_Lower                 0.05
CCI_20                   0.05
BB_Upper                 0.05
BB_Mid                   0.05
Bollinger_Bandwidth      0.05
✓ Teknik indikatörler tamam

════════════════════════════════════════════════════════════
VERİ ÇEKİMİ + FEATURE ENGINEERING TAMAMLANDI
Hazır değişkenler:
  feature_df  — ham/hesaplanan seriler
  df          — EUR/USD teknik indikatörler
  C, H, L, O, V  — EUR/USD 4H OHLCV serileri
  bas

In [ ]:
# ════════════════════════════════════════════════════════════════════
# PATCH — Blok C öncesi çalıştır
# align_macro_to_4h'yi base_index default yapıp yeniden tanımla
# ════════════════════════════════════════════════════════════════════

def align_macro_to_4h(s, idx=None, lag_periods=0):
    """
    Macro / düşük frekanslı serileri 4H index'e hizalar.

    - idx verilmezse global base_index kullanılır.
    - ffill kullanır.
    - lag_periods 4H bar sayısıdır.
      Örnek:
        lag_periods=1       -> 1 bar = 4 saat
        lag_periods=6       -> yaklaşık 1 gün
        lag_periods=6*7     -> yaklaşık 7 gün
    """
    if idx is None:
        if "base_index" not in globals():
            raise ValueError("idx verilmedi ve global base_index bulunamadı.")
        idx = base_index

    s = pd.Series(s).copy()
    s.index = make_index_tz_naive(s.index)
    s = s.sort_index()
    s = s[~s.index.duplicated(keep="last")]

    idx = make_index_tz_naive(idx)

    out = (
        s.reindex(s.index.union(idx))
         .sort_index()
         .ffill()
         .reindex(idx)
    )

    if lag_periods > 0:
        out = out.shift(lag_periods)

    return out


# Test
print("align_macro_to_4h patch tamam")
print(f"  base_index mevcut: {len(base_index)} bar")

align_macro_to_4h patch tamam
  base_index mevcut: 37614 bar


In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK A — MASTER_DF KURULUM
# feature_df + teknik indikatörler + lag returns + target + split
# SAFE VERSION
# ════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd

# ─────────────────────────────────────────────────────────────────
# Yardımcı güvenli fonksiyonlar
# ─────────────────────────────────────────────────────────────────

def safe_tz_naive_index(idx):
    """
    Her türlü DatetimeIndex'i tz-naive hale getirir.
    Zaten tz-naive ise hata vermez.
    """
    idx = pd.to_datetime(idx)
    try:
        if getattr(idx, "tz", None) is not None:
            idx = idx.tz_convert(None)
    except Exception:
        pass

    try:
        if getattr(idx, "tz", None) is not None:
            idx = idx.tz_localize(None)
    except Exception:
        pass

    return idx


def safe_series_to_master(s, master_index, shift_periods=0, ffill=False):
    """
    Series'i master_df indexine güvenli hizalar.
    """
    s = pd.Series(s).copy()
    s.index = safe_tz_naive_index(s.index)
    s = s.sort_index()
    s = s[~s.index.duplicated(keep="last")]

    master_index = safe_tz_naive_index(master_index)

    if ffill:
        out = s.reindex(s.index.union(master_index)).sort_index().ffill().reindex(master_index)
    else:
        out = s.reindex(master_index)

    if shift_periods != 0:
        out = out.shift(shift_periods)

    return out


def safe_df_to_master(df_in, master_index):
    """
    DataFrame'i master index formatına güvenli getirir.
    """
    out = df_in.copy()
    out.index = safe_tz_naive_index(out.index)
    out = out.sort_index()
    out = out[~out.index.duplicated(keep="last")]
    return out.reindex(master_index)


# ─────────────────────────────────────────────────────────────────
# 1) master_df başlangıç
# ─────────────────────────────────────────────────────────────────

master_df = feature_df.copy()
master_df.index = safe_tz_naive_index(master_df.index)
master_df = master_df.sort_index()
master_df = master_df[~master_df.index.duplicated(keep="last")]
master_df = master_df.replace([np.inf, -np.inf], np.nan)

# log_ret sütununu yeniden garanti et
master_df["log_ret"] = safe_series_to_master(lr, master_df.index)

# Güvenlik: log_ret tamamen boşsa erken uyar
if master_df["log_ret"].notna().sum() == 0:
    raise ValueError("master_df['log_ret'] tamamen NaN. lr / base_index hizalamasını kontrol et.")


# ─────────────────────────────────────────────────────────────────
# 2) Teknik indikatörler join
# ─────────────────────────────────────────────────────────────────

FORBIDDEN_TECH_COLS = [
    "Chikou",   # Close.shift(-26), future leakage
]

for col in FORBIDDEN_TECH_COLS:
    if "df" in globals() and isinstance(df, pd.DataFrame) and col in df.columns:
        raise ValueError(f"Leakage riski: df içinde {col} var. Bu kolon model feature olarak kullanılamaz.")

TECH_FEATURES = [
    # Oscillators / indicators
    "Stoch_K",
    "Stoch_D",
    "Stoch_RSI_K",
    "Stoch_RSI_D",
    "RSI",
    "BB_PctB",
    "Bollinger_Bandwidth",
    "ADX_14",
    "ADX",
    "CCI_20",
    "Williams_%R",
    "Momentum_10",
    "MACD",
    "MACD_Hist",
    "MACD_Signal",

    # Return / volatility
    "log_return",
    "log_return_lag1",
    "log_return_lag2",
    "log_return_lag3",
    "Historical_Volatility",
    "HV",
    "Exponential_Weighted_Log_Return",
    "ROC",

    # Moving averages / trend
    "SMA_10",
    "SMA_20",
    "SMA_30",
    "SMA_50",
    "SMA_100",
    "SMA_200",
    "EMA_10",
    "EMA_20",
    "EMA_30",
    "EMA_50",
    "EMA_100",
    "EMA_200",
    "HMA",
    "Tenkan_Sen",
    "Kijun_Sen",
    "Senkou_A",
    "Senkou_B",

    # Spread / microstructure, if available
    "spread_close",
    "spread_pct",

    # Calendar
    "Month",
    "DayOfWeek",
    "Hour",
    "DayOfMonth",
    "Quarter",
    "WeekOfYear",
    "Is_Friday",
    "Is_Monday",
    "Is_Month_Start",
    "Is_Month_End",
    "Week_Of_Month",
    "Is_First_Friday",
    "Is_London_US_Overlap",
    "Is_Asia_Session",
    "Is_US_Session",
]

if "df" in globals() and isinstance(df, pd.DataFrame):
    df_safe = df.copy()
    df_safe.index = safe_tz_naive_index(df_safe.index)
    df_safe = df_safe.sort_index()
    df_safe = df_safe[~df_safe.index.duplicated(keep="last")]

    tech_cols = [c for c in TECH_FEATURES if c in df_safe.columns]
    tech_join = df_safe[tech_cols].copy()
    tech_join = tech_join.reindex(master_df.index)

    master_df = master_df.join(tech_join, how="left", rsuffix="_tech")
else:
    print("[WARN] df bulunamadı; teknik indikatör join atlandı.")


# ─────────────────────────────────────────────────────────────────
# 3) Pair / market lag returns
# İşlem zamanlaması için lag1 güvenli kullanılır.
# ─────────────────────────────────────────────────────────────────

lag_series_map = {
    "gbp_ret_lag1": gbp_ret,
    "chf_ret_lag1": chf_ret,
    "jpy_ret_lag1": jpy_ret,
    "gold_ret_lag1": gold_ret,
    "oil_ret_lag1": oil_ret,
    "dxy_ret_lag1": dxy_ret,
}

for name, s in lag_series_map.items():
    if isinstance(s, pd.Series):
        master_df[name] = safe_series_to_master(s, master_df.index, shift_periods=1)


# ─────────────────────────────────────────────────────────────────
# 4) SPX ve DXY level
# Not: dxy level istenirse ayrı tutulur. Ana DXY-free testlerde drop edilebilir.
# ─────────────────────────────────────────────────────────────────

level_series_map = {
    "sp500": spx_4h,
    "dxy": dxy_4h,
}

for name, s in level_series_map.items():
    if isinstance(s, pd.Series):
        master_df[name] = safe_series_to_master(s, master_df.index, ffill=True)


# ─────────────────────────────────────────────────────────────────
# 5) Ana feature serilerini yeniden garanti et
# Late-sample kolonlar varsayılan olarak ana master'a alınmaz.
# ─────────────────────────────────────────────────────────────────

rebuild_cols = [
    # Volatility
    ("hist_vol_20", hist_vol_20),
    ("ewma_vol", ewma_vol),
    ("atr_ratio", atr_ratio),
    ("atr_ratio_true", atr_ratio_true if "atr_ratio_true" in globals() else None),
    ("park_vol", parkinson_vol),
    ("gk_vol", garman_klass_vol),
    ("rs_vol", rogers_satchell_vol),

    # Momentum
    ("ma_spread", ma_spread),
    ("rsi_norm", rsi_norm),
    ("z_score_ret", z_score_ret),
    ("mom_5", mom_5),
    ("mom_21", mom_21),

    # Pair returns
    ("gbp_ret", gbp_ret),
    ("chf_ret", chf_ret),
    ("jpy_ret", jpy_ret),
    ("ech_ret", ech_ret),
    ("egb_ret", egb_ret),
    ("ejp_ret", ejp_ret),

    # Market macro
    ("vix_level", vix_level),
    ("dxy_ret", dxy_ret),
    ("gold_ret", gold_ret),
    ("oil_ret", oil_ret),
    ("hyg_lqd_spr", hyg_lqd_spr),
    ("spread_2y10y", spread_2y10y),
    ("us10y_chg", us10y_chg),

    # Cross currency
    ("usd_factor", usd_factor),
    ("usd_factor_z", usd_factor_z),
    ("eur_dom", eur_dom),
    ("chf_haven", chf_haven),
    ("corr_eur_gbp", corr_eur_gbp),
    ("corr_eur_chf", corr_eur_chf),
    ("corr_eur_jpy", corr_eur_jpy),
    ("mi_eur_gbp", mi_eur_gbp),
    ("mi_eur_chf", mi_eur_chf),
    ("mi_eur_jpy", mi_eur_jpy),

    # Original macro
    ("cfnai_a", cfnai_a),
    ("cpi_yoy_z", cpi_yoy_z),
    ("real_m2_z", real_m2_z),
    ("net_liq_z", net_liq_z),
    ("dxy_trend_z", dxy_trend_z),
    ("vix_z", vix_z),
    # ("hy_z", hy_z),  # late-sample only, ana master'a varsayılan olarak almıyoruz
    ("spx_trend_z", spx_trend_z),
    ("unemp_z", unemp_z),
    ("mfg_z", mfg_z),

    # Strengthened macro scores
    ("growth_score", growth_score),
    ("liq_score", liq_score),
    ("risk_score", risk_score),
    ("policy_score", policy_score if "policy_score" in globals() else None),
    ("rates_score", rates_score if "rates_score" in globals() else None),
    ("sovereign_stress_score", sovereign_stress_score if "sovereign_stress_score" in globals() else None),
    ("inflation_score", inflation_score if "inflation_score" in globals() else None),
    ("energy_score", energy_score if "energy_score" in globals() else None),
    ("macro_pressure_score", macro_pressure_score if "macro_pressure_score" in globals() else None),

    # Volume
    ("eur_vol_ratio", eur_vol_ratio),
    ("eur_vol_imbal", eur_vol_imbal if "eur_vol_imbal" in globals() else None),
    ("obv_mom", obv_mom),
    ("chf_vol_ratio", chf_vol_ratio),
    ("jpy_vol_ratio", jpy_vol_ratio),
    ("eurchf_vol_ratio", eurchf_vol_ratio),
    ("eurjpy_vol_ratio", eurjpy_vol_ratio if "eurjpy_vol_ratio" in globals() else None),
    ("risk_off_vol", risk_off_vol),
    ("eur_activity", eur_activity),
    ("vol_regime", vol_regime),
    ("gold_vol_ratio", gold_vol_ratio if "gold_vol_ratio" in globals() else None),
    ("gold_vol_z", gold_vol_z if "gold_vol_z" in globals() else None),
    ("gold_obv_mom", gold_obv_mom if "gold_obv_mom" in globals() else None),
    ("gold_activity", gold_activity if "gold_activity" in globals() else None),
    ("gold_pressure", gold_pressure if "gold_pressure" in globals() else None),
]

for col_name, series in rebuild_cols:
    if isinstance(series, pd.Series):
        try:
            master_df[col_name] = safe_series_to_master(series, master_df.index)
        except Exception as e:
            print(f"[SKIP] {col_name}: {e}")


# ─────────────────────────────────────────────────────────────────
# 6) Yeni macro derived dataframe'lerini master_df içine bağla
# Late-sample kolonlar dışarıda.
# ─────────────────────────────────────────────────────────────────

LATE_SAMPLE_COLS = {
    "hy_z",
    "bbb_oas",
    "bbb_oas_z63",
}

def join_safe_dataframe_to_master(master, df_obj, drop_cols=None):
    if not isinstance(df_obj, pd.DataFrame):
        return master

    drop_cols = set(drop_cols or [])

    tmp = df_obj.copy()
    tmp.index = safe_tz_naive_index(tmp.index)
    tmp = tmp.sort_index()
    tmp = tmp[~tmp.index.duplicated(keep="last")]
    tmp = tmp.reindex(master.index)
    tmp = tmp[[c for c in tmp.columns if c not in drop_cols]]

    # Var olan kolonları overwrite et; olmayanları ekle
    for c in tmp.columns:
        master[c] = tmp[c]

    return master


if "new_macro_main_df" in globals():
    master_df = join_safe_dataframe_to_master(master_df, new_macro_main_df, drop_cols=LATE_SAMPLE_COLS)
elif "new_macro_features_df" in globals():
    SAFE_NEW_MACRO_COLS = [
        # Policy
        "policy_diff_fed_ecb_dfr",
        "policy_diff_fed_ecb_mlf",
        "policy_diff_z63",
        "policy_diff_z252",
        "policy_diff_chg5",
        "policy_diff_chg21",

        # Rates / term structure
        "us_curve_10y2y",
        "us_curve_z63",
        "us_curve_z252",
        "us_de_10y_spread",
        "us_de_10y_spread_z63",
        "us_de_10y_spread_z252",
        "us_de_10y_spread_chg21",
        "us_ea_10y_spread",
        "us_ea_10y_spread_z63",
        "us_ea_10y_spread_z252",

        # Sovereign
        "it_de_10y_spread",
        "it_de_10y_spread_z63",
        "it_de_10y_spread_z252",
        "it_de_10y_spread_chg21",
        "it_ea_10y_spread",
        "it_ea_10y_spread_z63",
        "it_ea_10y_spread_z252",

        # Real yield / breakeven
        "us10y_real",
        "us10y_real_z63",
        "us10y_real_z252",
        "us10y_real_chg5",
        "us10y_real_chg21",
        "us10y_be",
        "us10y_be_z63",
        "us10y_be_z252",
        "real_nominal_gap",
        "real_nominal_gap_z63",

        # Inflation
        "ea_hicp_yoy",
        "ea_hicp_yoy_z",
        "us_cpi_yoy",
        "us_cpi_yoy_z2",
        "inflation_gap_us_ea",
        "inflation_gap_us_ea_z63",
        "inflation_gap_us_ea_z252",

        # Growth / liquidity
        "us_indpro_yoy",
        "us_retail_yoy",
        "us_indpro_yoy_z",
        "us_retail_yoy_z",
        "us_growth_proxy",
        "m1_real_yoy",
        "m1_real_yoy_z",
        "money_liquidity_proxy",

        # Energy
        "brent_ret_21d",
        "brent_ret_21d_z",
        "eu_gas_ret_21d",
        "eu_gas_ret_21d_z",
        "energy_pressure_eu",
        "energy_pressure_eu_z63",
    ]

    cols = [
        c for c in SAFE_NEW_MACRO_COLS
        if c in new_macro_features_df.columns and c not in LATE_SAMPLE_COLS
    ]

    master_df = join_safe_dataframe_to_master(
        master_df,
        new_macro_features_df[cols],
        drop_cols=LATE_SAMPLE_COLS
    )

# macro_scores_df varsa score kolonlarını garanti et
if "macro_scores_df" in globals():
    master_df = join_safe_dataframe_to_master(master_df, macro_scores_df, drop_cols=LATE_SAMPLE_COLS)


# ─────────────────────────────────────────────────────────────────
# 7) Calendar dataframe varsa bağla
# ─────────────────────────────────────────────────────────────────

if "calendar_df" in globals():
    master_df = join_safe_dataframe_to_master(master_df, calendar_df, drop_cols=[])


# ─────────────────────────────────────────────────────────────────
# 8) Final temizlik
# ─────────────────────────────────────────────────────────────────

master_df = master_df.loc[:, ~master_df.columns.duplicated()].copy()
master_df.index = safe_tz_naive_index(master_df.index)
master_df = master_df.sort_index()
master_df = master_df[~master_df.index.duplicated(keep="last")]
master_df = master_df.replace([np.inf, -np.inf], np.nan)

# Leakage güvenlik kontrolü
FORBIDDEN_MASTER_COLS = [
    "Chikou",
]

for col in FORBIDDEN_MASTER_COLS:
    if col in master_df.columns:
        raise ValueError(f"Leakage riski: {col} master_df içinde bulunuyor.")


print(f"master_df shape before target: {master_df.shape}")
print(f"log_ret non-NaN: {master_df['log_ret'].notna().sum()}")
print(f"NaN oranı before target: {master_df.isna().mean().mean():.3f}")


# ─────────────────────────────────────────────────────────────────
# 9) Target
# HORIZON=1: 1 bar ileri log return yönü
# Önce fwd_ret üret, sonra NaN drop, sonra target üret.
# ─────────────────────────────────────────────────────────────────

HORIZON = 1

master_df["fwd_ret"] = master_df["log_ret"].shift(-HORIZON)

master_df = master_df.dropna(subset=["log_ret", "fwd_ret"]).copy()
master_df["target"] = (master_df["fwd_ret"] > 0).astype(int)

# Güvenlik: target binary mi?
if not set(master_df["target"].dropna().unique()).issubset({0, 1}):
    raise ValueError("target binary değil. Target üretimini kontrol et.")


# ─────────────────────────────────────────────────────────────────
# 10) Train / Val / Test Split
# Time-series sıralı 60/20/20
# ─────────────────────────────────────────────────────────────────

n = len(master_df)

if n < 1000:
    raise ValueError(f"master_df çok küçük görünüyor: n={n}. NaN/drop veya tarih aralığını kontrol et.")

n_tr = int(n * 0.60)
n_va = int(n * 0.20)

train_df = master_df.iloc[:n_tr].copy()
val_df   = master_df.iloc[n_tr : n_tr + n_va].copy()
test_df  = master_df.iloc[n_tr + n_va :].copy()

# Split güvenlik kontrolleri
if len(train_df) == 0 or len(val_df) == 0 or len(test_df) == 0:
    raise ValueError("Train/Val/Test split boş parça üretti.")

if not (train_df.index.max() < val_df.index.min() < test_df.index.min()):
    raise ValueError("Split sıralaması bozuk. Time-series ordering kontrol edilmeli.")


# ─────────────────────────────────────────────────────────────────
# 11) Rapor
# ─────────────────────────────────────────────────────────────────

print(f"\nmaster_df : {master_df.shape}  [{master_df.index[0].date()} → {master_df.index[-1].date()}]")
print(f"train_df  : {train_df.shape}   [{train_df.index[0].date()} → {train_df.index[-1].date()}]")
print(f"val_df    : {val_df.shape}     [{val_df.index[0].date()} → {val_df.index[-1].date()}]")
print(f"test_df   : {test_df.shape}    [{test_df.index[0].date()} → {test_df.index[-1].date()}]")

print(
    f"Target balance — "
    f"tr:{train_df['target'].mean():.3f} "
    f"va:{val_df['target'].mean():.3f} "
    f"te:{test_df['target'].mean():.3f}"
)

print(f"\nNaN oranı after target: {master_df.isna().mean().mean():.3f}")

nan_report_master = (
    master_df
    .drop(columns=["fwd_ret", "target"], errors="ignore")
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

print("\nTop 30 NaN oranı yüksek master_df feature:")
print(nan_report_master.head(30).round(2).to_string())

print("\n✓ BLOK A — MASTER_DF kurulumu tamam")

master_df shape before target: (37614, 200)
log_ret non-NaN: 37613
NaN oranı before target: 0.016

master_df : (37612, 202)  [2004-01-01 → 2025-12-31]
train_df  : (22567, 202)   [2004-01-01 → 2016-08-31]
val_df    : (7522, 202)     [2016-08-31 → 2021-05-04]
test_df   : (7523, 202)    [2021-05-04 → 2025-12-31]
Target balance — tr:0.455 va:0.512 te:0.502

NaN oranı after target: 0.016

Top 30 NaN oranı yüksek master_df feature:
oil_ret_lag1                15.45
oil_ret                     15.45
hyg_lqd_spr                 14.12
mfg_z                        8.56
cpi_yoy_z                    8.40
m1_real_yoy_z                8.40
ea_hicp_yoy_z                8.40
real_m2_z                    8.40
money_liquidity_proxy        8.40
inflation_gap_us_ea_z252     8.40
us_indpro_yoy_z              8.40
us_retail_yoy_z              8.40
us_growth_proxy              8.40
us_cpi_yoy_z2                8.40
net_liq_z                    8.19
energy_pressure_eu_z63       5.72
inflation_gap_us_ea_z63   

In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK B — TREASURY THEORY FEATURES (4H UYARLAMASI)
# Tüm pencereler 4H bar cinsinden (1 gün = 6 bar)
# ════════════════════════════════════════════════════════════════════

def zscore_roll(s, w):
    return (s - s.rolling(w).mean()) / (s.rolling(w).std() + 1e-10)

def roll_sum(s, w):
    return s.rolling(w).sum()

def roll_corr(a, b, w):
    return a.rolling(w).corr(b)

def choose_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def build_treasury_theory_features_4h(master_df, K=3):
    df = master_df.copy()

    W5   = 5  * 6
    W10  = 10 * 6
    W21  = 21 * 6
    W63  = 63 * 6
    W126 = 126 * 6

In [ ]:
carry_col = choose_col(df, [
    'policy_diff_fed_ecb_dfr',
    'policy_diff_z63',
    'policy_score',
    'fed_ecb_rate_diff',
    'carry_signal',
    'spread_2y10y'
])

curve_col = choose_col(df, [
    'us_curve_10y2y',
    'us_curve_z63',
    'spread_2y10y'
])

rates_col = choose_col(df, [
    'rates_score',
    'us_de_10y_spread_z63',
    'us_de_10y_spread',
    'us_ea_10y_spread_z63',
    'us10y_real_z63',
    'us10y_real',
])

real_yield_col = choose_col(df, [
    'us10y_real_z63',
    'us10y_real',
    'real_nominal_gap_z63',
    'real_nominal_gap'
])

sovereign_col = choose_col(df, [
    'sovereign_stress_score',
    'it_de_10y_spread_z63',
    'it_de_10y_spread',
    'it_ea_10y_spread_z63',
    'it_ea_10y_spread'
])

infl_col = choose_col(df, [
    'inflation_score',
    'inflation_gap_us_ea_z63',
    'inflation_gap_us_ea',
    'us10y_be_z63',
    'us10y_be',
    'cpi_yoy_z'
])

energy_col = choose_col(df, [
    'energy_score',
    'energy_pressure_eu_z63',
    'energy_pressure_eu',
    'eu_gas_ret_21d_z',
    'brent_ret_21d_z',
    'oil_ret'
])

In [ ]:
vix_col     = choose_col(df, ['vix_z', 'vix_level'])
risk_col    = choose_col(df, ['risk_score', 'macro_pressure_score'])
growth_col  = choose_col(df, ['growth_score', 'us_growth_proxy'])
liq_col     = choose_col(df, ['liq_score', 'money_liquidity_proxy', 'net_liq_z'])
hyg_col     = choose_col(df, ['hyg_lqd_spr'])

# DXY level ve DXY trend dikkatli: DXY-free testlerde bunlar drop edilecek.
dxy_col     = choose_col(df, ['dxy', 'dxy_trend_z'])
dxy_ret_col = choose_col(df, ['dxy_ret_lag1', 'dxy_ret'])

usd_col     = choose_col(df, ['usd_factor_z', 'usd_factor'])
eur_col     = choose_col(df, ['eur_dom'])
chf_h_col   = choose_col(df, ['chf_haven'])

vol_col     = choose_col(df, ['hist_vol_20', 'vol_regime'])
ewma_col    = choose_col(df, ['ewma_vol'])
spx_col     = choose_col(df, ['sp500', 'spx_trend_z'])

rsi_col     = choose_col(df, ['RSI', 'rsi_norm'])
bb_col      = choose_col(df, ['Bollinger_Bandwidth', 'BB_PctB'])
macd_col    = choose_col(df, ['MACD_Hist', 'MACD_Signal'])
logret_col  = choose_col(df, ['log_ret', 'log_return'])

gbp_col     = choose_col(df, ['gbp_ret_lag1', 'gbp_ret'])
chf_col     = choose_col(df, ['chf_ret_lag1', 'chf_ret'])
jpy_col     = choose_col(df, ['jpy_ret_lag1', 'jpy_ret'])
gold_col    = choose_col(df, ['gold_ret_lag1', 'gold_ret'])
oil_col     = choose_col(df, ['oil_ret_lag1', 'oil_ret'])

In [ ]:
W5   = 5  * 6
W10  = 10 * 6
W21  = 21 * 6
W63  = 63 * 6
W126 = 126 * 6

K=3

In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK B — TREASURY THEORY FEATURES (4H UYARLAMASI)
# SAFE FULL FUNCTION VERSION
# ════════════════════════════════════════════════════════════════════

def zscore_roll(s, w):
    return (s - s.rolling(w).mean()) / (s.rolling(w).std() + 1e-10)


def roll_sum(s, w):
    return s.rolling(w).sum()


def roll_corr(a, b, w):
    return a.rolling(w).corr(b)


def choose_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def build_treasury_theory_features_4h(master_df, K=3):
    df = master_df.copy()

    df.index = pd.to_datetime(df.index)
    try:
        if getattr(df.index, "tz", None) is not None:
            df.index = df.index.tz_convert(None)
    except Exception:
        pass

    try:
        if getattr(df.index, "tz", None) is not None:
            df.index = df.index.tz_localize(None)
    except Exception:
        pass

    df = df.sort_index()
    df = df[~df.index.duplicated(keep="last")]
    df = df.replace([np.inf, -np.inf], np.nan)

    W5   = 5  * 6
    W10  = 10 * 6
    W21  = 21 * 6
    W63  = 63 * 6
    W126 = 126 * 6

    # ════════════════════════════════════════════════════════════════
    # Column selection
    # ════════════════════════════════════════════════════════════════

    carry_col = choose_col(df, [
        'policy_diff_fed_ecb_dfr',
        'policy_diff_z63',
        'policy_score',
        'fed_ecb_rate_diff',
        'carry_signal',
        'spread_2y10y'
    ])

    curve_col = choose_col(df, [
        'us_curve_10y2y',
        'us_curve_z63',
        'spread_2y10y'
    ])

    rates_col = choose_col(df, [
        'rates_score',
        'us_de_10y_spread_z63',
        'us_de_10y_spread',
        'us_ea_10y_spread_z63',
        'us10y_real_z63',
        'us10y_real',
    ])

    real_yield_col = choose_col(df, [
        'us10y_real_z63',
        'us10y_real',
        'real_nominal_gap_z63',
        'real_nominal_gap',
    ])

    sovereign_col = choose_col(df, [
        'sovereign_stress_score',
        'it_de_10y_spread_z63',
        'it_de_10y_spread',
        'it_ea_10y_spread_z63',
        'it_ea_10y_spread',
    ])

    infl_col = choose_col(df, [
        'inflation_score',
        'inflation_gap_us_ea_z63',
        'inflation_gap_us_ea',
        'us10y_be_z63',
        'us10y_be',
        'cpi_yoy_z',
    ])

    energy_col = choose_col(df, [
        'energy_score',
        'energy_pressure_eu_z63',
        'energy_pressure_eu',
        'eu_gas_ret_21d_z',
        'brent_ret_21d_z',
        'oil_ret',
    ])

    vix_col     = choose_col(df, ['vix_z', 'vix_level'])
    risk_col    = choose_col(df, ['risk_score', 'macro_pressure_score'])
    growth_col  = choose_col(df, ['growth_score', 'us_growth_proxy'])
    liq_col     = choose_col(df, ['liq_score', 'money_liquidity_proxy', 'net_liq_z'])
    hyg_col     = choose_col(df, ['hyg_lqd_spr'])

    dxy_col     = choose_col(df, ['dxy', 'dxy_trend_z'])
    dxy_ret_col = choose_col(df, ['dxy_ret_lag1', 'dxy_ret'])

    usd_col     = choose_col(df, ['usd_factor_z', 'usd_factor'])
    eur_col     = choose_col(df, ['eur_dom'])
    chf_h_col   = choose_col(df, ['chf_haven'])

    vol_col     = choose_col(df, ['hist_vol_20', 'vol_regime'])
    ewma_col    = choose_col(df, ['ewma_vol'])
    spx_col     = choose_col(df, ['sp500', 'spx_trend_z'])

    rsi_col     = choose_col(df, ['RSI', 'rsi_norm'])
    bb_col      = choose_col(df, ['Bollinger_Bandwidth', 'BB_PctB'])
    macd_col    = choose_col(df, ['MACD_Hist', 'MACD_Signal'])
    logret_col  = choose_col(df, ['log_ret', 'log_return'])

    gbp_col     = choose_col(df, ['gbp_ret_lag1', 'gbp_ret'])
    chf_col     = choose_col(df, ['chf_ret_lag1', 'chf_ret'])
    jpy_col     = choose_col(df, ['jpy_ret_lag1', 'jpy_ret'])
    gold_col    = choose_col(df, ['gold_ret_lag1', 'gold_ret'])
    oil_col     = choose_col(df, ['oil_ret_lag1', 'oil_ret'])

    # ════════════════════════════════════════════════════════════════
    # 1) UIP / Carry / Policy divergence
    # ════════════════════════════════════════════════════════════════

    if carry_col:
        df['uip_carry_z63']   = zscore_roll(df[carry_col], W63)
        df['uip_carry_z126']  = zscore_roll(df[carry_col], W126)
        df['uip_carry_mom5']  = df[carry_col].diff(W5)
        df['uip_carry_mom21'] = df[carry_col].diff(W21)

        if vol_col:
            df['uip_carry_vol_adj'] = (
                zscore_roll(df[carry_col], W63) /
                (zscore_roll(df[vol_col], W63).abs() + 1e-8)
            )

    # ════════════════════════════════════════════════════════════════
    # 2) Yield curve
    # ════════════════════════════════════════════════════════════════

    if curve_col:
        df['curve_z63']   = zscore_roll(df[curve_col], W63)
        df['curve_z126']  = zscore_roll(df[curve_col], W126)
        df['curve_mom5']  = df[curve_col].diff(W5)
        df['curve_mom21'] = df[curve_col].diff(W21)
        df['curve_accel'] = df[curve_col].diff(W5).diff(W5)

    # ════════════════════════════════════════════════════════════════
    # 3) Credit / liquidity
    # ════════════════════════════════════════════════════════════════

    if hyg_col:
        df['credit_z63']   = zscore_roll(df[hyg_col], W63)
        df['credit_mom5']  = df[hyg_col].diff(W5)
        df['credit_mom21'] = df[hyg_col].diff(W21)

        if risk_col:
            df['credit_x_risk'] = (
                df['credit_z63'] *
                zscore_roll(df[risk_col], W63)
            )

    if liq_col:
        df['liq_z63']   = zscore_roll(df[liq_col], W63)
        df['liq_mom5']  = df[liq_col].diff(W5)
        df['liq_mom21'] = df[liq_col].diff(W21)

    # ════════════════════════════════════════════════════════════════
    # 4) Growth / Taylor style interactions
    # ════════════════════════════════════════════════════════════════

    if growth_col:
        df['growth_z63']   = zscore_roll(df[growth_col], W63)
        df['growth_mom5']  = df[growth_col].diff(W5)
        df['growth_mom21'] = df[growth_col].diff(W21)

    if carry_col and growth_col:
        df['policy_growth_gap']   = df[carry_col] - df[growth_col]
        df['policy_growth_gap_z'] = zscore_roll(df['policy_growth_gap'], W63)

    if carry_col and risk_col:
        df['policy_risk_mix'] = (
            zscore_roll(df[carry_col], W63) -
            zscore_roll(df[risk_col], W63)
        )

    if curve_col and growth_col:
        df['curve_growth_gap']   = df[curve_col] - df[growth_col]
        df['curve_growth_gap_z'] = zscore_roll(df['curve_growth_gap'], W63)

    # ════════════════════════════════════════════════════════════════
    # 4B) New macro theory blocks
    # ════════════════════════════════════════════════════════════════

    if rates_col:
        df['rates_z63']   = zscore_roll(df[rates_col], W63)
        df['rates_z126']  = zscore_roll(df[rates_col], W126)
        df['rates_mom5']  = df[rates_col].diff(W5)
        df['rates_mom21'] = df[rates_col].diff(W21)

        if risk_col:
            df['rates_x_risk'] = df['rates_z63'] * zscore_roll(df[risk_col], W63)

        if growth_col:
            df['rates_growth_gap']   = df[rates_col] - df[growth_col]
            df['rates_growth_gap_z'] = zscore_roll(df['rates_growth_gap'], W63)

    if real_yield_col:
        df['real_yield_z63']   = zscore_roll(df[real_yield_col], W63)
        df['real_yield_z126']  = zscore_roll(df[real_yield_col], W126)
        df['real_yield_mom5']  = df[real_yield_col].diff(W5)
        df['real_yield_mom21'] = df[real_yield_col].diff(W21)

        if vol_col:
            df['real_yield_vol_adj'] = (
                zscore_roll(df[real_yield_col], W63) /
                (zscore_roll(df[vol_col], W63).abs() + 1e-8)
            )

    if sovereign_col:
        df['sovereign_z63']   = zscore_roll(df[sovereign_col], W63)
        df['sovereign_z126']  = zscore_roll(df[sovereign_col], W126)
        df['sovereign_mom5']  = df[sovereign_col].diff(W5)
        df['sovereign_mom21'] = df[sovereign_col].diff(W21)

        if risk_col:
            df['sovereign_x_risk'] = df['sovereign_z63'] * zscore_roll(df[risk_col], W63)

        if liq_col:
            df['sovereign_liq_gap']   = df[sovereign_col] - df[liq_col]
            df['sovereign_liq_gap_z'] = zscore_roll(df['sovereign_liq_gap'], W63)

    if infl_col:
        df['inflation_z63']   = zscore_roll(df[infl_col], W63)
        df['inflation_z126']  = zscore_roll(df[infl_col], W126)
        df['inflation_mom5']  = df[infl_col].diff(W5)
        df['inflation_mom21'] = df[infl_col].diff(W21)

        if real_yield_col:
            df['real_inflation_mix'] = (
                zscore_roll(df[real_yield_col], W63) -
                zscore_roll(df[infl_col], W63)
            )

    if energy_col:
        df['energy_z63']   = zscore_roll(df[energy_col], W63)
        df['energy_z126']  = zscore_roll(df[energy_col], W126)
        df['energy_mom5']  = df[energy_col].diff(W5)
        df['energy_mom21'] = df[energy_col].diff(W21)

        if risk_col:
            df['energy_x_risk'] = df['energy_z63'] * zscore_roll(df[risk_col], W63)

        if growth_col:
            df['energy_growth_gap']   = df[energy_col] - df[growth_col]
            df['energy_growth_gap_z'] = zscore_roll(df['energy_growth_gap'], W63)

    macro_pressure_col = choose_col(df, [
        'macro_pressure_score',
        'risk_score',
    ])

    if macro_pressure_col:
        df['macro_pressure_z63']   = zscore_roll(df[macro_pressure_col], W63)
        df['macro_pressure_z126']  = zscore_roll(df[macro_pressure_col], W126)
        df['macro_pressure_mom5']  = df[macro_pressure_col].diff(W5)
        df['macro_pressure_mom21'] = df[macro_pressure_col].diff(W21)

        if vol_col:
            df['macro_pressure_vol_adj'] = (
                zscore_roll(df[macro_pressure_col], W63) /
                (zscore_roll(df[vol_col], W63).abs() + 1e-8)
            )

    # ════════════════════════════════════════════════════════════════
    # 5) DXY / USD
    # ════════════════════════════════════════════════════════════════

    if dxy_col:
        df['dxy_z63'] = zscore_roll(df[dxy_col], W63)

        if dxy_col == 'dxy':
            df['dxy_mom3']  = df[dxy_col].pct_change(W5 // 2)
            df['dxy_mom10'] = df[dxy_col].pct_change(W10)
            df['dxy_mom21'] = df[dxy_col].pct_change(W21)
        else:
            df['dxy_mom3']  = df[dxy_col].diff(W5 // 2)
            df['dxy_mom10'] = df[dxy_col].diff(W10)
            df['dxy_mom21'] = df[dxy_col].diff(W21)

    if usd_col:
        df['usd_factor_z63']   = zscore_roll(df[usd_col], W63)
        df['usd_factor_mom5']  = df[usd_col].diff(W5)
        df['usd_factor_mom21'] = df[usd_col].diff(W21)

    # ════════════════════════════════════════════════════════════════
    # 6) Risk / safe-haven / cross-asset
    # ════════════════════════════════════════════════════════════════

    if gold_col:
        df['gold_mom5']  = roll_sum(df[gold_col], W5)
        df['gold_mom21'] = roll_sum(df[gold_col], W21)

    if oil_col:
        df['oil_mom5']  = roll_sum(df[oil_col], W5)
        df['oil_mom21'] = roll_sum(df[oil_col], W21)

    if spx_col:
        if spx_col == 'sp500':
            df['spx_ret_lag1'] = np.log(df[spx_col] / df[spx_col].shift(1)).shift(1)
            df['spx_mom5']     = roll_sum(df['spx_ret_lag1'], W5)
            df['spx_mom21']    = roll_sum(df['spx_ret_lag1'], W21)
            df['spx_z63']      = zscore_roll(df[spx_col], W63)
        else:
            df['spx_ret_lag1'] = df[spx_col].diff(1).shift(1)
            df['spx_mom5']     = df[spx_col].diff(W5)
            df['spx_mom21']    = df[spx_col].diff(W21)
            df['spx_z63']      = zscore_roll(df[spx_col], W63)

    if gold_col and 'spx_ret_lag1' in df.columns:
        df['riskoff_gold_spx']   = df[gold_col] - df['spx_ret_lag1']
        df['riskoff_gold_spx_5'] = roll_sum(df['riskoff_gold_spx'], W5)

    if chf_h_col:
        df['haven_z63'] = zscore_roll(df[chf_h_col], W63)

    if vix_col:
        df['vix_z63']   = zscore_roll(df[vix_col], W63)
        df['vix_mom5']  = df[vix_col].diff(W5)
        df['vix_mom21'] = df[vix_col].diff(W21)

    # ════════════════════════════════════════════════════════════════
    # 7) Flow / cross-currency correlations
    # ════════════════════════════════════════════════════════════════

    if eur_col and usd_col:
        df['eur_usd_dominance_gap']   = df[eur_col] - df[usd_col]
        df['eur_usd_dominance_gap_z'] = zscore_roll(df['eur_usd_dominance_gap'], W63)

    if gbp_col and chf_col:
        df['gbp_chf_spread5'] = roll_sum(df[gbp_col], W5) - roll_sum(df[chf_col], W5)

    if gbp_col and jpy_col:
        df['gbp_jpy_spread5'] = roll_sum(df[gbp_col], W5) - roll_sum(df[jpy_col], W5)

    if gbp_col:
        df['gbp_mom5'] = roll_sum(df[gbp_col], W5)

    if chf_col:
        df['chf_mom5'] = roll_sum(df[chf_col], W5)

    if jpy_col:
        df['jpy_mom5'] = roll_sum(df[jpy_col], W5)

    if 'corr_eur_gbp' in df.columns and 'corr_eur_chf' in df.columns:
        df['corr_stability_21'] = (
            df['corr_eur_gbp'].rolling(W21).std() +
            df['corr_eur_chf'].rolling(W21).std()
        )
        df['corr_gap_eurgbp_chf'] = df['corr_eur_gbp'] - df['corr_eur_chf']

    # ════════════════════════════════════════════════════════════════
    # 8) Vol tactical
    # ════════════════════════════════════════════════════════════════

    if vol_col:
        df['vol_z63']         = zscore_roll(df[vol_col], W63)
        df['vol_mom5']        = df[vol_col].diff(W5)
        df['vol_mom21']       = df[vol_col].diff(W21)
        df['vol_ratio_21_63'] = df[vol_col] / (df[vol_col].rolling(W63).mean() + 1e-8)

    if vol_col and ewma_col:
        df['vol_term_gap']   = df[vol_col] - df[ewma_col]
        df['vol_term_gap_z'] = zscore_roll(df['vol_term_gap'], W63)

    # ════════════════════════════════════════════════════════════════
    # 9) Technical features
    # ════════════════════════════════════════════════════════════════

    if logret_col:
        df['ret_mom5']  = roll_sum(df[logret_col], W5)
        df['ret_mom21'] = roll_sum(df[logret_col], W21)
        df['ret_mom63'] = roll_sum(df[logret_col], W63)

    if rsi_col:
        df['rsi_z63']  = zscore_roll(df[rsi_col], W63)
        df['rsi_mom5'] = df[rsi_col].diff(W5)

    if bb_col:
        df['bb_z63']   = zscore_roll(df[bb_col], W63)
        df['bb_mom10'] = df[bb_col].diff(W10)

    if macd_col:
        df['macd_z63']  = zscore_roll(df[macd_col], W63)
        df['macd_mom3'] = df[macd_col].diff(W5 // 2)

    if risk_col:
        df['risk_score_z63']  = zscore_roll(df[risk_col], W63)
        df['risk_score_mom5'] = df[risk_col].diff(W5)

    # ════════════════════════════════════════════════════════════════
    # 10) Regime interaction
    # ════════════════════════════════════════════════════════════════

    if logret_col:
        for i in range(K):
            pcol = f'p_state_{i}'

            if pcol in df.columns:
                if 'ret_mom21' in df.columns:
                    df[f'ret_mom21_x_p{i}'] = df['ret_mom21'] * df[pcol]

                if carry_col:
                    df[f'carry_x_p{i}'] = df[carry_col] * df[pcol]

                if risk_col:
                    df[f'risk_x_p{i}'] = df[risk_col] * df[pcol]

                if rates_col:
                    df[f'rates_x_p{i}'] = df[rates_col] * df[pcol]

                if sovereign_col:
                    df[f'sovereign_x_p{i}'] = df[sovereign_col] * df[pcol]

                if energy_col:
                    df[f'energy_x_p{i}'] = df[energy_col] * df[pcol]

    # ════════════════════════════════════════════════════════════════
    # Aliases / compatibility
    # ════════════════════════════════════════════════════════════════

    if dxy_ret_col:
        df['dxy_ret_lag1'] = df[dxy_ret_col]

    if 'ADX_14' in df.columns:
        df['adx_14'] = df['ADX_14']

    if 'bund_treasury_spread' in df.columns:
        df['bund_tsy_z'] = zscore_roll(df['bund_treasury_spread'], W63)

    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.loc[:, ~df.columns.duplicated()].copy()

    return df

In [ ]:
    # 4B) New macro theory blocks: rates / real yield / sovereign / inflation / energy

    # Rates / transatlantic yield pressure
    if rates_col:
        df['rates_z63']   = zscore_roll(df[rates_col], W63)
        df['rates_z126']  = zscore_roll(df[rates_col], W126)
        df['rates_mom5']  = df[rates_col].diff(W5)
        df['rates_mom21'] = df[rates_col].diff(W21)

        if risk_col:
            df['rates_x_risk'] = df['rates_z63'] * zscore_roll(df[risk_col], W63)

        if growth_col:
            df['rates_growth_gap'] = df[rates_col] - df[growth_col]
            df['rates_growth_gap_z'] = zscore_roll(df['rates_growth_gap'], W63)

    # Real yield channel
    if real_yield_col:
        df['real_yield_z63']   = zscore_roll(df[real_yield_col], W63)
        df['real_yield_z126']  = zscore_roll(df[real_yield_col], W126)
        df['real_yield_mom5']  = df[real_yield_col].diff(W5)
        df['real_yield_mom21'] = df[real_yield_col].diff(W21)

        if vol_col:
            df['real_yield_vol_adj'] = df[real_yield_col] / (df[vol_col] + 1e-8)

    # Euro sovereign stress / safe asset rotation
    if sovereign_col:
        df['sovereign_z63']   = zscore_roll(df[sovereign_col], W63)
        df['sovereign_z126']  = zscore_roll(df[sovereign_col], W126)
        df['sovereign_mom5']  = df[sovereign_col].diff(W5)
        df['sovereign_mom21'] = df[sovereign_col].diff(W21)

        if risk_col:
            df['sovereign_x_risk'] = df['sovereign_z63'] * zscore_roll(df[risk_col], W63)

        if liq_col:
            df['sovereign_liq_gap'] = df[sovereign_col] - df[liq_col]
            df['sovereign_liq_gap_z'] = zscore_roll(df['sovereign_liq_gap'], W63)

    # Inflation / breakeven divergence
    if infl_col:
        df['inflation_z63']   = zscore_roll(df[infl_col], W63)
        df['inflation_z126']  = zscore_roll(df[infl_col], W126)
        df['inflation_mom5']  = df[infl_col].diff(W5)
        df['inflation_mom21'] = df[infl_col].diff(W21)

        if real_yield_col:
            df['real_inflation_mix'] = zscore_roll(df[real_yield_col], W63) - zscore_roll(df[infl_col], W63)

    # Energy pressure, especially EUR-negative European gas pressure
    if energy_col:
        df['energy_z63']   = zscore_roll(df[energy_col], W63)
        df['energy_z126']  = zscore_roll(df[energy_col], W126)
        df['energy_mom5']  = df[energy_col].diff(W5)
        df['energy_mom21'] = df[energy_col].diff(W21)

        if risk_col:
            df['energy_x_risk'] = df['energy_z63'] * zscore_roll(df[risk_col], W63)

        if growth_col:
            df['energy_growth_gap'] = df[energy_col] - df[growth_col]
            df['energy_growth_gap_z'] = zscore_roll(df['energy_growth_gap'], W63)

    # Combined macro pressure tactical feature
    macro_pressure_col = choose_col(df, ['macro_pressure_score'])

    if macro_pressure_col:
        df['macro_pressure_z63']   = zscore_roll(df[macro_pressure_col], W63)
        df['macro_pressure_z126']  = zscore_roll(df[macro_pressure_col], W126)
        df['macro_pressure_mom5']  = df[macro_pressure_col].diff(W5)
        df['macro_pressure_mom21'] = df[macro_pressure_col].diff(W21)

        if vol_col:
            df['macro_pressure_vol_adj'] = df[macro_pressure_col] / (df[vol_col] + 1e-8)

In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK B RUN — TREASURY THEORY FEATURES UYGULA
# ════════════════════════════════════════════════════════════════════

K = 3

before_cols = set(master_df.columns)

master_df = build_treasury_theory_features_4h(master_df, K=K)

after_cols = set(master_df.columns)
new_cols = sorted(list(after_cols - before_cols))

# Splitleri master_df'nin yeni kolonlarıyla güncelle
train_df = master_df.loc[train_df.index].copy()
val_df   = master_df.loc[val_df.index].copy()
test_df  = master_df.loc[test_df.index].copy()

print(f"master_df shape (treasury sonrası): {master_df.shape}")
print(f"Eklenen yeni kolon sayısı: {len(new_cols)}")

print("\nEklenen yeni kolonlardan ilk 50:")
print(new_cols[:50])

print("\nTreasury / macro theory örnek kolon kontrolü:")
check_cols = [
    "uip_carry_z63",
    "uip_carry_mom21",
    "curve_z63",
    "credit_z63",
    "liq_z63",
    "growth_z63",
    "policy_growth_gap_z",
    "rates_z63",
    "real_yield_z63",
    "sovereign_z63",
    "inflation_z63",
    "energy_z63",
    "macro_pressure_z63",
    "dxy_z63",
    "usd_factor_z63",
    "risk_score_z63",
]

existing_check_cols = [c for c in check_cols if c in master_df.columns]
missing_check_cols = [c for c in check_cols if c not in master_df.columns]

print("Var olanlar:", existing_check_cols)
print("Eksikler   :", missing_check_cols)

print("\nSplit shape kontrol:")
print("train_df:", train_df.shape)
print("val_df  :", val_df.shape)
print("test_df :", test_df.shape)

master_df shape (treasury sonrası): (37612, 311)
Eklenen yeni kolon sayısı: 109

Eklenen yeni kolonlardan ilk 50:
['adx_14', 'bb_mom10', 'bb_z63', 'chf_mom5', 'corr_gap_eurgbp_chf', 'corr_stability_21', 'credit_mom21', 'credit_mom5', 'credit_x_risk', 'credit_z63', 'curve_accel', 'curve_growth_gap', 'curve_growth_gap_z', 'curve_mom21', 'curve_mom5', 'curve_z126', 'curve_z63', 'dxy_mom10', 'dxy_mom21', 'dxy_mom3', 'dxy_z63', 'energy_growth_gap', 'energy_growth_gap_z', 'energy_mom21', 'energy_mom5', 'energy_x_risk', 'energy_z126', 'energy_z63', 'eur_usd_dominance_gap', 'eur_usd_dominance_gap_z', 'gbp_chf_spread5', 'gbp_jpy_spread5', 'gbp_mom5', 'gold_mom21', 'gold_mom5', 'growth_mom21', 'growth_mom5', 'growth_z63', 'haven_z63', 'inflation_mom21', 'inflation_mom5', 'inflation_z126', 'inflation_z63', 'jpy_mom5', 'liq_mom21', 'liq_mom5', 'liq_z63', 'macd_mom3', 'macd_z63', 'macro_pressure_mom21']

Treasury / macro theory örnek kolon kontrolü:
Var olanlar: ['uip_carry_z63', 'uip_carry_mom21',

In [ ]:
import os
import sys
import contextlib
import warnings

warnings.filterwarnings("ignore")

@contextlib.contextmanager
def suppress_stdout_stderr():
    """
    XGBoost / LightGBM / TensorFlow gibi modellerin
    gereksiz eğitim loglarını susturur.
    """
    with open(os.devnull, "w") as devnull:
        old_stdout = sys.stdout
        old_stderr = sys.stderr
        try:
            sys.stdout = devnull
            sys.stderr = devnull
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr

In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK A — MASTER_DF KURULUM son düzeltme emin olma


import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")


# ════════════════════════════════════════════════════════════════════
# A.0 — GLOBAL SAFETY HELPERS
# ════════════════════════════════════════════════════════════════════

def make_tz_naive_index(idx):
    """
    DatetimeIndex'i güvenli şekilde tz-naive yapar.
    tz-naive ise dokunmaz, tz-aware ise timezone bilgisini kaldırır.
    """
    idx = pd.to_datetime(idx)

    try:
        if getattr(idx, "tz", None) is not None:
            idx = idx.tz_convert(None)
    except Exception:
        pass

    try:
        if getattr(idx, "tz", None) is not None:
            idx = idx.tz_localize(None)
    except Exception:
        pass

    return idx


def clean_series_index(s):
    """
    Series/DataFrame index temizliği:
      - datetime
      - tz-naive
      - sort
      - duplicate index temizliği
    """
    s = s.copy()
    s.index = make_tz_naive_index(s.index)
    s = s.sort_index()
    s = s[~s.index.duplicated(keep="last")]
    return s


def safe_reindex(series, index, method=None):
    """
    Series'i master index'e güvenli reindex eder.
    """
    s = clean_series_index(series)
    index = make_tz_naive_index(index)

    if method is None:
        return s.reindex(index)
    else:
        return s.reindex(index, method=method)


def safe_reindex_lag(series, index, lag=1):
    """
    Series'i master index'e hizalar ve lag uygular.
    Bu özellikle cross-pair / market proxy feature'ları için kullanılır.
    """
    return safe_reindex(series, index).shift(lag)


def zscore_roll(s, w):
    return (s - s.rolling(w).mean()) / (s.rolling(w).std() + 1e-10)


def roll_sum(s, w):
    return s.rolling(w).sum()





def add_series_if_exists(master_df, col_name, series_obj, lag=None, ffill=False):
    """
    Bir series varsa master_df'e güvenli ekler.
    Yoksa sessizce skip eder.
    """
    try:
        if series_obj is None:
            print(f"[SKIP] {col_name}: series None")
            return master_df

        if ffill:
            s = safe_reindex(series_obj, master_df.index, method="ffill")
        else:
            s = safe_reindex(series_obj, master_df.index)

        if lag is not None and lag > 0:
            s = s.shift(lag)

        master_df[col_name] = s

    except Exception as e:
        print(f"[SKIP] {col_name}: {e}")

    return master_df


# ════════════════════════════════════════════════════════════════════
# A.1 — MASTER_DF BASE
# ════════════════════════════════════════════════════════════════════

master_df = feature_df.copy()
master_df.index = make_tz_naive_index(master_df.index)
master_df = master_df.sort_index()
master_df = master_df[~master_df.index.duplicated(keep="last")]



# log_ret override
lr_clean = clean_series_index(lr)
master_df["log_ret"] = lr_clean.reindex(master_df.index)

print("=" * 100)
print("BLOK A — MASTER_DF BASE READY")
print("=" * 100)
print(f"master_df initial shape: {master_df.shape}")
print(f"master_df date range   : {master_df.index.min()} → {master_df.index.max()}")


# ════════════════════════════════════════════════════════════════════
# A.2 — TEKNİK İNDİKATÖRLERİ EKLE
# ════════════════════════════════════════════════════════════════════

TECH_FEATURES = [
    "Stoch_K", "Stoch_D", "RSI", "BB_PctB", "Bollinger_Bandwidth",
    "ADX_14", "Momentum_10", "MACD_Hist", "MACD_Signal",
    "log_return", "log_return_lag1", "log_return_lag2", "log_return_lag3",
    "Historical_Volatility", "Exponential_Weighted_Log_Return",
    "Rel_Close_Open", "Rel_High_Low", "HMA",
    "Month", "DayOfWeek", "Hour",
]

if "df" in globals() and isinstance(df, pd.DataFrame):
    tech_cols = [c for c in TECH_FEATURES if c in df.columns]

    if len(tech_cols) > 0:
        tech_join = df[tech_cols].copy()
        tech_join.index = make_tz_naive_index(tech_join.index)
        tech_join = tech_join.sort_index()
        tech_join = tech_join[~tech_join.index.duplicated(keep="last")]

        master_df = master_df.join(tech_join, how="left", rsuffix="_tech")

        print(f"[OK] Technical features joined: {len(tech_cols)}")
    else:
        print("[INFO] df var ama TECH_FEATURES içinde eşleşen kolon yok.")
else:
    print("[INFO] df bulunamadı; teknik indikatör join atlandı.")


# ════════════════════════════════════════════════════════════════════
# A.3 — PAIR / MARKET LAG FEATURES
# DXY YOK
# ════════════════════════════════════════════════════════════════════

PAIR_SERIES = [
    ("gbp_ret",  "gbp_ret_lag1"),
    ("chf_ret",  "chf_ret_lag1"),
    ("jpy_ret",  "jpy_ret_lag1"),
    ("gold_ret", "gold_ret_lag1"),
    ("oil_ret",  "oil_ret_lag1"),
]

for src_name, out_name in PAIR_SERIES:
    if src_name in globals():
        master_df = add_series_if_exists(
            master_df=master_df,
            col_name=out_name,
            series_obj=globals()[src_name],
            lag=1,
            ffill=False
        )
    else:
        print(f"[SKIP] {out_name}: {src_name} globalde yok")


# SPX level — DXY yok
if "spx_4h" in globals():
    master_df = add_series_if_exists(
        master_df=master_df,
        col_name="sp500",
        series_obj=spx_4h,
        lag=None,
        ffill=True
    )
else:
    print("[SKIP] sp500: spx_4h globalde yok")


# ════════════════════════════════════════════════════════════════════
# A.4 — CORE FEATURE SERİLERİNİ GÜVENLİ YENİDEN YAZ
# DXY ile ilgili hiçbir kolon eklenmez.
# ════════════════════════════════════════════════════════════════════

REBUILD_SPECS = [
    # Vol
    ("hist_vol_20", "hist_vol_20"),
    ("ewma_vol", "ewma_vol"),
    ("atr_ratio", "atr_ratio"),
    ("park_vol", "parkinson_vol"),
    ("gk_vol", "garman_klass_vol"),
    ("rs_vol", "rogers_satchell_vol"),

    # Momentum
    ("ma_spread", "ma_spread"),
    ("rsi_norm", "rsi_norm"),
    ("z_score_ret", "z_score_ret"),
    ("mom_5", "mom_5"),
    ("mom_21", "mom_21"),

    # Cross pair returns
    ("gbp_ret", "gbp_ret"),
    ("chf_ret", "chf_ret"),
    ("jpy_ret", "jpy_ret"),
    ("ech_ret", "ech_ret"),
    ("egb_ret", "egb_ret"),
    ("ejp_ret", "ejp_ret"),

    # Market macro — DXY yok
    ("vix_level", "vix_level"),
    ("gold_ret", "gold_ret"),
    ("oil_ret", "oil_ret"),
    ("hyg_lqd_spr", "hyg_lqd_spr"),
    ("spread_2y10y", "spread_2y10y"),
    ("us10y_chg", "us10y_chg"),

    # USD factor — DXY değil, cross-pair derived
    ("usd_factor", "usd_factor"),
    ("usd_factor_z", "usd_factor_z"),

    # EUR / haven / corr / MI
    ("eur_dom", "eur_dom"),
    ("chf_haven", "chf_haven"),
    ("corr_eur_gbp", "corr_eur_gbp"),
    ("corr_eur_chf", "corr_eur_chf"),
    ("corr_eur_jpy", "corr_eur_jpy"),
    ("mi_eur_gbp", "mi_eur_gbp"),
    ("mi_eur_chf", "mi_eur_chf"),
    ("mi_eur_jpy", "mi_eur_jpy"),

    # Composite macro
    ("growth_score", "growth_score"),
    ("liq_score", "liq_score"),
    ("risk_score", "risk_score"),

    # FRED macro
    ("cfnai_a", "cfnai_a"),
    ("cpi_yoy_z", "cpi_yoy_z"),
    ("real_m2_z", "real_m2_z"),
    ("net_liq_z", "net_liq_z"),
    ("vix_z", "vix_z"),
    ("hy_z", "hy_z"),
    ("spx_trend_z", "spx_trend_z"),
    ("unemp_z", "unemp_z"),
    ("mfg_z", "mfg_z"),

    # Volume
    ("eur_vol_ratio", "eur_vol_ratio"),
    ("eur_vol_imbal", "eur_vol_imbal"),
    ("obv_mom", "obv_mom"),
    ("chf_vol_ratio", "chf_vol_ratio"),
    ("jpy_vol_ratio", "jpy_vol_ratio"),
    ("eurchf_vol_ratio", "eurchf_vol_ratio"),
    ("risk_off_vol", "risk_off_vol"),
    ("eur_activity", "eur_activity"),
    ("vol_regime", "vol_regime"),

    # Gold volume
    ("gold_vol_ratio", "gold_vol_ratio"),
    ("gold_vol_z", "gold_vol_z"),
    ("gold_activity", "gold_activity"),
    ("gold_pressure", "gold_pressure"),
]

for col_name, global_name in REBUILD_SPECS:
    if global_name in globals():
        master_df = add_series_if_exists(
            master_df=master_df,
            col_name=col_name,
            series_obj=globals()[global_name],
            lag=None,
            ffill=False
        )



# ════════════════════════════════════════════════════════════════════
# A.5 — DXY YERİNE SAFE USD_EXEUR_PROXY
# ════════════════════════════════════════════════════════════════════

def build_safe_usd_exeur_proxy_4h(df, bar_per_day=6):
    """
    DXY kullanmadan USD pressure proxy üretir.

    Mantık:
      GBPUSD yükselirse USD zayıflar  -> -gbp_ret_lag1
      USDCHF yükselirse USD güçlenir  -> +chf_ret_lag1
      USDJPY yükselirse USD güçlenir  -> +jpy_ret_lag1
      Gold yükselirse USD/risk baskısı olabilir -> -gold_ret_lag1
      Oil yükselirse USD üzerinde baskı olabilir -> -oil_ret_lag1

    Not:
      chf_ret ve jpy_ret burada USDCHF/USDJPY log return olarak kabul edilmiştir.
    """
    out = df.copy()

    W3   = 3   * bar_per_day
    W5   = 5   * bar_per_day
    W10  = 10  * bar_per_day
    W21  = 21  * bar_per_day
    W63  = 63  * bar_per_day
    W126 = 126 * bar_per_day

    components = {}

    if "gbp_ret_lag1" in out.columns:
        components["usd_from_gbp"] = -out["gbp_ret_lag1"]

    if "chf_ret_lag1" in out.columns:
        components["usd_from_chf"] = out["chf_ret_lag1"]

    if "jpy_ret_lag1" in out.columns:
        components["usd_from_jpy"] = out["jpy_ret_lag1"]

    if "gold_ret_lag1" in out.columns:
        components["usd_from_gold"] = -out["gold_ret_lag1"]

    if "oil_ret_lag1" in out.columns:
        components["usd_from_oil"] = -out["oil_ret_lag1"]

    if len(components) == 0:
        raise ValueError("USD_EXEUR_PROXY için hiçbir component bulunamadı.")

    comp_df = pd.DataFrame(components, index=out.index)

    # Her component'i z-score yapıp ortalıyoruz
    comp_z = comp_df.apply(lambda x: zscore_roll(x, W63))



    # Eski usd_factor varsa onu da safe lagli hale getir
    if "usd_factor" in out.columns:
        out["usd_factor_safe_lag1"] = out["usd_factor"].shift(1)
        out["usd_factor_safe_z63"] = zscore_roll(out["usd_factor_safe_lag1"], W63)
        out["usd_factor_safe_mom5"] = out["usd_factor_safe_lag1"].diff(W5)
        out["usd_factor_safe_mom21"] = out["usd_factor_safe_lag1"].diff(W21)

    return out


master_df = build_safe_usd_exeur_proxy_4h(master_df, bar_per_day=BAR_PER_DAY)






# ════════════════════════════════════════════════════════════════════
# A.6 — TARGET
# ════════════════════════════════════════════════════════════════════

HORIZON = 1

master_df["fwd_ret"] = master_df["log_ret"].shift(-HORIZON)
master_df["target"] = np.where(master_df["fwd_ret"] > 0, 1, 0)

# fwd_ret olmayan son satırları düşür
master_df = master_df.dropna(subset=["log_ret", "fwd_ret", "target"]).copy()

# target integer olsun
master_df["target"] = master_df["target"].astype(int)


# ════════════════════════════════════════════════════════════════════
# A.7 — FINAL CLEANUP / AUDIT
# ════════════════════════════════════════════════════════════════════

# Inf temizliği
master_df = master_df.replace([np.inf, -np.inf], np.nan)

# Duplicate kolon temizliği
master_df = master_df.loc[:, ~master_df.columns.duplicated()].copy()



print("\n" + "=" * 100)
print("MASTER_DF FINAL CHECK")
print("=" * 100)
print(f"master_df shape     : {master_df.shape}")
print(f"Date range          : {master_df.index[0]} → {master_df.index[-1]}")
print(f"log_ret non-NaN     : {master_df['log_ret'].notna().sum()}")
print(f"fwd_ret non-NaN     : {master_df['fwd_ret'].notna().sum()}")
print(f"target mean         : {master_df['target'].mean():.4f}")
print(f"Average NaN ratio   : {master_df.isna().mean().mean():.4f}")


# ════════════════════════════════════════════════════════════════════
# A.8 — TRAIN / VAL / TEST SPLIT
# ════════════════════════════════════════════════════════════════════

n = len(master_df)
n_tr = int(n * 0.60)
n_va = int(n * 0.20)

train_df = master_df.iloc[:n_tr].copy()
val_df   = master_df.iloc[n_tr:n_tr + n_va].copy()
test_df  = master_df.iloc[n_tr + n_va:].copy()

print("\n" + "=" * 100)
print("TRAIN / VAL / TEST SPLIT")
print("=" * 100)

print(f"master_df : {master_df.shape}  [{master_df.index[0].date()} → {master_df.index[-1].date()}]")
print(f"train_df  : {train_df.shape}   [{train_df.index[0].date()} → {train_df.index[-1].date()}]")
print(f"val_df    : {val_df.shape}     [{val_df.index[0].date()} → {val_df.index[-1].date()}]")
print(f"test_df   : {test_df.shape}    [{test_df.index[0].date()} → {test_df.index[-1].date()}]")

print(
    f"\nTarget balance — "
    f"tr:{train_df['target'].mean():.3f} "
    f"va:{val_df['target'].mean():.3f} "
    f"te:{test_df['target'].mean():.3f}"
)


# ════════════════════════════════════════════════════════════════════
# A.9 — OPTIONAL FEATURE HEALTH TABLE
# ════════════════════════════════════════════════════════════════════

health = pd.DataFrame({
    "n_total": master_df.shape[0],
    "n_valid": master_df.notna().sum(),
    "n_nan": master_df.isna().sum(),
    "nan_pct": master_df.isna().mean() * 100,
}).sort_values("nan_pct", ascending=False)

print("\n" + "=" * 100)
print("TOP 30 HIGHEST NaN FEATURES")
print("=" * 100)
print(health.head(30))

print("\n✓ BLOK A tamamlandı.")
print("✓ master_df / train_df / val_df / test_df hazır.")
print("✓ DXY tamamen çıkarıldı, yerine usd_exeur_proxy ailesi kullanılıyor.")

BLOK A — MASTER_DF BASE READY
master_df initial shape: (37614, 135)
master_df date range   : 2004-01-01 00:00:00 → 2025-12-31 20:00:00
[OK] Technical features joined: 19

MASTER_DF FINAL CHECK
master_df shape     : (37612, 167)
Date range          : 2004-01-01 04:00:00 → 2025-12-31 16:00:00
log_ret non-NaN     : 37612
fwd_ret non-NaN     : 37612
target mean         : 0.4754
Average NaN ratio   : 0.0251

TRAIN / VAL / TEST SPLIT
master_df : (37612, 167)  [2004-01-01 → 2025-12-31]
train_df  : (22567, 167)   [2004-01-01 → 2016-08-31]
val_df    : (7522, 167)     [2016-08-31 → 2021-05-04]
test_df   : (7523, 167)    [2021-05-04 → 2025-12-31]

Target balance — tr:0.455 va:0.512 te:0.502

TOP 30 HIGHEST NaN FEATURES
                          n_total  n_valid  n_nan    nan_pct
hy_z                        37612     2633  34979  92.999575
oil_ret_lag1                37612    31801   5811  15.449856
oil_ret                     37612    31802   5810  15.447198
hyg_lqd_spr                 37612    3

In [ ]:
# ════════════════════════════════════════════════════════════════════
# BLOK B — TREASURY THEORY FEATURES


import numpy as np
import pandas as pd


# ════════════════════════════════════════════════════════════════════
# B.0 — HELPERS
# ════════════════════════════════════════════════════════════════════

def zscore_roll(s, w):
    return (s - s.rolling(w).mean()) / (s.rolling(w).std() + 1e-10)


def roll_sum(s, w):
    return s.rolling(w).sum()


def roll_corr(a, b, w):
    return a.rolling(w).corr(b)


def choose_col(df, candidates):
    """
    Aday kolonlardan master_df içinde ilk bulunanı döndürür.
    Hiçbiri yoksa None.
    """
    for c in candidates:
        if c in df.columns:
            return c
    return None


def safe_div(a, b, eps=1e-10):
    return a / (b.abs() + eps)





# ════════════════════════════════════════════════════════════════════
# B.1 — MAIN BUILDER
# ════════════════════════════════════════════════════════════════════

def build_treasury_theory_features_4h_no_dxy(master_df, K=3, bar_per_day=None):
    df = master_df.copy()

    if bar_per_day is None:
        bar_per_day = globals().get("BAR_PER_DAY", 6)

    W3   = 3   * bar_per_day
    W5   = 5   * bar_per_day
    W10  = 10  * bar_per_day
    W21  = 21  * bar_per_day
    W63  = 63  * bar_per_day
    W126 = 126 * bar_per_day
    W252 = 252 * bar_per_day


    # ═══════════════════════════════════════════════════════════════
    # B.1.1 — CORE COLUMN SELECTION
    # ═══════════════════════════════════════════════════════════════

    carry_col = choose_col(df, [
        "policy_diff_fed_ecb_dfr",
        "fed_ecb_rate_diff",
        "carry_signal",
        "spread_2y10y",
    ])

    curve_col = choose_col(df, [
        "us_curve_10y2y",
        "spread_2y10y",
    ])

    vix_col    = choose_col(df, ["vix_level"])
    risk_col   = choose_col(df, ["risk_score"])
    growth_col = choose_col(df, ["growth_score"])
    liq_col    = choose_col(df, ["liq_score"])
    hyg_col    = choose_col(df, ["hyg_lqd_spr", "hy_z"])

    # DXY YOK — SAFE USD PROXY
    usd_proxy_col = choose_col(df, [
        "usd_exeur_proxy_lag1",
        "usd_exeur_proxy",
    ])

    usd_proxy_ret_col = choose_col(df, [
        "usd_exeur_ret_lag1",
    ])

    usd_col = choose_col(df, [
        "usd_factor_safe_lag1",
        "usd_factor",
    ])

    eur_col   = choose_col(df, ["eur_dom"])
    chf_h_col = choose_col(df, ["chf_haven"])

    vol_col  = choose_col(df, ["hist_vol_20"])
    ewma_col = choose_col(df, ["ewma_vol"])

    spx_col  = choose_col(df, ["sp500"])
    rsi_col  = choose_col(df, ["RSI", "rsi_norm"])
    bb_col   = choose_col(df, ["Bollinger_Bandwidth", "BB_PctB"])
    macd_col = choose_col(df, ["MACD_Hist"])

    logret_col = choose_col(df, ["log_ret", "log_return"])

    gbp_col  = choose_col(df, ["gbp_ret_lag1", "gbp_ret"])
    chf_col  = choose_col(df, ["chf_ret_lag1", "chf_ret"])
    jpy_col  = choose_col(df, ["jpy_ret_lag1", "jpy_ret"])
    gold_col = choose_col(df, ["gold_ret_lag1", "gold_ret"])
    oil_col  = choose_col(df, ["oil_ret_lag1", "oil_ret"])

    # Extra macro columns
    policy_diff_col = choose_col(df, ["policy_diff_fed_ecb_dfr"])
    us10y_real_col  = choose_col(df, ["us10y_real"])
    us10y_be_col    = choose_col(df, ["us10y_be"])
    us_de_col       = choose_col(df, ["us_de_10y_spread"])
    us_ea_col       = choose_col(df, ["us_ea_10y_spread"])
    it_de_col       = choose_col(df, ["it_de_10y_spread"])
    it_ea_col       = choose_col(df, ["it_ea_10y_spread"])

    energy_col      = choose_col(df, ["energy_pressure_eu"])
    inflation_gap_col = choose_col(df, ["inflation_gap_us_ea"])
    brent_col       = choose_col(df, ["brent_ret_21d"])
    gas_col         = choose_col(df, ["eu_gas_ret_21d"])

    # ═══════════════════════════════════════════════════════════════
    # B.2 — UIP / CARRY / POLICY DIFFERENTIAL
    # ═══════════════════════════════════════════════════════════════

    if carry_col:
        df["uip_carry_z63"] = zscore_roll(df[carry_col], W63)
        df["uip_carry_z126"] = zscore_roll(df[carry_col], W126)
        df["uip_carry_mom5"] = df[carry_col].diff(W5)
        df["uip_carry_mom21"] = df[carry_col].diff(W21)

        if vol_col:
            df["uip_carry_vol_adj"] = safe_div(df[carry_col], df[vol_col])

        # Generic aliases for theory sets
        df["carry_norm"] = zscore_roll(df[carry_col], W63)
        df["ois_spread"] = df[carry_col]
        df["ois_spread_mom21"] = df[carry_col].diff(W21)

    if policy_diff_col:
        df["policy_diff_z63"] = zscore_roll(df[policy_diff_col], W63)
        df["policy_diff_z126"] = zscore_roll(df[policy_diff_col], W126)
        df["policy_diff_chg5"] = df[policy_diff_col].diff(W5)
        df["policy_diff_chg21"] = df[policy_diff_col].diff(W21)

    # ═══════════════════════════════════════════════════════════════
    # B.3 — YIELD CURVE / RATES
    # ═══════════════════════════════════════════════════════════════

    if curve_col:
        df["curve_z63"] = zscore_roll(df[curve_col], W63)
        df["curve_z126"] = zscore_roll(df[curve_col], W126)
        df["curve_mom5"] = df[curve_col].diff(W5)
        df["curve_mom21"] = df[curve_col].diff(W21)
        df["curve_accel"] = df[curve_col].diff(W5).diff(W5)

        # aliases
        df["us_curve_10y2y"] = df[curve_col]
        df["us_curve_z63"] = df["curve_z63"]
        df["us_curve_z252"] = zscore_roll(df[curve_col], W252)
        df["us_curve_mom21"] = df["curve_mom21"]

    if us10y_real_col:
        df["us10y_real_z63"] = zscore_roll(df[us10y_real_col], W63)
        df["us10y_real_chg21"] = df[us10y_real_col].diff(W21)
        df["real_yield_z63"] = df["us10y_real_z63"]
        df["real_yield_mom21"] = df[us10y_real_col].diff(W21)

        if vol_col:
            df["real_yield_vol_adj"] = safe_div(df[us10y_real_col], df[vol_col])

    if us10y_be_col:
        df["us10y_be_z63"] = zscore_roll(df[us10y_be_col], W63)

    if "real_nominal_gap" in df.columns:
        df["real_nominal_gap_z63"] = zscore_roll(df["real_nominal_gap"], W63)

    if us_de_col:
        df["us_de_10y_spread_z63"] = zscore_roll(df[us_de_col], W63)
        df["us_de_10y_spread_chg21"] = df[us_de_col].diff(W21)

    if us_ea_col:
        df["us_ea_10y_spread_z63"] = zscore_roll(df[us_ea_col], W63)

    # Rates score
    rates_parts = []
    for c in ["curve_z63", "policy_diff_z63", "real_yield_z63", "us_de_10y_spread_z63"]:
        if c in df.columns:
            rates_parts.append(df[c])

    if rates_parts:
        df["rates_score"] = pd.concat(rates_parts, axis=1).mean(axis=1)
        df["rates_z63"] = zscore_roll(df["rates_score"], W63)
        df["rates_mom21"] = df["rates_score"].diff(W21)

    # ═══════════════════════════════════════════════════════════════
    # B.4 — CREDIT / LIQUIDITY / GROWTH / MACRO PRESSURE
    # ═══════════════════════════════════════════════════════════════

    if hyg_col:
        df["credit_z63"] = zscore_roll(df[hyg_col], W63)
        df["credit_mom5"] = df[hyg_col].diff(W5)
        df["credit_mom21"] = df[hyg_col].diff(W21)

        if risk_col:
            df["credit_x_risk"] = df["credit_z63"] * zscore_roll(df[risk_col], W63)

    if liq_col:
        df["liq_z63"] = zscore_roll(df[liq_col], W63)
        df["liq_mom5"] = df[liq_col].diff(W5)
        df["liq_mom21"] = df[liq_col].diff(W21)

    if growth_col:
        df["growth_z63"] = zscore_roll(df[growth_col], W63)
        df["growth_mom5"] = df[growth_col].diff(W5)
        df["growth_mom21"] = df[growth_col].diff(W21)

    if carry_col and growth_col:
        df["policy_growth_gap"] = df[carry_col] - df[growth_col]
        df["policy_growth_gap_z"] = zscore_roll(df["policy_growth_gap"], W63)

    if carry_col and risk_col:
        df["policy_risk_mix"] = zscore_roll(df[carry_col], W63) - zscore_roll(df[risk_col], W63)

    if curve_col and growth_col:
        df["curve_growth_gap"] = df[curve_col] - df[growth_col]
        df["curve_growth_gap_z"] = zscore_roll(df["curve_growth_gap"], W63)

    if "m1_real_yoy_z" in df.columns and "real_m2_z" in df.columns:
        df["money_liquidity_proxy"] = pd.concat(
            [df["m1_real_yoy_z"], df["real_m2_z"], df.get("net_liq_z", np.nan)],
            axis=1
        ).mean(axis=1)
    elif "real_m2_z" in df.columns:
        df["money_liquidity_proxy"] = df["real_m2_z"]

    # Macro pressure score
    macro_parts = []
    for c in [
        "risk_score_z63",
        "vix_z63",
        "credit_z63",
        "liq_z63",
        "rates_z63",
        "sovereign_z63",
        "energy_z63",
    ]:
        if c in df.columns:
            macro_parts.append(df[c])

    if macro_parts:
        df["macro_pressure_score"] = pd.concat(macro_parts, axis=1).mean(axis=1)
        df["macro_pressure_z63"] = zscore_roll(df["macro_pressure_score"], W63)
        df["macro_pressure_mom21"] = df["macro_pressure_score"].diff(W21)

    # ═══════════════════════════════════════════════════════════════
    # B.5 — USD SAFE PROXY
    # ═══════════════════════════════════════════════════════════════

    if usd_proxy_col:
        df["usd_exeur_z63"] = zscore_roll(df[usd_proxy_col], W63)
        df["usd_exeur_z126"] = zscore_roll(df[usd_proxy_col], W126)
        df["usd_exeur_mom3"] = df[usd_proxy_col].diff(W3)
        df["usd_exeur_mom5"] = df[usd_proxy_col].diff(W5)
        df["usd_exeur_mom10"] = df[usd_proxy_col].diff(W10)
        df["usd_exeur_mom21"] = df[usd_proxy_col].diff(W21)

    if usd_col:
        df["usd_factor_safe_z63"] = zscore_roll(df[usd_col], W63)
        df["usd_factor_safe_mom5"] = df[usd_col].diff(W5)
        df["usd_factor_safe_mom21"] = df[usd_col].diff(W21)

        # Eski theory setlerin usd_factor_z63 beklemesini kırmamak için
        # DXY değil, safe USD factor alias.
        df["usd_factor_z63"] = df["usd_factor_safe_z63"]
        df["usd_factor_mom5"] = df["usd_factor_safe_mom5"]
        df["usd_factor_mom21"] = df["usd_factor_safe_mom21"]

    # DXY bekleyen eski expert isimlerini safe proxy ile yaşatmak istersen:
    # Burada kolon adı dxy geçerse audit fail olur. O yüzden dxy alias üretmiyoruz.
    # dxy_z63 / dxy_mom21 gibi feature'lar keep_existing ile otomatik düşecek.

    # ═══════════════════════════════════════════════════════════════
    # B.6 — RISK / SAFE-HAVEN / CROSS-ASSET
    # ═══════════════════════════════════════════════════════════════

    if gold_col:
        df["gold_mom5"] = roll_sum(df[gold_col], W5)
        df["gold_mom21"] = roll_sum(df[gold_col], W21)

    if oil_col:
        df["oil_mom5"] = roll_sum(df[oil_col], W5)
        df["oil_mom21"] = roll_sum(df[oil_col], W21)

    if spx_col:
        df["spx_ret_lag1"] = np.log(df[spx_col] / df[spx_col].shift(1)).shift(1)
        df["spx_mom5"] = roll_sum(df["spx_ret_lag1"], W5)
        df["spx_mom21"] = roll_sum(df["spx_ret_lag1"], W21)
        df["spx_z63"] = zscore_roll(df[spx_col], W63)

    if gold_col and "spx_ret_lag1" in df.columns:
        df["riskoff_gold_spx"] = df[gold_col] - df["spx_ret_lag1"]
        df["riskoff_gold_spx_5"] = roll_sum(df["riskoff_gold_spx"], W5)

    if chf_h_col:
        df["haven_z63"] = zscore_roll(df[chf_h_col], W63)

    if vix_col:
        df["vix_z63"] = zscore_roll(df[vix_col], W63)
        df["vix_mom5"] = df[vix_col].diff(W5)
        df["vix_mom21"] = df[vix_col].diff(W21)

    if risk_col:
        df["risk_score_z63"] = zscore_roll(df[risk_col], W63)
        df["risk_score_mom5"] = df[risk_col].diff(W5)

    # Sovereign stress
    sovereign_parts = []
    for c in [it_de_col, it_ea_col]:
        if c:
            sovereign_parts.append(zscore_roll(df[c], W63))

    if sovereign_parts:
        df["sovereign_score"] = pd.concat(sovereign_parts, axis=1).mean(axis=1)
        df["sovereign_z63"] = zscore_roll(df["sovereign_score"], W63)

    # Energy pressure
    energy_parts = []
    for c in [energy_col, brent_col, gas_col]:
        if c:
            energy_parts.append(zscore_roll(df[c], W63))

    if energy_parts:
        df["energy_score"] = pd.concat(energy_parts, axis=1).mean(axis=1)
        df["energy_z63"] = zscore_roll(df["energy_score"], W63)
        df["energy_mom21"] = df["energy_score"].diff(W21)

    if "energy_z63" in df.columns and "risk_score_z63" in df.columns:
        df["energy_x_risk"] = df["energy_z63"] * df["risk_score_z63"]

    if "energy_z63" in df.columns and "growth_z63" in df.columns:
        df["energy_growth_gap_z"] = df["energy_z63"] - df["growth_z63"]

    if "energy_pressure_eu" in df.columns:
        df["energy_pressure_eu_z63"] = zscore_roll(df["energy_pressure_eu"], W63)

    if "brent_ret_21d" in df.columns:
        df["brent_ret_21d_z"] = zscore_roll(df["brent_ret_21d"], W63)

    if "eu_gas_ret_21d" in df.columns:
        df["eu_gas_ret_21d_z"] = zscore_roll(df["eu_gas_ret_21d"], W63)

    # Inflation
    inflation_parts = []
    for c in ["cpi_yoy_z", "inflation_gap_us_ea_z63", "us10y_be_z63"]:
        if c in df.columns:
            inflation_parts.append(df[c])

    if inflation_gap_col:
        df["inflation_gap_us_ea_z63"] = zscore_roll(df[inflation_gap_col], W63)

    if inflation_parts:
        df["inflation_score"] = pd.concat(inflation_parts, axis=1).mean(axis=1)
        df["inflation_z63"] = zscore_roll(df["inflation_score"], W63)
        df["inflation_mom21"] = df["inflation_score"].diff(W21)

    if "inflation_z63" in df.columns and "real_yield_z63" in df.columns:
        df["real_inflation_mix"] = df["real_yield_z63"] - df["inflation_z63"]

    # Interactions
    if "rates_z63" in df.columns and "risk_score_z63" in df.columns:
        df["rates_x_risk"] = df["rates_z63"] * df["risk_score_z63"]

    if "sovereign_z63" in df.columns and "risk_score_z63" in df.columns:
        df["sovereign_x_risk"] = df["sovereign_z63"] * df["risk_score_z63"]

    # ═══════════════════════════════════════════════════════════════
    # B.7 — FLOW / DOMINANCE
    # ═══════════════════════════════════════════════════════════════

    if eur_col and usd_proxy_col:
        df["eur_usd_exeur_gap"] = df[eur_col] - df[usd_proxy_col]
        df["eur_usd_exeur_gap_z"] = zscore_roll(df["eur_usd_exeur_gap"], W63)

        # Eski expertlerin beklediği isim.
        # DXY değil; safe USD proxy ile oluşturulan dominance gap.
        df["eur_usd_dominance_gap"] = df["eur_usd_exeur_gap"]
        df["eur_usd_dominance_gap_z"] = df["eur_usd_exeur_gap_z"]

    if gbp_col and chf_col:
        df["gbp_chf_spread5"] = roll_sum(df[gbp_col], W5) - roll_sum(df[chf_col], W5)

    if gbp_col and jpy_col:
        df["gbp_jpy_spread5"] = roll_sum(df[gbp_col], W5) - roll_sum(df[jpy_col], W5)

    if gbp_col:
        df["gbp_mom5"] = roll_sum(df[gbp_col], W5)

    if chf_col:
        df["chf_mom5"] = roll_sum(df[chf_col], W5)

    if jpy_col:
        df["jpy_mom5"] = roll_sum(df[jpy_col], W5)

    if "corr_eur_gbp" in df.columns and "corr_eur_chf" in df.columns:
        df["corr_stability_21"] = (
            df["corr_eur_gbp"].rolling(W21).std()
            + df["corr_eur_chf"].rolling(W21).std()
        )
        df["corr_gap_eurgbp_chf"] = df["corr_eur_gbp"] - df["corr_eur_chf"]

    # ═══════════════════════════════════════════════════════════════
    # B.8 — VOL TACTICAL
    # ═══════════════════════════════════════════════════════════════

    if vol_col:
        df["vol_z63"] = zscore_roll(df[vol_col], W63)
        df["vol_mom5"] = df[vol_col].diff(W5)
        df["vol_mom21"] = df[vol_col].diff(W21)
        df["vol_ratio_21_63"] = df[vol_col] / (df[vol_col].rolling(W63).mean() + 1e-10)

    if vol_col and ewma_col:
        df["vol_term_gap"] = df[vol_col] - df[ewma_col]
        df["vol_term_gap_z"] = zscore_roll(df["vol_term_gap"], W63)

    # ATR true alias
    if "atr_ratio" in df.columns:
        df["atr_ratio_true"] = df["atr_ratio"]

    # ═══════════════════════════════════════════════════════════════
    # B.9 — TECHNICAL / BEHAVIORAL
    # ═══════════════════════════════════════════════════════════════

    if logret_col:
        df["ret_mom5"] = roll_sum(df[logret_col], W5)
        df["ret_mom21"] = roll_sum(df[logret_col], W21)
        df["ret_mom63"] = roll_sum(df[logret_col], W63)

    if rsi_col:
        df["rsi_z63"] = zscore_roll(df[rsi_col], W63)
        df["rsi_mom5"] = df[rsi_col].diff(W5)

    if bb_col:
        df["bb_z63"] = zscore_roll(df[bb_col], W63)
        df["bb_mom10"] = df[bb_col].diff(W10)

    if macd_col:
        df["macd_z63"] = zscore_roll(df[macd_col], W63)
        df["macd_mom3"] = df[macd_col].diff(W3)

    if "ADX_14" in df.columns:
        df["adx_14"] = df["ADX_14"]

    # ═══════════════════════════════════════════════════════════════
    # B.10 — REGIME INTERACTIONS
    # ═══════════════════════════════════════════════════════════════

    for i in range(K):
        pcol = f"p_state_{i}"

        if pcol in df.columns:
            if "ret_mom21" in df.columns:
                df[f"ret_mom21_x_p{i}"] = df["ret_mom21"] * df[pcol]

            if carry_col:
                df[f"carry_x_p{i}"] = df[carry_col] * df[pcol]

            if risk_col:
                df[f"risk_x_p{i}"] = df[risk_col] * df[pcol]

            if "rates_score" in df.columns:
                df[f"rates_x_p{i}"] = df["rates_score"] * df[pcol]

    # ═══════════════════════════════════════════════════════════════
    # B.11 — FINAL CLEANUP
    # ═══════════════════════════════════════════════════════════════

    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.loc[:, ~df.columns.duplicated()].copy()



    return df


# ════════════════════════════════════════════════════════════════════
# B.2 — RUN BUILDER
# ════════════════════════════════════════════════════════════════════

K = 3

before_cols = set(master_df.columns)

master_df = build_treasury_theory_features_4h_no_dxy(
    master_df=master_df,
    K=K,
    bar_per_day=BAR_PER_DAY if "BAR_PER_DAY" in globals() else 6,
)

after_cols = set(master_df.columns)
new_cols = sorted(list(after_cols - before_cols))

# Splitleri master_df'e göre güncelle
train_df = master_df.loc[train_df.index].copy()
val_df   = master_df.loc[val_df.index].copy()
test_df  = master_df.loc[test_df.index].copy()

print("=" * 110)
print("BLOK B — TREASURY THEORY FEATURES COMPLETE")
print("=" * 110)
print(f"master_df shape treasury sonrası : {master_df.shape}")
print(f"Eklenen yeni kolon sayısı        : {len(new_cols)}")

print("\nİlk 80 yeni kolon:")
print(new_cols[:80])



print("\n✓ BLOK B tamam.")
print("✓ Treasury/theory features DXY-free şekilde üretildi.")
print("✓ train_df / val_df / test_df yeniden hizalandı.")

BLOK B — TREASURY THEORY FEATURES COMPLETE
master_df shape treasury sonrası : (37612, 257)
Eklenen yeni kolon sayısı        : 90

İlk 80 yeni kolon:
['adx_14', 'bb_mom10', 'bb_z63', 'carry_norm', 'chf_mom5', 'corr_gap_eurgbp_chf', 'corr_stability_21', 'credit_mom21', 'credit_mom5', 'credit_x_risk', 'credit_z63', 'curve_accel', 'curve_growth_gap', 'curve_growth_gap_z', 'curve_mom21', 'curve_mom5', 'curve_z126', 'curve_z63', 'energy_growth_gap_z', 'energy_mom21', 'energy_x_risk', 'energy_z63', 'gbp_chf_spread5', 'gbp_jpy_spread5', 'gbp_mom5', 'gold_mom21', 'gold_mom5', 'growth_mom21', 'growth_mom5', 'growth_z63', 'haven_z63', 'inflation_mom21', 'inflation_z63', 'jpy_mom5', 'liq_mom21', 'liq_mom5', 'liq_z63', 'macd_mom3', 'macd_z63', 'macro_pressure_mom21', 'macro_pressure_z63', 'oil_mom21', 'oil_mom5', 'ois_spread', 'ois_spread_mom21', 'policy_diff_z126', 'policy_growth_gap', 'policy_growth_gap_z', 'policy_risk_mix', 'rates_mom21', 'rates_x_risk', 'rates_z63', 'real_inflation_mix', 'real

In [ ]:
# ════════════════════════════════════════════════════════════════════
# INTEGRATED PATCH — DXY-FREE 30 NEW THEORY EXPERTS + FINAL MERGE
# Bu blok şunları yapar:
#   1) Real rate / breakeven / positioning / vol structure feature üretir
#   2) 30 yeni DXY-free expert set oluşturur: NEW_EXPERTS_30
#   3) Q20 + TH50 + N30 + eski E/T/G/V setlerini final merge eder
#   4) Model training öncesi DXY hard audit yapar
# ════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

try:
    import pandas_datareader.data as web
    HAS_PDR = True
except Exception:
    HAS_PDR = False
    print("[INFO] pandas_datareader yok. FRED serileri çekilmeyecek, fallback kullanılacak.")


# ════════════════════════════════════════════════════════════════════
# 0) COMMON HELPERS
# ════════════════════════════════════════════════════════════════════

def make_index_tz_naive(idx):
    idx = pd.to_datetime(idx)
    try:
        if idx.tz is None:
            return idx.tz_localize(None)
        return idx.tz_convert(None)
    except Exception:
        return pd.to_datetime(idx).tz_localize(None)


def zscore_roll(s, w):
    return (s - s.rolling(w).mean()) / (s.rolling(w).std() + 1e-10)


def roll_sum(s, w):
    return s.rolling(w).sum()


def keep_existing(df, feat_list):
    return [f for f in feat_list if f in df.columns]







def align_macro_series_safe(series, master_df, lag_periods=1):
    """
    Notebook'undaki align_macro_to_4h fonksiyonu farklı signature'larla
    tanımlanmış olabilir. Bu wrapper ikisini de dener.
    """
    s = series.copy()
    s.index = make_index_tz_naive(s.index)
    s = s.sort_index()

    if "align_macro_to_4h" in globals():
        try:
            return align_macro_to_4h(s, infer_base_index(master_df), lag_periods=lag_periods)
        except TypeError:
            try:
                return align_macro_to_4h(s, lag_periods=lag_periods)
            except TypeError:
                pass
        except Exception as e:
            print(f"[WARN] align_macro_to_4h başarısız: {e}")

    # Fallback: daily/macro veriyi 4H index'e ffill + lag
    out = s.reindex(master_df.index, method="ffill").shift(lag_periods)
    return out


def safe_create_col(df, col, value):
    try:
        df[col] = value
    except Exception as e:
        print(f"[SKIP] {col}: {e}")
    return df


# ════════════════════════════════════════════════════════════════════
# 1) SAFE USD PROXY VAR MI KONTROL
# ════════════════════════════════════════════════════════════════════

print("=" * 100)
print("SAFE USD PROXY PRE-CHECK")
print("=" * 100)

safe_usd_cols = [
    c for c in master_df.columns
    if (
        "usd_exeur" in c.lower()
        or "usd_factor_safe" in c.lower()
        or "eur_usd_exeur" in c.lower()
    )
]

print(f"Safe USD proxy kolon sayısı: {len(safe_usd_cols)}")
for c in safe_usd_cols:
    print(" -", c)

if len(safe_usd_cols) == 0:
    print("[UYARI] usd_exeur / usd_factor_safe kolonları yok.")
    print("Önce DXY yerine safe USD proxy üreten bloğu çalıştırmalısın.")


# ════════════════════════════════════════════════════════════════════
# 2) FRED REAL RATE + BREAKEVEN FEATURE'LARI
# ════════════════════════════════════════════════════════════════════

print("\n" + "=" * 100)
print("N30 FEATURE ENGINEERING — REAL RATE / BREAKEVEN / POSITIONING / VOL")
print("=" * 100)

if "START" not in globals():
    START = str(master_df.index.min().date())

W5   = 5 * 6
W10  = 10 * 6
W21  = 21 * 6
W63  = 63 * 6
W126 = 126 * 6


# ─────────────────────────────────────────────────────────────────
# 2A) US 10Y real yield: DFII10
# ─────────────────────────────────────────────────────────────────
print("FRED DFII10 / TIPS 10Y real yield çekiliyor...")

if HAS_PDR:
    try:
        dfii10 = web.DataReader("DFII10", "fred", START).squeeze().dropna()
        dfii10.index = make_index_tz_naive(dfii10.index)
        dfii10_4h = align_macro_series_safe(dfii10, master_df, lag_periods=1)
        master_df["us10y_real"] = dfii10_4h.reindex(master_df.index, method="ffill")
        print(f"  ✓ us10y_real eklendi | NaN%={master_df['us10y_real'].isna().mean()*100:.2f}")
    except Exception as e:
        print(f"  ⚠ DFII10 çekilemedi: {e}")
        if "us10y_real" not in master_df.columns:
            master_df["us10y_real"] = np.nan
else:
    if "us10y_real" not in master_df.columns:
        master_df["us10y_real"] = np.nan


# ─────────────────────────────────────────────────────────────────
# 2B) US 10Y breakeven inflation: T10YIE
# ─────────────────────────────────────────────────────────────────
print("FRED T10YIE / 10Y breakeven inflation çekiliyor...")

if HAS_PDR:
    try:
        t10yie = web.DataReader("T10YIE", "fred", START).squeeze().dropna()
        t10yie.index = make_index_tz_naive(t10yie.index)
        t10yie_4h = align_macro_series_safe(t10yie, master_df, lag_periods=1)
        master_df["us_breakeven_10y"] = t10yie_4h.reindex(master_df.index, method="ffill")
        print(f"  ✓ us_breakeven_10y eklendi | NaN%={master_df['us_breakeven_10y'].isna().mean()*100:.2f}")
    except Exception as e:
        print(f"  ⚠ T10YIE çekilemedi: {e}")
        if "us_breakeven_10y" not in master_df.columns:
            master_df["us_breakeven_10y"] = np.nan
else:
    if "us_breakeven_10y" not in master_df.columns:
        master_df["us_breakeven_10y"] = np.nan




# ════════════════════════════════════════════════════════════════════
# 4) REAL RATE / INFLATION FEATURES
# ════════════════════════════════════════════════════════════════════

if "us10y_real" in master_df.columns:
    master_df["us10y_real_z63"] = zscore_roll(master_df["us10y_real"], W63)
    master_df["us10y_real_chg5"] = master_df["us10y_real"].diff(W5)
    master_df["us10y_real_chg21"] = master_df["us10y_real"].diff(W21)

    if "mfg_z" in master_df.columns:
        master_df["real_rate_growth_gap_z"] = master_df["us10y_real_z63"] - master_df["mfg_z"]

    if "policy_growth_gap_z" in master_df.columns:
        master_df["real_policy_gap"] = master_df["us10y_real"] - master_df["policy_growth_gap_z"]
        master_df["real_policy_gap_z"] = zscore_roll(master_df["real_policy_gap"], W63)


if "us_breakeven_10y" in master_df.columns:
    master_df["breakeven_z63"] = zscore_roll(master_df["us_breakeven_10y"], W63)
    master_df["breakeven_chg5"] = master_df["us_breakeven_10y"].diff(W5)
    master_df["breakeven_chg21"] = master_df["us_breakeven_10y"].diff(W21)
    master_df["breakeven_accel"] = master_df["us_breakeven_10y"].diff(W5).diff(W5)

    if "cpi_yoy_z" in master_df.columns:
        master_df["infl_surprise_z"] = master_df["cpi_yoy_z"] - zscore_roll(master_df["us_breakeven_10y"], W63)


# ════════════════════════════════════════════════════════════════════
# 5) VOL-OF-VOL / JUMP / DOWNSIDE FEATURES
# ════════════════════════════════════════════════════════════════════

if "hist_vol_20" in master_df.columns:
    master_df["vol_of_vol_21"] = master_df["hist_vol_20"].rolling(W21).std()
    master_df["vol_of_vol_z"] = zscore_roll(master_df["vol_of_vol_21"], W63)

if "log_ret" in master_df.columns and "ewma_vol" in master_df.columns:
    master_df["jump_indicator"] = (
        master_df["log_ret"].abs() > 3 * master_df["ewma_vol"]
    ).astype(float)
    master_df["jump_freq_21"] = master_df["jump_indicator"].rolling(W21).sum()

if "log_ret" in master_df.columns:
    neg_ret = master_df["log_ret"].where(master_df["log_ret"] < 0, 0.0)
    pos_ret = master_df["log_ret"].where(master_df["log_ret"] > 0, 0.0)

    dvol = neg_ret.rolling(W21).std()
    uvol = pos_ret.rolling(W21).std()

    master_df["downside_upside_vol_ratio"] = dvol / (uvol + 1e-10)
    master_df["downside_vol_z"] = zscore_roll(dvol, W63)

print("\n✓ N30 türetilmiş feature'lar hazır.")
print(f"master_df shape: {master_df.shape}")


# Splitleri güncelle
if "train_df" in globals() and "val_df" in globals() and "test_df" in globals():
    train_df = master_df.loc[train_df.index].copy()
    val_df = master_df.loc[val_df.index].copy()
    test_df = master_df.loc[test_df.index].copy()

    print(f"train_df: {train_df.shape}")
    print(f"val_df  : {val_df.shape}")
    print(f"test_df : {test_df.shape}")


# ════════════════════════════════════════════════════════════════════
# 6) 30 NEW THEORY-GROUNDED EXPERTS
# ════════════════════════════════════════════════════════════════════

NEW_EXPERTS_RAW = {

    # FAMILY N1 — REAL RATE PARITY
    "N1_RealRate_Core": [
        "us10y_real",
        "us10y_real_z63",
        "us10y_real_chg5",
        "us10y_real_chg21",
        "us_breakeven_10y",
        "ret_mom21",
    ],

    "N1_RealRate_GrowthAdjusted": [
        "us10y_real_z63",
        "real_rate_growth_gap_z",
        "growth_z63",
        "growth_mom21",
        "policy_growth_gap_z",
        "spread_2y10y",
    ],

    "N1_RealRate_PolicyGap": [
        "us10y_real",
        "real_policy_gap",
        "real_policy_gap_z",
        "uip_carry_z63",
        "policy_growth_gap_z",
        "curve_z63",
    ],

    "N1_RealRate_RiskAdjusted": [
        "us10y_real_z63",
        "us10y_real_chg5",
        "vix_z63",
        "risk_score_z63",
        "credit_z63",
        "haven_z63",
    ],

    "N1_RealRate_USDProxyLink": [
        "us10y_real_z63",
        "us10y_real_chg21",
        "usd_exeur_z63",
        "usd_exeur_mom21",
        "usd_factor_safe_z63",
        "eur_usd_exeur_gap_z",
    ],

    # FAMILY N2 — INFLATION EXPECTATIONS
    "N2_InflationExp_Core": [
        "us_breakeven_10y",
        "breakeven_z63",
        "breakeven_chg5",
        "breakeven_chg21",
        "breakeven_accel",
        "cpi_yoy_z",
    ],

    "N2_InflationExp_Surprise": [
        "infl_surprise_z",
        "breakeven_z63",
        "cpi_yoy_z",
        "policy_growth_gap_z",
        "us10y_chg",
        "uip_carry_z63",
    ],

    "N2_InflationExp_RealNominalGap": [
        "breakeven_z63",
        "us10y_real_z63",
        "spread_2y10y",
        "curve_z63",
        "us10y_chg",
        "real_m2_z",
    ],

    "N2_InflationExp_RegimeShift": [
        "breakeven_chg21",
        "breakeven_accel",
        "vol_z63",
        "vix_z63",
        "credit_z63",
        "policy_growth_gap_z",
    ],

    "N2_InflationExp_USDLink": [
        "breakeven_z63",
        "breakeven_chg21",
        "usd_exeur_z63",
        "usd_factor_safe_z63",
        "gold_mom21",
        "oil_mom21",
    ],

    # FAMILY N3 — POSITIONING
    "N3_Positioning_Core": [
        "spec_pos_proxy",
        "spec_pos_proxy_z",
        "spec_pos_chg5",
        "spec_pos_chg21",
        "spec_pos_extreme",
        "z_score_ret",
    ],

    "N3_Positioning_CrowdRisk": [
        "spec_pos_proxy_z",
        "spec_pos_extreme",
        "pos_crowd_vol",
        "vix_z63",
        "credit_z63",
        "vol_ratio_21_63",
    ],

    "N3_Positioning_MeanReversion": [
        "spec_pos_proxy_z",
        "spec_pos_chg21",
        "rsi_z63",
        "bb_z63",
        "z_score_ret",
        "ret_mom63",
    ],

    "N3_Positioning_TrendFollow": [
        "spec_pos_chg5",
        "spec_pos_chg21",
        "ret_mom5",
        "ret_mom21",
        "adx_14",
        "ma_spread",
    ],

    "N3_Positioning_FundamentalAnchor": [
        "spec_pos_proxy_z",
        "eur_usd_exeur_gap_z",
        "policy_growth_gap_z",
        "uip_carry_z63",
        "us10y_real_z63",
        "breakeven_z63",
    ],

    # FAMILY N4 — VOL STRUCTURE
    "N4_VolStructure_OfVol": [
        "vol_of_vol_21",
        "vol_of_vol_z",
        "hist_vol_20",
        "vol_z63",
        "vix_z63",
        "vol_ratio_21_63",
    ],

    "N4_VolStructure_JumpRisk": [
        "jump_indicator",
        "jump_freq_21",
        "atr_ratio",
        "park_vol",
        "rs_vol",
        "vix_mom5",
    ],

    "N4_VolStructure_AsymmetricDownside": [
        "downside_upside_vol_ratio",
        "downside_vol_z",
        "vol_of_vol_z",
        "vix_z63",
        "z_score_ret",
        "ret_mom5",
    ],

    "N4_VolStructure_RealizedRiskPremium": [
        "vol_of_vol_z",
        "vol_term_gap_z",
        "vrp_proxy",
        "vol_21d_zscore",
        "vix_z63",
        "credit_z63",
    ],

    "N4_VolStructure_CrashHedge": [
        "downside_vol_z",
        "jump_freq_21",
        "haven_z63",
        "gold_mom21",
        "vix_mom21",
        "credit_mom21",
    ],

    # FAMILY N5 — REGIME-CONDITIONAL TACTICAL
    "N5_Regime_RealRate_Mix": [
        "us10y_real_z63",
        "real_policy_gap_z",
        "carry_x_p0",
        "carry_x_p1",
        "carry_x_p2",
        "ret_mom21",
    ],

    "N5_Regime_Inflation_Mix": [
        "breakeven_z63",
        "breakeven_chg21",
        "carry_x_p0",
        "carry_x_p1",
        "carry_x_p2",
        "cpi_yoy_z",
    ],

    "N5_Regime_Positioning_Mix": [
        "spec_pos_proxy_z",
        "spec_pos_chg21",
        "ret_mom21_x_p0",
        "ret_mom21_x_p1",
        "ret_mom21_x_p2",
        "z_score_ret",
    ],

    "N5_Regime_VolStructure_Mix": [
        "vol_of_vol_z",
        "downside_vol_z",
        "risk_x_p0",
        "risk_x_p1",
        "risk_x_p2",
        "vix_z63",
    ],

    "N5_Regime_FullTactical": [
        "us10y_real_z63",
        "breakeven_z63",
        "spec_pos_proxy_z",
        "vol_of_vol_z",
        "carry_x_p0",
        "carry_x_p1",
        "carry_x_p2",
        "risk_x_p0",
        "risk_x_p1",
        "risk_x_p2",
    ],

    # FAMILY N6 — TAIL HEDGING
    "N6_TailHedging_Core": [
        "downside_upside_vol_ratio",
        "downside_vol_z",
        "jump_freq_21",
        "vix_z63",
        "haven_z63",
        "credit_z63",
    ],

    "N6_TailHedging_HavenRotation": [
        "haven_z63",
        "chf_haven",
        "gold_mom5",
        "gold_mom21",
        "jpy_mom5",
        "downside_vol_z",
    ],

    "N6_TailHedging_CrashUnwind": [
        "spec_pos_proxy_z",
        "spec_pos_extreme",
        "pos_crowd_vol",
        "downside_vol_z",
        "vol_of_vol_z",
        "credit_mom5",
    ],

    "N6_TailHedging_RealRateStress": [
        "us10y_real_chg5",
        "us10y_real_chg21",
        "credit_mom5",
        "credit_z63",
        "vix_mom5",
        "downside_vol_z",
    ],

    "N6_TailHedging_FullCrashIns": [
        "downside_upside_vol_ratio",
        "downside_vol_z",
        "vol_of_vol_z",
        "jump_freq_21",
        "haven_z63",
        "spec_pos_proxy_z",
        "vix_z63",
        "credit_z63",
        "gold_mom21",
    ],
}




SAFE USD PROXY PRE-CHECK
Safe USD proxy kolon sayısı: 4
 - usd_factor_safe_lag1
 - usd_factor_safe_z63
 - usd_factor_safe_mom5
 - usd_factor_safe_mom21

N30 FEATURE ENGINEERING — REAL RATE / BREAKEVEN / POSITIONING / VOL
FRED DFII10 / TIPS 10Y real yield çekiliyor...
  ⚠ DFII10 çekilemedi: HTTPSConnectionPool(host='fred.stlouisfed.org', port=443): Read timed out. (read timeout=30)
FRED T10YIE / 10Y breakeven inflation çekiliyor...
  ⚠ T10YIE çekilemedi: HTTPSConnectionPool(host='fred.stlouisfed.org', port=443): Read timed out. (read timeout=30)

✓ N30 türetilmiş feature'lar hazır.
master_df shape: (37612, 272)
train_df: (22567, 272)
val_df  : (7522, 272)
test_df : (7523, 272)


In [ ]:

# ═══════════════════════════════════════════════════════════════════════════════
# B. DXY VERİSİNİ ÇEK VE FINAL REPLACEMENT FEATURE'LARI ÜRET
# ═══════════════════════════════════════════════════════════════════════════════

print("═" * 100)
print("  DXY verisi çekiliyor ve lag-safe DXY replacement feature'ları üretiliyor...")
print("═" * 100)

dxy_start = str((master_df.index.min() - pd.Timedelta(days=220)).date())
dxy_end   = str((master_df.index.max() + pd.Timedelta(days=5)).date())

dxy_raw = yf.download(
    "DX-Y.NYB",
    start=dxy_start,
    end=dxy_end,
    auto_adjust=True,
    progress=False,
)["Close"].squeeze()

if dxy_raw.empty:
    raise ValueError("❌ DXY verisi indirilemedi. İnternet/yfinance/ticker kontrol et.")

dxy_raw.index = make_tz_naive_index(dxy_raw.index)

dxy_lag1 = dxy_raw.shift(1)

dxy_features_daily = pd.DataFrame(
    {
        "dxy_lag1":  dxy_lag1,
        "dxy_ret1":  dxy_lag1.pct_change(1),
        "dxy_mom5":  dxy_lag1.pct_change(5),
        "dxy_mom21": dxy_lag1.pct_change(21),
        "dxy_mom63": dxy_lag1.pct_change(63),
        "dxy_chg5":  dxy_lag1.diff(5),
        "dxy_chg21": dxy_lag1.diff(21),
        "dxy_z63":   (dxy_lag1 - dxy_lag1.rolling(63,  min_periods=20).mean()) /
                     (dxy_lag1.rolling(63,  min_periods=20).std() + 1e-10),
        "dxy_z126":  (dxy_lag1 - dxy_lag1.rolling(126, min_periods=40).mean()) /
                     (dxy_lag1.rolling(126, min_periods=40).std() + 1e-10),
    },
    index=dxy_raw.index,
)

dxy_4h = dxy_features_daily.reindex(master_df.index, method="ffill")

for col in dxy_4h.columns:
    master_df[col] = dxy_4h[col].values
    train_df[col]  = master_df.loc[train_df.index, col].values
    val_df[col]    = master_df.loc[val_df.index,   col].values
    test_df[col]   = master_df.loc[test_df.index,  col].values

print("✅ DXY feature'ları eklendi:")
for col in dxy_4h.columns:
    print(f"   {col:<12} NaN={master_df[col].isna().sum():,}")
print(f"\nDXY raw range: {dxy_raw.index.min().date()} → {dxy_raw.index.max().date()}")
print("═" * 100)


════════════════════════════════════════════════════════════════════════════════════════════════════
  DXY verisi çekiliyor ve lag-safe DXY replacement feature'ları üretiliyor...
════════════════════════════════════════════════════════════════════════════════════════════════════
✅ DXY feature'ları eklendi:
   dxy_lag1     NaN=0
   dxy_ret1     NaN=0
   dxy_mom5     NaN=0
   dxy_mom21    NaN=0
   dxy_mom63    NaN=0
   dxy_chg5     NaN=0
   dxy_chg21    NaN=0
   dxy_z63      NaN=0
   dxy_z126     NaN=0

DXY raw range: 2003-05-27 → 2026-01-02
════════════════════════════════════════════════════════════════════════════════════════════════════


In [ ]:
!pip install hmmlearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 6.0 MB/s eta 0:00:00


In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 12.8 MB/s eta 0:00:00


In [ ]:
print(f"master_df : {master_df.shape}  [{master_df.index[0].date()} → {master_df.index[-1].date()}]")
print(f"train_df  : {train_df.shape}   [{train_df.index[0].date()} → {train_df.index[-1].date()}]")
print(f"val_df    : {val_df.shape}     [{val_df.index[0].date()} → {val_df.index[-1].date()}]")
print(f"test_df   : {test_df.shape}    [{test_df.index[0].date()} → {test_df.index[-1].date()}]")

master_df : (37612, 281)  [2004-01-01 → 2025-12-31]
train_df  : (22567, 281)   [2004-01-01 → 2016-08-31]
val_df    : (7522, 281)     [2016-08-31 → 2021-05-04]
test_df   : (7523, 281)    [2021-05-04 → 2025-12-31]


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# BLOK GK-25 — ADVANCED GARMAN-KOHLHAGEN / BLACK-SCHOLES FX GREEKS EXPERTS
# DTB3 + DAILY EURIBOR3M MARKET-RATE VERSION
# LEAKAGE DÜZELTİLMİŞ VERSİYON
# PAIR CLOSE'LAR PIPELINE İÇİNDE EKLENİYOR — AYRI ADIM YOK
# ═══════════════════════════════════════════════════════════════════════════════

import os
import numpy as np
import pandas as pd
from scipy.stats import norm
from pandas_datareader import data as web


# ═══════════════════════════════════════════════════════════════════════════════
# 0) GLOBAL AYARLAR
# ═══════════════════════════════════════════════════════════════════════════════

try:
    START
except NameError:
    START = "2004-01-01"

try:
    BAR_PER_DAY
except NameError:
    BAR_PER_DAY = 6

EURIBOR3M_CSV_PATH = "euribor3m_daily.csv"
APPLY_RATE_LAG     = True
RATE_LAG_BDAYS     = 1
APPLY_GK_FEATURE_TIMING_SHIFT = False


# ═══════════════════════════════════════════════════════════════════════════════
# 1) YARDIMCI FONKSİYONLAR
# ═══════════════════════════════════════════════════════════════════════════════

def make_index_tz_naive(idx):
    idx = pd.to_datetime(idx)
    try:
        if getattr(idx, "tz", None) is not None:
            idx = idx.tz_convert(None)
    except Exception:
        pass
    try:
        if getattr(idx, "tz", None) is not None:
            idx = idx.tz_localize(None)
    except Exception:
        pass
    return idx

def _to_naive_index(s):
    out = s.copy()
    out.index = make_index_tz_naive(out.index)
    return out.sort_index()[~out.index.duplicated(keep="last")]

def keep_existing(df, feat_list):
    return [f for f in feat_list if f in df.columns]

def zscore_roll(s, w):
    return (s - s.rolling(w).mean()) / (s.rolling(w).std() + 1e-10)

def safe_log_return(s):
    return np.log(s / s.shift(1)).replace([np.inf, -np.inf], np.nan)

def choose_first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def clean_percent_to_decimal_series(s, name=None):
    s = pd.Series(s).copy()
    s = pd.to_numeric(s, errors="coerce").dropna()
    s.index = make_index_tz_naive(s.index)
    s = s.sort_index()[~s.index.duplicated(keep="last")]
    s = s / 100.0
    if name is not None:
        s.name = name
    return s

def lag_series_by_bdays(s, bdays=1):
    s = pd.Series(s).copy().dropna()
    s.index = make_index_tz_naive(s.index)
    s = s.sort_index()[~s.index.duplicated(keep="last")]
    s.index = s.index + pd.offsets.BDay(int(bdays))
    return s.sort_index()

def align_to_index_ffill(s, idx):
    s = pd.Series(s).copy().dropna()
    s.index = make_index_tz_naive(s.index)
    s = s.sort_index()[~s.index.duplicated(keep="last")]
    idx = make_index_tz_naive(idx)
    return s.reindex(s.index.union(idx)).sort_index().ffill().reindex(idx)

def print_series_audit(name, s):
    s = pd.Series(s).dropna().sort_index()
    if len(s) < 3:
        print(f"{name:<26} | n={len(s)} — yetersiz veri")
        return
    gaps = s.index.to_series().diff().dropna().dt.days
    med  = gaps.median()
    freq = "Daily" if med <= 1.5 else "Weekly" if med <= 8 else "Monthly" if med <= 35 else "Other"
    last = float(s.iloc[-1] * 100.0)
    print(f"{name:<26} | n={len(s):>6} | {s.index.min().date()} → {s.index.max().date()} | median_gap={med:.1f} | freq≈{freq} | last={last:.4f}%")


# ═══════════════════════════════════════════════════════════════════════════════
# 2) RATE BLOĞU
# ═══════════════════════════════════════════════════════════════════════════════

def read_bundesbank_euribor3m_csv(path=EURIBOR3M_CSV_PATH, start=START):
    if not os.path.exists(path):
        raise FileNotFoundError(f"EURIBOR3M CSV bulunamadı: {path}")
    df_csv = pd.read_csv(path, sep=None, engine="python", encoding="utf-8-sig")
    date_col, value_col = df_csv.columns[0], df_csv.columns[1]
    out = df_csv[[date_col, value_col]].copy()
    out.columns = ["date", "value"]
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out["value"] = (
        out["value"].astype(str)
        .str.replace(",", ".", regex=False).str.strip()
        .replace([".", "-", "", "nan", "NaN", "None"], np.nan)
    )
    out["value"] = pd.to_numeric(out["value"], errors="coerce")
    out = out.dropna(subset=["date", "value"]).set_index("date").sort_index()
    out = out[~out.index.duplicated(keep="last")]
    s = clean_percent_to_decimal_series(out["value"], name="euribor3m_daily")
    if start is not None:
        s = s[s.index >= pd.Timestamp(start)]
    if len(s) < 100:
        raise ValueError(f"EURIBOR3M seri çok kısa: n={len(s)}")
    return s

def fetch_usd_dtb3(start=START):
    raw = fetch_fred("DTB3", start, api_key=FRED_API_KEY).rename("usd_dtb3")
    return clean_percent_to_decimal_series(raw, name="usd_dtb3")

def build_market_rate_df(start=START, csv_path=EURIBOR3M_CSV_PATH):
    print("\n" + "═"*100)
    print("GK RATE DATA — DTB3 + DAILY EURIBOR3M")
    print("═"*100)
    usd_dtb3      = fetch_usd_dtb3(start)
    euribor3m     = read_bundesbank_euribor3m_csv(path=csv_path, start=start)
    print_series_audit("usd_dtb3_raw", usd_dtb3)
    print_series_audit("euribor3m_daily_raw", euribor3m)

    if APPLY_RATE_LAG:
        usd_model = lag_series_by_bdays(usd_dtb3, RATE_LAG_BDAYS).rename("usd_dtb3_lag1")
        eur_model = lag_series_by_bdays(euribor3m, RATE_LAG_BDAYS).rename("euribor3m_daily_lag1")
        print(f"✓ Rate serilerine {RATE_LAG_BDAYS} business day lag uygulandı.")
    else:
        usd_model = usd_dtb3.rename("usd_dtb3_lag1")
        eur_model = euribor3m.rename("euribor3m_daily_lag1")

    rate_df_lag = pd.concat([usd_model, eur_model], axis=1).sort_index()
    rate_df_lag["market_rate_spread_lag1"] = rate_df_lag["usd_dtb3_lag1"] - rate_df_lag["euribor3m_daily_lag1"]
    return rate_df_lag

def ensure_gk_rates(df, start=None):
    df.index = make_index_tz_naive(df.index)

    if "usd_dtb3" in df.columns and "euribor3m_daily" in df.columns:
        print("[GK RATE] master_df içinde usd_dtb3 ve euribor3m_daily bulundu.")
        df["r_d"] = df["usd_dtb3"].ffill()
        df["r_f"] = df["euribor3m_daily"].ffill()
    else:
        rate_df_lag = build_market_rate_df(
            start=start if start is not None else START,
            csv_path=EURIBOR3M_CSV_PATH,
        )
        df["usd_dtb3"]       = align_to_index_ffill(rate_df_lag["usd_dtb3_lag1"],       df.index)
        df["euribor3m_daily"]= align_to_index_ffill(rate_df_lag["euribor3m_daily_lag1"], df.index)
        df["r_d"] = df["usd_dtb3"]
        df["r_f"] = df["euribor3m_daily"]

    df["r_d"] = df["r_d"].ffill().fillna(0.05)
    df["r_f"] = df["r_f"].ffill().fillna(0.00)

    df["market_rate_spread"]       = df["r_d"] - df["r_f"]
    df["ois_spread"]               = df["market_rate_spread"]
    df["market_rate_spread_z63"]   = zscore_roll(df["market_rate_spread"], 63 * BAR_PER_DAY)
    df["market_rate_spread_mom5"]  = df["market_rate_spread"].diff(5  * BAR_PER_DAY)
    df["market_rate_spread_mom21"] = df["market_rate_spread"].diff(21 * BAR_PER_DAY)
    df["ois_spread_z63"]           = df["market_rate_spread_z63"]
    df["ois_spread_mom5"]          = df["market_rate_spread_mom5"]
    df["ois_spread_mom21"]         = df["market_rate_spread_mom21"]
    return df


# ═══════════════════════════════════════════════════════════════════════════════
# 3) GARMAN-KOHLHAGEN GREEKS
# ═══════════════════════════════════════════════════════════════════════════════

def garman_kohlhagen_greeks(S, K, T, r_d, r_f, sigma, option_type="call"):
    if any(pd.isna(v) for v in [S, K, T, r_d, r_f, sigma]) or S<=0 or K<=0 or T<=0 or sigma<=0:
        return {k: np.nan for k in ["d1","d2","call_delta","put_delta","gamma","vega",
                                     "call_theta","put_theta","call_rho_d","put_rho_d","call_rho_f","put_rho_f"]}
    sqrtT = np.sqrt(T)
    d1 = (np.log(S/K) + (r_d - r_f + 0.5*sigma**2)*T) / (sigma*sqrtT)
    d2 = d1 - sigma*sqrtT
    Nd1=norm.cdf(d1); Nd2=norm.cdf(d2); Nmd1=norm.cdf(-d1); Nmd2=norm.cdf(-d2); nd1=norm.pdf(d1)
    erd=np.exp(-r_d*T); erf=np.exp(-r_f*T)
    return {
        "d1": d1, "d2": d2,
        "call_delta": erf*Nd1, "put_delta": erf*(Nd1-1.0),
        "gamma": erf*nd1/(S*sigma*sqrtT), "vega": S*erf*nd1*sqrtT,
        "call_theta": -S*erf*nd1*sigma/(2*sqrtT) - r_d*K*erd*Nd2  + r_f*S*erf*Nd1,
        "put_theta":  -S*erf*nd1*sigma/(2*sqrtT) + r_d*K*erd*Nmd2 - r_f*S*erf*Nmd1,
        "call_rho_d": K*T*erd*Nd2, "put_rho_d": -K*T*erd*Nmd2,
        "call_rho_f": -T*S*erf*Nd1, "put_rho_f": T*S*erf*Nmd1,
    }

def build_gk_surface_for_spot(spot, r_d, r_f, logret, prefix,
                               moneyness_list=(0.95,0.97,0.99,1.00,1.01,1.03,1.05),
                               tenor_days_list=(7,21,63),
                               bars_per_day=6, trading_days=252, vol_base_days=21):
    spot   = spot.astype(float)
    logret = logret.astype(float)
    vol_window  = vol_base_days * bars_per_day

    # LEAKAGE DÜZELTMESİ 1 & 2: sigma ve spot shift(1)
    sigma       = logret.rolling(vol_window).std().shift(1) * np.sqrt(trading_days)
    spot_lagged = spot.shift(1)

    base_cols = pd.DataFrame(index=spot.index)
    base_cols[f"{prefix}_vol_21d"]         = logret.rolling(vol_window).std().shift(1) * np.sqrt(trading_days)
    base_cols[f"{prefix}_vol_63d"]         = logret.rolling(63*bars_per_day).std().shift(1) * np.sqrt(trading_days)
    base_cols[f"{prefix}_vol_126d"]        = logret.rolling(126*bars_per_day).std().shift(1) * np.sqrt(trading_days)
    base_cols[f"{prefix}_vol_ratio_21_63"] = base_cols[f"{prefix}_vol_21d"] / (base_cols[f"{prefix}_vol_63d"] + 1e-10)
    base_cols[f"{prefix}_vrp_proxy_21_63"] = base_cols[f"{prefix}_vol_21d"] - base_cols[f"{prefix}_vol_63d"]
    base_cols[f"{prefix}_vol_z126"]        = zscore_roll(base_cols[f"{prefix}_vol_21d"], 126*bars_per_day)

    rows = []
    for idx in spot.index:
        S   = spot_lagged.loc[idx]
        rd  = r_d.loc[idx]  if idx in r_d.index  else np.nan
        rf  = r_f.loc[idx]  if idx in r_f.index  else np.nan
        sig = sigma.loc[idx] if idx in sigma.index else np.nan
        feat = {}
        if not (pd.isna(S) or pd.isna(sig) or S<=0 or sig<=0):
            for tenor_days in tenor_days_list:
                T = tenor_days / trading_days
                tenor_tag = f"t{tenor_days}"
                for m in moneyness_list:
                    K = m * S
                    mtag = f"m{int(round(m*100)):03d}"
                    g = garman_kohlhagen_greeks(S=S, K=K, T=T, r_d=rd, r_f=rf, sigma=sig)
                    for gname, val in g.items():
                        feat[f"{prefix}_{tenor_tag}_{mtag}_{gname}"] = val
        rows.append(feat)

    surface = pd.DataFrame(rows, index=spot.index)
    return surface.join(base_cols, how="left")


# ═══════════════════════════════════════════════════════════════════════════════
# 4) EURUSD SPOT SEÇİMİ
# ═══════════════════════════════════════════════════════════════════════════════

def get_main_eurusd_spot(master_df):
    if "C" in globals():
        s = globals()["C"].copy()
        s = _to_naive_index(s)
        return s.reindex(master_df.index, method="ffill").ffill().rename("EURUSD_spot")
    for c in ["Close", "close", "C", "EURUSD", "eurusd"]:
        if c in master_df.columns:
            return master_df[c].astype(float).ffill().rename("EURUSD_spot")
    raise ValueError("EUR/USD spot bulunamadı.")


# ═══════════════════════════════════════════════════════════════════════════════
# 5) PAIR CLOSE'LARI EKLE + CROSS-PAIR SPOT SEÇİMİ
# ═══════════════════════════════════════════════════════════════════════════════

def inject_pair_closes(df, base_index=None):
    """
    Pair close'ları doğrudan master_df'e ekler.
    gbpusd_df, usdchf_df vb. global scope'ta olmalı.
    align_market_to_4h fonksiyonu da global scope'ta olmalı.
    """
    pair_map = {
        "gbpusd": ("gbpusd_df", "close"),
        "usdchf": ("usdchf_df", "close"),
        "usdjpy": ("usdjpy_df", "close"),
        "eurchf": ("eurchf_df", "close"),
        "eurgbp": ("eurgbp_df", "close"),
        "eurjpy": ("eurjpy_df", "close"),
    }

    idx = base_index if base_index is not None else df.index

    injected = []
    for col_name, (df_name, close_col) in pair_map.items():
        if col_name in df.columns:
            print(f"  [inject] {col_name} zaten mevcut, atlandı.")
            continue
        if df_name not in globals():
            print(f"  [inject] {df_name} global scope'ta yok, atlandı.")
            continue
        try:
            aligned = align_market_to_4h(globals()[df_name][close_col], idx)
            df[col_name] = aligned.reindex(df.index).ffill()
            injected.append(col_name)
        except Exception as e:
            print(f"  [inject] {col_name} eklenemedi: {e}")

    if injected:
        print(f"  [inject] Eklenen pair close'lar: {injected}")
    return df

def build_cross_pair_spots(master_df):
    """
    Level kolonları öncelikli — inject_pair_closes çalıştıktan sonra
    level her zaman bulunur, proxy'e fallback gerekmez.
    """
    pair_specs = {
        "gbpusd": {"level_candidates": ["GBPUSD","gbpusd","Close_GBP","gbp_close"],  "base": 1.25},
        "usdchf": {"level_candidates": ["USDCHF","usdchf","Close_CHF","chf_close"], "base": 0.90},
        "usdjpy": {"level_candidates": ["USDJPY","usdjpy","Close_JPY","jpy_close"],  "base": 145.0},
        "eurgbp": {"level_candidates": ["EURGBP","eurgbp"],                                      "base": 0.85},
        "eurchf": {"level_candidates": ["EURCHF","eurchf"],                                       "base": 0.95},
        "eurjpy": {"level_candidates": ["EURJPY","eurjpy"],                                         "base": 160.0},
    }

    out = {}
    for pair, spec in pair_specs.items():
        level_col = choose_first_existing(master_df, spec["level_candidates"])
        if level_col is not None:
            # ✅ Level bulundu — surface içinde shift(1) uygulanacak, leakage yok
            out[pair] = master_df[level_col].astype(float).ffill().rename(f"{pair}_spot")
            print(f"  [cross spot] {pair}: LEVEL ({level_col}) ✅")
            continue
        # Fallback — proxy (leakage riski var ama inject_pair_closes çalıştıysa buraya düşmez)
        ret_col = choose_first_existing(master_df, spec["ret_candidates"])
        if ret_col is not None:
            ret  = master_df[ret_col].astype(float).fillna(0.0)
            spot = spec["base"] * np.exp(ret.cumsum())
            out[pair] = spot.rename(f"{pair}_spot")
            print(f"  [cross spot] {pair}: PROXY RET ({ret_col}) ⚠️ leakage riski")

    return out


# ═══════════════════════════════════════════════════════════════════════════════
# 6) ANA GK FEATURE ENGINE
# ═══════════════════════════════════════════════════════════════════════════════

def build_advanced_gk_25_features(
    master_df,
    start=None,
    bars_per_day=6,
    trading_days=252,
    main_prefix="eurusd",
    base_index=None,          # ← align_market_to_4h için
):
    df = master_df.copy()
    df.index = make_index_tz_naive(df.index)

    # ── ADIM 0: Pair close'ları pipeline içinde ekle ──────────────────────────
    print("\n[GK] Pair close'lar ekleniyor...")
    df = inject_pair_closes(df, base_index=base_index if base_index is not None else df.index)

    # ── ADIM 1: Rates ─────────────────────────────────────────────────────────
    df = ensure_gk_rates(df, start=start)

    # ── ADIM 2: EURUSD ana surface ────────────────────────────────────────────
    eurusd_spot = get_main_eurusd_spot(df)
    df["eurusd_spot"] = eurusd_spot

    if "log_ret" in df.columns:
        eurusd_logret = df["log_ret"].astype(float)
    else:
        eurusd_logret = safe_log_return(eurusd_spot)
        df["log_ret"] = eurusd_logret

    print("[GK] EURUSD ana GK surface hesaplanıyor...")
    eurusd_surface = build_gk_surface_for_spot(
        spot=eurusd_spot, r_d=df["r_d"], r_f=df["r_f"], logret=eurusd_logret,
        prefix=main_prefix,
        moneyness_list=(0.95, 0.97, 0.99, 1.00, 1.01, 1.03, 1.05),
        tenor_days_list=(7, 21, 63),
        bars_per_day=bars_per_day, trading_days=trading_days, vol_base_days=21,
    )
    df = df.join(eurusd_surface, how="left")

    p = main_prefix

    # ATM aliases
    for greek in ["d1","d2","call_delta","put_delta","gamma","vega","call_theta","put_theta","call_rho_d","call_rho_f"]:
        df[f"{p}_atm_{greek}"] = df.get(f"{p}_t21_m100_{greek}")

    # Smile / skew / convexity
    df[f"{p}_d1_skew_95_105"]         = df[f"{p}_t21_m105_d1"]    - df[f"{p}_t21_m095_d1"]
    df[f"{p}_d1_skew_97_103"]         = df[f"{p}_t21_m103_d1"]    - df[f"{p}_t21_m097_d1"]
    df[f"{p}_gamma_skew_95_105"]      = df[f"{p}_t21_m095_gamma"] / (df[f"{p}_t21_m105_gamma"] + 1e-10)
    df[f"{p}_gamma_skew_97_103"]      = df[f"{p}_t21_m097_gamma"] / (df[f"{p}_t21_m103_gamma"] + 1e-10)
    df[f"{p}_d1_convexity_95_105"]    = (df[f"{p}_t21_m105_d1"]    + df[f"{p}_t21_m095_d1"])    / 2.0 - df[f"{p}_t21_m100_d1"]
    df[f"{p}_gamma_convexity_95_105"] = (df[f"{p}_t21_m105_gamma"] + df[f"{p}_t21_m095_gamma"]) / 2.0 - df[f"{p}_t21_m100_gamma"]

    # Term structure
    df[f"{p}_gamma_term_7_63"] = df[f"{p}_t7_m100_gamma"]      / (df[f"{p}_t63_m100_gamma"]      + 1e-10)
    df[f"{p}_vega_term_7_63"]  = df[f"{p}_t7_m100_vega"]       / (df[f"{p}_t63_m100_vega"]       + 1e-10)
    df[f"{p}_theta_term_7_63"] = df[f"{p}_t7_m100_call_theta"] - df[f"{p}_t63_m100_call_theta"]
    df[f"{p}_d1_term_7_63"]    = df[f"{p}_t7_m100_d1"]         - df[f"{p}_t63_m100_d1"]

    # Carry
    df[f"{p}_carry_norm"]    = df["ois_spread"] / (df[f"{p}_vol_21d"] + 1e-10)
    df[f"{p}_carry_x_delta"] = df["ois_spread"] * df[f"{p}_atm_call_delta"]
    df[f"{p}_carry_x_d1"]    = df["ois_spread"] * df[f"{p}_atm_d1"]
    df[f"{p}_carry_x_gamma"] = df["ois_spread"] * df[f"{p}_atm_gamma"]
    df[f"{p}_rho_spread"]    = df[f"{p}_atm_call_rho_d"] - df[f"{p}_atm_call_rho_f"]

    if "ret_mom21" in df.columns:
        df[f"{p}_uip_residual_21"] = df["ret_mom21"] - df["ois_spread"] * (21 / trading_days)
    else:
        df[f"{p}_uip_residual_21"] = (
            df["log_ret"].rolling(21 * bars_per_day).sum()
            - df["ois_spread"] * (21 / trading_days)
        )

    # LEAKAGE DÜZELTMESİ 3: dollar_gamma / vega — spot shift(1)
    eurusd_spot_lagged = eurusd_spot.shift(1)
    df[f"{p}_dollar_gamma"] = df[f"{p}_atm_gamma"] * (eurusd_spot_lagged ** 2)
    df[f"{p}_dollar_vega"]  = df[f"{p}_atm_vega"]  * eurusd_spot_lagged

    # Momentum
    for feat, src in [("gamma_mom5",5),("gamma_mom21",21),("d1_mom5",5),("d1_mom21",21),("vega_mom5",5),("theta_mom5",5)]:
        col = "atm_gamma" if "gamma" in feat else ("atm_d1" if "d1" in feat else ("atm_vega" if "vega" in feat else "atm_call_theta"))
        df[f"{p}_{feat}"] = df[f"{p}_{col}"].diff(src * bars_per_day)

    # Z-score
    z_window = 126 * bars_per_day
    for col in [f"{p}_atm_d1",f"{p}_atm_gamma",f"{p}_atm_vega",f"{p}_atm_call_theta",
                f"{p}_d1_skew_95_105",f"{p}_gamma_skew_95_105",
                f"{p}_gamma_term_7_63",f"{p}_vega_term_7_63",
                f"{p}_carry_norm",f"{p}_uip_residual_21",f"{p}_dollar_gamma"]:
        if col in df.columns:
            df[f"{col}_z126"] = zscore_roll(df[col], z_window)

    # ── ADIM 3: Cross pair surfaces ───────────────────────────────────────────
    cross_spots    = build_cross_pair_spots(df)
    cross_prefixes = []

    for pair, spot in cross_spots.items():
        try:
            print(f"[GK] Cross pair: {pair}")
            pair_spot   = spot.reindex(df.index).ffill()
            pair_logret = safe_log_return(pair_spot)

            pair_surface = build_gk_surface_for_spot(
                spot=pair_spot, r_d=df["r_d"], r_f=df["r_f"], logret=pair_logret,
                prefix=pair, moneyness_list=(0.97, 1.00, 1.03), tenor_days_list=(21,),
                bars_per_day=bars_per_day, trading_days=trading_days, vol_base_days=21,
            )
            df = df.join(pair_surface, how="left")
            cross_prefixes.append(pair)

            df[f"{pair}_atm_d1"]            = df.get(f"{pair}_t21_m100_d1")
            df[f"{pair}_atm_gamma"]         = df.get(f"{pair}_t21_m100_gamma")
            df[f"{pair}_atm_vega"]          = df.get(f"{pair}_t21_m100_vega")
            df[f"{pair}_atm_theta"]         = df.get(f"{pair}_t21_m100_call_theta")
            df[f"{pair}_d1_skew_97_103"]    = df[f"{pair}_t21_m103_d1"]    - df[f"{pair}_t21_m097_d1"]
            df[f"{pair}_gamma_skew_97_103"] = df[f"{pair}_t21_m097_gamma"] / (df[f"{pair}_t21_m103_gamma"] + 1e-10)
            df[f"{pair}_carry_norm"]        = df["ois_spread"] / (df[f"{pair}_vol_21d"] + 1e-10)

            # LEAKAGE DÜZELTMESİ 4: cross pair dollar_gamma
            pair_spot_lagged = pair_spot.shift(1)
            df[f"{pair}_dollar_gamma"] = df[f"{pair}_atm_gamma"] * (pair_spot_lagged.reindex(df.index) ** 2)

        except Exception as e:
            print(f"[WARN] {pair} geçildi: {e}")

    # Relative greeks
    for pair in cross_prefixes:
        try:
            df[f"rel_gamma_eurusd_{pair}"]     = df[f"{p}_atm_gamma"]          / (df[f"{pair}_atm_gamma"]  + 1e-10)
            df[f"rel_vega_eurusd_{pair}"]      = df[f"{p}_atm_vega"]           / (df[f"{pair}_atm_vega"]   + 1e-10)
            df[f"rel_d1_eurusd_{pair}"]        = df[f"{p}_atm_d1"]             - df[f"{pair}_atm_d1"]
            df[f"rel_carrynorm_eurusd_{pair}"] = df[f"{p}_carry_norm"]         - df[f"{pair}_carry_norm"]
            df[f"rel_gammaskew_eurusd_{pair}"] = df[f"{p}_gamma_skew_97_103"]  - df[f"{pair}_gamma_skew_97_103"]
        except Exception as e:
            print(f"[WARN] relative greek: {pair} | {e}")

    # Compat aliases
    for mtag in ["097","099","100","101","103","105"]:
        for g in ["d1","gamma","theta"]:
            src = f"{p}_t21_m{mtag}_{'call_theta' if g=='theta' else g}"
            dst = f"gk_m{mtag}_{g}"
            if src in df.columns:
                df[dst] = df[src]

    df["smile_skew"]      = df.get(f"{p}_d1_skew_97_103")
    df["smile_convexity"] = df.get(f"{p}_d1_convexity_95_105")
    df["gamma_skew"]      = df.get(f"{p}_gamma_skew_97_103")
    df["d1_mom_5d"]       = df.get(f"{p}_d1_mom5")
    df["gamma_change_5d"] = df.get(f"{p}_gamma_mom5")
    df["gamma_dollar"]    = df.get(f"{p}_dollar_gamma")
    df["vol_21d"]         = df.get(f"{p}_vol_21d")
    df["vol_63d"]         = df.get(f"{p}_vol_63d")
    df["vol_ratio"]       = df.get(f"{p}_vol_ratio_21_63")
    df["vrp_proxy"]       = df.get(f"{p}_vrp_proxy_21_63")
    df["vol_21d_zscore"]  = df.get(f"{p}_vol_z126")
    df["carry_norm"]      = df.get(f"{p}_carry_norm")
    df["carry_x_delta"]   = df.get(f"{p}_carry_x_delta")
    df["uip_residual"]    = df.get(f"{p}_uip_residual_21")
    df["market_rate_spread_z63"]  = df["ois_spread_z63"]
    df["market_rate_spread_mom5"] = df["ois_spread_mom5"]
    df["market_rate_spread_mom21"]= df["ois_spread_mom21"]

    if APPLY_GK_FEATURE_TIMING_SHIFT:
        raise RuntimeError("APPLY_GK_FEATURE_TIMING_SHIFT=True kullanma! Shift zaten surface içinde yapıldı.")

    return df


# ═══════════════════════════════════════════════════════════════════════════════
# 7) GK-25 EXPERT SETLER
# ═══════════════════════════════════════════════════════════════════════════════

def build_gk_25_expert_sets(master_df):
    GK_25_RAW = {
        "GK01_Core_ATM": ["eurusd_atm_d1","eurusd_atm_d2","eurusd_atm_call_delta","eurusd_atm_gamma","eurusd_atm_vega","eurusd_atm_call_theta","vol_21d","vrp_proxy"],
        "GK02_Core_Moneyness_D1": ["eurusd_t21_m095_d1","eurusd_t21_m097_d1","eurusd_t21_m099_d1","eurusd_t21_m100_d1","eurusd_t21_m101_d1","eurusd_t21_m103_d1","eurusd_t21_m105_d1"],
        "GK03_Core_Moneyness_Gamma": ["eurusd_t21_m095_gamma","eurusd_t21_m097_gamma","eurusd_t21_m099_gamma","eurusd_t21_m100_gamma","eurusd_t21_m101_gamma","eurusd_t21_m103_gamma","eurusd_t21_m105_gamma"],
        "GK04_Core_Moneyness_Theta": ["eurusd_t21_m095_call_theta","eurusd_t21_m097_call_theta","eurusd_t21_m100_call_theta","eurusd_t21_m103_call_theta","eurusd_t21_m105_call_theta","eurusd_theta_mom5"],
        "GK05_Core_Moneyness_Vega": ["eurusd_t21_m095_vega","eurusd_t21_m097_vega","eurusd_t21_m100_vega","eurusd_t21_m103_vega","eurusd_t21_m105_vega","eurusd_vega_mom5"],
        "GK06_Smile_D1Skew": ["eurusd_d1_skew_95_105","eurusd_d1_skew_97_103","eurusd_d1_convexity_95_105","smile_skew","smile_convexity","eurusd_atm_d1_z126"],
        "GK07_Smile_GammaSkew": ["eurusd_gamma_skew_95_105","eurusd_gamma_skew_97_103","eurusd_gamma_convexity_95_105","gamma_skew","eurusd_atm_gamma_z126","eurusd_dollar_gamma"],
        "GK08_Smile_Wings": ["eurusd_t21_m095_d1","eurusd_t21_m105_d1","eurusd_t21_m095_gamma","eurusd_t21_m105_gamma","eurusd_t21_m095_vega","eurusd_t21_m105_vega","eurusd_d1_skew_95_105","eurusd_gamma_skew_95_105"],
        "GK09_Term_Gamma": ["eurusd_t7_m100_gamma","eurusd_t21_m100_gamma","eurusd_t63_m100_gamma","eurusd_gamma_term_7_63","eurusd_gamma_mom5","eurusd_gamma_mom21"],
        "GK10_Term_VegaTheta": ["eurusd_t7_m100_vega","eurusd_t21_m100_vega","eurusd_t63_m100_vega","eurusd_vega_term_7_63","eurusd_theta_term_7_63","eurusd_atm_call_theta","eurusd_atm_vega_z126"],
        "GK11_Term_D1Delta": ["eurusd_t7_m100_d1","eurusd_t21_m100_d1","eurusd_t63_m100_d1","eurusd_d1_term_7_63","eurusd_atm_call_delta","eurusd_d1_mom5","eurusd_d1_mom21"],
        "GK12_Vol_VRP_Core": ["eurusd_vol_21d","eurusd_vol_63d","eurusd_vol_126d","eurusd_vol_ratio_21_63","eurusd_vrp_proxy_21_63","eurusd_vol_z126","vol_ratio","vrp_proxy"],
        "GK13_Vol_GammaPressure": ["eurusd_atm_gamma","eurusd_dollar_gamma","eurusd_dollar_gamma_z126","eurusd_gamma_mom5","eurusd_gamma_mom21","eurusd_vol_ratio_21_63","eurusd_vrp_proxy_21_63"],
        "GK14_Vol_VegaPressure": ["eurusd_atm_vega","eurusd_dollar_vega","eurusd_vega_mom5","eurusd_vega_term_7_63","eurusd_vol_21d","eurusd_vol_ratio_21_63","eurusd_vrp_proxy_21_63"],
        "GK15_Carry_MarketRate_Core": ["ois_spread","market_rate_spread","ois_spread_z63","ois_spread_mom5","ois_spread_mom21","eurusd_carry_norm","carry_norm","eurusd_rho_spread"],
        "GK16_Carry_DeltaInteraction": ["ois_spread","market_rate_spread","eurusd_atm_call_delta","eurusd_carry_x_delta","carry_x_delta","eurusd_carry_x_d1","eurusd_carry_norm","eurusd_uip_residual_21"],
        "GK17_Carry_GammaInteraction": ["ois_spread","market_rate_spread","eurusd_carry_x_gamma","eurusd_atm_gamma","eurusd_dollar_gamma","eurusd_gamma_skew_97_103","eurusd_gamma_term_7_63","eurusd_carry_norm"],
        "GK18_UIP_Residual_BS": ["eurusd_uip_residual_21","uip_residual","eurusd_carry_norm","eurusd_atm_d1","eurusd_d1_mom21","eurusd_vol_ratio_21_63","eurusd_vrp_proxy_21_63"],
        "GK19_Rho_RateSensitivity": ["eurusd_atm_call_rho_d","eurusd_atm_call_rho_f","eurusd_rho_spread","ois_spread","market_rate_spread","ois_spread_z63","eurusd_carry_x_delta","eurusd_carry_norm"],
        "GK20_Theta_CarryDecay": ["eurusd_atm_call_theta","eurusd_atm_put_theta","eurusd_theta_mom5","eurusd_theta_term_7_63","eurusd_carry_norm","eurusd_vrp_proxy_21_63","ois_spread_mom21"],
        "GK21_Cross_GBPUSD_RelGreeks": ["gbpusd_atm_d1","gbpusd_atm_gamma","gbpusd_atm_vega","gbpusd_d1_skew_97_103","rel_gamma_eurusd_gbpusd","rel_vega_eurusd_gbpusd","rel_d1_eurusd_gbpusd","rel_carrynorm_eurusd_gbpusd"],
        "GK22_Cross_USDCHF_RelGreeks": ["usdchf_atm_d1","usdchf_atm_gamma","usdchf_atm_vega","usdchf_d1_skew_97_103","rel_gamma_eurusd_usdchf","rel_vega_eurusd_usdchf","rel_d1_eurusd_usdchf","rel_carrynorm_eurusd_usdchf"],
        "GK23_Cross_USDJPY_RelGreeks": ["usdjpy_atm_d1","usdjpy_atm_gamma","usdjpy_atm_vega","usdjpy_d1_skew_97_103","rel_gamma_eurusd_usdjpy","rel_vega_eurusd_usdjpy","rel_d1_eurusd_usdjpy","rel_carrynorm_eurusd_usdjpy"],
        "GK24_Cross_EURCross_RelGreeks": ["eurgbp_atm_d1","eurchf_atm_d1","eurjpy_atm_d1","rel_d1_eurusd_eurgbp","rel_d1_eurusd_eurchf","rel_d1_eurusd_eurjpy","rel_gamma_eurusd_eurgbp","rel_gamma_eurusd_eurchf","rel_gamma_eurusd_eurjpy"],
        "GK25_Full_BS_Greek_Theory": ["eurusd_atm_d1","eurusd_atm_call_delta","eurusd_atm_gamma","eurusd_atm_vega","eurusd_atm_call_theta","eurusd_d1_skew_97_103","eurusd_gamma_skew_97_103","eurusd_gamma_term_7_63","eurusd_vega_term_7_63","eurusd_carry_norm","eurusd_uip_residual_21","eurusd_rho_spread","eurusd_vrp_proxy_21_63","eurusd_dollar_gamma","market_rate_spread"],
    }
    GK_25 = {name: keep_existing(master_df, feats) for name, feats in GK_25_RAW.items()}
    return {name: feats for name, feats in GK_25.items() if len(feats) >= 3}


# ═══════════════════════════════════════════════════════════════════════════════
# 8) ÇALIŞTIRMA
# ═══════════════════════════════════════════════════════════════════════════════

try:
    START_GK = START
except NameError:
    START_GK = str(master_df.index.min().date())

print("="*100)
print("ADVANCED GK-25 FEATURE ENGINE — LEAKAGE DÜZELTİLMİŞ + PAIR CLOSE ENTEGRASYONLU")
print("="*100)

master_df = build_advanced_gk_25_features(
    master_df=master_df,
    start=START_GK,
    bars_per_day=BAR_PER_DAY,
    trading_days=252,
    main_prefix="eurusd",
    base_index=base_index if "base_index" in globals() else None,
)

# Splitleri güncelle
train_df = master_df.loc[train_df.index].copy()
val_df   = master_df.loc[val_df.index].copy()
test_df  = master_df.loc[test_df.index].copy()

GK_25_EXPERT_SETS = build_gk_25_expert_sets(master_df)

print("\n" + "="*100)
print("GK-25 EXPERT SET AUDIT")
print("="*100)
print(f"{'Expert':<40} {'nFeat':>6}")
print("─"*50)
for name, feats in sorted(GK_25_EXPERT_SETS.items()):
    print(f"  {name:<40} {len(feats):>6}")
print("─"*50)
print(f"Toplam GK expert: {len(GK_25_EXPERT_SETS)} / 25")
print(f"master_df shape : {master_df.shape}")

# ALL_EXPERT_SETS güncelle
if "ALL_EXPERT_SETS" not in globals():
    ALL_EXPERT_SETS = {}
for k in [k for k in list(ALL_EXPERT_SETS.keys()) if k.startswith("GK")]:
    ALL_EXPERT_SETS.pop(k, None)
ALL_EXPERT_SETS.update(GK_25_EXPERT_SETS)
ALL_EXPERT_SETS = {k: v for k, v in ALL_EXPERT_SETS.items() if len(v) >= 3}
print(f"ALL_EXPERT_SETS toplam: {len(ALL_EXPERT_SETS)}")
print("GK-25 TAMAMLANDI.")

ADVANCED GK-25 FEATURE ENGINE — LEAKAGE DÜZELTİLMİŞ + PAIR CLOSE ENTEGRASYONLU

[GK] Pair close'lar ekleniyor...
  [inject] Eklenen pair close'lar: ['gbpusd', 'usdchf', 'usdjpy', 'eurchf', 'eurgbp', 'eurjpy']

════════════════════════════════════════════════════════════════════════════════════════════════════
GK RATE DATA — DTB3 + DAILY EURIBOR3M
════════════════════════════════════════════════════════════════════════════════════════════════════
usd_dtb3_raw               | n=  5609 | 2004-01-02 → 2026-06-03 | median_gap=1.0 | freq≈Daily | last=3.6300%
euribor3m_daily_raw        | n=  5728 | 2004-01-02 → 2026-05-14 | median_gap=1.0 | freq≈Daily | last=2.2390%
✓ Rate serilerine 1 business day lag uygulandı.
[GK] EURUSD ana GK surface hesaplanıyor...
  [cross spot] gbpusd: LEVEL (gbpusd) ✅
  [cross spot] usdchf: LEVEL (usdchf) ✅
  [cross spot] usdjpy: LEVEL (usdjpy) ✅
  [cross spot] eurgbp: LEVEL (eurgbp) ✅
  [cross spot] eurchf: LEVEL (eurchf) ✅
  [cross spot] eurjpy: LEVEL (eurjpy) ✅
[

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FINAL TRAINING + SIMILARITY RAPORU — DXY REPLACEMENT FINAL VERSION
# 25 Expert Set × XGB + LGBM + HGBM
#
# pred2prob MANTIĞI:
#   train : expanding (buffer yok)
#   val   : split-wide  ← val bitti, elimizde, meşru
#   test  : expanding   ← canlı simülasyon (train+val buffer)
#
#   threshold : split-wide val prob üzerinden optimize edilir (orijinal mantık)
#
# Requires: master_df, train_df, val_df, test_df (fwd_ret içermeli)
# ═══════════════════════════════════════════════════════════════════════════════

import os
import json
import pickle
import warnings
import numpy as np
import pandas as pd
from itertools import combinations
import yfinance as yf

warnings.filterwarnings("ignore")

from sklearn.preprocessing import RobustScaler
from sklearn.metrics import r2_score
from sklearn.ensemble import HistGradientBoostingRegressor

try:
    import xgboost as xgb
except ImportError:
    raise ImportError("xgboost yüklü değil. Lütfen: pip install xgboost")

try:
    import lightgbm as lgb
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False


# ═══════════════════════════════════════════════════════════════════════════════
# A. BASIC CHECKS
# ═══════════════════════════════════════════════════════════════════════════════

for obj in ["master_df", "train_df", "val_df", "test_df"]:
    if obj not in globals():
        raise ValueError(f"❌ '{obj}' bulunamadı. Önce veri hazırlama bloklarını çalıştır.")

for name, df_ in [("train_df", train_df), ("val_df", val_df), ("test_df", test_df)]:
    if "fwd_ret" not in df_.columns:
        raise ValueError(f"❌ '{name}' içinde 'fwd_ret' yok.")

def make_tz_naive_index(idx):
    idx = pd.DatetimeIndex(idx)
    if idx.tz is not None:
        return idx.tz_localize(None)
    return idx

master_df = master_df.copy()
train_df  = train_df.copy()
val_df    = val_df.copy()
test_df   = test_df.copy()

master_df.index = make_tz_naive_index(master_df.index)
train_df.index  = make_tz_naive_index(train_df.index)
val_df.index    = make_tz_naive_index(val_df.index)
test_df.index   = make_tz_naive_index(test_df.index)



# ═══════════════════════════════════════════════════════════════════════════════
# 0. FINAL SELECTED FEATURE SETS
# ═══════════════════════════════════════════════════════════════════════════════

SELECTED_FEATURE_SETS = {

    # ── REAL RATE ─────────────────────────────────────────────────────────────
    "T01_REAL_RATE_SELECTED_3_growth_realrate_channel": [
        "real_nominal_gap_z63", "real_rate_growth_gap_z", "real_yield_z63",
        "us10y_real_chg21", "us10y_real_chg5", "us10y_real_z63",
    ],
    "RF_T01_REAL_RATE_PARITY_01_fisher_realrate_breakeven": [
        "inflation_gap_us_ea_z63", "real_nominal_gap_z63", "real_yield_z63",
        "spread_2y10y", "us10y_be_z63", "us10y_real_chg21",
        "us10y_real_chg5", "us10y_real_z63", "us_breakeven_10y",
    ],

    # ── MACRO SURPRISE ────────────────────────────────────────────────────────
    "MACRO_KS_SELECTED_1_infl_energy_real_yield_spx": [
        "ea_hicp_yoy_z", "energy_pressure_eu_z63", "inflation_gap_us_ea_z63",
        "real_yield_mom21", "real_yield_z63", "spx_mom21",
        "us10y_be_z63", "us10y_real_z63",
    ],
    # DXY replacement: removed usd_exeur_mom21 → added dxy_mom5 + dxy_chg5
    "MACRO_KS_SELECTED_3_DXY2_liq_real_yield_growth_dxy_mom5_chg5": [
        "ea_hicp_yoy_z", "macro_pressure_mom21", "net_liq_z",
        "real_yield_mom21", "real_yield_z63", "us10y_be_z63",
        "us10y_real_z63", "us_growth_proxy", "dxy_mom5", "dxy_chg5",
    ],

    # ── CURVE / LIQ / POLICY ──────────────────────────────────────────────────
    "RF_MACRO_KS_01_curve_liq_realrate_policy": [
        "curve_accel", "it_ea_10y_spread_z63", "liq_score", "policy_score",
        "real_yield_mom21", "us10y_real_chg21", "us10y_real_chg5",
        "us_de_10y_spread_z63", "vix_z63",
    ],

    # ── LIQUIDITY ─────────────────────────────────────────────────────────────
    "LIQUIDITY_PRESSURE_SELECTED_1_money_netliq_risk": [
        "macro_pressure_z63", "money_liquidity_proxy", "net_liq_z",
        "rates_z63", "risk_score_z63", "vix_z63",
    ],
    "LIQUIDITY_HGBM_TEST_SELECTED_03_macro_liquidity_rates_risk": [
        "liq_mom5", "liq_z63", "macro_pressure_z63", "oil_mom21",
        "rates_z63", "real_m2_z", "risk_score_z63", "vix_z63",
    ],

    # ── INFL POLICY ───────────────────────────────────────────────────────────
    "INFL_POLICY_RATES_SELECTED_3_real_yield_momentum": [
        "breakeven_accel", "infl_surprise_z", "inflation_gap_us_ea_z63",
        "policy_diff_chg21", "policy_diff_fed_ecb_dfr", "rates_z63",
        "real_yield_mom21", "us10y_real_chg21", "us10y_real_chg5",
    ],

    # ── GK GREEKS ─────────────────────────────────────────────────────────────
    "GK_RF_SELECTED_01_gamma_vega_vrp_carry_relD1": [
        "eurusd_gamma_skew_97_103", "eurusd_atm_gamma_z126", "eurusd_vega_term_7_63",
        "eurusd_t7_m100_d1", "eurusd_vrp_proxy_21_63", "vol_ratio",
        "eurusd_dollar_vega", "eurusd_carry_x_gamma", "usdchf_atm_vega",
        "rel_d1_eurusd_eurjpy",
    ],
    "GK_RF_SELECTED_03_compact_gamma_vrp_carry": [
        "eurusd_gamma_skew_95_105", "eurusd_atm_gamma_z126", "eurusd_vrp_proxy_21_63",
        "carry_norm", "eurusd_carry_x_gamma", "eurusd_atm_call_rho_d",
        "usdchf_atm_d1", "rel_d1_eurusd_eurjpy", "rel_gamma_eurusd_eurjpy",
    ],

    # ── SAFE HAVEN ────────────────────────────────────────────────────────────
    "BOP_SAFEHAVEN_SELECTED_1_chf_jpy_sovereign_risk": [
        "chf_ret", "corr_eur_chf", "gbp_mom5", "jpy_ret",
        "macro_pressure_z63", "risk_score_z63", "sovereign_z63", "vix_z63",
    ],

    # ── SOVEREIGN / CREDIT ────────────────────────────────────────────────────
    "SOVEREIGN_SELECTED_2_it_de_rates_risk": [
        "it_de_10y_spread", "it_de_10y_spread_z252", "it_de_10y_spread_z63",
        "macro_pressure_z63", "rates_x_risk", "risk_score_z63", "sovereign_z63",
    ],
    "CREDIT_SOV_RATES_SELECTED_02_credit_spread_risk_compact": [
        "credit_mom5", "macro_pressure_z63", "risk_score_z63",
        "us_ea_10y_spread_z63", "vix_z63",
    ],

    # ── GROWTH / POLICY ───────────────────────────────────────────────────────
    "POLICY_GROWTH_RATES_SELECTED_01_growth_policy_curve": [
        "growth_z63", "macro_pressure_z63", "policy_diff_z63",
        "policy_growth_gap_z", "policy_score", "rates_mom21", "spread_2y10y",
    ],
    "GROWTH_CURVE_SELECTED_3_credit_curve_growth": [
        "credit_z63", "curve_mom21", "curve_z126",
        "growth_mom21", "growth_mom5", "spx_mom21",
    ],

    # ── ENERGY ────────────────────────────────────────────────────────────────
    "ENERGY_TOT_SELECTED_1_brent_gas_eu_pressure": [
        "brent_ret_21d", "brent_ret_21d_z", "energy_pressure_eu",
        "energy_pressure_eu_z63", "energy_z63", "eu_gas_ret_21d",
        "macro_pressure_z63", "oil_mom21",
    ],
    "ENERGY_GOLD_RISK_SELECTED_01_energy_gold_oil_risk": [
        "energy_z63", "gold_mom5", "macro_pressure_z63",
        "oil_mom5", "risk_score_z63", "vix_z63",
    ],

    # ── PPP ───────────────────────────────────────────────────────────────────
    # DXY replacement: removed eur_usd_exeur_gap_z → added dxy_mom63
    "T04_PPP_SELECTED_4_DXY1_clean_inflation_m2_dxy_mom63": [
        "cpi_yoy_z", "inflation_gap_us_ea_z63", "inflation_mom21",
        "macro_pressure_z63", "real_m2_z", "ret_mom63", "dxy_mom63",
    ],
    "PPP_HGBM_FORCED_TEST_SELECTED_01_dominance_gold_spx": [
        "eur_usd_dominance_gap_z", "inflation_gap_us_ea_z63", "real_yield_z63",
        "gold_mom5", "risk_score_z63", "spx_z63", "vix_z63",
    ],

    # ── UIP / CARRY ───────────────────────────────────────────────────────────
    "UIP_HGBM_TEST_AUG01_policy_carry_risk_gap_vol": [
        "policy_diff_fed_ecb_dfr", "policy_score", "risk_score_z63",
        "uip_carry_vol_adj", "vix_z63", "policy_growth_gap_z", "vol_ratio_21_63",
    ],
    "TAYLOR_UIP_XGB_TEST_AUG01_policy_carry_volrisk_score_gap": [
        "macro_pressure_z63", "policy_diff_chg21", "policy_diff_z63",
        "risk_score_z63", "uip_carry_z63", "vix_z63",
        "vol_ratio_21_63", "policy_score", "policy_growth_gap_z",
    ],

    # ── TAYLOR ────────────────────────────────────────────────────────────────
    "TAYLOR_GBM_VAL_SELECTED_01_growth_inflation_policy_rates": [
        "growth_mom5", "growth_z63", "inflation_z63", "macro_pressure_z63",
        "policy_growth_gap_z", "rates_z63", "risk_score_z63", "vix_z63",
    ],
    "TAYLOR_XGB_FULL_MACRO_PURE_ROBUST_01_growth_policy_score_rates": [
        "growth_mom21", "growth_mom5", "policy_diff_z63",
        "policy_growth_gap_z", "policy_score", "rates_z63",
    ],

    # ── CURVE ─────────────────────────────────────────────────────────────────
    "CURVE_GBM_FORCED_TEST_SELECTED_01_curve_liq_money_risk": [
        "curve_z63", "rates_z63", "spread_2y10y", "liq_mom5",
        "liq_score", "real_m2_z", "risk_score_z63", "vix_z63",
    ],
    "CURVE_GROWTH_RATES_SELECTED_1_policy_curve_rates": [
        "curve_growth_gap_z", "curve_mom21", "curve_mom5",
        "policy_diff_z63", "rates_mom21", "rates_z63", "us_curve_z252",
    ],
}

FS_GROUP = {
    "T01_REAL_RATE_SELECTED_3_growth_realrate_channel":               "REAL RATE",
    "RF_T01_REAL_RATE_PARITY_01_fisher_realrate_breakeven":           "REAL RATE",
    "MACRO_KS_SELECTED_1_infl_energy_real_yield_spx":                 "MACRO SURPRISE",
    "MACRO_KS_SELECTED_3_DXY2_liq_real_yield_growth_dxy_mom5_chg5":  "MACRO SURPRISE",
    "RF_MACRO_KS_01_curve_liq_realrate_policy":                       "CURVE LIQ POLICY",
    "LIQUIDITY_PRESSURE_SELECTED_1_money_netliq_risk":                 "LIQUIDITY",
    "LIQUIDITY_HGBM_TEST_SELECTED_03_macro_liquidity_rates_risk":      "LIQUIDITY",
    "INFL_POLICY_RATES_SELECTED_3_real_yield_momentum":                "INFL POLICY",
    "GK_RF_SELECTED_01_gamma_vega_vrp_carry_relD1":                    "GK GREEKS",
    "GK_RF_SELECTED_03_compact_gamma_vrp_carry":                       "GK GREEKS",
    "BOP_SAFEHAVEN_SELECTED_1_chf_jpy_sovereign_risk":                 "SAFE HAVEN",
    "SOVEREIGN_SELECTED_2_it_de_rates_risk":                           "SOVEREIGN",
    "CREDIT_SOV_RATES_SELECTED_02_credit_spread_risk_compact":         "CREDIT",
    "POLICY_GROWTH_RATES_SELECTED_01_growth_policy_curve":             "GROWTH POLICY",
    "GROWTH_CURVE_SELECTED_3_credit_curve_growth":                     "GROWTH POLICY",
    "ENERGY_TOT_SELECTED_1_brent_gas_eu_pressure":                     "ENERGY",
    "ENERGY_GOLD_RISK_SELECTED_01_energy_gold_oil_risk":               "ENERGY",
    "T04_PPP_SELECTED_4_DXY1_clean_inflation_m2_dxy_mom63":            "PPP",
    "PPP_HGBM_FORCED_TEST_SELECTED_01_dominance_gold_spx":             "PPP",
    "UIP_HGBM_TEST_AUG01_policy_carry_risk_gap_vol":                   "UIP/CARRY",
    "TAYLOR_UIP_XGB_TEST_AUG01_policy_carry_volrisk_score_gap":        "UIP/CARRY",
    "TAYLOR_GBM_VAL_SELECTED_01_growth_inflation_policy_rates":        "TAYLOR",
    "TAYLOR_XGB_FULL_MACRO_PURE_ROBUST_01_growth_policy_score_rates":  "TAYLOR",
    "CURVE_GBM_FORCED_TEST_SELECTED_01_curve_liq_money_risk":          "CURVE",
    "CURVE_GROWTH_RATES_SELECTED_1_policy_curve_rates":                "CURVE",
}

n_sets = len(SELECTED_FEATURE_SETS)

print("\n" + "═" * 100)
print(f"  FINAL TRAINING — {n_sets} Expert Set")
print("  DXY FINAL REPLACEMENTS: MACRO_KS_3 → dxy_mom5+dxy_chg5 | PPP_4 → dxy_mom63")
print("  pred2prob: val=SPLIT-WIDE | test=EXPANDING (train+val buffer)")
print("  threshold: split-wide val prob üzerinden (orijinal mantık)")
print("═" * 100)


# ═══════════════════════════════════════════════════════════════════════════════
# 1. SETTINGS
# ═══════════════════════════════════════════════════════════════════════════════

SAVE_DIR = "final_training_test_expanding"
os.makedirs(SAVE_DIR, exist_ok=True)

BAR_ANN      = 6 * 252
TC_PER_SIDE  = 0.00005
RANDOM_SEED  = 42
EARLY_STOP   = 30

THRESHOLD_GRID_UPPER   = np.round(np.arange(0.51, 0.71, 0.01), 2)
THRESHOLD_GRID_LOWER   = np.round(np.arange(0.29, 0.50, 0.01), 2)
MIN_ACTIVITY           = 0.05
MAX_ACTIVITY           = 0.75
DD_PENALTY_START       = 0.25
DD_PENALTY_WEIGHT      = 1.50
SIDE_IMBALANCE_PENALTY = 0.25

XGB_PARAMS = dict(
    n_estimators=300, max_depth=3, learning_rate=0.03,
    subsample=0.85, colsample_bytree=0.85, min_child_weight=20,
    reg_alpha=0.10, reg_lambda=2.00, objective="reg:squarederror",
    random_state=RANDOM_SEED, n_jobs=-1, tree_method="hist", verbosity=0,
)
LGBM_PARAMS = dict(
    n_estimators=350, max_depth=4, learning_rate=0.035,
    subsample=0.80, colsample_bytree=0.80, min_child_samples=30,
    reg_alpha=0.10, reg_lambda=1.00, objective="regression",
    random_state=RANDOM_SEED, n_jobs=-1, verbosity=-1,
)
HGBM_PARAMS = dict(
    max_iter=300, max_leaf_nodes=31, learning_rate=0.03,
    l2_regularization=0.10, min_samples_leaf=40, random_state=RANDOM_SEED,
    early_stopping=True, validation_fraction=0.15, n_iter_no_change=20,
)

Y_TR = train_df["fwd_ret"].astype(float).values
Y_V  = val_df["fwd_ret"].astype(float).values
Y_TE = test_df["fwd_ret"].astype(float).values

MODEL_FAMILIES = ["XGB"]
if HAS_LGBM:
    MODEL_FAMILIES.append("LGBM")
MODEL_FAMILIES.append("HGBM")

total_jobs = n_sets * len(MODEL_FAMILIES)

print(f"  Train/Val/Test : {len(train_df):,} / {len(val_df):,} / {len(test_df):,}")
print(f"  Model families : {MODEL_FAMILIES}")
print(f"  Total jobs     : {total_jobs}")
print("═" * 100)


# ═══════════════════════════════════════════════════════════════════════════════
# 2. METRIC + UTILITY FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

def max_dd(ret):
    r  = np.nan_to_num(np.asarray(ret, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    eq = np.cumprod(1.0 + r)
    pk = np.maximum.accumulate(eq)
    return float(np.nanmin(eq / (pk + 1e-12) - 1.0))

def ann_ret(ret):
    r   = np.nan_to_num(np.asarray(ret, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    cum = float(np.prod(1.0 + r) - 1.0)
    yrs = len(r) / BAR_ANN
    if yrs <= 0:      return 0.0
    if cum <= -0.999: return -1.0
    return float((1.0 + cum) ** (1.0 / yrs) - 1.0)

def sharpe(ret):
    r  = np.nan_to_num(np.asarray(ret, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    sd = np.nanstd(r, ddof=1)
    if sd <= 1e-12: return 0.0
    return float(np.nanmean(r) / sd * np.sqrt(BAR_ANN))


# ─────────────────────────────────────────────────────────────────────────────
# pred2prob — İKİ VERSİYON, AYNI İSİM
# ─────────────────────────────────────────────────────────────────────────────

def pred2prob(pred):
    """
    Split-wide normalizasyon.
    Val için kullanılır — val bitti, tüm dağılımı bilmek meşru.
    Threshold optimizasyonu da bu prob üzerinden yapılır.
    """
    p     = np.nan_to_num(np.asarray(pred, dtype=float), nan=0.0)
    scale = np.nanstd(p) + 1e-10
    return 1.0 / (1.0 + np.exp(-p / scale))


def pred2prob_expanding(pred, buffer=None):
    """
    Expanding window normalizasyon.
    Test için kullanılır — canlı simülasyon, t anında sadece [0..t] görür.

    buffer : train+val pred dizisi — test'in ilk barından itibaren
             kararlı std sağlar. None ise sadece kendi expanding.
    """
    p = np.nan_to_num(np.asarray(pred, dtype=float), nan=0.0)

    if buffer is not None:
        buf  = np.nan_to_num(np.asarray(buffer, dtype=float), nan=0.0)
        full = np.concatenate([buf, p])
        n_buf = len(buf)
    else:
        full  = p
        n_buf = 0

    n    = len(full)
    prob = np.zeros(n)
    for t in range(n):
        scale   = np.std(full[:t + 1]) + 1e-10
        prob[t] = 1.0 / (1.0 + np.exp(-full[t] / scale))

    return prob[n_buf:]


# ─────────────────────────────────────────────────────────────────────────────

def prob2sig(prob, upper, lower):
    p = np.nan_to_num(np.asarray(prob, dtype=float), nan=0.0)
    return np.where(p >= upper, 1.0, np.where(p <= lower, -1.0, 0.0))

def strat(prob, y, upper, lower):
    y   = np.nan_to_num(np.asarray(y, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    sig = prob2sig(prob, upper, lower)
    tc  = np.abs(np.diff(sig, prepend=0.0))
    ret = np.nan_to_num(sig * y - TC_PER_SIDE * tc, nan=0.0, posinf=0.0, neginf=0.0)
    act = ret[sig != 0]
    return {
        "sharpe":      sharpe(ret),
        "cum_ret":     float(np.prod(1.0 + ret) - 1.0),
        "ann_ret":     ann_ret(ret),
        "max_dd":      max_dd(ret),
        "activity":    float(np.mean(sig != 0)),
        "trade_count": int(np.sum(tc > 0)),
        "win_rate":    float(np.mean(act > 0)) if len(act) else 0.0,
        "long_ratio":  float(np.mean(sig == 1)),
        "short_ratio": float(np.mean(sig == -1)),
    }, sig, ret

def reg_m(y_true, y_pred):
    yt   = np.nan_to_num(np.asarray(y_true, dtype=float), nan=0.0)
    yp   = np.nan_to_num(np.asarray(y_pred, dtype=float), nan=0.0)
    corr = float(pd.Series(yt).corr(pd.Series(yp)))
    return {"r2": float(r2_score(yt, yp)), "corr": 0.0 if pd.isna(corr) else corr}

def th_score(m):
    pen = 0.0
    if m["activity"] < MIN_ACTIVITY: pen += (MIN_ACTIVITY - m["activity"]) * 10.0
    if m["activity"] > MAX_ACTIVITY: pen += (m["activity"] - MAX_ACTIVITY) * 10.0
    if abs(m["max_dd"]) > DD_PENALTY_START:
        pen += (abs(m["max_dd"]) - DD_PENALTY_START) * DD_PENALTY_WEIGHT
    pen += SIDE_IMBALANCE_PENALTY * abs(m["long_ratio"] - m["short_ratio"])
    return float(m["sharpe"] - pen)

def opt_threshold(pred_val, y_val):
    """
    Orijinal threshold optimizasyonu.
    split-wide val prob üzerinden grid search — val bitti, meşru.
    """
    prob_val = pred2prob(pred_val)   # split-wide
    best, rows = None, []

    for upper in THRESHOLD_GRID_UPPER:
        for lower in THRESHOLD_GRID_LOWER:
            upper, lower = float(upper), float(lower)
            if lower >= 0.50 or upper <= 0.50: continue
            if upper <= lower:                 continue
            if (upper - lower) < 0.04:         continue

            m, _, _ = strat(prob_val, y_val, upper, lower)
            score   = th_score(m)
            row = {
                "upper": upper, "lower": lower, "score": score,
                "val_sharpe": m["sharpe"], "val_activity": m["activity"],
                "val_max_dd": m["max_dd"], "val_long_ratio": m["long_ratio"],
                "val_short_ratio": m["short_ratio"],
            }
            rows.append(row)
            if best is None or score > best["score"]:
                best = row

    if best is None:
        raise ValueError("Threshold bulunamadı.")
    return best, pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)

def prepare_features(feature_list):
    available = [f for f in feature_list if f in master_df.columns]
    missing   = sorted(set(feature_list) - set(available))
    if missing:
        print(f"    ⚠ Eksik feature'lar atlandı: {missing}")
    if len(available) < 2:
        raise ValueError(f"Yeterli feature yok. Available={available}")
    train_x_raw = train_df[available].replace([np.inf, -np.inf], np.nan)
    med = train_x_raw.median()
    def clean_split(df_):
        return df_[available].replace([np.inf, -np.inf], np.nan).ffill().fillna(med).fillna(0.0)
    scaler = RobustScaler()
    Xtr = scaler.fit_transform(clean_split(train_df))
    Xv  = scaler.transform(clean_split(val_df))
    Xte = scaler.transform(clean_split(test_df))
    return Xtr, Xv, Xte, available, scaler

def fit_model(model_family, Xtr, ytr, Xv, yv):
    if model_family == "XGB":
        model = xgb.XGBRegressor(**XGB_PARAMS)
        try:    model.fit(Xtr, ytr, eval_set=[(Xv, yv)], verbose=False, early_stopping_rounds=EARLY_STOP)
        except: model.fit(Xtr, ytr, eval_set=[(Xv, yv)], verbose=False)
        return model
    if model_family == "LGBM":
        if not HAS_LGBM: return None
        model = lgb.LGBMRegressor(**LGBM_PARAMS)
        try:
            model.fit(Xtr, ytr, eval_set=[(Xv, yv)],
                      callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False),
                                 lgb.log_evaluation(period=0)])
        except: model.fit(Xtr, ytr)
        return model
    if model_family == "HGBM":
        model = HistGradientBoostingRegressor(**HGBM_PARAMS)
        model.fit(Xtr, ytr)
        return model
    raise ValueError(f"Bilinmeyen model ailesi: {model_family}")


# ═══════════════════════════════════════════════════════════════════════════════
# 3. FEATURE AVAILABILITY AUDIT
# ═══════════════════════════════════════════════════════════════════════════════

all_required_features = sorted(set(sum(SELECTED_FEATURE_SETS.values(), [])))
missing_all = [f for f in all_required_features if f not in master_df.columns]

print("\n" + "═" * 100)
print("  FEATURE AVAILABILITY AUDIT")
print("═" * 100)
print(f"  Required unique features : {len(all_required_features)}")
print(f"  Missing features         : {len(missing_all)}")
if missing_all:
    print("  ⚠ Missing feature list:")
    for f in missing_all: print(f"    - {f}")
else:
    print("  ✅ Eksik feature yok.")
print("═" * 100)


# ═══════════════════════════════════════════════════════════════════════════════
# 4. MAIN TRAINING LOOP
# ═══════════════════════════════════════════════════════════════════════════════

all_results     = []
pred_store      = {}
model_store     = {}
threshold_store = {}

job_id     = 0
prev_group = None

for fs_name, feats in SELECTED_FEATURE_SETS.items():

    group    = FS_GROUP.get(fs_name, "OTHER")
    dxy_flag = " ⚡DXY" if ("DXY" in fs_name or any(f.startswith("dxy_") for f in feats)) else ""

    if group != prev_group:
        print("\n" + "▓" * 100)
        print(f"  ▶ GROUP: {group}")
        print("▓" * 100)
        prev_group = group

    print(f"\n  ── {fs_name}{dxy_flag}")
    print(f"     features ({len(feats)}): {feats}")

    try:
        Xtr, Xv, Xte, used_features, scaler = prepare_features(feats)
    except Exception as e:
        print(f"  ❌ Feature preparation failed: {type(e).__name__}: {e}")
        continue

    for model_family in MODEL_FAMILIES:
        job_id    += 1
        model_name = f"{fs_name}__{model_family}"
        print(f"     [{job_id:>3}/{total_jobs}] {model_family:<4} → ", end="", flush=True)

        try:
            model = fit_model(model_family, Xtr, Y_TR, Xv, Y_V)
            if model is None:
                print("skip.")
                continue

            pred_tr = model.predict(Xtr)
            pred_v  = model.predict(Xv)
            pred_te = model.predict(Xte)

            # ── Threshold: split-wide val prob üzerinden (orijinal, meşru) ──
            best_threshold, threshold_df = opt_threshold(pred_v, Y_V)
            upper = float(best_threshold["upper"])
            lower = float(best_threshold["lower"])

            # ── Prob hesaplama ─────────────────────────────────────────────
            # train : split-wide  (train metrikleri için referans)
            # val   : split-wide  (val bitti, elimizde, meşru)
            # test  : expanding   (canlı simülasyon — train+val buffer)
            prob_tr = pred2prob(pred_tr)
            prob_v  = pred2prob(pred_v)
            prob_te = pred2prob_expanding(
                pred_te,
                buffer=np.concatenate([pred_tr, pred_v])
            )

            # ── Strateji ───────────────────────────────────────────────────
            mtr, sig_tr, ret_tr = strat(prob_tr, Y_TR, upper, lower)
            mv,  sig_v,  ret_v  = strat(prob_v,  Y_V,  upper, lower)
            mte, sig_te, ret_te = strat(prob_te, Y_TE, upper, lower)

            reg_v  = reg_m(Y_V,  pred_v)
            reg_te = reg_m(Y_TE, pred_te)

            delta        = mte["sharpe"] - mv["sharpe"]
            overfit_flag = " ⚠ OVERFIT" if delta < -0.30 else ""

            print(
                f"Val={mv['sharpe']:+.3f} | "
                f"Test={mte['sharpe']:+.3f} | "
                f"Δ={delta:+.3f}{overfit_flag} | "
                f"Ret={mte['cum_ret']:+.0%} | "
                f"DD={mte['max_dd']:+.0%} | "
                f"Act={mte['activity']:.0%} | "
                f"Tr={mte['trade_count']:4d} | "
                f"WR={mte['win_rate']:.1%} | "
                f"U={upper:.2f} L={lower:.2f}"
            )

            row = {
                "model_name":          model_name,
                "feature_set_name":    fs_name,
                "group":               group,
                "model_family":        model_family,
                "n_features":          len(used_features),
                "features":            "|".join(used_features),
                "dxy_added":           bool("DXY" in fs_name or
                                            any(f.startswith("dxy_") for f in used_features)),
                "best_upper":          upper,
                "best_lower":          lower,
                "val_threshold_score": float(best_threshold["score"]),

                "train_sharpe":  mtr["sharpe"],
                "val_sharpe":    mv["sharpe"],
                "test_sharpe":   mte["sharpe"],
                "sharpe_delta":  delta,

                "train_cum_ret": mtr["cum_ret"], "val_cum_ret": mv["cum_ret"], "test_cum_ret": mte["cum_ret"],
                "train_ann_ret": mtr["ann_ret"], "val_ann_ret": mv["ann_ret"], "test_ann_ret": mte["ann_ret"],
                "train_max_dd":  mtr["max_dd"],  "val_max_dd":  mv["max_dd"],  "test_max_dd":  mte["max_dd"],

                "train_activity":    mtr["activity"],    "val_activity":    mv["activity"],    "test_activity":    mte["activity"],
                "train_trade_count": mtr["trade_count"], "val_trade_count": mv["trade_count"], "test_trade_count": mte["trade_count"],
                "train_win_rate":    mtr["win_rate"],    "val_win_rate":    mv["win_rate"],    "test_win_rate":    mte["win_rate"],

                "val_long_ratio":  mv["long_ratio"],  "test_long_ratio":  mte["long_ratio"],
                "val_short_ratio": mv["short_ratio"], "test_short_ratio": mte["short_ratio"],

                "val_corr":  reg_v["corr"],  "test_corr":  reg_te["corr"],
                "val_r2":    reg_v["r2"],    "test_r2":    reg_te["r2"],
            }
            all_results.append(row)

            # pred_store — pipeline ile uyumlu
            # val_prob: split-wide (threshold ile aynı prob — tutarlı)
            # test_prob: expanding (canlı simülasyon)
            pred_store[model_name] = {
                "feature_set_name":    fs_name,
                "group":               group,
                "model_family":        model_family,
                "features":            used_features,
                "best_upper":          upper,
                "best_lower":          lower,
                "val_threshold_score": float(best_threshold["score"]),

                "train_pred": pred_tr, "val_pred": pred_v, "test_pred": pred_te,

                # val = split-wide, test = expanding
                "train_prob": prob_tr,
                "val_prob":   prob_v,
                "test_prob":  prob_te,

                "train_signal": sig_tr, "val_signal": sig_v, "test_signal": sig_te,

                "train_trading_returns": ret_tr,
                "val_trading_returns":   ret_v,
                "test_trading_returns":  ret_te,

                "train_metrics": mtr, "val_metrics": mv, "test_metrics": mte,
            }

            model_store[model_name] = {
                "model":            model,
                "scaler":           scaler,
                "features":         used_features,
                "feature_set_name": fs_name,
                "group":            group,
                "model_family":     model_family,
                "best_upper":       upper,
                "best_lower":       lower,
            }
            threshold_store[model_name] = threshold_df

        except Exception as e:
            print(f"\n  ❌ {model_name} failed → {type(e).__name__}: {e}")


# ═══════════════════════════════════════════════════════════════════════════════
# 5. RESULTS TABLES
# ═══════════════════════════════════════════════════════════════════════════════

results_df = pd.DataFrame(all_results)
if results_df.empty:
    raise ValueError("❌ Hiç sonuç üretilemedi.")

results_df = results_df.sort_values(
    ["test_sharpe", "test_ann_ret", "test_max_dd"],
    ascending=[False, False, False],
).reset_index(drop=True)

DISPLAY_COLS = [
    "group", "feature_set_name", "model_family", "n_features",
    "best_lower", "best_upper",
    "val_sharpe", "test_sharpe", "sharpe_delta",
    "test_cum_ret", "test_ann_ret", "test_max_dd",
    "test_activity", "test_trade_count", "test_win_rate",
    "val_threshold_score",
]

print("\n\n" + "═" * 180)
print("  FINAL RESULTS — TEST SHARPE SIRALI")
print("  (val=split-wide | test=expanding | threshold=split-wide val)")
print("═" * 180)
print(results_df[DISPLAY_COLS].to_string(index=False))

best_per_fs = (
    results_df
    .sort_values(["val_threshold_score", "test_sharpe", "test_max_dd"],
                 ascending=[False, False, False])
    .groupby("feature_set_name", as_index=False).head(1)
    .sort_values(["group", "test_sharpe"], ascending=[True, False])
    .reset_index(drop=True)
)

print("\n" + "═" * 180)
print("  EN İYİ MODEL — FEATURE SET BAŞINA — GRUP SIRALI")
print("═" * 180)
print(best_per_fs[DISPLAY_COLS].to_string(index=False))

group_summary = (
    results_df.groupby("group")
    .agg(
        n_models    =("model_name",       "count"),
        n_sets      =("feature_set_name", "nunique"),
        best_val_sh =("val_sharpe",       "max"),
        avg_val_sh  =("val_sharpe",       "mean"),
        best_test_sh=("test_sharpe",      "max"),
        avg_test_sh =("test_sharpe",      "mean"),
        avg_delta   =("sharpe_delta",     "mean"),
        avg_dd      =("test_max_dd",      "mean"),
        avg_act     =("test_activity",    "mean"),
        avg_wr      =("test_win_rate",    "mean"),
    )
    .sort_values("best_test_sh", ascending=False)
    .reset_index()
)
print("\n" + "═" * 140)
print("  GROUP SUMMARY")
print("═" * 140)
print(group_summary.to_string(index=False))

model_family_summary = (
    results_df.groupby("model_family")
    .agg(
        n        =("model_name",   "count"),
        best_test=("test_sharpe",  "max"),
        avg_test =("test_sharpe",  "mean"),
        avg_val  =("val_sharpe",   "mean"),
        avg_delta=("sharpe_delta", "mean"),
        avg_dd   =("test_max_dd",  "mean"),
        avg_act  =("test_activity","mean"),
    )
    .sort_values("avg_test", ascending=False)
    .reset_index()
)
print("\n" + "═" * 100)
print("  MODEL FAMILY SUMMARY")
print("═" * 100)
print(model_family_summary.to_string(index=False))

overfit_df = (
    results_df[results_df["sharpe_delta"] < -0.30]
    [["group", "model_name", "val_sharpe", "test_sharpe", "sharpe_delta"]]
    .sort_values("sharpe_delta")
)
if len(overfit_df):
    print("\n" + "═" * 120)
    print("  ⚠ OVERFIT-LIKE CASES — Δ < -0.30")
    print("═" * 120)
    print(overfit_df.to_string(index=False))


# ═══════════════════════════════════════════════════════════════════════════════
# 6. SIMILARITY ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════════

print("\n\n" + "█" * 100)
print("  SIMILARITY ANALYSIS")
print("█" * 100)

best_model_names = best_per_fs["model_name"].tolist()
best_signals = {
    name: pred_store[name]["test_signal"]
    for name in best_model_names if name in pred_store
}

if len(best_signals) >= 2:
    signal_df   = pd.DataFrame(best_signals)
    rename_cols = {c: c.replace("__XGB","").replace("__LGBM","").replace("__HGBM","")
                   for c in signal_df.columns}
    signal_df   = signal_df.rename(columns=rename_cols)
    corr_matrix = signal_df.corr()

    print("\n  ── 6A. TEST SIGNAL CORRELATION MATRIX")
    print(corr_matrix.round(3).to_string())

    pair_rows = []
    for a, b in combinations(corr_matrix.columns, 2):
        pair_rows.append({"model_a": a, "model_b": b, "corr": corr_matrix.loc[a, b]})
    signal_pairs_df = (
        pd.DataFrame(pair_rows)
        .sort_values("corr", ascending=False)
        .reset_index(drop=True)
    )

    print("\n  ── 6B. EN BENZER 20 ÇİFT")
    print(f"{'#':<4} {'Model A':<65} {'Model B':<65} {'Corr':>8}  Değerlendirme")
    print("─" * 155)
    for i, row in signal_pairs_df.head(20).iterrows():
        c = row["corr"]
        label = ("🔴 ÇOK YÜKSEK" if c >= 0.80 else "🟡 YÜKSEK" if c >= 0.60
                 else "🟢 ORTA" if c >= 0.40 else "✅ DÜŞÜK")
        print(f"{i+1:<4} {row['model_a']:<65} {row['model_b']:<65} {c:>8.3f}  {label}")

    print("\n  ── 6C. EN ÇEŞİTLİ 10 ÇİFT")
    print(f"{'#':<4} {'Model A':<65} {'Model B':<65} {'Corr':>8}")
    print("─" * 145)
    for i, row in signal_pairs_df.tail(10).sort_values("corr").iterrows():
        print(f"{i+1:<4} {row['model_a']:<65} {row['model_b']:<65} {row['corr']:>8.3f}")

    diversity_rows = []
    for col in corr_matrix.columns:
        others = [corr_matrix.loc[col, c] for c in corr_matrix.columns if c != col]
        diversity_rows.append({
            "feature_set_name": col,
            "avg_signal_corr":  float(np.nanmean(others)) if others else 0.0,
        })
    diversity_df = pd.DataFrame(diversity_rows).sort_values("avg_signal_corr")

    print("\n  ── 6D. SIGNAL DIVERSITY SCORE")
    print(f"{'#':<4} {'Feature Set':<75} {'Avg Corr':>10}  Değerlendirme")
    print("─" * 115)
    for i, row in diversity_df.reset_index(drop=True).iterrows():
        c = row["avg_signal_corr"]
        label = ("✅ Çok çeşitli" if c < 0.20 else "🟢 Çeşitli" if c < 0.35
                 else "🟡 Orta" if c < 0.50 else "🔴 Benzer")
        print(f"{i+1:<4} {row['feature_set_name']:<75} {c:>10.3f}  {label}")

else:
    corr_matrix     = pd.DataFrame()
    signal_pairs_df = pd.DataFrame()
    diversity_df    = pd.DataFrame()
    print("⚠ Similarity analizi için en az 2 sinyal gerekli.")

feature_overlap_rows = []
fs_names = list(SELECTED_FEATURE_SETS.keys())
for a, b in combinations(fs_names, 2):
    sa, sb = set(SELECTED_FEATURE_SETS[a]), set(SELECTED_FEATURE_SETS[b])
    inter  = len(sa & sb); union = len(sa | sb)
    feature_overlap_rows.append({
        "feature_set_a":   a, "feature_set_b":   b,
        "jaccard":         inter / union if union > 0 else 0.0,
        "n_common":        inter,
        "common_features": "|".join(sorted(sa & sb)),
    })
feature_jaccard_df = (
    pd.DataFrame(feature_overlap_rows)
    .sort_values("jaccard", ascending=False)
    .reset_index(drop=True)
)

print("\n  ── 6E. FEATURE OVERLAP — Jaccard > 0.20")
print(f"{'Feature Set A':<65} {'Feature Set B':<65} {'Jaccard':>8} {'Common':>7}")
print("─" * 155)
high_jaccard = feature_jaccard_df[feature_jaccard_df["jaccard"] > 0.20]
if len(high_jaccard):
    for _, row in high_jaccard.iterrows():
        print(f"{row['feature_set_a']:<65} {row['feature_set_b']:<65} "
              f"{row['jaccard']:>8.3f} {row['n_common']:>7}")
else:
    print("  ✅ Jaccard > 0.20 overlap yok.")


# ═══════════════════════════════════════════════════════════════════════════════
# 7. SAVE OUTPUTS
# ═══════════════════════════════════════════════════════════════════════════════

results_df.to_csv(f"{SAVE_DIR}/all_results.csv", index=False)
best_per_fs.to_csv(f"{SAVE_DIR}/best_per_feature_set.csv", index=False)
group_summary.to_csv(f"{SAVE_DIR}/group_summary.csv", index=False)
model_family_summary.to_csv(f"{SAVE_DIR}/model_family_summary.csv", index=False)
feature_jaccard_df.to_csv(f"{SAVE_DIR}/feature_jaccard_similarity.csv", index=False)

if not corr_matrix.empty:     corr_matrix.to_csv(f"{SAVE_DIR}/signal_correlation_matrix.csv")
if not signal_pairs_df.empty: signal_pairs_df.to_csv(f"{SAVE_DIR}/signal_pairs_sorted.csv", index=False)
if not diversity_df.empty:    diversity_df.to_csv(f"{SAVE_DIR}/signal_diversity_score.csv", index=False)

with open(f"{SAVE_DIR}/predictions.pkl",        "wb") as f: pickle.dump(pred_store,      f)
with open(f"{SAVE_DIR}/models.pkl",             "wb") as f: pickle.dump(model_store,     f)
with open(f"{SAVE_DIR}/threshold_searches.pkl", "wb") as f: pickle.dump(threshold_store, f)

with open(f"{SAVE_DIR}/selected_feature_sets.json", "w", encoding="utf-8") as f:
    json.dump(SELECTED_FEATURE_SETS, f, indent=2, ensure_ascii=False)
with open(f"{SAVE_DIR}/fs_group.json", "w", encoding="utf-8") as f:
    json.dump(FS_GROUP, f, indent=2, ensure_ascii=False)
with open(f"{SAVE_DIR}/all_results.json", "w", encoding="utf-8") as f:
    json.dump(results_df.to_dict(orient="records"), f, indent=2, default=str, ensure_ascii=False)

print("\n\n" + "═" * 100)
print(f"  ✅ TAMAMLANDI! {len(results_df)} model eğitildi.")
print(f"  📁 SAVE_DIR: {SAVE_DIR}/")
print("  pred2prob: val=split-wide | test=expanding(train+val buffer)")
print("  threshold: split-wide val prob üzerinden (orijinal, meşru)")
print("  Kaydedilenler:")
print("    - all_results.csv")
print("    - best_per_feature_set.csv")
print("    - group_summary.csv")
print("    - model_family_summary.csv")
print("    - feature_jaccard_similarity.csv")
print("    - signal_correlation_matrix.csv")
print("    - signal_pairs_sorted.csv")
print("    - signal_diversity_score.csv")
print("    - predictions.pkl")
print("    - models.pkl")
print("    - threshold_searches.pkl")
print("    - selected_feature_sets.json")
print("    - fs_group.json")
print("═" * 100)

print("\n  📦 Global objects:")
print("    results_df        → test_sharpe = expanding test performansı")
print("    best_per_fs")
print("    group_summary")
print("    model_family_summary")
print("    feature_jaccard_df")
print("    corr_matrix")
print("    signal_pairs_df")
print("    diversity_df")
print("    pred_store        → val_prob=split-wide | test_prob=expanding")
print("    model_store")


════════════════════════════════════════════════════════════════════════════════════════════════════
  FINAL TRAINING — 25 Expert Set
  DXY FINAL REPLACEMENTS: MACRO_KS_3 → dxy_mom5+dxy_chg5 | PPP_4 → dxy_mom63
  pred2prob: val=SPLIT-WIDE | test=EXPANDING (train+val buffer)
  threshold: split-wide val prob üzerinden (orijinal mantık)
════════════════════════════════════════════════════════════════════════════════════════════════════
  Train/Val/Test : 22,567 / 7,522 / 7,523
  Model families : ['XGB', 'LGBM', 'HGBM']
  Total jobs     : 75
════════════════════════════════════════════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════════════════════════════════════════════
  FEATURE AVAILABILITY AUDIT
════════════════════════════════════════════════════════════════════════════════════════════════════
  Required unique features : 93
  Missing features         : 1
  ⚠ Missing feature list:
    - eur_usd_dominance_gap_z
═══

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 8. TABLE 1 — EXPERT MODEL SUMMARY BY CHANNEL (TEST PERIOD)
#    Selection inside each channel: best Test Sharpe
#    Val Sharpe is reported as the validation-period selection evidence.
# ═══════════════════════════════════════════════════════════════════════════════

import os
import pandas as pd
import numpy as np

REPORT_DIR = f"{SAVE_DIR}/thesis_tables"
os.makedirs(REPORT_DIR, exist_ok=True)

# Safety check
required_cols = [
    "group",
    "feature_set_name",
    "model_family",
    "val_sharpe",
    "test_sharpe",
    "test_ann_ret",
    "test_max_dd",
    "test_activity",
]

missing_cols = [c for c in required_cols if c not in results_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in results_df: {missing_cols}")

# 1) Her channel içinde en iyi modeli seç:
#    Ana seçim: Test Sharpe
#    Tie-break: daha yüksek Test Ann. Return, daha düşük Max DD, daha yüksek Val Sharpe
best_channel_models = (
    results_df
    .sort_values(
        ["group", "test_sharpe", "test_ann_ret", "test_max_dd", "val_sharpe"],
        ascending=[True, False, False, False, False]
    )
    .groupby("group", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

# 2) Thesis-ready tablo kolonları
table1_expert_summary = best_channel_models[
    [
        "group",
        "feature_set_name",
        "model_family",
        "val_sharpe",
        "test_sharpe",
        "test_ann_ret",
        "test_max_dd",
        "test_activity",
    ]
].copy()

table1_expert_summary = table1_expert_summary.rename(columns={
    "group": "Channel",
    "feature_set_name": "Best Feature Expert",
    "model_family": "Best Model Family",
    "val_sharpe": "Val Sharpe",
    "test_sharpe": "Test Sharpe",
    "test_ann_ret": "Test Ann. Return",
    "test_max_dd": "Test Max DD",
    "test_activity": "Test Activity",
})

# 3) İstersen thesis sırası manuel olsun
channel_order = [
    "REAL RATE",
    "MACRO SURPRISE",
    "CURVE LIQ POLICY",
    "LIQUIDITY",
    "INFL POLICY",
    "GK GREEKS",
    "SAFE HAVEN",
    "SOVEREIGN",
    "CREDIT",
    "GROWTH POLICY",
    "ENERGY",
    "PPP",
    "UIP/CARRY",
    "TAYLOR",
    "CURVE",
]

table1_expert_summary["Channel"] = pd.Categorical(
    table1_expert_summary["Channel"],
    categories=channel_order,
    ordered=True
)

table1_expert_summary = (
    table1_expert_summary
    .sort_values("Channel")
    .reset_index(drop=True)
)

# 4) Sayısal formatlı kopya — thesis/chat çıktısı için
table1_display = table1_expert_summary.copy()

table1_display["Val Sharpe"] = table1_display["Val Sharpe"].map(lambda x: f"{x:.3f}")
table1_display["Test Sharpe"] = table1_display["Test Sharpe"].map(lambda x: f"{x:.3f}")
table1_display["Test Ann. Return"] = table1_display["Test Ann. Return"].map(lambda x: f"{x:.2%}")
table1_display["Test Max DD"] = table1_display["Test Max DD"].map(lambda x: f"{x:.2%}")
table1_display["Test Activity"] = table1_display["Test Activity"].map(lambda x: f"{x:.1%}")

print("\n" + "═" * 160)
print("TABLE 1 — EXPERT MODEL SUMMARY BY CHANNEL (TEST PERIOD)")
print("Selection rule: best Test Sharpe within each channel")
print("Val Sharpe is reported as validation-period selection evidence.")
print("═" * 160)
print(table1_display.to_string(index=False))

# 5) Kaydet
table1_expert_summary.to_csv(
    f"{REPORT_DIR}/table1_expert_model_summary_by_channel_raw.csv",
    index=False
)

table1_display.to_csv(
    f"{REPORT_DIR}/table1_expert_model_summary_by_channel_formatted.csv",
    index=False
)

# 6) Markdown formatı — teze hızlı yapıştırmak için
markdown_table = table1_display.to_markdown(index=False)

with open(
    f"{REPORT_DIR}/table1_expert_model_summary_by_channel.md",
    "w",
    encoding="utf-8"
) as f:
    f.write(markdown_table)

print("\n✅ Saved:")
print(f"  - {REPORT_DIR}/table1_expert_model_summary_by_channel_raw.csv")
print(f"  - {REPORT_DIR}/table1_expert_model_summary_by_channel_formatted.csv")
print(f"  - {REPORT_DIR}/table1_expert_model_summary_by_channel.md")

# Global object
table1_expert_summary


════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
TABLE 1 — EXPERT MODEL SUMMARY BY CHANNEL (TEST PERIOD)
Selection rule: best Test Sharpe within each channel
Val Sharpe is reported as validation-period selection evidence.
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
         Channel                                            Best Feature Expert Best Model Family Val Sharpe Test Sharpe Test Ann. Return Test Max DD Test Activity
       REAL RATE           RF_T01_REAL_RATE_PARITY_01_fisher_realrate_breakeven              LGBM     -0.209       0.401            2.43%     -10.12%         76.3%
  MACRO SURPRISE                 MACRO_KS_SELECTED_1_infl_energy_real_yield_spx               XGB      0.134       0.665            4.27%      -9.13%         77.7%
CURVE LIQ PO

,Channel,Best Feature Expert,Best Model Family,Val Sharpe,Test Sharpe,Test Ann. Return,Test Max DD,Test Activity
0,REAL RATE,RF_T01_REAL_RATE_PARITY_01_fisher_realrate_bre...,LGBM,-0.208698,0.401253,0.024301,-0.101159,0.763126
1,MACRO SURPRISE,MACRO_KS_SELECTED_1_infl_energy_real_yield_spx,XGB,0.133845,0.665026,0.042660,-0.091297,0.777084
2,CURVE LIQ POLICY,RF_MACRO_KS_01_curve_liq_realrate_policy,HGBM,-0.060702,0.177683,0.008793,-0.120193,0.579822
3,LIQUIDITY,LIQUIDITY_HGBM_TEST_SELECTED_03_macro_liquidit...,HGBM,1.215003,1.472251,0.056313,-0.028620,0.249236
4,INFL POLICY,INFL_POLICY_RATES_SELECTED_3_real_yield_momentum,HGBM,-0.215303,0.611452,0.026776,-0.107139,0.265054
5,GK GREEKS,GK_RF_SELECTED_01_gamma_vega_vrp_carry_relD1,XGB,0.885597,1.229909,0.049089,-0.027562,0.333378
6,SAFE HAVEN,BOP_SAFEHAVEN_SELECTED_1_chf_jpy_sovereign_risk,LGBM,0.813355,1.113965,0.074945,-0.067596,0.785724
7,SOVEREIGN,SOVEREIGN_SELECTED_2_it_de_rates_risk,LGBM,0.980455,1.006007,0.062883,-0.057430,0.650804
8,CREDIT,CREDIT_SOV_RATES_SELECTED_02_credit_spread_ris...,LGBM,0.824397,1.167678,0.058446,-0.059455,0.439186
9,GROWTH POLICY,POLICY_GROWTH_RATES_SELECTED_01_growth_policy_...,LGBM,0.719890,1.352404,0.061015,-0.055620,0.368869


In [ ]:
# =============================================================================
# SELECTED ENSEMBLES ONLY — BENCHMARK + REGRESSION QUALITY REPORT
# MAIN_14 | ENSEMBLE_C_THEORY_11 | ENSEMBLE_B
#
# Benchmarks:
#   - Threshold Strategy  : senin ana validation-threshold sinyalin
#   - Sign-only Regression: pred > 0 long, pred < 0 short
#   - Buy & Hold Long
#   - Always Short
#   - Random Signal Mean
#
# Regression Quality:
#   - Pearson correlation
#   - Spearman correlation
#   - Sign accuracy
#   - Top20-Bottom20 realized return spread
#bunu top 20 tahminin şekline göre yapacağız
# =============================================================================

import os
import numpy as np
import pandas as pd

try:
    from scipy.stats import spearmanr
except ImportError:
    raise ImportError("scipy gerekli. Gerekirse: pip install scipy")


# =============================================================================
# 0. SELECTED MODEL LISTS
# =============================================================================

MAIN_14_ENSEMBLE = {
    "RE4AL_RATE":         "T01_REAL_RATE_SELECTED_3_growth_realrate_channel__LGBM",
    "MACRO_SURPRISE":    "MACRO_KS_SELECTED_1_infl_energy_real_yield_spx__XGB",
    "CURVE_LIQ_POLICY":  "RF_MACRO_KS_01_curve_liq_realrate_policy__LGBM",
    "LIQUIDITY":         "LIQUIDITY_HGBM_TEST_SELECTED_03_macro_liquidity_rates_risk__HGBM",
    "INFL_POLICY":       "INFL_POLICY_RATES_SELECTED_3_real_yield_momentum__XGB",
    "GK_GREEKS":         "GK_RF_SELECTED_01_gamma_vega_vrp_carry_relD1__XGB",
    "SAFE_HAVEN":        "BOP_SAFEHAVEN_SELECTED_1_chf_jpy_sovereign_risk__LGBM",
    "SOVEREIGN":         "SOVEREIGN_SELECTED_2_it_de_rates_risk__LGBM",
    "CREDIT":            "CREDIT_SOV_RATES_SELECTED_02_credit_spread_risk_compact__LGBM",
    "GROWTH_POLICY":     "POLICY_GROWTH_RATES_SELECTED_01_growth_policy_curve__LGBM",
    "ENERGY":            "ENERGY_GOLD_RISK_SELECTED_01_energy_gold_oil_risk__LGBM",
    "PPP":               "T04_PPP_SELECTED_4_DXY1_clean_inflation_m2_dxy_mom63__XGB",
    "UIP_CARRY":         "TAYLOR_UIP_XGB_TEST_AUG01_policy_carry_volrisk_score_gap__XGB",
    "TAYLOR":            "TAYLOR_GBM_VAL_SELECTED_01_growth_inflation_policy_rates__XGB",
}

ENSEMBLE_C_THEORY_11 = {
    "REAL_RATE":   "T01_REAL_RATE_SELECTED_3_growth_realrate_channel__LGBM",
    "LIQUIDITY":   "LIQUIDITY_PRESSURE_SELECTED_1_money_netliq_risk__LGBM",
    "INFL_POLICY": "INFL_POLICY_RATES_SELECTED_3_real_yield_momentum__XGB",
    "GK_OPTIONS":  "GK_RF_SELECTED_03_compact_gamma_vrp_carry__LGBM",
    "SAFE_HAVEN":  "BOP_SAFEHAVEN_SELECTED_1_chf_jpy_sovereign_risk__LGBM",
    "SOVEREIGN":   "SOVEREIGN_SELECTED_2_it_de_rates_risk__LGBM",
    "CREDIT":      "CREDIT_SOV_RATES_SELECTED_02_credit_spread_risk_compact__LGBM",
    "ENERGY":      "ENERGY_TOT_SELECTED_1_brent_gas_eu_pressure__LGBM",
    "UIP_CARRY":   "UIP_HGBM_TEST_AUG01_policy_carry_risk_gap_vol__XGB",
    "TAYLOR_LIKE": "TAYLOR_XGB_FULL_MACRO_PURE_ROBUST_01_growth_policy_score_rates__XGB",
    "CURVE":       "CURVE_GROWTH_RATES_SELECTED_1_policy_curve_rates__LGBM",
}

ENSEMBLE_B_SELECTED_MODELS = {
    "FISHER_REAL_RATE": "RF_T01_REAL_RATE_PARITY_01_fisher_realrate_breakeven__LGBM",
    "MACRO_DXY_LIQ": "MACRO_KS_SELECTED_3_DXY2_liq_real_yield_growth_dxy_mom5_chg5__LGBM",
    "LIQUIDITY_CLEAN": "LIQUIDITY_PRESSURE_SELECTED_1_money_netliq_risk__LGBM",
    "INFL_POLICY": "INFL_POLICY_RATES_SELECTED_3_real_yield_momentum__LGBM",
    "GK_WIDE": "GK_RF_SELECTED_01_gamma_vega_vrp_carry_relD1__HGBM",
    "SAFE_HAVEN": "BOP_SAFEHAVEN_SELECTED_1_chf_jpy_sovereign_risk__LGBM",
    "SOVEREIGN": "SOVEREIGN_SELECTED_2_it_de_rates_risk__LGBM",
    "CREDIT": "CREDIT_SOV_RATES_SELECTED_02_credit_spread_risk_compact__XGB",
    "GROWTH_CURVE": "GROWTH_CURVE_SELECTED_3_credit_curve_growth__LGBM",
    "ENERGY": "ENERGY_TOT_SELECTED_1_brent_gas_eu_pressure__LGBM",
    "PPP_ALT": "PPP_HGBM_FORCED_TEST_SELECTED_01_dominance_gold_spx__LGBM",
    "TAYLOR_UIP": "TAYLOR_UIP_XGB_TEST_AUG01_policy_carry_volrisk_score_gap__XGB",
    "TAYLOR_RISK": "TAYLOR_GBM_VAL_SELECTED_01_growth_inflation_policy_rates__HGBM",
    "CURVE_LIQ_MONEY": "CURVE_GBM_FORCED_TEST_SELECTED_01_curve_liq_money_risk__HGBM",
}


SELECTED_ENSEMBLES = {
    "MAIN_14": MAIN_14_ENSEMBLE,
    "THEORY_11": ENSEMBLE_C_THEORY_11,
    "ENSEMBLE_B": ENSEMBLE_B_SELECTED_MODELS,
}


# =============================================================================
# 1. REQUIRED GLOBAL CHECKS
# =============================================================================

required_objects = ["pred_store", "test_df"]
for obj in required_objects:
    if obj not in globals():
        raise ValueError(f"❌ '{obj}' bulunamadı. Önce ana training kodunu çalıştır.")

if "fwd_ret" not in test_df.columns:
    raise ValueError("❌ test_df içinde 'fwd_ret' yok.")

Y_TEST = test_df["fwd_ret"].astype(float).values

BAR_ANN_LOCAL = globals().get("BAR_ANN", 6 * 252)
TC_LOCAL      = globals().get("TC_PER_SIDE", 0.00005)

RANDOM_SEED_LOCAL = 42
N_RANDOM_RUNS     = 500


# =============================================================================
# 2. UTILITY FUNCTIONS
# =============================================================================

def safe_array(x):
    return np.nan_to_num(np.asarray(x, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)


def calc_max_dd(ret):
    r = safe_array(ret)
    equity = np.cumprod(1.0 + r)
    peak = np.maximum.accumulate(equity)
    dd = equity / (peak + 1e-12) - 1.0
    return float(np.min(dd))


def calc_ann_ret(ret, bar_ann=BAR_ANN_LOCAL):
    r = safe_array(ret)
    cum_ret = float(np.prod(1.0 + r) - 1.0)
    years = len(r) / bar_ann

    if years <= 0:
        return 0.0
    if cum_ret <= -0.999:
        return -1.0

    return float((1.0 + cum_ret) ** (1.0 / years) - 1.0)


def calc_sharpe(ret, bar_ann=BAR_ANN_LOCAL):
    r = safe_array(ret)
    sd = np.std(r, ddof=1)

    if sd <= 1e-12:
        return 0.0

    return float(np.mean(r) / sd * np.sqrt(bar_ann))


def signal_to_returns(signal, y, tc_per_side=TC_LOCAL):
    sig = safe_array(signal)
    y = safe_array(y)

    turnover = np.abs(np.diff(sig, prepend=0.0))
    ret = sig * y - tc_per_side * turnover

    return safe_array(ret)


def summarize_strategy(ensemble_name, channel_name, model_name, benchmark_type, signal, y, tc_per_side=TC_LOCAL):
    sig = safe_array(signal)
    y = safe_array(y)
    ret = signal_to_returns(sig, y, tc_per_side=tc_per_side)

    active_mask = sig != 0
    active_ret = ret[active_mask]

    long_mask = sig == 1
    short_mask = sig == -1
    flat_mask = sig == 0

    correct_direction = (
        ((sig == 1) & (y > 0)) |
        ((sig == -1) & (y < 0))
    )

    directional_accuracy = (
        float(np.mean(correct_direction[active_mask]))
        if np.sum(active_mask) > 0 else 0.0
    )

    long_accuracy = (
        float(np.mean(y[long_mask] > 0))
        if np.sum(long_mask) > 0 else 0.0
    )

    short_accuracy = (
        float(np.mean(y[short_mask] < 0))
        if np.sum(short_mask) > 0 else 0.0
    )

    long_avg_fwd_ret = (
        float(np.mean(y[long_mask]))
        if np.sum(long_mask) > 0 else 0.0
    )

    short_avg_fwd_ret = (
        float(np.mean(y[short_mask]))
        if np.sum(short_mask) > 0 else 0.0
    )

    active_avg_fwd_ret = (
        float(np.mean(y[active_mask]))
        if np.sum(active_mask) > 0 else 0.0
    )

    return {
        "ensemble": ensemble_name,
        "channel": channel_name,
        "model_name": model_name,
        "benchmark_type": benchmark_type,

        "sharpe": calc_sharpe(ret),
        "cum_ret": float(np.prod(1.0 + ret) - 1.0),
        "ann_ret": calc_ann_ret(ret),
        "max_dd": calc_max_dd(ret),

        "activity": float(np.mean(active_mask)),
        "trade_count": int(np.sum(np.abs(np.diff(sig, prepend=0.0)) > 0)),
        "win_rate_active_ret": float(np.mean(active_ret > 0)) if len(active_ret) else 0.0,

        "directional_accuracy": directional_accuracy,
        "long_accuracy": long_accuracy,
        "short_accuracy": short_accuracy,

        "long_ratio": float(np.mean(long_mask)),
        "short_ratio": float(np.mean(short_mask)),
        "flat_ratio": float(np.mean(flat_mask)),

        "long_avg_fwd_ret": long_avg_fwd_ret,
        "short_avg_fwd_ret": short_avg_fwd_ret,
        "active_avg_fwd_ret": active_avg_fwd_ret,
    }


def regression_quality_metrics(ensemble_name, channel_name, model_name, pred, y):
    pred = safe_array(pred)
    y = safe_array(y)

    pearson_corr = pd.Series(pred).corr(pd.Series(y))
    pearson_corr = 0.0 if pd.isna(pearson_corr) else float(pearson_corr)

    sp = spearmanr(pred, y, nan_policy="omit")
    spearman_corr = 0.0 if pd.isna(sp.correlation) else float(sp.correlation)
    spearman_pval = 1.0 if pd.isna(sp.pvalue) else float(sp.pvalue)

    non_zero_mask = (pred != 0) & (y != 0)
    sign_accuracy = (
        float(np.mean(np.sign(pred[non_zero_mask]) == np.sign(y[non_zero_mask])))
        if np.sum(non_zero_mask) > 0 else 0.0
    )

    pred_long_mask = pred > 0
    pred_short_mask = pred < 0

    pred_long_accuracy = (
        float(np.mean(y[pred_long_mask] > 0))
        if np.sum(pred_long_mask) > 0 else 0.0
    )

    pred_short_accuracy = (
        float(np.mean(y[pred_short_mask] < 0))
        if np.sum(pred_short_mask) > 0 else 0.0
    )

    pred_long_avg_ret = (
        float(np.mean(y[pred_long_mask]))
        if np.sum(pred_long_mask) > 0 else 0.0
    )

    pred_short_avg_ret = (
        float(np.mean(y[pred_short_mask]))
        if np.sum(pred_short_mask) > 0 else 0.0
    )

    q_low = np.nanquantile(pred, 0.20)
    q_high = np.nanquantile(pred, 0.80)

    bottom_mask = pred <= q_low
    top_mask = pred >= q_high

    bottom_avg_ret = float(np.mean(y[bottom_mask])) if np.sum(bottom_mask) else 0.0
    top_avg_ret = float(np.mean(y[top_mask])) if np.sum(top_mask) else 0.0

    return {
        "ensemble": ensemble_name,
        "channel": channel_name,
        "model_name": model_name,

        "pearson_corr": pearson_corr,
        "spearman_corr": spearman_corr,
        "spearman_pval": spearman_pval,

        "sign_accuracy": sign_accuracy,
        "pred_long_accuracy": pred_long_accuracy,
        "pred_short_accuracy": pred_short_accuracy,

        "pred_long_avg_fwd_ret": pred_long_avg_ret,
        "pred_short_avg_fwd_ret": pred_short_avg_ret,

        "top20_avg_fwd_ret": top_avg_ret,
        "bottom20_avg_fwd_ret": bottom_avg_ret,
        "top_bottom_spread": top_avg_ret - bottom_avg_ret,
    }


def buy_and_hold_signal(n):
    return np.ones(n)


def always_short_signal(n):
    return -np.ones(n)


def sign_only_signal(pred):
    pred = safe_array(pred)
    return np.where(pred > 0, 1.0, np.where(pred < 0, -1.0, 0.0))


def random_signal(n, activity=None, long_ratio=0.5, seed=42):
    rng = np.random.default_rng(seed)

    if activity is None:
        return rng.choice([-1.0, 1.0], size=n, replace=True, p=[1.0 - long_ratio, long_ratio])

    activity = float(np.clip(activity, 0.0, 1.0))
    flat_prob = 1.0 - activity
    long_prob = activity * long_ratio
    short_prob = activity * (1.0 - long_ratio)

    return rng.choice(
        [-1.0, 0.0, 1.0],
        size=n,
        replace=True,
        p=[short_prob, flat_prob, long_prob]
    )


def get_threshold_signal_from_pred_store(model_name, obj):
    if "test_signal" in obj:
        return safe_array(obj["test_signal"])

    if "test_prob" in obj and "best_upper" in obj and "best_lower" in obj:
        prob = safe_array(obj["test_prob"])
        upper = float(obj["best_upper"])
        lower = float(obj["best_lower"])
        return np.where(prob >= upper, 1.0, np.where(prob <= lower, -1.0, 0.0))

    raise ValueError(f"{model_name} için test_signal veya test_prob+threshold bulunamadı.")


# =============================================================================
# 3. RUN REPORT ONLY FOR SELECTED MODELS
# =============================================================================

benchmark_rows = []
quality_rows = []
random_rows = []
missing_rows = []

n_test = len(Y_TEST)

seen_keys = set()

for ensemble_name, ensemble_dict in SELECTED_ENSEMBLES.items():

    print("\n" + "═" * 120)
    print(f"RUNNING SELECTED BENCHMARK REPORT → {ensemble_name}")
    print("═" * 120)

    for channel_name, model_name in ensemble_dict.items():

        unique_key = (ensemble_name, channel_name, model_name)

        if model_name not in pred_store:
            print(f"⚠ Missing in pred_store: {ensemble_name} | {channel_name} | {model_name}")
            missing_rows.append({
                "ensemble": ensemble_name,
                "channel": channel_name,
                "model_name": model_name,
                "reason": "model_name_not_found_in_pred_store"
            })
            continue

        obj = pred_store[model_name]

        if "test_pred" not in obj:
            print(f"⚠ Missing test_pred: {ensemble_name} | {channel_name} | {model_name}")
            missing_rows.append({
                "ensemble": ensemble_name,
                "channel": channel_name,
                "model_name": model_name,
                "reason": "test_pred_missing"
            })
            continue

        pred_test = safe_array(obj["test_pred"])

        if len(pred_test) != n_test:
            print(f"⚠ Length mismatch: {ensemble_name} | {channel_name} | {model_name}")
            missing_rows.append({
                "ensemble": ensemble_name,
                "channel": channel_name,
                "model_name": model_name,
                "reason": f"length_mismatch_pred_{len(pred_test)}_test_{n_test}"
            })
            continue

        try:
            threshold_signal = get_threshold_signal_from_pred_store(model_name, obj)
        except Exception as e:
            print(f"⚠ Signal error: {ensemble_name} | {channel_name} | {model_name} | {e}")
            missing_rows.append({
                "ensemble": ensemble_name,
                "channel": channel_name,
                "model_name": model_name,
                "reason": str(e)
            })
            continue

        print(f"✅ {channel_name:<20} {model_name}")

        # ---------------------------------------------------------------------
        # A) Regression quality
        # ---------------------------------------------------------------------
        quality_rows.append(
            regression_quality_metrics(
                ensemble_name=ensemble_name,
                channel_name=channel_name,
                model_name=model_name,
                pred=pred_test,
                y=Y_TEST
            )
        )

        # ---------------------------------------------------------------------
        # B) Benchmark strategies
        # ---------------------------------------------------------------------
        benchmark_rows.append(
            summarize_strategy(
                ensemble_name=ensemble_name,
                channel_name=channel_name,
                model_name=model_name,
                benchmark_type="THRESHOLD_STRATEGY",
                signal=threshold_signal,
                y=Y_TEST
            )
        )

        benchmark_rows.append(
            summarize_strategy(
                ensemble_name=ensemble_name,
                channel_name=channel_name,
                model_name=model_name,
                benchmark_type="SIGN_ONLY_REGRESSION",
                signal=sign_only_signal(pred_test),
                y=Y_TEST
            )
        )

        benchmark_rows.append(
            summarize_strategy(
                ensemble_name=ensemble_name,
                channel_name=channel_name,
                model_name=model_name,
                benchmark_type="BUY_AND_HOLD_LONG",
                signal=buy_and_hold_signal(n_test),
                y=Y_TEST
            )
        )

        benchmark_rows.append(
            summarize_strategy(
                ensemble_name=ensemble_name,
                channel_name=channel_name,
                model_name=model_name,
                benchmark_type="ALWAYS_SHORT",
                signal=always_short_signal(n_test),
                y=Y_TEST
            )
        )

        # ---------------------------------------------------------------------
        # C) Random benchmark, activity and long-ratio matched
        # ---------------------------------------------------------------------
        threshold_activity = float(np.mean(threshold_signal != 0))

        if np.sum(threshold_signal != 0) > 0:
            threshold_long_ratio_active = float(np.mean(threshold_signal[threshold_signal != 0] == 1))
        else:
            threshold_long_ratio_active = 0.5

        model_random_metrics = []

        for i in range(N_RANDOM_RUNS):
            sig_random = random_signal(
                n=n_test,
                activity=threshold_activity,
                long_ratio=threshold_long_ratio_active,
                seed=RANDOM_SEED_LOCAL + i
            )

            m_rand = summarize_strategy(
                ensemble_name=ensemble_name,
                channel_name=channel_name,
                model_name=model_name,
                benchmark_type="RANDOM_SIGNAL",
                signal=sig_random,
                y=Y_TEST
            )

            m_rand["random_run"] = i
            model_random_metrics.append(m_rand)
            random_rows.append(m_rand)

        random_df_model = pd.DataFrame(model_random_metrics)

        random_summary = {
            "ensemble": ensemble_name,
            "channel": channel_name,
            "model_name": model_name,
            "benchmark_type": f"RANDOM_SIGNAL_MEAN_{N_RANDOM_RUNS}",

            "sharpe": random_df_model["sharpe"].mean(),
            "cum_ret": random_df_model["cum_ret"].mean(),
            "ann_ret": random_df_model["ann_ret"].mean(),
            "max_dd": random_df_model["max_dd"].mean(),

            "activity": random_df_model["activity"].mean(),
            "trade_count": random_df_model["trade_count"].mean(),
            "win_rate_active_ret": random_df_model["win_rate_active_ret"].mean(),

            "directional_accuracy": random_df_model["directional_accuracy"].mean(),
            "long_accuracy": random_df_model["long_accuracy"].mean(),
            "short_accuracy": random_df_model["short_accuracy"].mean(),

            "long_ratio": random_df_model["long_ratio"].mean(),
            "short_ratio": random_df_model["short_ratio"].mean(),
            "flat_ratio": random_df_model["flat_ratio"].mean(),

            "long_avg_fwd_ret": random_df_model["long_avg_fwd_ret"].mean(),
            "short_avg_fwd_ret": random_df_model["short_avg_fwd_ret"].mean(),
            "active_avg_fwd_ret": random_df_model["active_avg_fwd_ret"].mean(),
        }

        benchmark_rows.append(random_summary)


# =============================================================================
# 4. BUILD DATAFRAMES
# =============================================================================

benchmark_selected_df = pd.DataFrame(benchmark_rows)
quality_selected_df = pd.DataFrame(quality_rows)
random_selected_all_df = pd.DataFrame(random_rows)
missing_selected_df = pd.DataFrame(missing_rows)

if benchmark_selected_df.empty:
    raise ValueError("❌ Hiç benchmark sonucu üretilemedi. Model isimleri pred_store ile eşleşmiyor olabilir.")

# Optional metadata merge from results_df
if "results_df" in globals() and isinstance(results_df, pd.DataFrame):
    meta_cols = ["model_name", "group", "feature_set_name", "model_family"]
    if all(c in results_df.columns for c in meta_cols):
        meta = results_df[meta_cols].drop_duplicates("model_name")

        benchmark_selected_df = benchmark_selected_df.merge(
            meta, on="model_name", how="left"
        )

        quality_selected_df = quality_selected_df.merge(
            meta, on="model_name", how="left"
        )


# =============================================================================
# 5. PIVOT COMPARISON
# =============================================================================

pivot_cols = [
    "sharpe",
    "ann_ret",
    "cum_ret",
    "max_dd",
    "activity",
    "trade_count",
    "directional_accuracy",
    "long_accuracy",
    "short_accuracy",
    "active_avg_fwd_ret",
]

comparison_selected_df = (
    benchmark_selected_df
    .pivot_table(
        index=["ensemble", "channel", "model_name"],
        columns="benchmark_type",
        values=pivot_cols,
        aggfunc="first"
    )
)

comparison_selected_df.columns = [
    f"{metric}__{bench}" for metric, bench in comparison_selected_df.columns
]

comparison_selected_df = comparison_selected_df.reset_index()

random_col = f"sharpe__RANDOM_SIGNAL_MEAN_{N_RANDOM_RUNS}"

if "sharpe__THRESHOLD_STRATEGY" in comparison_selected_df.columns and "sharpe__SIGN_ONLY_REGRESSION" in comparison_selected_df.columns:
    comparison_selected_df["delta_sharpe_threshold_vs_sign"] = (
        comparison_selected_df["sharpe__THRESHOLD_STRATEGY"] -
        comparison_selected_df["sharpe__SIGN_ONLY_REGRESSION"]
    )

if "sharpe__THRESHOLD_STRATEGY" in comparison_selected_df.columns and random_col in comparison_selected_df.columns:
    comparison_selected_df["delta_sharpe_threshold_vs_random"] = (
        comparison_selected_df["sharpe__THRESHOLD_STRATEGY"] -
        comparison_selected_df[random_col]
    )

if "ann_ret__THRESHOLD_STRATEGY" in comparison_selected_df.columns and "ann_ret__SIGN_ONLY_REGRESSION" in comparison_selected_df.columns:
    comparison_selected_df["delta_ann_ret_threshold_vs_sign"] = (
        comparison_selected_df["ann_ret__THRESHOLD_STRATEGY"] -
        comparison_selected_df["ann_ret__SIGN_ONLY_REGRESSION"]
    )

if "max_dd__THRESHOLD_STRATEGY" in comparison_selected_df.columns and "max_dd__SIGN_ONLY_REGRESSION" in comparison_selected_df.columns:
    comparison_selected_df["delta_dd_threshold_vs_sign"] = (
        comparison_selected_df["max_dd__THRESHOLD_STRATEGY"] -
        comparison_selected_df["max_dd__SIGN_ONLY_REGRESSION"]
    )

comparison_selected_df = comparison_selected_df.merge(
    quality_selected_df,
    on=["ensemble", "channel", "model_name"],
    how="left",
    suffixes=("", "_quality")
)

sort_cols = []
ascending = []

if "ensemble" in comparison_selected_df.columns:
    sort_cols.append("ensemble")
    ascending.append(True)

if "sharpe__THRESHOLD_STRATEGY" in comparison_selected_df.columns:
    sort_cols.append("sharpe__THRESHOLD_STRATEGY")
    ascending.append(False)

comparison_selected_df = comparison_selected_df.sort_values(
    sort_cols,
    ascending=ascending
).reset_index(drop=True)


# =============================================================================
# 6. ENSEMBLE-LEVEL SUMMARY
# =============================================================================

ensemble_summary_rows = []

for ensemble_name in comparison_selected_df["ensemble"].unique():

    temp = comparison_selected_df[comparison_selected_df["ensemble"] == ensemble_name].copy()

    row = {
        "ensemble": ensemble_name,
        "n_models": len(temp),
    }

    for col in [
        "sharpe__THRESHOLD_STRATEGY",
        "sharpe__SIGN_ONLY_REGRESSION",
        random_col,
        "sharpe__BUY_AND_HOLD_LONG",
        "sharpe__ALWAYS_SHORT",
        "delta_sharpe_threshold_vs_sign",
        "delta_sharpe_threshold_vs_random",
        "spearman_corr",
        "sign_accuracy",
        "top_bottom_spread",
        "ann_ret__THRESHOLD_STRATEGY",
        "max_dd__THRESHOLD_STRATEGY",
        "activity__THRESHOLD_STRATEGY",
        "trade_count__THRESHOLD_STRATEGY",
    ]:
        if col in temp.columns:
            row[f"avg_{col}"] = temp[col].mean()
            row[f"best_{col}"] = temp[col].max()

    ensemble_summary_rows.append(row)

ensemble_selected_summary_df = pd.DataFrame(ensemble_summary_rows)


# =============================================================================
# 7. PRINT REPORTS
# =============================================================================

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 260)

print("\n\n" + "═" * 180)
print("SELECTED MODELS — REGRESSION QUALITY REPORT")
print("═" * 180)

quality_cols = [
    "ensemble", "channel", "group", "feature_set_name", "model_family", "model_name",
    "pearson_corr", "spearman_corr", "spearman_pval",
    "sign_accuracy",
    "pred_long_accuracy", "pred_short_accuracy",
    "top20_avg_fwd_ret", "bottom20_avg_fwd_ret", "top_bottom_spread",
]
quality_cols = [c for c in quality_cols if c in quality_selected_df.columns]

print(
    quality_selected_df[quality_cols]
    .sort_values(["ensemble", "spearman_corr", "top_bottom_spread"], ascending=[True, False, False])
    .round(6)
    .to_string(index=False)
)


print("\n\n" + "═" * 180)
print("SELECTED MODELS — BENCHMARK STRATEGY REPORT")
print("═" * 180)

benchmark_cols = [
    "ensemble", "channel", "group", "feature_set_name", "model_family", "model_name",
    "benchmark_type",
    "sharpe", "ann_ret", "cum_ret", "max_dd",
    "activity", "trade_count",
    "directional_accuracy", "long_accuracy", "short_accuracy",
    "long_ratio", "short_ratio", "flat_ratio",
]
benchmark_cols = [c for c in benchmark_cols if c in benchmark_selected_df.columns]

print(
    benchmark_selected_df[benchmark_cols]
    .sort_values(["ensemble", "channel", "sharpe"], ascending=[True, True, False])
    .round(6)
    .to_string(index=False)
)


print("\n\n" + "═" * 180)
print("SELECTED MODELS — MAIN COMPARISON REPORT")
print("Positive delta_sharpe_threshold_vs_sign means validation-threshold conversion improved sign-only regression.")
print("═" * 180)

main_cols = [
    "ensemble", "channel", "group", "feature_set_name", "model_family", "model_name",

    "spearman_corr",
    "sign_accuracy",
    "top_bottom_spread",

    "sharpe__THRESHOLD_STRATEGY",
    "sharpe__SIGN_ONLY_REGRESSION",
    random_col,
    "sharpe__BUY_AND_HOLD_LONG",
    "sharpe__ALWAYS_SHORT",

    "delta_sharpe_threshold_vs_sign",
    "delta_sharpe_threshold_vs_random",
    "delta_ann_ret_threshold_vs_sign",
    "delta_dd_threshold_vs_sign",

    "ann_ret__THRESHOLD_STRATEGY",
    "ann_ret__SIGN_ONLY_REGRESSION",

    "max_dd__THRESHOLD_STRATEGY",
    "max_dd__SIGN_ONLY_REGRESSION",

    "activity__THRESHOLD_STRATEGY",
    "activity__SIGN_ONLY_REGRESSION",

    "trade_count__THRESHOLD_STRATEGY",
    "trade_count__SIGN_ONLY_REGRESSION",

    "directional_accuracy__THRESHOLD_STRATEGY",
    "directional_accuracy__SIGN_ONLY_REGRESSION",
]
main_cols = [c for c in main_cols if c in comparison_selected_df.columns]

print(
    comparison_selected_df[main_cols]
    .round(6)
    .to_string(index=False)
)


print("\n\n" + "═" * 180)
print("ENSEMBLE-LEVEL SUMMARY")
print("═" * 180)
print(
    ensemble_selected_summary_df
    .round(6)
    .to_string(index=False)
)


if not missing_selected_df.empty:
    print("\n\n" + "═" * 140)
    print("⚠ MISSING SELECTED MODELS")
    print("Bu modeller pred_store içinde bulunamadı veya test_pred/signal problemi var.")
    print("═" * 140)
    print(missing_selected_df.to_string(index=False))


# =============================================================================
# 8. SAVE OUTPUTS
# =============================================================================

SAVE_SELECTED_BENCH_DIR = os.path.join(
    globals().get("SAVE_DIR", "final_training_test_expanding"),
    "selected_ensemble_benchmarks"
)

os.makedirs(SAVE_SELECTED_BENCH_DIR, exist_ok=True)

benchmark_selected_df.to_csv(
    f"{SAVE_SELECTED_BENCH_DIR}/selected_benchmark_strategy_report.csv",
    index=False
)

quality_selected_df.to_csv(
    f"{SAVE_SELECTED_BENCH_DIR}/selected_regression_quality_report.csv",
    index=False
)

comparison_selected_df.to_csv(
    f"{SAVE_SELECTED_BENCH_DIR}/selected_threshold_vs_benchmark_comparison.csv",
    index=False
)

random_selected_all_df.to_csv(
    f"{SAVE_SELECTED_BENCH_DIR}/selected_random_signal_all_runs.csv",
    index=False
)

ensemble_selected_summary_df.to_csv(
    f"{SAVE_SELECTED_BENCH_DIR}/selected_ensemble_level_summary.csv",
    index=False
)

if not missing_selected_df.empty:
    missing_selected_df.to_csv(
        f"{SAVE_SELECTED_BENCH_DIR}/missing_selected_models.csv",
        index=False
    )


print("\n" + "═" * 120)
print("✅ SELECTED ENSEMBLE BENCHMARK REPORT TAMAMLANDI")
print(f"📁 SAVE DIR: {SAVE_SELECTED_BENCH_DIR}")
print("Kaydedilenler:")
print("  - selected_benchmark_strategy_report.csv")
print("  - selected_regression_quality_report.csv")
print("  - selected_threshold_vs_benchmark_comparison.csv")
print("  - selected_random_signal_all_runs.csv")
print("  - selected_ensemble_level_summary.csv")
if not missing_selected_df.empty:
    print("  - missing_selected_models.csv")
print("═" * 120)


# =============================================================================
# 9. QUICK BEST VIEW
# =============================================================================

quick_cols = [
    "ensemble", "channel", "model_family", "model_name",
    "spearman_corr",
    "sign_accuracy",
    "top_bottom_spread",
    "sharpe__THRESHOLD_STRATEGY",
    "sharpe__SIGN_ONLY_REGRESSION",
    random_col,
    "delta_sharpe_threshold_vs_sign",
    "delta_sharpe_threshold_vs_random",
    "ann_ret__THRESHOLD_STRATEGY",
    "max_dd__THRESHOLD_STRATEGY",
    "activity__THRESHOLD_STRATEGY",
    "trade_count__THRESHOLD_STRATEGY",
]

quick_cols = [c for c in quick_cols if c in comparison_selected_df.columns]

selected_best_view = (
    comparison_selected_df[quick_cols]
    .sort_values(["ensemble", "sharpe__THRESHOLD_STRATEGY"], ascending=[True, False])
    .reset_index(drop=True)
)

print("\n\n" + "═" * 180)
print("QUICK BEST VIEW — SELECTED ENSEMBLES")
print("═" * 180)
print(selected_best_view.round(6).to_string(index=False))

In [ ]:
# =============================================================================
# FINAL SELECTED REPORT — THRESHOLD VS SIGN-ONLY
#
# Required columns:
#   threshold_sharpe
#   sign_only_sharpe
#   delta_sharpe
#   sign_only_accuracy
#   threshold_after_accuracy
#   DD
#   Sortino
#   Calmar
# =============================================================================

import os
import numpy as np
import pandas as pd


# =============================================================================
# 0. CHECKS
# =============================================================================

required_objects = ["pred_store", "test_df", "SELECTED_ENSEMBLES"]

for obj in required_objects:
    if obj not in globals():
        raise ValueError(f"❌ '{obj}' bulunamadı. Önce selected benchmark kodunu çalıştır.")

if "fwd_ret" not in test_df.columns:
    raise ValueError("❌ test_df içinde 'fwd_ret' yok.")

Y_TEST = test_df["fwd_ret"].astype(float).values

BAR_ANN_LOCAL = globals().get("BAR_ANN", 6 * 252)
TC_LOCAL = globals().get("TC_PER_SIDE", 0.00005)


# =============================================================================
# 1. METRIC FUNCTIONS
# =============================================================================

def safe_array(x):
    return np.nan_to_num(
        np.asarray(x, dtype=float),
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )


def calc_sharpe(ret, bar_ann=BAR_ANN_LOCAL):
    r = safe_array(ret)
    sd = np.std(r, ddof=1)

    if sd <= 1e-12:
        return 0.0

    return float(np.mean(r) / sd * np.sqrt(bar_ann))


def calc_sortino(ret, bar_ann=BAR_ANN_LOCAL):
    """
    Sortino Ratio:
        annualized mean return / annualized downside deviation

    Sadece negatif returnlerin std'si kullanılır.
    """
    r = safe_array(ret)
    downside = r[r < 0]

    if len(downside) == 0:
        return np.nan

    downside_std = np.std(downside, ddof=1)

    if downside_std <= 1e-12:
        return np.nan

    return float(np.mean(r) / downside_std * np.sqrt(bar_ann))


def calc_max_dd(ret):
    r = safe_array(ret)

    equity = np.cumprod(1.0 + r)
    peak = np.maximum.accumulate(equity)

    dd = equity / (peak + 1e-12) - 1.0

    return float(np.min(dd))


def calc_ann_ret(ret, bar_ann=BAR_ANN_LOCAL):
    r = safe_array(ret)

    cum_ret = float(np.prod(1.0 + r) - 1.0)
    years = len(r) / bar_ann

    if years <= 0:
        return 0.0

    if cum_ret <= -0.999:
        return -1.0

    return float((1.0 + cum_ret) ** (1.0 / years) - 1.0)


def calc_calmar(ret, bar_ann=BAR_ANN_LOCAL):
    """
    Calmar Ratio:
        annualized return / abs(max drawdown)
    """
    ann = calc_ann_ret(ret, bar_ann=bar_ann)
    dd = calc_max_dd(ret)

    if abs(dd) <= 1e-12:
        return np.nan

    return float(ann / abs(dd))


def signal_to_returns(signal, y, tc_per_side=TC_LOCAL):
    sig = safe_array(signal)
    y = safe_array(y)

    turnover = np.abs(np.diff(sig, prepend=0.0))
    ret = sig * y - tc_per_side * turnover

    return safe_array(ret)


def sign_only_signal(pred):
    pred = safe_array(pred)

    return np.where(
        pred > 0,
        1.0,
        np.where(pred < 0, -1.0, 0.0)
    )


def get_threshold_signal_from_pred_store(model_name, obj):
    """
    Ana pipeline'daki threshold sinyali varsa onu kullanır.
    Yoksa test_prob + best_upper/best_lower ile tekrar üretir.
    """
    if "test_signal" in obj:
        return safe_array(obj["test_signal"])

    if "test_prob" in obj and "best_upper" in obj and "best_lower" in obj:
        prob = safe_array(obj["test_prob"])
        upper = float(obj["best_upper"])
        lower = float(obj["best_lower"])

        return np.where(
            prob >= upper,
            1.0,
            np.where(prob <= lower, -1.0, 0.0)
        )

    raise ValueError(f"{model_name} için test_signal veya test_prob+threshold bulunamadı.")


def directional_accuracy(signal, y):
    """
    Accuracy sadece pozisyon alınan barlarda hesaplanır.

    Long doğru:
        signal = +1 and fwd_ret > 0

    Short doğru:
        signal = -1 and fwd_ret < 0
    """
    sig = safe_array(signal)
    y = safe_array(y)

    active = sig != 0

    if np.sum(active) == 0:
        return 0.0

    correct = (
        ((sig == 1) & (y > 0)) |
        ((sig == -1) & (y < 0))
    )

    return float(np.mean(correct[active]))


def strategy_metrics(signal, y):
    sig = safe_array(signal)
    ret = signal_to_returns(sig, y)

    return {
        "sharpe": calc_sharpe(ret),
        "ann_ret": calc_ann_ret(ret),
        "cum_ret": float(np.prod(1.0 + ret) - 1.0),
        "dd": calc_max_dd(ret),
        "sortino": calc_sortino(ret),
        "calmar": calc_calmar(ret),
        "accuracy": directional_accuracy(sig, y),
        "activity": float(np.mean(sig != 0)),
        "trade_count": int(np.sum(np.abs(np.diff(sig, prepend=0.0)) > 0)),
        "long_ratio": float(np.mean(sig == 1)),
        "short_ratio": float(np.mean(sig == -1)),
        "flat_ratio": float(np.mean(sig == 0)),
    }


# =============================================================================
# 2. BUILD FINAL REPORT
# =============================================================================

final_rows = []
missing_rows = []

n_test = len(Y_TEST)

for ensemble_name, ensemble_dict in SELECTED_ENSEMBLES.items():

    for channel_name, model_name in ensemble_dict.items():

        if model_name not in pred_store:
            missing_rows.append({
                "ensemble": ensemble_name,
                "channel": channel_name,
                "model_name": model_name,
                "reason": "model_not_found_in_pred_store",
            })
            continue

        obj = pred_store[model_name]

        if "test_pred" not in obj:
            missing_rows.append({
                "ensemble": ensemble_name,
                "channel": channel_name,
                "model_name": model_name,
                "reason": "test_pred_missing",
            })
            continue

        pred_test = safe_array(obj["test_pred"])

        if len(pred_test) != n_test:
            missing_rows.append({
                "ensemble": ensemble_name,
                "channel": channel_name,
                "model_name": model_name,
                "reason": f"length_mismatch_pred_{len(pred_test)}_test_{n_test}",
            })
            continue

        try:
            threshold_sig = get_threshold_signal_from_pred_store(model_name, obj)
        except Exception as e:
            missing_rows.append({
                "ensemble": ensemble_name,
                "channel": channel_name,
                "model_name": model_name,
                "reason": str(e),
            })
            continue

        sign_sig = sign_only_signal(pred_test)

        threshold_m = strategy_metrics(threshold_sig, Y_TEST)
        sign_m = strategy_metrics(sign_sig, Y_TEST)

        row = {
            "ensemble": ensemble_name,
            "channel": channel_name,
            "model_name": model_name,

            # Core requested columns
            "threshold_sharpe": threshold_m["sharpe"],
            "sign_only_sharpe": sign_m["sharpe"],
            "delta_sharpe": threshold_m["sharpe"] - sign_m["sharpe"],

            "sign_only_accuracy": sign_m["accuracy"],
            "threshold_after_accuracy": threshold_m["accuracy"],

            "threshold_dd": threshold_m["dd"],
            "sign_only_dd": sign_m["dd"],
            "delta_dd": threshold_m["dd"] - sign_m["dd"],

            "threshold_sortino": threshold_m["sortino"],
            "sign_only_sortino": sign_m["sortino"],
            "delta_sortino": threshold_m["sortino"] - sign_m["sortino"],

            "threshold_calmar": threshold_m["calmar"],
            "sign_only_calmar": sign_m["calmar"],
            "delta_calmar": threshold_m["calmar"] - sign_m["calmar"],

            # Helpful extra columns
            "threshold_ann_ret": threshold_m["ann_ret"],
            "sign_only_ann_ret": sign_m["ann_ret"],
            "threshold_cum_ret": threshold_m["cum_ret"],
            "sign_only_cum_ret": sign_m["cum_ret"],

            "threshold_activity": threshold_m["activity"],
            "sign_only_activity": sign_m["activity"],

            "threshold_trade_count": threshold_m["trade_count"],
            "sign_only_trade_count": sign_m["trade_count"],

            "threshold_long_ratio": threshold_m["long_ratio"],
            "threshold_short_ratio": threshold_m["short_ratio"],
            "threshold_flat_ratio": threshold_m["flat_ratio"],

            "sign_only_long_ratio": sign_m["long_ratio"],
            "sign_only_short_ratio": sign_m["short_ratio"],
            "sign_only_flat_ratio": sign_m["flat_ratio"],
        }

        # Metadata merge from results_df if available
        if "results_df" in globals() and isinstance(results_df, pd.DataFrame):
            if "model_name" in results_df.columns:
                meta = results_df[results_df["model_name"] == model_name]

                if len(meta) > 0:
                    meta_row = meta.iloc[0]

                    for col in ["group", "feature_set_name", "model_family", "best_lower", "best_upper"]:
                        if col in meta_row.index:
                            row[col] = meta_row[col]

        final_rows.append(row)


final_selected_threshold_report = pd.DataFrame(final_rows)
missing_final_selected_report = pd.DataFrame(missing_rows)

if final_selected_threshold_report.empty:
    raise ValueError("❌ Final rapor boş. Model isimleri pred_store ile eşleşmiyor olabilir.")


# =============================================================================
# 3. COLUMN ORDER
# =============================================================================

ordered_cols = [
    "ensemble",
    "channel",
    "group",
    "feature_set_name",
    "model_family",
    "model_name",

    "threshold_sharpe",
    "sign_only_sharpe",
    "delta_sharpe",

    "sign_only_accuracy",
    "threshold_after_accuracy",

    "threshold_dd",
    "sign_only_dd",
    "delta_dd",

    "threshold_sortino",
    "sign_only_sortino",
    "delta_sortino",

    "threshold_calmar",
    "sign_only_calmar",
    "delta_calmar",

    "threshold_ann_ret",
    "sign_only_ann_ret",
    "threshold_cum_ret",
    "sign_only_cum_ret",

    "threshold_activity",
    "sign_only_activity",

    "threshold_trade_count",
    "sign_only_trade_count",

    "threshold_long_ratio",
    "threshold_short_ratio",
    "threshold_flat_ratio",

    "sign_only_long_ratio",
    "sign_only_short_ratio",
    "sign_only_flat_ratio",

    "best_lower",
    "best_upper",
]

ordered_cols = [c for c in ordered_cols if c in final_selected_threshold_report.columns]

final_selected_threshold_report = final_selected_threshold_report[ordered_cols]


# =============================================================================
# 4. SORT + PRINT
# =============================================================================

final_selected_threshold_report = final_selected_threshold_report.sort_values(
    ["ensemble", "threshold_sharpe", "delta_sharpe", "threshold_calmar"],
    ascending=[True, False, False, False]
).reset_index(drop=True)

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 280)

print("\n" + "═" * 190)
print("FINAL SELECTED REPORT — THRESHOLD VS SIGN-ONLY")
print("Main columns: threshold_sharpe | sign_only_sharpe | delta | accuracy | DD | Sortino | Calmar")
print("═" * 190)

print(
    final_selected_threshold_report
    .round(6)
    .to_string(index=False)
)


# =============================================================================
# 5. ENSEMBLE SUMMARY
# =============================================================================

summary_cols = [
    "threshold_sharpe",
    "sign_only_sharpe",
    "delta_sharpe",

    "sign_only_accuracy",
    "threshold_after_accuracy",

    "threshold_dd",
    "sign_only_dd",

    "threshold_sortino",
    "sign_only_sortino",

    "threshold_calmar",
    "sign_only_calmar",

    "threshold_ann_ret",
    "sign_only_ann_ret",

    "threshold_activity",
    "threshold_trade_count",
]

available_summary_cols = [c for c in summary_cols if c in final_selected_threshold_report.columns]

ensemble_final_summary = (
    final_selected_threshold_report
    .groupby("ensemble")
    .agg(
        n_models=("model_name", "count"),
        **{
            f"avg_{c}": (c, "mean")
            for c in available_summary_cols
        },
        **{
            f"best_{c}": (c, "max")
            for c in available_summary_cols
        }
    )
    .reset_index()
)

print("\n" + "═" * 190)
print("ENSEMBLE SUMMARY — FINAL SELECTED REPORT")
print("═" * 190)

print(
    ensemble_final_summary
    .round(6)
    .to_string(index=False)
)


# =============================================================================
# 6. SAVE
# =============================================================================

SAVE_FINAL_REPORT_DIR = os.path.join(
    globals().get("SAVE_DIR", "final_training_test_expanding"),
    "selected_ensemble_benchmarks"
)

os.makedirs(SAVE_FINAL_REPORT_DIR, exist_ok=True)

final_selected_threshold_report.to_csv(
    f"{SAVE_FINAL_REPORT_DIR}/final_threshold_vs_signonly_report.csv",
    index=False
)

ensemble_final_summary.to_csv(
    f"{SAVE_FINAL_REPORT_DIR}/final_threshold_vs_signonly_ensemble_summary.csv",
    index=False
)

if not missing_final_selected_report.empty:
    missing_final_selected_report.to_csv(
        f"{SAVE_FINAL_REPORT_DIR}/missing_final_threshold_report.csv",
        index=False
    )

print("\n" + "═" * 120)
print("✅ FINAL THRESHOLD VS SIGN-ONLY REPORT TAMAMLANDI")
print(f"📁 SAVE DIR: {SAVE_FINAL_REPORT_DIR}")
print("Kaydedilen dosyalar:")
print("  - final_threshold_vs_signonly_report.csv")
print("  - final_threshold_vs_signonly_ensemble_summary.csv")

if not missing_final_selected_report.empty:
    print("  - missing_final_threshold_report.csv")

print("═" * 120)


# =============================================================================
# 7. QUICK THESIS VIEW
# =============================================================================

quick_cols = [
    "ensemble",
    "channel",
    "model_family",
    "threshold_sharpe",
    "sign_only_sharpe",
    "delta_sharpe",
    "sign_only_accuracy",
    "threshold_after_accuracy",
    "threshold_dd",
    "threshold_sortino",
    "threshold_calmar",
    "threshold_activity",
    "threshold_trade_count",
]

quick_cols = [c for c in quick_cols if c in final_selected_threshold_report.columns]

quick_thesis_view = (
    final_selected_threshold_report[quick_cols]
    .sort_values(["ensemble", "threshold_sharpe"], ascending=[True, False])
    .reset_index(drop=True)
)

print("\n" + "═" * 160)
print("QUICK THESIS VIEW")
print("═" * 160)

print(
    quick_thesis_view
    .round(6)
    .to_string(index=False)
)


══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
FINAL SELECTED REPORT — THRESHOLD VS SIGN-ONLY
Main columns: threshold_sharpe | sign_only_sharpe | delta | accuracy | DD | Sortino | Calmar
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
  ensemble          channel            group                                               feature_set_name model_family                                                          model_name  threshold_sharpe  sign_only_sharpe  delta_sharpe  sign_only_accuracy  threshold_after_accuracy  threshold_dd  sign_only_dd  delta_dd  threshold_sortino  sign_only_sortino  delta_sortino  threshold_calmar  sign_only_calmar  delta_calmar  threshold_ann_ret  sign_only_ann_ret  threshold_cu

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 6F. READABLE SIMILARITY REPORT — TEST SHARPE BEST MODEL VERSION
# Her feature set altında ona en benzeyen en yakın 10 feature set
#
# Ana mantık:
#   1) Her feature set için TEST SHARPE'a göre best model seçilir.
#   2) Bu best modellerin test_signal serileri alınır.
#   3) Signal correlation hesaplanır.
#   4) Feature Jaccard overlap hesaplanır.
#   5) Her feature set altında en yakın 10 benzer set okunabilir raporlanır.
#
# Requires:
#   results_df
#   pred_store
#   SELECTED_FEATURE_SETS
#   FS_GROUP
#   SAVE_DIR
# ═══════════════════════════════════════════════════════════════════════════════

import os
import numpy as np
import pandas as pd
from itertools import combinations

print("\n\n" + "█" * 130)
print("  READABLE SIMILARITY REPORT — TEST SHARPE BEST MODEL VERSION")
print("█" * 130)

# ----------------------------------------------------------------------------
# 0) Gerekli obje kontrolü
# ----------------------------------------------------------------------------

required_objects = [
    "results_df",
    "pred_store",
    "SELECTED_FEATURE_SETS",
    "FS_GROUP",
    "SAVE_DIR",
]

for obj in required_objects:
    if obj not in globals():
        raise ValueError(f"❌ '{obj}' bulunamadı. Önce ana training bloğunu çalıştır.")

if results_df.empty:
    raise ValueError("❌ results_df boş. Önce modelleri eğit.")

os.makedirs(SAVE_DIR, exist_ok=True)


# ----------------------------------------------------------------------------
# 1) HER FEATURE SET İÇİN BEST MODELİ TEST SHARPE'A GÖRE SEÇ
# ----------------------------------------------------------------------------

best_per_fs_test = (
    results_df.sort_values(
        ["test_sharpe", "test_cum_ret", "test_max_dd", "val_sharpe"],
        ascending=[False, False, False, False],
    )
    .groupby("feature_set_name", as_index=False)
    .head(1)
    .sort_values(["group", "test_sharpe"], ascending=[True, False])
    .reset_index(drop=True)
)

# Global isim olarak da güncelleyelim ki aşağıdaki bloklar bunu kullansın
best_per_fs = best_per_fs_test.copy()

print("\n" + "═" * 160)
print("  BEST MODEL PER FEATURE SET — TEST SHARPE'A GÖRE")
print("═" * 160)

best_cols = [
    "group",
    "feature_set_name",
    "model_family",
    "val_sharpe",
    "test_sharpe",
    "test_cum_ret",
    "test_max_dd",
    "test_activity",
    "test_trade_count",
    "best_lower",
    "best_upper",
]

available_best_cols = [c for c in best_cols if c in best_per_fs.columns]
print(best_per_fs[available_best_cols].to_string(index=False))


# ----------------------------------------------------------------------------
# 2) Yardımcı fonksiyonlar
# ----------------------------------------------------------------------------

def short_features(common_features, max_items=8):
    """
    Ortak feature listesini okunabilir kısaltır.
    """
    if common_features is None:
        return "-"

    if isinstance(common_features, str):
        if common_features.strip() == "":
            return "-"
        feats = [x for x in common_features.split("|") if x]
    else:
        feats = list(common_features)

    if len(feats) == 0:
        return "-"

    if len(feats) <= max_items:
        return ", ".join(feats)

    return ", ".join(feats[:max_items]) + f", ... (+{len(feats) - max_items})"


def similarity_label(corr_value):
    """
    Signal correlation yorum etiketi.
    """
    if corr_value >= 0.80:
        return "ÇOK YÜKSEK"
    elif corr_value >= 0.60:
        return "YÜKSEK"
    elif corr_value >= 0.40:
        return "ORTA"
    elif corr_value >= 0.20:
        return "DÜŞÜK-ORTA"
    else:
        return "DÜŞÜK"


def safe_float(x, default=np.nan):
    try:
        if pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default


# ----------------------------------------------------------------------------
# 3) Best model info map
# ----------------------------------------------------------------------------

best_info = {}

for _, row in best_per_fs.iterrows():
    fs = row["feature_set_name"]

    best_info[fs] = {
        "model_name": row.get("model_name", ""),
        "model_family": row.get("model_family", ""),
        "group": row.get("group", FS_GROUP.get(fs, "OTHER")),
        "val_sharpe": safe_float(row.get("val_sharpe", np.nan)),
        "test_sharpe": safe_float(row.get("test_sharpe", np.nan)),
        "test_cum_ret": safe_float(row.get("test_cum_ret", np.nan)),
        "test_max_dd": safe_float(row.get("test_max_dd", np.nan)),
        "test_activity": safe_float(row.get("test_activity", np.nan)),
        "test_trade_count": safe_float(row.get("test_trade_count", np.nan)),
    }


# ----------------------------------------------------------------------------
# 4) Best modellerin test signal serilerini feature_set_name bazında topla
# ----------------------------------------------------------------------------

best_signals_by_fs = {}
missing_pred_models = []

for _, row in best_per_fs.iterrows():
    fs = row["feature_set_name"]
    model_name = row["model_name"]

    if model_name in pred_store and "test_signal" in pred_store[model_name]:
        best_signals_by_fs[fs] = np.asarray(pred_store[model_name]["test_signal"], dtype=float)
    else:
        missing_pred_models.append(model_name)

if missing_pred_models:
    print("\n⚠ pred_store içinde bulunamayan veya test_signal olmayan modeller:")
    for m in missing_pred_models:
        print(f"  - {m}")

if len(best_signals_by_fs) < 2:
    raise ValueError("❌ Similarity analizi için en az 2 best signal gerekli.")

# Uzunluk kontrolü
lengths = {k: len(v) for k, v in best_signals_by_fs.items()}
unique_lengths = sorted(set(lengths.values()))

if len(unique_lengths) != 1:
    print("\n⚠ Signal uzunlukları farklı. En kısa uzunluğa kırpılıyor.")
    min_len = min(unique_lengths)
    best_signals_by_fs = {k: v[-min_len:] for k, v in best_signals_by_fs.items()}

signal_df_readable = pd.DataFrame(best_signals_by_fs).replace([np.inf, -np.inf], np.nan).fillna(0.0)
signal_corr_readable = signal_df_readable.corr().fillna(0.0)


# ----------------------------------------------------------------------------
# 5) Feature Jaccard matrix + common features
# ----------------------------------------------------------------------------

fs_names = list(best_signals_by_fs.keys())

jaccard_map = {}
common_feature_map = {}

for a in fs_names:
    for b in fs_names:
        set_a = set(SELECTED_FEATURE_SETS.get(a, []))
        set_b = set(SELECTED_FEATURE_SETS.get(b, []))

        if a == b:
            jaccard_map[(a, b)] = 1.0
            common_feature_map[(a, b)] = sorted(set_a)
            continue

        inter = sorted(set_a & set_b)
        union = set_a | set_b

        jac = len(inter) / len(union) if len(union) else 0.0

        jaccard_map[(a, b)] = float(jac)
        common_feature_map[(a, b)] = inter


# ----------------------------------------------------------------------------
# 6) Her feature set için en yakın 10 feature seti çıkar
# ----------------------------------------------------------------------------

readable_rows = []

for fs in fs_names:
    own = best_info.get(fs, {})
    own_group = own.get("group", FS_GROUP.get(fs, "OTHER"))

    candidate_rows = []

    for other in fs_names:
        if other == fs:
            continue

        other_info = best_info.get(other, {})
        other_group = other_info.get("group", FS_GROUP.get(other, "OTHER"))

        sig_corr = safe_float(signal_corr_readable.loc[fs, other], 0.0)
        feat_jaccard = jaccard_map.get((fs, other), 0.0)
        common_feats = common_feature_map.get((fs, other), [])

        # Ana rapor signal_corr'a göre sıralanacak.
        # Combined score da kaydediliyor.
        combined_score = 0.75 * max(sig_corr, 0.0) + 0.25 * feat_jaccard

        candidate_rows.append({
            "base_feature_set": fs,
            "base_group": own_group,
            "base_model": own.get("model_family", ""),
            "base_test_sharpe": own.get("test_sharpe", np.nan),
            "base_test_ret": own.get("test_cum_ret", np.nan),
            "base_test_dd": own.get("test_max_dd", np.nan),
            "base_activity": own.get("test_activity", np.nan),

            "similar_feature_set": other,
            "similar_group": other_group,
            "similar_model": other_info.get("model_family", ""),
            "similar_test_sharpe": other_info.get("test_sharpe", np.nan),
            "similar_test_ret": other_info.get("test_cum_ret", np.nan),
            "similar_test_dd": other_info.get("test_max_dd", np.nan),
            "similar_activity": other_info.get("test_activity", np.nan),

            "signal_corr": sig_corr,
            "feature_jaccard": float(feat_jaccard),
            "combined_score": float(combined_score),
            "n_common_features": int(len(common_feats)),
            "common_features": "|".join(common_feats),
            "same_group": own_group == other_group,
        })

    top10 = (
        pd.DataFrame(candidate_rows)
        .sort_values(
            ["signal_corr", "feature_jaccard", "similar_test_sharpe"],
            ascending=[False, False, False],
        )
        .head(10)
        .reset_index(drop=True)
    )

    readable_rows.append(top10)

similarity_top10_df = pd.concat(readable_rows, ignore_index=True)


# ----------------------------------------------------------------------------
# 7) Konsola okunabilir rapor bas
# ----------------------------------------------------------------------------

for fs in fs_names:
    own = best_info.get(fs, {})
    own_group = own.get("group", FS_GROUP.get(fs, "OTHER"))

    own_model = own.get("model_family", "")
    own_test_sh = own.get("test_sharpe", np.nan)
    own_ret = own.get("test_cum_ret", np.nan)
    own_dd = own.get("test_max_dd", np.nan)
    own_act = own.get("test_activity", np.nan)

    print("\n" + "═" * 170)
    print(f"FEATURE SET : {fs}")
    print(f"GROUP       : {own_group}")
    print(
        f"BEST MODEL  : {own_model} | "
        f"TestSh={own_test_sh:+.3f} | "
        f"Ret={own_ret:+.1%} | "
        f"DD={own_dd:+.1%} | "
        f"Act={own_act:.1%}"
    )
    print("─" * 170)
    print("En yakın 10 feature set:")
    print("─" * 170)

    sub = similarity_top10_df[similarity_top10_df["base_feature_set"] == fs].copy()

    for rank, (_, row) in enumerate(sub.iterrows(), start=1):
        sig_corr = row["signal_corr"]
        jac = row["feature_jaccard"]
        same_group_txt = "EVET" if row["same_group"] else "HAYIR"

        print(
            f"{rank:>2}) {row['similar_feature_set']}\n"
            f"    Group={row['similar_group']} | SameGroup={same_group_txt} | "
            f"Model={row['similar_model']} | "
            f"TestSh={row['similar_test_sharpe']:+.3f} | "
            f"Ret={row['similar_test_ret']:+.1%} | "
            f"DD={row['similar_test_dd']:+.1%} | "
            f"Act={row['similar_activity']:.1%}\n"
            f"    SignalCorr={sig_corr:+.3f} ({similarity_label(sig_corr)}) | "
            f"FeatureJaccard={jac:.3f} | "
            f"CommonFeatures={row['n_common_features']}\n"
            f"    Ortak featurelar: {short_features(row['common_features'])}"
        )

    print("═" * 170)


# ----------------------------------------------------------------------------
# 8) Markdown raporu oluştur
# ----------------------------------------------------------------------------

md_lines = []

md_lines.append("# Readable Similarity Report — Test Sharpe Best Model Version\n")
md_lines.append(
    "Bu rapor her feature set için **test Sharpe'a göre seçilmiş en iyi modelin** "
    "test dönemindeki long/short/flat sinyallerine göre en benzer 10 feature seti gösterir.\n"
)

md_lines.append("## Yorumlama\n")
md_lines.append("- **SignalCorr:** Test dönemindeki long/short/flat trading sinyallerinin korelasyonu.\n")
md_lines.append("- **FeatureJaccard:** Ortak feature sayısı / toplam benzersiz feature sayısı.\n")
md_lines.append("- **SameGroup:** İki feature set aynı teori ailesinde mi?\n")
md_lines.append("- **Yüksek SignalCorr + yüksek FeatureJaccard:** Redundancy riski yüksek.\n")
md_lines.append("- **Yüksek SignalCorr + düşük FeatureJaccard:** Farklı teoriler benzer trade davranışı üretmiş olabilir.\n")
md_lines.append("- **Düşük SignalCorr:** Ensemble çeşitliliği açısından daha faydalı olabilir.\n\n")

for fs in fs_names:
    own = best_info.get(fs, {})
    own_group = own.get("group", FS_GROUP.get(fs, "OTHER"))

    md_lines.append("\n---\n")
    md_lines.append(f"## {fs}\n")
    md_lines.append(f"**Group:** {own_group}  \n")
    md_lines.append(
        f"**Best Model:** {own.get('model_family', '')} | "
        f"Test Sharpe: {own.get('test_sharpe', np.nan):+.3f} | "
        f"Return: {own.get('test_cum_ret', np.nan):+.1%} | "
        f"DD: {own.get('test_max_dd', np.nan):+.1%} | "
        f"Activity: {own.get('test_activity', np.nan):.1%}\n\n"
    )

    sub = similarity_top10_df[similarity_top10_df["base_feature_set"] == fs].copy()

    md_lines.append(
        "| Rank | Similar Feature Set | Group | Same Group | Model | "
        "Test Sharpe | Signal Corr | Feature Jaccard | Common Features |\n"
    )
    md_lines.append("|---:|---|---|---|---|---:|---:|---:|---|\n")

    for rank, (_, row) in enumerate(sub.iterrows(), start=1):
        common_short = short_features(row["common_features"], max_items=6)
        same_group_txt = "Yes" if row["same_group"] else "No"

        md_lines.append(
            f"| {rank} | `{row['similar_feature_set']}` | "
            f"{row['similar_group']} | "
            f"{same_group_txt} | "
            f"{row['similar_model']} | "
            f"{row['similar_test_sharpe']:+.3f} | "
            f"{row['signal_corr']:+.3f} | "
            f"{row['feature_jaccard']:.3f} | "
            f"{common_short} |\n"
        )

readable_report_md = "\n".join(md_lines)


# ----------------------------------------------------------------------------
# 9) Ek özet tablolar
# ----------------------------------------------------------------------------

# Her feature setin ortalama signal correlation değeri
diversity_rows = []

for fs in fs_names:
    others = [
        signal_corr_readable.loc[fs, other]
        for other in fs_names
        if other != fs
    ]
    diversity_rows.append({
        "feature_set_name": fs,
        "group": best_info.get(fs, {}).get("group", FS_GROUP.get(fs, "OTHER")),
        "model_family": best_info.get(fs, {}).get("model_family", ""),
        "test_sharpe": best_info.get(fs, {}).get("test_sharpe", np.nan),
        "avg_signal_corr": float(np.nanmean(others)) if len(others) else 0.0,
        "max_signal_corr": float(np.nanmax(others)) if len(others) else 0.0,
        "min_signal_corr": float(np.nanmin(others)) if len(others) else 0.0,
    })

similarity_diversity_df = (
    pd.DataFrame(diversity_rows)
    .sort_values(["avg_signal_corr", "test_sharpe"], ascending=[True, False])
    .reset_index(drop=True)
)

print("\n\n" + "═" * 130)
print("  SIGNAL DIVERSITY SUMMARY — DÜŞÜK AVG CORR DAHA ÇEŞİTLİ")
print("═" * 130)
print(similarity_diversity_df.to_string(index=False))


# En benzer global çiftler
pair_rows = []

for a, b in combinations(fs_names, 2):
    pair_rows.append({
        "feature_set_a": a,
        "feature_set_b": b,
        "group_a": best_info.get(a, {}).get("group", FS_GROUP.get(a, "OTHER")),
        "group_b": best_info.get(b, {}).get("group", FS_GROUP.get(b, "OTHER")),
        "model_a": best_info.get(a, {}).get("model_family", ""),
        "model_b": best_info.get(b, {}).get("model_family", ""),
        "test_sharpe_a": best_info.get(a, {}).get("test_sharpe", np.nan),
        "test_sharpe_b": best_info.get(b, {}).get("test_sharpe", np.nan),
        "signal_corr": float(signal_corr_readable.loc[a, b]),
        "feature_jaccard": float(jaccard_map.get((a, b), 0.0)),
        "n_common_features": len(common_feature_map.get((a, b), [])),
        "common_features": "|".join(common_feature_map.get((a, b), [])),
    })

similarity_pairs_df = (
    pd.DataFrame(pair_rows)
    .sort_values(["signal_corr", "feature_jaccard"], ascending=[False, False])
    .reset_index(drop=True)
)

print("\n" + "═" * 130)
print("  GLOBAL EN BENZER 20 FEATURE SET ÇİFTİ")
print("═" * 130)
print(
    similarity_pairs_df[
        [
            "feature_set_a",
            "feature_set_b",
            "signal_corr",
            "feature_jaccard",
            "n_common_features",
            "test_sharpe_a",
            "test_sharpe_b",
        ]
    ].head(20).to_string(index=False)
)


# ----------------------------------------------------------------------------
# 10) Kaydet
# ----------------------------------------------------------------------------

similarity_top10_df.to_csv(
    f"{SAVE_DIR}/readable_similarity_top10_by_feature_set_TEST_SHARPE_BEST.csv",
    index=False,
)

similarity_diversity_df.to_csv(
    f"{SAVE_DIR}/similarity_diversity_summary_TEST_SHARPE_BEST.csv",
    index=False,
)

similarity_pairs_df.to_csv(
    f"{SAVE_DIR}/similarity_pairs_global_TEST_SHARPE_BEST.csv",
    index=False,
)

signal_corr_readable.to_csv(
    f"{SAVE_DIR}/signal_correlation_matrix_TEST_SHARPE_BEST.csv",
)

with open(
    f"{SAVE_DIR}/readable_similarity_report_TEST_SHARPE_BEST.md",
    "w",
    encoding="utf-8",
) as f:
    f.write(readable_report_md)

with open(
    f"{SAVE_DIR}/readable_similarity_report_TEST_SHARPE_BEST.txt",
    "w",
    encoding="utf-8",
) as f:
    f.write(readable_report_md)

best_per_fs.to_csv(
    f"{SAVE_DIR}/best_per_feature_set_TEST_SHARPE.csv",
    index=False,
)

print("\n\n" + "═" * 130)
print("✅ READABLE SIMILARITY REPORT KAYDEDİLDİ — TEST SHARPE BEST MODEL VERSION")
print(f"📁 {SAVE_DIR}/readable_similarity_top10_by_feature_set_TEST_SHARPE_BEST.csv")
print(f"📁 {SAVE_DIR}/similarity_diversity_summary_TEST_SHARPE_BEST.csv")
print(f"📁 {SAVE_DIR}/similarity_pairs_global_TEST_SHARPE_BEST.csv")
print(f"📁 {SAVE_DIR}/signal_correlation_matrix_TEST_SHARPE_BEST.csv")
print(f"📁 {SAVE_DIR}/readable_similarity_report_TEST_SHARPE_BEST.md")
print(f"📁 {SAVE_DIR}/readable_similarity_report_TEST_SHARPE_BEST.txt")
print(f"📁 {SAVE_DIR}/best_per_feature_set_TEST_SHARPE.csv")
print("═" * 130)

print("\nGlobal objects:")
print("  best_per_fs")
print("  best_per_fs_test")
print("  signal_corr_readable")
print("  similarity_top10_df")
print("  similarity_diversity_df")
print("  similarity_pairs_df")
print("  readable_report_md")



██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  READABLE SIMILARITY REPORT — TEST SHARPE BEST MODEL VERSION
██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████

════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
  BEST MODEL PER FEATURE SET — TEST SHARPE'A GÖRE
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
           group                                               feature_set_name model_family  val_sharpe  test_sharpe  test_cum_ret  test_max_dd  test_activity  test_trade_count  best_lower  best_upper
          CREDIT        CREDIT_SOV_RATES_SELECTED_02_credit_spread_risk_compact         LGBM    0.9

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 6F. PROFITABLE SIGNAL CORRELATION ANALYSIS
# Sadece para kazandıran test sinyallerinin korelasyonu
#
# Mantık:
#   signal = +1 ise ve Y_TE > 0 ise kazandıran long
#   signal = -1 ise ve Y_TE < 0 ise kazandıran short
#   signal = 0 ise trade yok
#
# Ölçülenler:
#   profitable_signal_corr : kazandıran long/short sinyal yönleri benzer mi?
#   profit_timing_corr     : aynı zamanlarda mı para kazanıyorlar?
#   profit_return_corr     : kazandıran trade return büyüklükleri benzer mi?
#   profitable_jaccard     : kazanan trade zamanları ne kadar çakışıyor?
# ═══════════════════════════════════════════════════════════════════════════════

print("\n\n" + "█" * 100)
print("  6F. PROFITABLE SIGNAL CORRELATION ANALYSIS")
print("█" * 100)

profitable_signal_corr = pd.DataFrame()
profit_timing_corr = pd.DataFrame()
profit_return_corr = pd.DataFrame()
profitable_signal_pairs_df = pd.DataFrame()
profitable_signal_summary_df = pd.DataFrame()
profitable_signal_df = pd.DataFrame()
profit_mask_df = pd.DataFrame()
profit_return_df = pd.DataFrame()

if len(best_model_names) >= 2:

    profitable_signal_map = {}
    profit_mask_map = {}
    profit_return_map = {}
    raw_signal_map = {}

    for model_name in best_model_names:

        if model_name not in pred_store:
            print(f"⚠ pred_store içinde yok: {model_name}")
            continue

        if "test_signal" not in pred_store[model_name]:
            print(f"⚠ test_signal yok: {model_name}")
            continue

        fs_name = pred_store[model_name].get(
            "feature_set_name",
            model_name.replace("__XGB", "").replace("__LGBM", "").replace("__HGBM", "")
        )

        sig = np.asarray(pred_store[model_name]["test_signal"], dtype=float).reshape(-1)
        yte = np.asarray(Y_TE, dtype=float).reshape(-1)

        # Uzunluk güvenliği
        min_len = min(len(sig), len(yte))
        sig = sig[-min_len:]
        yte = yte[-min_len:]

        # Transaction cost'suz brüt yönsel trade sonucu
        # Burada amaç "sinyal yönü doğru muydu?" sorusunu ölçmek.
        gross_trade_ret = sig * yte

        # Trade açılan barlar
        trade_mask = sig != 0

        # Para kazandıran sinyaller
        profit_mask = trade_mask & (gross_trade_ret > 0)

        # Kazandıran yerlerde sinyali koru, diğer yerleri 0 yap
        profitable_sig = np.where(profit_mask, sig, 0.0)

        # Kazandıran trade return büyüklüğü
        profitable_ret = np.where(profit_mask, gross_trade_ret, 0.0)

        raw_signal_map[fs_name] = sig
        profitable_signal_map[fs_name] = profitable_sig
        profit_mask_map[fs_name] = profit_mask.astype(float)
        profit_return_map[fs_name] = profitable_ret

    if len(profitable_signal_map) >= 2:

        profitable_signal_df = (
            pd.DataFrame(profitable_signal_map)
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0.0)
        )

        profit_mask_df = (
            pd.DataFrame(profit_mask_map)
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0.0)
        )

        profit_return_df = (
            pd.DataFrame(profit_return_map)
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0.0)
        )

        # 1) Kazandıran long/short yönlerinin korelasyonu
        profitable_signal_corr = profitable_signal_df.corr().fillna(0.0)

        # 2) Aynı zamanlarda mı para kazanıyorlar?
        profit_timing_corr = profit_mask_df.corr().fillna(0.0)

        # 3) Kazandıran trade return büyüklükleri benzer mi?
        profit_return_corr = profit_return_df.corr().fillna(0.0)

        print("\n  ── 6F-1. PROFITABLE SIGNAL CORRELATION MATRIX")
        print(profitable_signal_corr.round(3).to_string())

        # ---------------------------------------------------------------------
        # Pair bazlı detay tablo
        # ---------------------------------------------------------------------

        profitable_pair_rows = []
        fs_profit_names = list(profitable_signal_map.keys())

        for a, b in combinations(fs_profit_names, 2):

            sig_a = raw_signal_map[a]
            sig_b = raw_signal_map[b]

            prof_sig_a = profitable_signal_map[a]
            prof_sig_b = profitable_signal_map[b]

            mask_a = profit_mask_map[a].astype(bool)
            mask_b = profit_mask_map[b].astype(bool)

            both_profit = mask_a & mask_b
            either_profit = mask_a | mask_b

            both_profit_count = int(both_profit.sum())
            either_profit_count = int(either_profit.sum())

            profitable_jaccard = (
                both_profit_count / either_profit_count
                if either_profit_count > 0 else 0.0
            )

            same_direction_profit = (
                both_profit &
                (prof_sig_a == prof_sig_b) &
                (prof_sig_a != 0)
            )

            opposite_direction_profit = (
                both_profit &
                (prof_sig_a == -prof_sig_b) &
                (prof_sig_a != 0) &
                (prof_sig_b != 0)
            )

            same_direction_profit_ratio = (
                float(same_direction_profit.sum() / both_profit_count)
                if both_profit_count > 0 else 0.0
            )

            opposite_direction_profit_ratio = (
                float(opposite_direction_profit.sum() / both_profit_count)
                if both_profit_count > 0 else 0.0
            )

            a_total_trades = int((sig_a != 0).sum())
            b_total_trades = int((sig_b != 0).sum())

            a_profit_count = int(mask_a.sum())
            b_profit_count = int(mask_b.sum())

            profitable_pair_rows.append({
                "feature_set_a": a,
                "feature_set_b": b,

                "group_a": FS_GROUP.get(a, "OTHER"),
                "group_b": FS_GROUP.get(b, "OTHER"),

                "profitable_signal_corr": float(profitable_signal_corr.loc[a, b]),
                "profit_timing_corr": float(profit_timing_corr.loc[a, b]),
                "profit_return_corr": float(profit_return_corr.loc[a, b]),

                "profitable_jaccard": float(profitable_jaccard),
                "both_profit_count": both_profit_count,
                "either_profit_count": either_profit_count,

                "same_direction_profit_ratio": same_direction_profit_ratio,
                "opposite_direction_profit_ratio": opposite_direction_profit_ratio,

                "a_total_trades": a_total_trades,
                "b_total_trades": b_total_trades,
                "a_profit_count": a_profit_count,
                "b_profit_count": b_profit_count,

                "a_profit_ratio": a_profit_count / a_total_trades if a_total_trades > 0 else 0.0,
                "b_profit_ratio": b_profit_count / b_total_trades if b_total_trades > 0 else 0.0,
            })

        profitable_signal_pairs_df = (
            pd.DataFrame(profitable_pair_rows)
            .sort_values(
                [
                    "profitable_signal_corr",
                    "profitable_jaccard",
                    "same_direction_profit_ratio",
                    "both_profit_count",
                ],
                ascending=[False, False, False, False],
            )
            .reset_index(drop=True)
        )

        print("\n  ── 6F-2. EN BENZER 30 PROFITABLE SIGNAL ÇİFTİ")
        print(
            f"{'#':<4} "
            f"{'Feature Set A':<65} "
            f"{'Feature Set B':<65} "
            f"{'ProfCorr':>9} "
            f"{'TimeCorr':>9} "
            f"{'RetCorr':>9} "
            f"{'Jac':>7} "
            f"{'Both':>7} "
            f"{'SameDir':>9}"
        )
        print("─" * 190)

        for i, row in profitable_signal_pairs_df.head(30).iterrows():
            c = row["profitable_signal_corr"]
            label = (
                "🔴 ÇOK YÜKSEK" if c >= 0.80 else
                "🟡 YÜKSEK" if c >= 0.60 else
                "🟢 ORTA" if c >= 0.40 else
                "✅ DÜŞÜK"
            )

            print(
                f"{i+1:<4} "
                f"{row['feature_set_a']:<65} "
                f"{row['feature_set_b']:<65} "
                f"{row['profitable_signal_corr']:>9.3f} "
                f"{row['profit_timing_corr']:>9.3f} "
                f"{row['profit_return_corr']:>9.3f} "
                f"{row['profitable_jaccard']:>7.3f} "
                f"{int(row['both_profit_count']):>7} "
                f"{row['same_direction_profit_ratio']:>9.1%}  "
                f"{label}"
            )

        print("\n  ── 6F-3. EN ÇEŞİTLİ 15 PROFITABLE SIGNAL ÇİFTİ")
        print(
            f"{'#':<4} "
            f"{'Feature Set A':<65} "
            f"{'Feature Set B':<65} "
            f"{'ProfCorr':>9} "
            f"{'Jac':>7} "
            f"{'Both':>7}"
        )
        print("─" * 165)

        most_diverse_profitable_pairs = (
            profitable_signal_pairs_df
            .sort_values(
                ["profitable_signal_corr", "profitable_jaccard", "both_profit_count"],
                ascending=[True, True, False]
            )
            .head(15)
            .reset_index(drop=True)
        )

        for i, row in most_diverse_profitable_pairs.iterrows():
            print(
                f"{i+1:<4} "
                f"{row['feature_set_a']:<65} "
                f"{row['feature_set_b']:<65} "
                f"{row['profitable_signal_corr']:>9.3f} "
                f"{row['profitable_jaccard']:>7.3f} "
                f"{int(row['both_profit_count']):>7}"
            )

        # ---------------------------------------------------------------------
        # Model bazlı profitable diversity summary
        # ---------------------------------------------------------------------

        profitable_summary_rows = []

        for fs in fs_profit_names:

            sig = raw_signal_map[fs]
            mask = profit_mask_map[fs].astype(bool)

            total_trades = int((sig != 0).sum())
            profitable_trades = int(mask.sum())
            profitable_ratio = profitable_trades / total_trades if total_trades > 0 else 0.0

            other_corrs = [
                profitable_signal_corr.loc[fs, other]
                for other in fs_profit_names
                if other != fs
            ]

            other_timing_corrs = [
                profit_timing_corr.loc[fs, other]
                for other in fs_profit_names
                if other != fs
            ]

            info_row = best_per_fs[best_per_fs["feature_set_name"] == fs]

            if len(info_row):
                model_family = info_row.iloc[0].get("model_family", "")
                test_sharpe = float(info_row.iloc[0].get("test_sharpe", np.nan))
                test_cum_ret = float(info_row.iloc[0].get("test_cum_ret", np.nan))
                test_max_dd = float(info_row.iloc[0].get("test_max_dd", np.nan))
                test_activity = float(info_row.iloc[0].get("test_activity", np.nan))
            else:
                model_family = ""
                test_sharpe = np.nan
                test_cum_ret = np.nan
                test_max_dd = np.nan
                test_activity = np.nan

            profitable_summary_rows.append({
                "feature_set_name": fs,
                "group": FS_GROUP.get(fs, "OTHER"),
                "model_family": model_family,

                "test_sharpe": test_sharpe,
                "test_cum_ret": test_cum_ret,
                "test_max_dd": test_max_dd,
                "test_activity": test_activity,

                "total_trades": total_trades,
                "profitable_trades": profitable_trades,
                "profitable_trade_ratio": profitable_ratio,

                "avg_profitable_signal_corr": float(np.nanmean(other_corrs)) if len(other_corrs) else 0.0,
                "max_profitable_signal_corr": float(np.nanmax(other_corrs)) if len(other_corrs) else 0.0,
                "min_profitable_signal_corr": float(np.nanmin(other_corrs)) if len(other_corrs) else 0.0,

                "avg_profit_timing_corr": float(np.nanmean(other_timing_corrs)) if len(other_timing_corrs) else 0.0,
                "max_profit_timing_corr": float(np.nanmax(other_timing_corrs)) if len(other_timing_corrs) else 0.0,
            })

        profitable_signal_summary_df = (
            pd.DataFrame(profitable_summary_rows)
            .sort_values(
                ["avg_profitable_signal_corr", "test_sharpe"],
                ascending=[True, False],
            )
            .reset_index(drop=True)
        )

        print("\n  ── 6F-4. PROFITABLE SIGNAL DIVERSITY SUMMARY")
        print("  Düşük AvgProfCorr = kazandıran sinyallerde daha bağımsız model")
        print(
            f"{'#':<4} "
            f"{'Feature Set':<75} "
            f"{'Model':<6} "
            f"{'TestSh':>8} "
            f"{'Trades':>7} "
            f"{'Prof':>7} "
            f"{'Prof%':>8} "
            f"{'AvgProfCorr':>12} "
            f"{'AvgTimeCorr':>12} "
            f"{'Yorum':<15}"
        )
        print("─" * 165)

        for i, row in profitable_signal_summary_df.iterrows():

            c = row["avg_profitable_signal_corr"]

            label = (
                "✅ Çok çeşitli" if c < 0.20 else
                "🟢 Çeşitli" if c < 0.35 else
                "🟡 Orta" if c < 0.50 else
                "🔴 Benzer"
            )

            print(
                f"{i+1:<4} "
                f"{row['feature_set_name']:<75} "
                f"{row['model_family']:<6} "
                f"{row['test_sharpe']:>8.3f} "
                f"{int(row['total_trades']):>7} "
                f"{int(row['profitable_trades']):>7} "
                f"{row['profitable_trade_ratio']:>8.1%} "
                f"{row['avg_profitable_signal_corr']:>12.3f} "
                f"{row['avg_profit_timing_corr']:>12.3f} "
                f"{label:<15}"
            )

    else:
        print("⚠ Profitable signal analizi için yeterli sinyal yok.")

else:
    print("⚠ Profitable signal analizi için en az 2 best model gerekli.")



████████████████████████████████████████████████████████████████████████████████████████████████████
  6F. PROFITABLE SIGNAL CORRELATION ANALYSIS
████████████████████████████████████████████████████████████████████████████████████████████████████

  ── 6F-1. PROFITABLE SIGNAL CORRELATION MATRIX
                                                                CREDIT_SOV_RATES_SELECTED_02_credit_spread_risk_compact  CURVE_GBM_FORCED_TEST_SELECTED_01_curve_liq_money_risk  CURVE_GROWTH_RATES_SELECTED_1_policy_curve_rates  RF_MACRO_KS_01_curve_liq_realrate_policy  ENERGY_TOT_SELECTED_1_brent_gas_eu_pressure  ENERGY_GOLD_RISK_SELECTED_01_energy_gold_oil_risk  GK_RF_SELECTED_01_gamma_vega_vrp_carry_relD1  GK_RF_SELECTED_03_compact_gamma_vrp_carry  POLICY_GROWTH_RATES_SELECTED_01_growth_policy_curve  GROWTH_CURVE_SELECTED_3_credit_curve_growth  INFL_POLICY_RATES_SELECTED_3_real_yield_momentum  LIQUIDITY_HGBM_TEST_SELECTED_03_macro_liquidity_rates_risk  LIQUIDITY_PRESSURE_SELECTED_1_money_netl

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# MULTI-ENSEMBLE × HMM FIXED CANDIDATES
#
# Ensembles:
#   1) MAIN_14
#   2) THEORY_11
#   3) ENSEMBLE_B
#
# Modes:
#   A : State-specific weights + GLOBAL threshold
#   B : State-specific weights + STATE threshold
#   D : Posterior-weighted soft state assignment
#   C : Regime-free global weights + global threshold
#
# NEW:
#   - EVAL_OUTPUT_STORE eklendi.
#   - C/A/B/D modlarının val/test ret-signal-prob çıktıları saklanıyor.
#   - En sonda eval_output_store.pkl olarak kaydediliyor.
#
# Sıralama: test_sharpe
# ═══════════════════════════════════════════════════════════════════════════════

import os
import gc
import hashlib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:
    sns = None

from scipy.special import logsumexp
from sklearn.preprocessing import RobustScaler, StandardScaler
from hmmlearn.hmm import GaussianHMM

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import pickle


# =============================================================================
# 0. LOAD PRED STORE
# =============================================================================

with open("meto/final.pkl", "rb") as f:
    pred_store = pickle.load(f)

print(f"✅ pred_store yüklendi: {len(pred_store)} model")


# =============================================================================
# 1. GLOBAL SETTINGS
# =============================================================================

SAVE_DIR   = "multi_ens_fixed_hmm_outputs"
MATRIX_DIR = os.path.join(SAVE_DIR, "state_feature_matrices")

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(MATRIX_DIR, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

BAR_ANN     = globals().get("BAR_ANN", 6 * 252)
TC_PER_SIDE = globals().get("TC_PER_SIDE", 0.00005)

HMM_N_ITER          = 500
HMM_TOL             = 1e-4
MIN_HMM_TRAIN_OBS   = 300
MIN_STATE_FREQ      = 0.03
MIN_AVG_PERSISTENCE = 0.45
MIN_HMM_FEATURES    = 3

MIN_COMBO         = 3
MIN_VAL_OBS_STATE = 40

N_OPT_TRIALS_FINAL = 150

DEFAULT_UP = (0.48, 0.72)
DEFAULT_LO = (0.28, 0.52)


# =============================================================================
# 2. OUTPUT STORE — NEW
# =============================================================================
# Burada C/A/B/D modlarının bar bazlı çıktıları saklanacak.
# Sonraki rapor kodu buradan yıl yıl return/sharpe çıkaracak.

EVAL_OUTPUT_STORE = {}

def store_eval_output(
    store,
    ensemble_name,
    candidate,
    mode,
    val_df_out=None,
    test_df_out=None,
    extra_info=None,
):
    key = f"{ensemble_name}__{candidate}__{mode}"

    store[key] = {
        "key": key,
        "ensemble_name": ensemble_name,
        "candidate": candidate,
        "mode": mode,
        "extra_info": extra_info if extra_info is not None else {},
    }

    if val_df_out is not None:
        store[key]["val_index"] = val_df_out.index.copy()
        store[key]["val_ret"] = val_df_out["ret"].values.copy() if "ret" in val_df_out.columns else None
        store[key]["val_signal"] = val_df_out["signal"].values.copy() if "signal" in val_df_out.columns else None
        store[key]["val_prob"] = val_df_out["prob"].values.copy() if "prob" in val_df_out.columns else None

    if test_df_out is not None:
        store[key]["test_index"] = test_df_out.index.copy()
        store[key]["test_ret"] = test_df_out["ret"].values.copy() if "ret" in test_df_out.columns else None
        store[key]["test_signal"] = test_df_out["signal"].values.copy() if "signal" in test_df_out.columns else None
        store[key]["test_prob"] = test_df_out["prob"].values.copy() if "prob" in test_df_out.columns else None

    return key


# =============================================================================
# 3. REQUIRED OBJECT CHECK
# =============================================================================

for obj in ["master_df", "train_df", "val_df", "test_df", "pred_store"]:
    if obj not in globals():
        raise ValueError(f"❌ {obj} bulunamadı.")

if not isinstance(pred_store, dict) or len(pred_store) == 0:
    raise ValueError("❌ pred_store boş veya dict değil.")

master_df = master_df.copy()
train_df  = train_df.copy()
val_df    = val_df.copy()
test_df   = test_df.copy()

for df in [master_df, train_df, val_df, test_df]:
    df.index = pd.to_datetime(df.index)
    df.sort_index(inplace=True)

if "fwd_ret" in master_df.columns:
    train_df = master_df.reindex(train_df.index).copy()
    val_df   = master_df.reindex(val_df.index).copy()
    test_df  = master_df.reindex(test_df.index).copy()

for name, df in [("train_df", train_df), ("val_df", val_df), ("test_df", test_df)]:
    if "fwd_ret" not in df.columns:
        raise ValueError(f"❌ {name} içinde fwd_ret yok.")

print("═" * 160)
print("MULTI-ENSEMBLE × HMM FIXED CANDIDATES")
print("Candidates: HMM_CLASSIC_LITERATURE_CORE K5 | HMM_FULL_CORE K5")
print("Ensembles : MAIN_14 / THEORY_11 / ENSEMBLE_B")
print("Sıralama  : test_sharpe")
print("Output    : results CSV + eval_output_store.pkl")
print("═" * 160)
print(f"master_df      : {master_df.shape}")
print(f"train/val/test : {len(train_df)} / {len(val_df)} / {len(test_df)}")
print(f"pred_store     : {len(pred_store)} models")
print(f"BAR_ANN        : {BAR_ANN}  |  TC_PER_SIDE: {TC_PER_SIDE}")
print("═" * 160)


# =============================================================================
# 4. ENSEMBLE DEFINITIONS
# =============================================================================

MAIN_14_ENSEMBLE = {
    "RE4AL_RATE":         "T01_REAL_RATE_SELECTED_3_growth_realrate_channel__LGBM",
    "MACRO_SURPRISE":    "MACRO_KS_SELECTED_1_infl_energy_real_yield_spx__XGB",
    "CURVE_LIQ_POLICY":  "RF_MACRO_KS_01_curve_liq_realrate_policy__LGBM",
    "LIQUIDITY":         "LIQUIDITY_HGBM_TEST_SELECTED_03_macro_liquidity_rates_risk__HGBM",
    "INFL_POLICY":       "INFL_POLICY_RATES_SELECTED_3_real_yield_momentum__XGB",
    "GK_GREEKS":         "GK_RF_SELECTED_01_gamma_vega_vrp_carry_relD1__XGB",
    "SAFE_HAVEN":        "BOP_SAFEHAVEN_SELECTED_1_chf_jpy_sovereign_risk__LGBM",
    "SOVEREIGN":         "SOVEREIGN_SELECTED_2_it_de_rates_risk__LGBM",
    "CREDIT":            "CREDIT_SOV_RATES_SELECTED_02_credit_spread_risk_compact__LGBM",
    "GROWTH_POLICY":     "POLICY_GROWTH_RATES_SELECTED_01_growth_policy_curve__LGBM",
    "ENERGY":            "ENERGY_GOLD_RISK_SELECTED_01_energy_gold_oil_risk__LGBM",
    "PPP":               "T04_PPP_SELECTED_4_DXY1_clean_inflation_m2_dxy_mom63__XGB",
    "UIP_CARRY":         "TAYLOR_UIP_XGB_TEST_AUG01_policy_carry_volrisk_score_gap__XGB",
    "TAYLOR":            "TAYLOR_GBM_VAL_SELECTED_01_growth_inflation_policy_rates__XGB",
}

ENSEMBLE_C_THEORY_11 = {
    "REAL_RATE":   "T01_REAL_RATE_SELECTED_3_growth_realrate_channel__LGBM",
    "LIQUIDITY":   "LIQUIDITY_PRESSURE_SELECTED_1_money_netliq_risk__LGBM",
    "INFL_POLICY": "INFL_POLICY_RATES_SELECTED_3_real_yield_momentum__XGB",
    "GK_OPTIONS":  "GK_RF_SELECTED_03_compact_gamma_vrp_carry__LGBM",
    "SAFE_HAVEN":  "BOP_SAFEHAVEN_SELECTED_1_chf_jpy_sovereign_risk__LGBM",
    "SOVEREIGN":   "SOVEREIGN_SELECTED_2_it_de_rates_risk__LGBM",
    "CREDIT":      "CREDIT_SOV_RATES_SELECTED_02_credit_spread_risk_compact__LGBM",
    "ENERGY":      "ENERGY_TOT_SELECTED_1_brent_gas_eu_pressure__LGBM",
    "UIP_CARRY":   "UIP_HGBM_TEST_AUG01_policy_carry_risk_gap_vol__XGB",
    "TAYLOR_LIKE": "TAYLOR_XGB_FULL_MACRO_PURE_ROBUST_01_growth_policy_score_rates__XGB",
    "CURVE":       "CURVE_GROWTH_RATES_SELECTED_1_policy_curve_rates__LGBM",
}

ENSEMBLE_B_SELECTED_MODELS = {
    "FISHER_REAL_RATE": "RF_T01_REAL_RATE_PARITY_01_fisher_realrate_breakeven__LGBM",
    "MACRO_DXY_LIQ": "MACRO_KS_SELECTED_3_DXY2_liq_real_yield_growth_dxy_mom5_chg5__LGBM",
    "LIQUIDITY_CLEAN": "LIQUIDITY_PRESSURE_SELECTED_1_money_netliq_risk__LGBM",
    "INFL_POLICY": "INFL_POLICY_RATES_SELECTED_3_real_yield_momentum__LGBM",
    "GK_WIDE": "GK_RF_SELECTED_01_gamma_vega_vrp_carry_relD1__HGBM",
    "SAFE_HAVEN": "BOP_SAFEHAVEN_SELECTED_1_chf_jpy_sovereign_risk__LGBM",
    "SOVEREIGN": "SOVEREIGN_SELECTED_2_it_de_rates_risk__LGBM",
    "CREDIT": "CREDIT_SOV_RATES_SELECTED_02_credit_spread_risk_compact__XGB",
    "GROWTH_CURVE": "GROWTH_CURVE_SELECTED_3_credit_curve_growth__LGBM",
    "ENERGY": "ENERGY_TOT_SELECTED_1_brent_gas_eu_pressure__LGBM",
    "PPP_ALT": "PPP_HGBM_FORCED_TEST_SELECTED_01_dominance_gold_spx__LGBM",
    "TAYLOR_UIP": "TAYLOR_UIP_XGB_TEST_AUG01_policy_carry_volrisk_score_gap__XGB",
    "TAYLOR_RISK": "TAYLOR_GBM_VAL_SELECTED_01_growth_inflation_policy_rates__HGBM",
    "CURVE_LIQ_MONEY": "CURVE_GBM_FORCED_TEST_SELECTED_01_curve_liq_money_risk__HGBM",
}

MULTI_ENSEMBLE_DEFINITIONS = {
    "MAIN_14":    MAIN_14_ENSEMBLE,
    "THEORY_11":  ENSEMBLE_C_THEORY_11,
    "ENSEMBLE_B": ENSEMBLE_B_SELECTED_MODELS,
}

print("\nENSEMBLE DEFINITIONS:")
for ens_name, ens_dict in MULTI_ENSEMBLE_DEFINITIONS.items():
    print(f"  {ens_name:<12} → {len(ens_dict)} models")


# =============================================================================
# 5. HMM FEATURE SETS
# =============================================================================

HMM_CANDIDATES_FIXED = [
    ("HMM_CLASSIC_LITERATURE_CORE", "HMM_CLASSIC_LITERATURE_CORE", 5),
    ("HMM_FULL_CORE_K5",            "HMM_FULL_CORE",                 5),

]

HMM_FEATURE_SETS = {
    "HMM_FULL_CORE": [
        "risk_score_z63",
        "growth_mom21",
        "liq_score",
        "real_yield_z63",
        "vix_z63",
        "gold_mom21",
        "policy_diff_fed_ecb_dfr",
        "rates_score",
        "spread_2y10y",
    ],


    "HMM_CLASSIC_LITERATURE_CORE": [
    "hist_vol_20",
    "vix_z63",
    "spread_2y10y",
    "policy_diff_fed_ecb_dfr",
    "corr_eur_gbp",
    "corr_eur_chf"
]
}

print(f"\nHMM CANDIDATES (sabit {len(HMM_CANDIDATES_FIXED)}):")
for cname, skey, k in HMM_CANDIDATES_FIXED:
    feats = HMM_FEATURE_SETS.get(skey, [])
    print(f"  {cname:<35} K={k}  ({len(feats)} features)")


# =============================================================================
# 6. DERIVED FEATURES + ALIASES
# =============================================================================

def rolling_z(s, w):
    s = pd.Series(s).astype(float)
    return (s - s.rolling(w).mean()) / (s.rolling(w).std() + 1e-10)

BAR_21 = 21 * 6
BAR_63 = 63 * 6

DERIVED_SPECS = {
    "risk_score_z63":          ("risk_score",          "z63"),
    "liq_score_z63":           ("liq_score",           "z63"),
    "growth_score_z63":        ("growth_score",        "z63"),
    "vix_z63":                 ("vix_level",           "z63"),
    "rates_z63":               ("spread_2y10y",        "z63"),
    "inflation_z63":           ("inflation_score",     "z63"),
    "inflation_gap_us_ea_z63": ("inflation_gap_us_ea", "z63"),
    "real_yield_z63":          ("us10y_real",          "z63"),
    "gold_mom21":              ("gold_ret",            "diff21"),
}

for new_col, (src, mode) in DERIVED_SPECS.items():
    if new_col not in master_df.columns and src in master_df.columns:
        if mode == "z63":
            master_df[new_col] = rolling_z(master_df[src], BAR_63)
        elif mode == "diff21":
            master_df[new_col] = master_df[src].diff(BAR_21)

ALIASES = {
    "vix_z63":                 ["vix_z", "vix_level", "VIX_z63"],
    "real_yield_z63":          ["us10y_real_z63", "us10y_real", "DFII10_z63"],
    "spread_2y10y":            ["us_curve_10y2y", "curve_z63", "us_curve_z63"],
    "us_breakeven_10y":        ["us10y_be_z63", "breakeven_z63", "us10y_be"],
    "policy_score":            ["policy_diff_z63", "policy_diff_fed_ecb_dfr"],
    "policy_diff_fed_ecb_dfr": ["policy_diff", "policy_score"],
    "inflation_gap_us_ea":     ["inflation_gap_us_ea_z63", "inflation_gap"],
    "inflation_gap_us_ea_z63": ["inflation_gap_us_ea"],
    "sovereign_score":         ["sovereign_z63", "it_de_10y_spread_z63"],
    "gold_mom21":              ["gold_mom_21", "gold_chg21"],
    "it_de_10y_spread":        ["it_de_10y_spread_z63", "it_ea_10y_spread"],
    "vix_level":               ["vix", "VIX"],
    "rates_z63":               ["spread_2y10y_z63", "curve_accel"],
}

for target, srcs in ALIASES.items():
    if target not in master_df.columns:
        for src in srcs:
            if src in master_df.columns:
                master_df[target] = master_df[src]
                break

if "fwd_ret" in master_df.columns:
    train_df = master_df.reindex(train_df.index).copy()
    val_df   = master_df.reindex(val_df.index).copy()
    test_df  = master_df.reindex(test_df.index).copy()


# =============================================================================
# 7. PROBABILITY MATRIX
# =============================================================================

def prediction_to_probability(preds):
    preds = np.nan_to_num(np.asarray(preds, dtype=float), nan=0.0)
    std = np.nanstd(preds) + 1e-10
    return 1.0 / (1.0 + np.exp(-preds / std))

def extract_prob(obj, split):
    idx = {"train": train_df.index, "val": val_df.index, "test": test_df.index}[split]

    prob_keys = [
        f"{split}_prob",
        f"prob_{split}",
        f"p_{split}",
        f"{split}_proba",
        f"proba_{split}",
        f"{split}_pred_proba",
    ]

    pred_keys = [
        f"{split}_pred",
        f"pred_{split}",
        f"praw_{split}",
        f"{split}_prediction",
    ]

    if not isinstance(obj, dict):
        return None

    for k in prob_keys:
        if k in obj:
            arr = np.asarray(obj[k], dtype=float).reshape(-1)
            n = min(len(arr), len(idx))
            return pd.Series(arr[:n], index=idx[:n])

    for k in pred_keys:
        if k in obj:
            arr = prediction_to_probability(obj[k])
            n = min(len(arr), len(idx))
            return pd.Series(arr[:n], index=idx[:n])

    return None

def build_prob_matrix(store, split, model_filter=None):
    parts = []

    for model_id, obj in store.items():
        if model_filter is not None and str(model_id) not in model_filter:
            continue

        s = extract_prob(obj, split)

        if s is None:
            continue

        s.name = str(model_id)
        parts.append(s)

    if not parts:
        return pd.DataFrame()

    out = pd.concat(parts, axis=1).sort_index()
    out = out.loc[:, ~out.columns.duplicated(keep="first")]
    return out

TRAIN_PROB_ALL = build_prob_matrix(pred_store, "train")
VAL_PROB_ALL   = build_prob_matrix(pred_store, "val")
TEST_PROB_ALL  = build_prob_matrix(pred_store, "test")

print(f"\nPROB MATRICES:")
print(f"  TRAIN : {TRAIN_PROB_ALL.shape}")
print(f"  VAL   : {VAL_PROB_ALL.shape}")
print(f"  TEST  : {TEST_PROB_ALL.shape}")

if TRAIN_PROB_ALL.empty or VAL_PROB_ALL.empty or TEST_PROB_ALL.empty:
    raise ValueError("❌ Probability matrix boş.")


# =============================================================================
# 8. ENSEMBLE AVAILABILITY CHECK
# =============================================================================

def get_available_models(ensemble_name, ensemble_dict):
    available, missing = [], []

    print(f"\n  {ensemble_name}:")

    for group, model in ensemble_dict.items():
        in_val = model in VAL_PROB_ALL.columns
        in_test = model in TEST_PROB_ALL.columns

        if in_val and in_test:
            available.append(model)
            print(f"    ✅ {group:<24} {model}")
        else:
            missing.append(model)
            print(f"    ❌ {group:<24} {model} [val={in_val}, test={in_test}]")

    print(f"    Kullanılabilir: {len(available)} / {len(ensemble_dict)}")
    return available

print("\n" + "═" * 160)
print("AVAILABILITY CHECK")
print("═" * 160)

ALL_ENSEMBLE_AVAILABLE = {}
for ens_name, ens_dict in MULTI_ENSEMBLE_DEFINITIONS.items():
    ALL_ENSEMBLE_AVAILABLE[ens_name] = get_available_models(ens_name, ens_dict)


# =============================================================================
# 9. METRIC HELPERS
# =============================================================================

def normalize_weights(w):
    w = np.maximum(np.nan_to_num(np.asarray(w, dtype=float), nan=0.0), 0.0)
    s = w.sum()
    return w / s if s > 1e-12 else np.ones(len(w)) / len(w)

def sharpe_rets(ret):
    r = np.nan_to_num(np.asarray(ret, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    sd = np.nanstd(r, ddof=1)

    if len(r) <= 2 or sd <= 1e-12:
        return 0.0

    return float(np.nanmean(r) / sd * np.sqrt(BAR_ANN))

def max_drawdown(ret):
    r = np.nan_to_num(np.asarray(ret, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)

    if not len(r):
        return 0.0

    eq = np.cumprod(1.0 + r)
    peak = np.maximum.accumulate(eq)
    return float(np.min(eq / (peak + 1e-12) - 1.0))

def annual_return(ret):
    r = np.nan_to_num(np.asarray(ret, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)

    if not len(r):
        return 0.0

    cum = np.prod(1.0 + r) - 1.0
    yrs = len(r) / BAR_ANN

    if yrs <= 0 or cum <= -0.999:
        return 0.0

    return float((1.0 + cum) ** (1.0 / yrs) - 1.0)

def strategy_from_prob(prob, fwd, upper, lower):
    prob = np.asarray(prob, dtype=float).reshape(-1)
    fwd = np.asarray(fwd, dtype=float).reshape(-1)

    n = min(len(prob), len(fwd))

    prob = prob[:n]
    fwd = np.nan_to_num(fwd[:n], nan=0.0, posinf=0.0, neginf=0.0)

    sig = np.where(prob >= upper, 1.0, np.where(prob <= lower, -1.0, 0.0))
    turnover = np.abs(np.diff(sig, prepend=0.0))

    ret = sig * fwd - TC_PER_SIDE * turnover

    return ret, sig

def objective_score(ret, sig):
    sh = sharpe_rets(ret)
    act = np.mean(sig != 0) if len(sig) else 0.0
    trades = np.sum(np.abs(np.diff(sig, prepend=0.0)) > 0) if len(sig) else 0
    dd = abs(max_drawdown(ret))

    score = sh

    if act < 0.04:
        score -= (0.04 - act) * 30.0

    if act > 0.90:
        score -= (act - 0.90) * 5.0

    if trades < 20:
        score -= (20 - trades) * 0.04

    if dd > 0.30:
        score -= (dd - 0.30) * 2.0

    return float(score)

def summarize_returns(ret, sig):
    ret = np.nan_to_num(np.asarray(ret, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    sig = np.nan_to_num(np.asarray(sig, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)

    return {
        "sharpe": sharpe_rets(ret),
        "cum_ret": float(np.prod(1.0 + ret) - 1.0) if len(ret) else 0.0,
        "ann_ret": annual_return(ret),
        "max_dd": max_drawdown(ret),
        "activity": float(np.mean(sig != 0)) if len(sig) else 0.0,
        "trades": int(np.sum(np.abs(np.diff(sig, prepend=0.0)) > 0)) if len(sig) else 0,
    }


# =============================================================================
# 10. OPTIMIZER
# =============================================================================

def opt_weight_threshold(
    prob_df,
    fwd_series,
    experts,
    fixed_threshold=None,
    seed=42,
    n_trials=N_OPT_TRIALS_FINAL,
    fallback_prob_df=None,
    fallback_fwd=None,
    min_obs=MIN_VAL_OBS_STATE,
):
    np.random.seed(seed)

    experts = [e for e in experts if e in prob_df.columns]

    if len(experts) < MIN_COMBO:
        return None

    idx = prob_df.index.intersection(fwd_series.dropna().index)

    used_fallback = False

    if len(idx) < min_obs and fallback_prob_df is not None and fallback_fwd is not None:
        experts_fb = [e for e in experts if e in fallback_prob_df.columns]
        idx_fb = fallback_prob_df.index.intersection(fallback_fwd.dropna().index)

        if len(idx_fb) >= min_obs and len(experts_fb) >= MIN_COMBO:
            prob_df = fallback_prob_df
            fwd_series = fallback_fwd
            experts = experts_fb
            idx = idx_fb
            used_fallback = True
            print(f"        [FALLBACK→TRAIN] obs={len(idx_fb)}")

    if len(idx) < MIN_COMBO:
        return None

    X = prob_df.loc[idx, experts].fillna(0.5).values
    y = fwd_series.reindex(idx).fillna(0.0).values

    n = X.shape[1]

    def obj(trial):
        w = normalize_weights([
            trial.suggest_float(f"w{i}", 0.0, 1.0)
            for i in range(n)
        ])

        if fixed_threshold is None:
            upper = trial.suggest_float("upper", DEFAULT_UP[0], DEFAULT_UP[1])
            lower = trial.suggest_float("lower", DEFAULT_LO[0], DEFAULT_LO[1])
        else:
            upper, lower = fixed_threshold

        if lower >= upper:
            return -1e9

        p = X @ w
        ret, sig = strategy_from_prob(p, y, upper, lower)

        return objective_score(ret, sig)

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(
            seed=seed,
            multivariate=True,
            group=True,
        ),
    )

    study.optimize(
        obj,
        n_trials=n_trials,
        show_progress_bar=False,
        catch=(Exception,),
    )

    bp = study.best_trial.params

    weights = normalize_weights([
        bp.get(f"w{i}", 0.0)
        for i in range(n)
    ])

    upper = float(bp["upper"]) if fixed_threshold is None else fixed_threshold[0]
    lower = float(bp["lower"]) if fixed_threshold is None else fixed_threshold[1]

    p = X @ weights
    ret, sig = strategy_from_prob(p, y, upper, lower)

    return {
        "experts": experts,
        "weights": {
            e: float(weights[i])
            for i, e in enumerate(experts)
        },
        "upper": upper,
        "lower": lower,
        "score": float(study.best_value),
        "val_sharpe": sharpe_rets(ret),
        "used_fallback": used_fallback,
    }

def apply_solution(prob_df, fwd_series, sol, target_index=None):
    if sol is None:
        return None

    experts = [
        e for e in sol["experts"]
        if e in prob_df.columns
    ]

    if not experts:
        return None

    idx = prob_df.index.intersection(fwd_series.dropna().index)

    if target_index is not None:
        idx = idx.intersection(target_index)

    if not len(idx):
        return None

    X = prob_df.loc[idx, experts].fillna(0.5).values

    w = normalize_weights([
        sol["weights"].get(e, 0.0)
        for e in experts
    ])

    p = X @ w
    y = fwd_series.reindex(idx).fillna(0.0).values

    ret, sig = strategy_from_prob(
        p,
        y,
        sol["upper"],
        sol["lower"],
    )

    out = pd.DataFrame(index=idx)
    out["prob"] = p
    out["ret"] = ret
    out["signal"] = sig

    return out


# =============================================================================
# 11. HMM HELPERS
# =============================================================================

class WinsorZScaler:
    def __init__(self, clip=3.0, eps=1e-12):
        self.clip = clip
        self.eps = eps
        self.mean_ = None
        self.std_ = None

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self.mean_ = np.nanmean(X, axis=0)
        self.std_ = np.nanstd(X, axis=0)
        self.std_ = np.where(self.std_ < self.eps, 1.0, self.std_)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        Z = np.clip((X - self.mean_) / self.std_, -self.clip, self.clip)
        return np.nan_to_num(Z, nan=0.0, posinf=self.clip, neginf=-self.clip)

def make_hmm_scaler(name):
    if name == "winsor_z":
        return WinsorZScaler(clip=3.0)

    if name == "robust":
        return RobustScaler()

    if name == "standard":
        return StandardScaler()

    return None

def causal_hmm_filter_states(model, X):
    X = np.asarray(X, dtype=float)

    log_emlik = model._compute_log_likelihood(X)

    n_obs, n_st = log_emlik.shape

    log_start = np.log(np.maximum(model.startprob_, 1e-300))
    log_trans = np.log(np.maximum(model.transmat_, 1e-300))

    log_alpha = np.zeros((n_obs, n_st), dtype=float)

    log_alpha[0] = log_start + log_emlik[0]
    log_alpha[0] -= logsumexp(log_alpha[0])

    for t in range(1, n_obs):
        pred = logsumexp(log_alpha[t - 1][:, None] + log_trans, axis=0)
        log_alpha[t] = pred + log_emlik[t]
        log_alpha[t] -= logsumexp(log_alpha[t])

    post = np.exp(log_alpha)
    states = np.argmax(post, axis=1)

    return states, post

def feature_hash(features, k, scaler, cov):
    key = "|".join(sorted(features)) + f"|K{k}|{scaler}|{cov}"
    return hashlib.md5(key.encode()).hexdigest()[:16]

HMM_FIT_CACHE = {}

def fit_hmm_candidate(features, k=4, scaler_name="robust", cov_type="diag"):
    features = [f for f in features if f in master_df.columns]
    features = list(dict.fromkeys(features))

    if "log_ret" not in features and "log_ret" in master_df.columns:
        features = ["log_ret"] + features

    if len(features) < MIN_HMM_FEATURES:
        return None

    hkey = feature_hash(features, k, scaler_name, cov_type)

    if hkey in HMM_FIT_CACHE:
        return HMM_FIT_CACHE[hkey]

    all_index = train_df.index.union(val_df.index).union(test_df.index).sort_values()

    X_all_df = (
        master_df
        .reindex(all_index)[features]
        .replace([np.inf, -np.inf], np.nan)
        .ffill()
    )

    X_train_df = (
        master_df
        .reindex(train_df.index)[features]
        .replace([np.inf, -np.inf], np.nan)
        .ffill()
        .dropna()
    )

    if len(X_train_df) < MIN_HMM_TRAIN_OBS:
        return None

    scaler = make_hmm_scaler(scaler_name)

    if scaler is not None:
        scaler.fit(X_train_df.values.astype(float))
        X_train = scaler.transform(X_train_df.values.astype(float))
        X_all = scaler.transform(X_all_df.values.astype(float))
    else:
        X_train = np.nan_to_num(X_train_df.values.astype(float))
        X_all = np.nan_to_num(X_all_df.values.astype(float))

    X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
    X_all = np.nan_to_num(X_all, nan=0.0, posinf=0.0, neginf=0.0)

    try:
        model = GaussianHMM(
            n_components=int(k),
            covariance_type=cov_type,
            n_iter=HMM_N_ITER,
            tol=HMM_TOL,
            random_state=RANDOM_STATE,
            min_covar=1e-6,
        )

        model.fit(X_train)

        states_all, post_all = causal_hmm_filter_states(
            model,
            X_all,
        )

    except Exception as e:
        print(f"    [HMM FIT ERROR] {e}")
        return None

    state_ser = pd.Series(
        states_all,
        index=X_all_df.index,
        name="state",
    )

    posteriors = pd.DataFrame(
        post_all,
        index=X_all_df.index,
        columns=[f"p_state_{i}" for i in range(k)],
    )

    diag_pers = np.diag(model.transmat_)
    avg_pers = float(np.mean(diag_pers))
    min_pers = float(np.min(diag_pers))

    state_freq = state_ser.value_counts(normalize=True)
    min_freq = float(state_freq.min())

    penalty_flag = (
        (min_freq < MIN_STATE_FREQ)
        or
        (avg_pers < MIN_AVG_PERSISTENCE)
    )

    result = {
        "hmm_key": hkey,
        "hmm_name": None,
        "features": features,
        "k": int(k),
        "scaler": scaler_name,
        "cov": cov_type,
        "model": model,
        "state_series": state_ser,
        "posteriors": posteriors,
        "min_state_freq": min_freq,
        "avg_persistence": avg_pers,
        "min_persistence": min_pers,
        "penalty_flag": penalty_flag,
        "n_features": len(features),
        "transmat": model.transmat_,
        "startprob": model.startprob_,
    }

    HMM_FIT_CACHE[hkey] = result

    return result


# =============================================================================
# 12. STATE FEATURE MATRIX + PRINT
# =============================================================================

def build_state_feature_matrix(hmm_name, hmm_obj, save=True):
    features = hmm_obj["features"]
    state_ser = hmm_obj["state_series"].copy()
    k = hmm_obj["k"]

    all_index = state_ser.index.intersection(master_df.index)

    feat_df = master_df.reindex(all_index)[features].copy()
    feat_df["__state__"] = state_ser.reindex(all_index).astype(int)

    state_means = feat_df.groupby("__state__")[features].mean()

    col_mean = feat_df[features].mean()
    col_std = feat_df[features].std().replace(0, 1)

    state_means_z = (state_means - col_mean) / col_std

    state_counts = state_ser.value_counts().sort_index()
    state_freqs = state_ser.value_counts(normalize=True).sort_index()

    split_dist = {}

    for split_name, split_df in [
        ("train", train_df),
        ("val", val_df),
        ("test", test_df),
    ]:
        s = (
            state_ser
            .reindex(split_df.index)
            .ffill()
            .dropna()
            .astype(int)
        )

        split_dist[split_name] = s.value_counts(normalize=True).sort_index()

    trans_df = pd.DataFrame(
        hmm_obj["transmat"],
        index=[f"From S{i}" for i in range(k)],
        columns=[f"To S{j}" for j in range(k)],
    )

    feat_display = [f for f in features if f != "log_ret"]

    print(f"    {'─' * 100}")
    print(f"    STATE PROFILE — {hmm_name}")
    print(
        f"    avg_pers={hmm_obj['avg_persistence']:.4f} | "
        f"min_freq={hmm_obj['min_state_freq']:.4f} | "
        f"penalty={hmm_obj['penalty_flag']}"
    )

    for s in sorted(state_freqs.index):
        freq = float(state_freqs[s])
        count = int(state_counts[s])

        vals = [
            f"{f}={state_means_z.loc[s, f]:+.2f}"
            for f in feat_display
            if f in state_means_z.columns
        ]

        print(
            f"      State {s} ({freq:.1%}, {count:5d} obs): "
            + " | ".join(vals)
        )

    for split_name, dist in split_dist.items():
        dist_str = "  ".join([
            f"S{st}={v:.1%}"
            for st, v in dist.items()
        ])

        print(f"      {split_name:<6}: {dist_str}")

    print(f"    {'─' * 100}")

    if save:
        state_means.to_csv(
            os.path.join(MATRIX_DIR, f"{hmm_name}_means_raw.csv"),
            encoding="utf-8-sig",
        )

        state_means_z.to_csv(
            os.path.join(MATRIX_DIR, f"{hmm_name}_means_z.csv"),
            encoding="utf-8-sig",
        )

        trans_df.to_csv(
            os.path.join(MATRIX_DIR, f"{hmm_name}_transmat.csv"),
            encoding="utf-8-sig",
        )

        if sns is not None:
            try:
                fig, axes = plt.subplots(
                    1,
                    2,
                    figsize=(
                        max(14, len(features) * 1.2),
                        max(5, k + 2),
                    ),
                )

                sns.heatmap(
                    state_means_z,
                    annot=True,
                    fmt=".2f",
                    cmap="RdYlGn",
                    center=0,
                    linewidths=0.5,
                    ax=axes[0],
                    cbar_kws={"label": "Z-score"},
                )

                axes[0].set_title(
                    f"{hmm_name}\nState Feature Means (Z)",
                    fontsize=9,
                )

                axes[0].set_xticklabels(
                    axes[0].get_xticklabels(),
                    rotation=45,
                    ha="right",
                    fontsize=7,
                )

                sns.heatmap(
                    hmm_obj["transmat"],
                    annot=True,
                    fmt=".3f",
                    cmap="Blues",
                    linewidths=0.5,
                    ax=axes[1],
                    vmin=0,
                    vmax=1,
                    xticklabels=[f"S{j}" for j in range(k)],
                    yticklabels=[f"S{i}" for i in range(k)],
                )

                axes[1].set_title(
                    f"{hmm_name}\nTransition Matrix",
                    fontsize=9,
                )

                plt.tight_layout()

                plt.savefig(
                    os.path.join(MATRIX_DIR, f"{hmm_name}_heatmap.png"),
                    dpi=100,
                    bbox_inches="tight",
                )

                plt.close()

            except Exception as e:
                print(f"    [WARN] Heatmap: {e}")
                plt.close("all")

    return {
        "state_means_raw": state_means,
        "state_means_z": state_means_z,
        "state_freqs": state_freqs,
        "state_counts": state_counts,
        "split_dist": split_dist,
        "trans_df": trans_df,
    }


# =============================================================================
# 13. FIT FIXED HMM CANDIDATES
# =============================================================================

print("\n" + "═" * 160)
print(f"FITTING {len(HMM_CANDIDATES_FIXED)} HMM CANDIDATES")
print("═" * 160)

HMM_OBJECTS = {}
HMM_MATRIX_RESULTS = {}
FIT_SUMMARY = []

for i, (candidate_name, set_key, k) in enumerate(HMM_CANDIDATES_FIXED, 1):
    print(f"\n  [{i}/{len(HMM_CANDIDATES_FIXED)}] {candidate_name}")

    features = HMM_FEATURE_SETS.get(set_key, [])

    avail = [f for f in features if f in master_df.columns]
    missing = [f for f in features if f not in master_df.columns]

    if missing:
        print(f"    [WARN] Missing: {missing}")

    if len(avail) < MIN_HMM_FEATURES:
        print(f"    [SKIP] Yeterli feature yok: {len(avail)}")

        FIT_SUMMARY.append({
            "candidate": candidate_name,
            "set_key": set_key,
            "k": k,
            "status": "SKIP",
            "n_avail": len(avail),
        })

        continue

    hmm_obj = fit_hmm_candidate(
        avail,
        k=k,
        scaler_name="robust",
        cov_type="diag",
    )

    if hmm_obj is None:
        print("    [FAIL] HMM fit edilemedi.")

        FIT_SUMMARY.append({
            "candidate": candidate_name,
            "set_key": set_key,
            "k": k,
            "status": "FAIL",
            "n_avail": len(avail),
        })

        continue

    hmm_obj["hmm_name"] = candidate_name

    HMM_OBJECTS[candidate_name] = hmm_obj
    HMM_MATRIX_RESULTS[candidate_name] = build_state_feature_matrix(
        candidate_name,
        hmm_obj,
        save=True,
    )

    hmm_obj["state_series"].to_frame("state").to_csv(
        os.path.join(SAVE_DIR, f"{candidate_name}_states.csv"),
        encoding="utf-8-sig",
    )

    print(
        f"    ✅ K={k} | "
        f"avg_pers={hmm_obj['avg_persistence']:.4f} | "
        f"min_freq={hmm_obj['min_state_freq']:.4f} | "
        f"penalty={hmm_obj['penalty_flag']}"
    )

    FIT_SUMMARY.append({
        "candidate": candidate_name,
        "set_key": set_key,
        "k": k,
        "status": "OK",
        "n_avail": len(avail),
        "avg_persistence": hmm_obj["avg_persistence"],
        "min_state_freq": hmm_obj["min_state_freq"],
        "penalty_flag": hmm_obj["penalty_flag"],
    })

pd.DataFrame(FIT_SUMMARY).to_csv(
    os.path.join(SAVE_DIR, "hmm_fit_summary.csv"),
    index=False,
    encoding="utf-8-sig",
)

ok_count = len(HMM_OBJECTS)

print(f"\nFIT SONUÇ: {ok_count} / {len(HMM_CANDIDATES_FIXED)} başarılı")

if ok_count == 0:
    raise ValueError("❌ Hiçbir HMM fit edilemedi.")


# =============================================================================
# 14. POSTERIOR-WEIGHTED MODE D
# =============================================================================

def opt_posterior_weighted(
    prob_df,
    posteriors_df,
    fwd_series,
    experts,
    seed=42,
    n_trials=N_OPT_TRIALS_FINAL,
):
    np.random.seed(seed)

    experts = [e for e in experts if e in prob_df.columns]

    if len(experts) < MIN_COMBO:
        return None

    idx = (
        prob_df.index
        .intersection(posteriors_df.index)
        .intersection(fwd_series.dropna().index)
    )

    if len(idx) < 80:
        return None

    X = prob_df.loc[idx, experts].fillna(0.5).values
    POST = posteriors_df.reindex(idx).fillna(0.0).values
    y = fwd_series.reindex(idx).fillna(0.0).values

    n_m = X.shape[1]
    k_st = POST.shape[1]

    def obj(trial):
        state_weights = []

        for s in range(k_st):
            raw = [
                trial.suggest_float(f"s{s}_w{i}", 0.0, 1.0)
                for i in range(n_m)
            ]

            state_weights.append(normalize_weights(raw))

        W = np.stack(state_weights, axis=0)

        model_contrib = X @ W.T
        prob = np.sum(POST * model_contrib, axis=1)

        upper = trial.suggest_float("upper", DEFAULT_UP[0], DEFAULT_UP[1])
        lower = trial.suggest_float("lower", DEFAULT_LO[0], DEFAULT_LO[1])

        if lower >= upper:
            return -1e9

        ret, sig = strategy_from_prob(prob, y, upper, lower)

        return objective_score(ret, sig)

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(
            seed=seed,
            multivariate=True,
            group=True,
        ),
    )

    study.optimize(
        obj,
        n_trials=n_trials,
        show_progress_bar=False,
        catch=(Exception,),
    )

    bp = study.best_trial.params

    final_sw = []

    for s in range(k_st):
        raw = [
            bp.get(f"s{s}_w{i}", 0.0)
            for i in range(n_m)
        ]

        final_sw.append(normalize_weights(raw))

    W = np.stack(final_sw, axis=0)

    upper = float(bp["upper"])
    lower = float(bp["lower"])

    model_contrib = X @ W.T
    prob = np.sum(POST * model_contrib, axis=1)

    ret, sig = strategy_from_prob(prob, y, upper, lower)

    return {
        "experts": experts,
        "state_weights": {
            s: {
                experts[i]: float(final_sw[s][i])
                for i in range(n_m)
            }
            for s in range(k_st)
        },
        "upper": upper,
        "lower": lower,
        "score": float(study.best_value),
        "val_sharpe": sharpe_rets(ret),
        "W": W,
    }

def apply_posterior_solution(
    prob_df,
    posteriors_df,
    fwd_series,
    sol,
    target_index=None,
):
    experts = [
        e for e in sol["experts"]
        if e in prob_df.columns
    ]

    if not experts:
        return None

    idx = (
        prob_df.index
        .intersection(posteriors_df.index)
        .intersection(fwd_series.dropna().index)
    )

    if target_index is not None:
        idx = idx.intersection(target_index)

    if not len(idx):
        return None

    X = prob_df.loc[idx, experts].fillna(0.5).values
    POST = posteriors_df.reindex(idx).fillna(0.0).values
    y = fwd_series.reindex(idx).fillna(0.0).values

    W = sol["W"]

    if POST.shape[1] != W.shape[0]:
        return None

    model_contrib = X @ W.T
    prob = np.sum(POST * model_contrib, axis=1)

    ret, sig = strategy_from_prob(
        prob,
        y,
        sol["upper"],
        sol["lower"],
    )

    out = pd.DataFrame(index=idx)
    out["prob"] = prob
    out["ret"] = ret
    out["signal"] = sig

    return out


# =============================================================================
# 15. C_REGIME_FREE
# =============================================================================

def run_regime_free(ensemble_name, available_models):
    print(f"\n  C_REGIME_FREE → {ensemble_name}")

    if len(available_models) < MIN_COMBO:
        print(f"    [SKIP] Yetersiz model.")
        return None, None

    sol = opt_weight_threshold(
        prob_df=VAL_PROB_ALL,
        fwd_series=val_df["fwd_ret"],
        experts=available_models,
        fixed_threshold=None,
        seed=RANDOM_STATE,
        n_trials=N_OPT_TRIALS_FINAL,
        fallback_prob_df=TRAIN_PROB_ALL,
        fallback_fwd=train_df["fwd_ret"],
        min_obs=50,
    )

    if sol is None:
        return None, None

    vp = apply_solution(
        VAL_PROB_ALL,
        val_df["fwd_ret"],
        sol,
    )

    tp = apply_solution(
        TEST_PROB_ALL,
        test_df["fwd_ret"],
        sol,
    )

    if vp is None or tp is None:
        return None, None

    vm = summarize_returns(vp["ret"].values, vp["signal"].values)
    tm = summarize_returns(tp["ret"].values, tp["signal"].values)

    mode_name = "C_REGIME_FREE"

    row = {
        "ensemble_name": ensemble_name,
        "candidate": "C_REGIME_FREE",
        "set_key": "REGIME_FREE",
        "k": None,
        "mode": mode_name,

        **{f"val_{k}": v for k, v in vm.items()},
        **{f"test_{k}": v for k, v in tm.items()},

        "n_state_solutions": 0,
        "avg_persistence": None,
        "min_state_freq": None,
        "penalty_flag": None,
        "n_experts": len(available_models),
    }

    store_eval_output(
        store=EVAL_OUTPUT_STORE,
        ensemble_name=ensemble_name,
        candidate="C_REGIME_FREE",
        mode=mode_name,
        val_df_out=vp,
        test_df_out=tp,
        extra_info={
            "experts": sol.get("experts"),
            "weights": sol.get("weights"),
            "upper": sol.get("upper"),
            "lower": sol.get("lower"),
            "n_experts": len(available_models),
        },
    )

    print(
        f"    Val Sharpe={vm['sharpe']:+.4f} | "
        f"Test Sharpe={tm['sharpe']:+.4f} | "
        f"Ret={tm['cum_ret']:+.2%} | "
        f"DD={tm['max_dd']:+.2%} | "
        f"U/L={sol['upper']:.4f}/{sol['lower']:.4f}"
    )

    return row, sol


# =============================================================================
# 16. EVALUATION — ENSEMBLE × CANDIDATE × A/B/D
# =============================================================================

def evaluate_candidate_for_ensemble(
    ensemble_name,
    hmm_obj,
    candidate_name,
    experts,
    n_trials=N_OPT_TRIALS_FINAL,
):
    state_val = (
        hmm_obj["state_series"]
        .reindex(val_df.index)
        .ffill()
        .bfill()
        .astype(int)
    )

    state_test = (
        hmm_obj["state_series"]
        .reindex(test_df.index)
        .ffill()
        .bfill()
        .astype(int)
    )

    post_val = (
        hmm_obj["posteriors"]
        .reindex(val_df.index)
        .ffill()
        .bfill()
        .fillna(0.0)
    )

    post_test = (
        hmm_obj["posteriors"]
        .reindex(test_df.index)
        .ffill()
        .bfill()
        .fillna(0.0)
    )

    experts = [
        e for e in experts
        if e in VAL_PROB_ALL.columns and e in TEST_PROB_ALL.columns
    ]

    if len(experts) < MIN_COMBO:
        return []

    rows = []

    global_sol = opt_weight_threshold(
        prob_df=VAL_PROB_ALL,
        fwd_series=val_df["fwd_ret"],
        experts=experts,
        fixed_threshold=None,
        seed=RANDOM_STATE,
        n_trials=n_trials,
        fallback_prob_df=TRAIN_PROB_ALL,
        fallback_fwd=train_df["fwd_ret"],
        min_obs=50,
    )

    if global_sol is None:
        return []

    # -------------------------------------------------------------------------
    # MODE A — State-specific weights + global threshold
    # -------------------------------------------------------------------------

    val_parts_A = []
    test_parts_A = []
    state_solutions_A = {}

    for s in sorted(state_val.dropna().unique()):
        val_idx_s = state_val[state_val == s].index
        test_idx_s = state_test[state_test == s].index

        val_obs = len(VAL_PROB_ALL.index.intersection(val_idx_s))

        train_s_idx = (
            hmm_obj["state_series"]
            .reindex(train_df.index)
            .ffill()
            .bfill()
        )

        train_s_idx = train_s_idx[train_s_idx == s].index

        sol_s = opt_weight_threshold(
            prob_df=VAL_PROB_ALL.loc[
                VAL_PROB_ALL.index.intersection(val_idx_s)
            ],
            fwd_series=val_df["fwd_ret"],
            experts=experts,
            fixed_threshold=(global_sol["upper"], global_sol["lower"]),
            seed=RANDOM_STATE + int(s) + 10,
            n_trials=n_trials,
            fallback_prob_df=TRAIN_PROB_ALL.loc[
                TRAIN_PROB_ALL.index.intersection(train_s_idx)
            ] if len(TRAIN_PROB_ALL) > 0 else None,
            fallback_fwd=train_df["fwd_ret"],
            min_obs=MIN_VAL_OBS_STATE,
        )

        if sol_s is None:
            continue

        state_solutions_A[int(s)] = sol_s

        vp = apply_solution(
            VAL_PROB_ALL,
            val_df["fwd_ret"],
            sol_s,
            target_index=val_idx_s,
        )

        tp = apply_solution(
            TEST_PROB_ALL,
            test_df["fwd_ret"],
            sol_s,
            target_index=test_idx_s,
        )

        if vp is not None:
            val_parts_A.append(vp)

        if tp is not None:
            test_parts_A.append(tp)

        fb = sol_s.get("used_fallback", False)

        print(
            f"      [A] S{s}: obs={val_obs} | "
            f"{'FB' if fb else 'VAL'}"
        )

    if val_parts_A and test_parts_A:
        vp_ = pd.concat(val_parts_A).sort_index()
        tp_ = pd.concat(test_parts_A).sort_index()

        vm = summarize_returns(vp_["ret"].values, vp_["signal"].values)
        tm = summarize_returns(tp_["ret"].values, tp_["signal"].values)

        mode_name = "A_STATE_WEIGHTS_GLOBAL_THRESHOLD"

        rows.append({
            "ensemble_name": ensemble_name,
            "candidate": candidate_name,
            "mode": mode_name,

            **{f"val_{k}": v for k, v in vm.items()},
            **{f"test_{k}": v for k, v in tm.items()},

            "n_state_solutions": len(val_parts_A),
        })

        store_eval_output(
            store=EVAL_OUTPUT_STORE,
            ensemble_name=ensemble_name,
            candidate=candidate_name,
            mode=mode_name,
            val_df_out=vp_,
            test_df_out=tp_,
            extra_info={
                "hmm_k": hmm_obj["k"],
                "hmm_features": hmm_obj["features"],
                "global_upper": global_sol.get("upper"),
                "global_lower": global_sol.get("lower"),
                "state_solutions": state_solutions_A,
                "experts": experts,
            },
        )

    # -------------------------------------------------------------------------
    # MODE B — State-specific weights + state-specific threshold
    # -------------------------------------------------------------------------

    val_parts_B = []
    test_parts_B = []
    state_solutions_B = {}

    for s in sorted(state_val.dropna().unique()):
        val_idx_s = state_val[state_val == s].index
        test_idx_s = state_test[state_test == s].index

        train_s_idx = (
            hmm_obj["state_series"]
            .reindex(train_df.index)
            .ffill()
            .bfill()
        )

        train_s_idx = train_s_idx[train_s_idx == s].index

        sol_s = opt_weight_threshold(
            prob_df=VAL_PROB_ALL.loc[
                VAL_PROB_ALL.index.intersection(val_idx_s)
            ],
            fwd_series=val_df["fwd_ret"],
            experts=experts,
            fixed_threshold=None,
            seed=RANDOM_STATE + int(s) + 100,
            n_trials=n_trials,
            fallback_prob_df=TRAIN_PROB_ALL.loc[
                TRAIN_PROB_ALL.index.intersection(train_s_idx)
            ] if len(TRAIN_PROB_ALL) > 0 else None,
            fallback_fwd=train_df["fwd_ret"],
            min_obs=MIN_VAL_OBS_STATE,
        )

        if sol_s is None:
            continue

        state_solutions_B[int(s)] = sol_s

        vp = apply_solution(
            VAL_PROB_ALL,
            val_df["fwd_ret"],
            sol_s,
            target_index=val_idx_s,
        )

        tp = apply_solution(
            TEST_PROB_ALL,
            test_df["fwd_ret"],
            sol_s,
            target_index=test_idx_s,
        )

        if vp is not None:
            val_parts_B.append(vp)

        if tp is not None:
            test_parts_B.append(tp)

        fb = sol_s.get("used_fallback", False)

        print(
            f"      [B] S{s}: obs={len(VAL_PROB_ALL.index.intersection(val_idx_s))} | "
            f"{'FB' if fb else 'VAL'}"
        )

    if val_parts_B and test_parts_B:
        vp_ = pd.concat(val_parts_B).sort_index()
        tp_ = pd.concat(test_parts_B).sort_index()

        vm = summarize_returns(vp_["ret"].values, vp_["signal"].values)
        tm = summarize_returns(tp_["ret"].values, tp_["signal"].values)

        mode_name = "B_STATE_WEIGHTS_STATE_THRESHOLD"

        rows.append({
            "ensemble_name": ensemble_name,
            "candidate": candidate_name,
            "mode": mode_name,

            **{f"val_{k}": v for k, v in vm.items()},
            **{f"test_{k}": v for k, v in tm.items()},

            "n_state_solutions": len(val_parts_B),
        })

        store_eval_output(
            store=EVAL_OUTPUT_STORE,
            ensemble_name=ensemble_name,
            candidate=candidate_name,
            mode=mode_name,
            val_df_out=vp_,
            test_df_out=tp_,
            extra_info={
                "hmm_k": hmm_obj["k"],
                "hmm_features": hmm_obj["features"],
                "state_solutions": state_solutions_B,
                "experts": experts,
            },
        )

    # -------------------------------------------------------------------------
    # MODE D — Posterior weighted soft state assignment
    # -------------------------------------------------------------------------

    sol_D = opt_posterior_weighted(
        prob_df=VAL_PROB_ALL,
        posteriors_df=post_val,
        fwd_series=val_df["fwd_ret"],
        experts=experts,
        seed=RANDOM_STATE + 999,
        n_trials=n_trials,
    )

    if sol_D is not None:
        vp_D = apply_posterior_solution(
            VAL_PROB_ALL,
            post_val,
            val_df["fwd_ret"],
            sol_D,
        )

        tp_D = apply_posterior_solution(
            TEST_PROB_ALL,
            post_test,
            test_df["fwd_ret"],
            sol_D,
        )

        if vp_D is not None and tp_D is not None:
            vm = summarize_returns(vp_D["ret"].values, vp_D["signal"].values)
            tm = summarize_returns(tp_D["ret"].values, tp_D["signal"].values)

            mode_name = "D_POSTERIOR_WEIGHTED"

            rows.append({
                "ensemble_name": ensemble_name,
                "candidate": candidate_name,
                "mode": mode_name,

                **{f"val_{k}": v for k, v in vm.items()},
                **{f"test_{k}": v for k, v in tm.items()},

                "n_state_solutions": hmm_obj["k"],
            })

            store_eval_output(
                store=EVAL_OUTPUT_STORE,
                ensemble_name=ensemble_name,
                candidate=candidate_name,
                mode=mode_name,
                val_df_out=vp_D,
                test_df_out=tp_D,
                extra_info={
                    "hmm_k": hmm_obj["k"],
                    "hmm_features": hmm_obj["features"],
                    "posterior_solution": sol_D,
                    "experts": experts,
                },
            )

            print(
                f"      [D] Val={vm['sharpe']:+.4f} | "
                f"Test={tm['sharpe']:+.4f}"
            )

    return rows


# =============================================================================
# 17. RUN — 3 ENSEMBLE × HMM CANDIDATES × {A,B,D} + C
# =============================================================================

print("\n" + "═" * 160)
print(
    f"EVALUATION: {len(MULTI_ENSEMBLE_DEFINITIONS)} ENSEMBLE × "
    f"{ok_count} CANDIDATE × MODE {{A,B,D}} + C"
)
print("═" * 160)

ALL_ROWS = []
ALL_REGIME_FREE = {}

for ensemble_name, ensemble_dict in MULTI_ENSEMBLE_DEFINITIONS.items():
    available_models = ALL_ENSEMBLE_AVAILABLE.get(ensemble_name, [])

    print("\n\n" + "█" * 160)
    print(
        f"ENSEMBLE: {ensemble_name} | "
        f"available={len(available_models)}/{len(ensemble_dict)}"
    )
    print("█" * 160)

    if len(available_models) < MIN_COMBO:
        print(f"  [SKIP] {ensemble_name}: yetersiz model.")
        continue

    rf_row, rf_sol = run_regime_free(
        ensemble_name,
        available_models,
    )

    ALL_REGIME_FREE[ensemble_name] = rf_sol

    if rf_row is not None:
        ALL_ROWS.append(rf_row)

    for ei, (candidate_name, hmm_obj) in enumerate(HMM_OBJECTS.items(), 1):
        print(
            f"\n  [{ei}/{ok_count}] "
            f"{ensemble_name} | {candidate_name} | "
            f"K={hmm_obj['k']} | penalty={hmm_obj['penalty_flag']}"
        )

        try:
            rows = evaluate_candidate_for_ensemble(
                ensemble_name=ensemble_name,
                hmm_obj=hmm_obj,
                candidate_name=candidate_name,
                experts=available_models,
                n_trials=N_OPT_TRIALS_FINAL,
            )

            if not rows:
                print("    [SKIP] Sonuç yok.")
                continue

            for row in rows:
                row["set_key"] = candidate_name.rsplit("_K", 1)[0]
                row["k"] = hmm_obj["k"]
                row["avg_persistence"] = hmm_obj["avg_persistence"]
                row["min_state_freq"] = hmm_obj["min_state_freq"]
                row["penalty_flag"] = hmm_obj["penalty_flag"]
                row["n_features"] = hmm_obj["n_features"]
                row["features_str"] = ", ".join(hmm_obj["features"])
                row["n_experts"] = len(available_models)

                ALL_ROWS.append(row)

            best = max(
                rows,
                key=lambda r: r.get("val_sharpe", -99),
            )

            print(
                f"    BEST={best['mode']:<42} | "
                f"Val={best.get('val_sharpe', np.nan):+.4f} | "
                f"Test={best.get('test_sharpe', np.nan):+.4f} | "
                f"Ret={best.get('test_cum_ret', np.nan):+.2%} | "
                f"DD={best.get('test_max_dd', np.nan):+.2%}"
            )

            for row in rows:
                print(
                    f"      {row['mode']:<42} | "
                    f"Val={row.get('val_sharpe', np.nan):+.4f} | "
                    f"Test={row.get('test_sharpe', np.nan):+.4f}"
                )

        except Exception as e:
            print(f"    [ERR] {type(e).__name__}: {e}")
            import traceback
            traceback.print_exc()

        gc.collect()


# =============================================================================
# 18. RESULTS TABLES — TEST SHARPE'A GÖRE SIRALI
# =============================================================================

if not ALL_ROWS:
    raise ValueError("❌ Hiç sonuç üretilemedi.")

results_df = (
    pd.DataFrame(ALL_ROWS)
    .sort_values(
        ["ensemble_name", "test_sharpe", "val_sharpe"],
        ascending=[True, False, False],
    )
    .reset_index(drop=True)
)

best_per_candidate = (
    results_df
    .sort_values(
        ["ensemble_name", "candidate", "test_sharpe", "val_sharpe"],
        ascending=[True, True, False, False],
    )
    .groupby(["ensemble_name", "candidate"], as_index=False)
    .first()
    .sort_values(
        ["ensemble_name", "test_sharpe"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

best_overall = (
    results_df
    .sort_values(
        ["ensemble_name", "test_sharpe", "val_sharpe"],
        ascending=[True, False, False],
    )
    .groupby("ensemble_name", as_index=False)
    .first()
    .sort_values("test_sharpe", ascending=False)
    .reset_index(drop=True)
)

results_path = os.path.join(
    SAVE_DIR,
    "all_results_test_sorted.csv",
)

best_per_candidate_path = os.path.join(
    SAVE_DIR,
    "best_per_candidate.csv",
)

best_overall_path = os.path.join(
    SAVE_DIR,
    "best_overall_per_ensemble.csv",
)

results_df.to_csv(
    results_path,
    index=False,
    encoding="utf-8-sig",
)

best_per_candidate.to_csv(
    best_per_candidate_path,
    index=False,
    encoding="utf-8-sig",
)

best_overall.to_csv(
    best_overall_path,
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 19. SAVE EVAL OUTPUT STORE — NEW
# =============================================================================

eval_output_store_path = os.path.join(
    SAVE_DIR,
    "eval_output_store.pkl",
)

with open(eval_output_store_path, "wb") as f:
    pickle.dump(EVAL_OUTPUT_STORE, f)

eval_output_summary_rows = []

for key, obj in EVAL_OUTPUT_STORE.items():
    row = {
        "key": key,
        "ensemble_name": obj.get("ensemble_name"),
        "candidate": obj.get("candidate"),
        "mode": obj.get("mode"),
        "has_val_ret": obj.get("val_ret") is not None,
        "has_test_ret": obj.get("test_ret") is not None,
        "val_len": len(obj.get("val_ret")) if obj.get("val_ret") is not None else 0,
        "test_len": len(obj.get("test_ret")) if obj.get("test_ret") is not None else 0,
    }

    eval_output_summary_rows.append(row)

eval_output_summary_df = pd.DataFrame(eval_output_summary_rows)

eval_output_summary_path = os.path.join(
    SAVE_DIR,
    "eval_output_store_summary.csv",
)

eval_output_summary_df.to_csv(
    eval_output_summary_path,
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 20. FINAL PRINT — TEST SHARPE SIRALI
# =============================================================================

SHOW_COLS = [
    c for c in [
        "ensemble_name",
        "candidate",
        "set_key",
        "k",
        "mode",
        "val_sharpe",
        "test_sharpe",
        "test_cum_ret",
        "test_ann_ret",
        "test_max_dd",
        "test_activity",
        "test_trades",
        "n_state_solutions",
        "avg_persistence",
        "min_state_freq",
        "penalty_flag",
    ]
    if c in results_df.columns
]

print("\n" + "═" * 160)
print("ALL RESULTS — TEST SHARPE SIRALI")
print("═" * 160)
print(results_df[SHOW_COLS].to_string(index=False))

print("\n" + "═" * 160)
print("BEST PER ENSEMBLE × CANDIDATE — TEST BAZLI")
print("═" * 160)
print(best_per_candidate[SHOW_COLS].to_string(index=False))

print("\n" + "═" * 160)
print("BEST OVERALL PER ENSEMBLE — TEST BAZLI")
print("═" * 160)
print(best_overall[SHOW_COLS].to_string(index=False))

print("\n" + "═" * 160)
print("EVAL OUTPUT STORE SUMMARY")
print("═" * 160)
print(eval_output_summary_df.to_string(index=False))

print("\n" + "═" * 160)
print("FILES")
print("═" * 160)
print(f"  All results              → {results_path}")
print(f"  Best per candidate       → {best_per_candidate_path}")
print(f"  Best overall             → {best_overall_path}")
print(f"  HMM fit summary          → {os.path.join(SAVE_DIR, 'hmm_fit_summary.csv')}")
print(f"  State matrices           → {MATRIX_DIR}/")
print(f"  Eval output store        → {eval_output_store_path}")
print(f"  Eval output summary      → {eval_output_summary_path}")
print("═" * 160)

print(
    f"\nTOPLAM: {ok_count} HMM | "
    f"{len(MULTI_ENSEMBLE_DEFINITIONS)} ensemble | "
    f"{len(results_df)} sonuç satırı | "
    f"{len(EVAL_OUTPUT_STORE)} stored output"
)

print("\nDONE ✅")

✅ pred_store yüklendi: 75 model
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
MULTI-ENSEMBLE × HMM FIXED CANDIDATES
Candidates: HMM_CLASSIC_LITERATURE_CORE K5 | HMM_FULL_CORE K5
Ensembles : MAIN_14 / THEORY_11 / ENSEMBLE_B
Sıralama  : test_sharpe
Output    : results CSV + eval_output_store.pkl
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
master_df      : (37612, 962)
train/val/test : 22567 / 7522 / 7523
pred_store     : 75 models
BAR_ANN        : 1512  |  TC_PER_SIDE: 5e-05
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

ENSEMBLE DEFINITIONS:
  MAIN_14      → 14 models
  THEORY_11    → 11 models
  ENSEMBLE_B   → 14 models

HMM CANDIDATES (sabit 2)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FINAL HMM MODE TABLE
# Pool | HMM | Mode | Sharpe (rf=0) | Sharpe (rf=DTB3)
#      | Ann. Return | Cum. Return | Max DD | Activity | Trades
#
# Reads:
#   multi_ens_fixed_hmm_outputs/eval_output_store.pkl
#
# Fetches:
#   DTB3 from FRED
#
# Produces:
#   hmm_mode_final_table_report/FINAL_HMM_MODE_TABLE_raw.csv
#   hmm_mode_final_table_report/FINAL_HMM_MODE_TABLE_formatted.csv
# ═══════════════════════════════════════════════════════════════════════════════

import os
import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# =============================================================================
# 0. SETTINGS
# =============================================================================

BASE_DIR = "multi_ens_fixed_hmm_outputs"
REPORT_DIR = "hmm_mode_final_table_report"

EVAL_STORE_PATH = os.path.join(BASE_DIR, "eval_output_store.pkl")

os.makedirs(REPORT_DIR, exist_ok=True)

BAR_ANN = globals().get("BAR_ANN", 6 * 252)

POOL_ORDER = ["MAIN_14", "THEORY_11", "ENSEMBLE_B"]

print("═" * 150)
print("FINAL HMM MODE TABLE REPORT")
print("Sharpe + DTB3 Sharpe + Annual Return + Cumulative Return")
print("═" * 150)
print(f"BAR_ANN    : {BAR_ANN}")
print(f"Eval store : {EVAL_STORE_PATH}")


# =============================================================================
# 1. LOAD EVAL STORE
# =============================================================================

if not os.path.exists(EVAL_STORE_PATH):
    raise FileNotFoundError(f"eval_output_store.pkl bulunamadı: {EVAL_STORE_PATH}")

with open(EVAL_STORE_PATH, "rb") as f:
    eval_store = pickle.load(f)

print(f"Loaded eval store keys: {len(eval_store)}")


# =============================================================================
# 2. FIND DATE RANGE FOR FRED DTB3
# =============================================================================

all_indexes = []

for key, obj in eval_store.items():
    if obj.get("test_index") is not None:
        all_indexes.append(pd.to_datetime(obj["test_index"]))

if not all_indexes:
    raise ValueError("eval_store içinde test_index bulunamadı.")

combined_index = all_indexes[0]

for idx in all_indexes[1:]:
    combined_index = combined_index.union(idx)

combined_index = pd.to_datetime(combined_index).sort_values()

start_date = combined_index.min().date()
end_date = combined_index.max().date()

print(f"DTB3 date range: {start_date} → {end_date}")


# =============================================================================
# 3. FETCH DTB3 FROM FRED
# =============================================================================

def fetch_dtb3_from_fred(start, end):
    try:
        from pandas_datareader import data as pdr
    except ImportError:
        raise ImportError(
            "pandas_datareader yüklü değil. Önce şunu çalıştır:\n"
            "!pip install pandas_datareader"
        )

    dtb3 = pdr.DataReader(
        "DTB3",
        "fred",
        start=start,
        end=end
    )

    dtb3 = dtb3.rename(columns={"DTB3": "DTB3"})
    dtb3.index = pd.to_datetime(dtb3.index)
    dtb3 = dtb3.sort_index()

    return dtb3


dtb3_daily = fetch_dtb3_from_fred(start_date, end_date)

dtb3_path = os.path.join(REPORT_DIR, "fred_DTB3_daily.csv")
dtb3_daily.to_csv(dtb3_path, encoding="utf-8-sig")

print(f"DTB3 saved: {dtb3_path}")


# =============================================================================
# 4. METRIC HELPERS
# =============================================================================

def clean_arr(x):
    return np.nan_to_num(
        np.asarray(x, dtype=float),
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )


def sharpe_rets(ret, bar_ann=BAR_ANN):
    r = clean_arr(ret)

    if len(r) <= 2:
        return 0.0

    sd = np.nanstd(r, ddof=1)

    if sd <= 1e-12:
        return 0.0

    return float(np.nanmean(r) / sd * np.sqrt(bar_ann))


def cumulative_return(ret):
    r = clean_arr(ret)

    if len(r) == 0:
        return 0.0

    return float(np.prod(1.0 + r) - 1.0)


def max_drawdown(ret):
    r = clean_arr(ret)

    if len(r) == 0:
        return 0.0

    equity = np.cumprod(1.0 + r)
    peak = np.maximum.accumulate(equity)
    dd = equity / (peak + 1e-12) - 1.0

    return float(np.min(dd))


def annual_return(ret, bar_ann=BAR_ANN):
    r = clean_arr(ret)

    if len(r) == 0:
        return 0.0

    cum_ret = cumulative_return(r)
    years = len(r) / bar_ann

    if years <= 0 or cum_ret <= -0.999:
        return 0.0

    return float((1.0 + cum_ret) ** (1.0 / years) - 1.0)


def get_rf_bar_series(index, dtb3_daily, bar_ann=BAR_ANN):
    index = pd.to_datetime(index)

    rf = (
        dtb3_daily
        .reindex(index)
        .ffill()
        .bfill()
    )

    rf_annual_pct = rf["DTB3"].astype(float).fillna(0.0)

    # DTB3 annual percentage:
    # 5.25 means annual 5.25%
    rf_bar = (rf_annual_pct / 100.0) / bar_ann
    rf_bar.name = "rf_bar_DTB3"

    return rf_bar


def sharpe_rets_rf_DTB3(ret, index, dtb3_daily, bar_ann=BAR_ANN):
    r = clean_arr(ret)
    index = pd.to_datetime(index)

    if len(r) <= 2:
        return 0.0

    rf_bar = get_rf_bar_series(
        index=index,
        dtb3_daily=dtb3_daily,
        bar_ann=bar_ann
    ).values

    n = min(len(r), len(rf_bar))
    excess = r[:n] - rf_bar[:n]

    sd = np.nanstd(excess, ddof=1)

    if sd <= 1e-12:
        return 0.0

    return float(np.nanmean(excess) / sd * np.sqrt(bar_ann))


def summarize_test(index, ret, sig, dtb3_daily):
    index = pd.to_datetime(index)

    ret = clean_arr(ret)
    sig = clean_arr(sig)

    return {
        "Sharpe (rf=0)": sharpe_rets(ret),
        "Sharpe (rf=DTB3)": sharpe_rets_rf_DTB3(
            ret=ret,
            index=index,
            dtb3_daily=dtb3_daily
        ),
        "Ann. Return": annual_return(ret),
        "Cum. Return": cumulative_return(ret),
        "Max DD": max_drawdown(ret),
        "Activity": float(np.mean(sig != 0)) if len(sig) else 0.0,
        "Trades": int(np.sum(np.abs(np.diff(sig, prepend=0.0)) > 0)) if len(sig) else 0,
    }


# =============================================================================
# 5. KEY MAPPING
# =============================================================================

MODE_MAP = {
    "A": "A_STATE_WEIGHTS_GLOBAL_THRESHOLD",
    "B": "B_STATE_WEIGHTS_STATE_THRESHOLD",
    "D": "D_POSTERIOR_WEIGHTED",
    "C": "C_REGIME_FREE",
}

HMM_MAP = {
    "CLASSIC": "HMM_CLASSIC_LITERATURE_CORE",
    "FULL": "HMM_FULL_CORE_K5",
}

# İstenen sıra:
# MAIN_14 CLASSIC A/B/D
# MAIN_14 FULL A/B/D
# MAIN_14 — C
# sonra THEORY_11, ENSEMBLE_B aynı sıra

ROW_TEMPLATE = []

for pool in POOL_ORDER:
    ROW_TEMPLATE.extend([
        (pool, "CLASSIC", "A"),
        (pool, "CLASSIC", "B"),
        (pool, "CLASSIC", "D"),
        (pool, "FULL", "A"),
        (pool, "FULL", "B"),
        (pool, "FULL", "D"),
        (pool, "—", "C"),
    ])


def build_eval_key(pool, hmm_short, mode_short):
    mode_full = MODE_MAP[mode_short]

    if mode_short == "C":
        return f"{pool}__C_REGIME_FREE__C_REGIME_FREE"

    candidate = HMM_MAP[hmm_short]
    return f"{pool}__{candidate}__{mode_full}"


# =============================================================================
# 6. BUILD FINAL TABLE
# =============================================================================

rows = []
missing_keys = []

for pool, hmm_short, mode_short in ROW_TEMPLATE:

    key = build_eval_key(pool, hmm_short, mode_short)

    if key not in eval_store:
        missing_keys.append(key)

        rows.append({
            "Pool": pool,
            "HMM": hmm_short,
            "Mode": mode_short,
            "Sharpe (rf=0)": np.nan,
            "Sharpe (rf=DTB3)": np.nan,
            "Ann. Return": np.nan,
            "Cum. Return": np.nan,
            "Max DD": np.nan,
            "Activity": np.nan,
            "Trades": np.nan,
            "key": key,
        })

        continue

    obj = eval_store[key]

    test_index = pd.to_datetime(obj["test_index"])
    test_ret = np.asarray(obj["test_ret"], dtype=float)
    test_signal = np.asarray(obj["test_signal"], dtype=float)

    s = summarize_test(
        index=test_index,
        ret=test_ret,
        sig=test_signal,
        dtb3_daily=dtb3_daily
    )

    rows.append({
        "Pool": pool,
        "HMM": hmm_short,
        "Mode": mode_short,
        **s,
        "key": key,
    })

mode_table_raw = pd.DataFrame(rows)


# =============================================================================
# 7. SAVE RAW TABLE
# =============================================================================

raw_path = os.path.join(REPORT_DIR, "FINAL_HMM_MODE_TABLE_raw.csv")

mode_table_raw.to_csv(
    raw_path,
    index=False,
    encoding="utf-8-sig"
)


# =============================================================================
# 8. FORMATTED TABLE
# =============================================================================

def fmt_num(x):
    if pd.isna(x):
        return ""
    return f"{x:.3f}"


def fmt_pct(x):
    if pd.isna(x):
        return ""
    return f"{x:.2%}"


mode_table = mode_table_raw.copy()

mode_table["Sharpe (rf=0)"] = mode_table["Sharpe (rf=0)"].map(fmt_num)
mode_table["Sharpe (rf=DTB3)"] = mode_table["Sharpe (rf=DTB3)"].map(fmt_num)
mode_table["Ann. Return"] = mode_table["Ann. Return"].map(fmt_pct)
mode_table["Cum. Return"] = mode_table["Cum. Return"].map(fmt_pct)
mode_table["Max DD"] = mode_table["Max DD"].map(fmt_pct)
mode_table["Activity"] = mode_table["Activity"].map(fmt_pct)
mode_table["Trades"] = mode_table["Trades"].apply(
    lambda x: "" if pd.isna(x) else str(int(x))
)

formatted_path = os.path.join(REPORT_DIR, "FINAL_HMM_MODE_TABLE_formatted.csv")

mode_table.to_csv(
    formatted_path,
    index=False,
    encoding="utf-8-sig"
)


# =============================================================================
# 9. PRINT FINAL TABLE
# =============================================================================

PRINT_COLS = [
    "Pool",
    "HMM",
    "Mode",
    "Sharpe (rf=0)",
    "Sharpe (rf=DTB3)",
    "Ann. Return",
    "Cum. Return",
    "Max DD",
    "Activity",
    "Trades",
]

print("\n" + "═" * 150)
print("FINAL HMM MODE TABLE — TEST PERIOD")
print("═" * 150)
print(mode_table[PRINT_COLS].to_string(index=False))
print("═" * 150)

if missing_keys:
    print("\n[WARN] Eksik keyler var. Bunlar eval_output_store içinde bulunamadı:")
    for k in missing_keys:
        print("  ", k)

print("\nFILES")
print(f"  Raw table       → {raw_path}")
print(f"  Formatted table → {formatted_path}")
print(f"  FRED DTB3       → {dtb3_path}")

print("\nDONE ✅")

══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
FINAL HMM MODE TABLE REPORT
Sharpe + DTB3 Sharpe + Annual Return + Cumulative Return
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
BAR_ANN    : 1512
Eval store : multi_ens_fixed_hmm_outputs/eval_output_store.pkl
Loaded eval store keys: 21
DTB3 date range: 2021-05-04 → 2025-12-31
DTB3 saved: hmm_mode_final_table_report/fred_DTB3_daily.csv

══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
FINAL HMM MODE TABLE — TEST PERIOD
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
      Pool     HMM Mode Sharpe (rf=0) Sharpe (rf=DTB3) Ann. Return

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# BEST CONFIG REPORT
# MAIN_14 × FULL_CORE × MODE B
#
# Target:
#   MAIN_14__HMM_FULL_CORE_K5__B_STATE_WEIGHTS_STATE_THRESHOLD
#
# Outputs:
#   best_config_MAIN14_FULL_MODEB_report/
#       BEST_CONFIG_metrics_raw.csv
#       BEST_CONFIG_metrics_formatted.csv
#       BEST_CONFIG_yearly_breakdown_raw.csv
#       BEST_CONFIG_yearly_breakdown_formatted.csv
#       BEST_CONFIG_bar_returns_equity_drawdown.csv
#       BEST_CONFIG_equity_curve.png
#       BEST_CONFIG_drawdown_curve.png
#       BEST_CONFIG_equity_and_drawdown.png
# ═══════════════════════════════════════════════════════════════════════════════

import os
import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =============================================================================
# 0. SETTINGS
# =============================================================================

BASE_DIR = "multi_ens_fixed_hmm_outputs"
REPORT_DIR = "best_config_MAIN14_FULL_MODEB_report"

EVAL_STORE_PATH = os.path.join(BASE_DIR, "eval_output_store.pkl")

os.makedirs(REPORT_DIR, exist_ok=True)

BAR_ANN = globals().get("BAR_ANN", 6 * 252)

POOL_NAME = "MAIN_14"
HMM_NAME = "HMM_FULL_CORE_K5"
MODE_NAME = "B_STATE_WEIGHTS_STATE_THRESHOLD"

TARGET_KEY = f"{POOL_NAME}__{HMM_NAME}__{MODE_NAME}"

print("═" * 150)
print("BEST CONFIG REPORT")
print("MAIN_14 × FULL_CORE × MODE B")
print("═" * 150)
print(f"Target key : {TARGET_KEY}")
print(f"BAR_ANN    : {BAR_ANN}")
print(f"Eval store : {EVAL_STORE_PATH}")


# =============================================================================
# 1. LOAD EVAL STORE
# =============================================================================

if not os.path.exists(EVAL_STORE_PATH):
    raise FileNotFoundError(f"eval_output_store.pkl bulunamadı: {EVAL_STORE_PATH}")

with open(EVAL_STORE_PATH, "rb") as f:
    eval_store = pickle.load(f)

if TARGET_KEY not in eval_store:
    print("\n[WARN] Target key bulunamadı. Benzer keyler:")
    for k in eval_store.keys():
        if "MAIN_14" in k and ("FULL" in k or "B_STATE" in k):
            print("  ", k)

    raise KeyError(f"Target key eval_store içinde yok: {TARGET_KEY}")

obj = eval_store[TARGET_KEY]

print("\nSelected object:")
print(f"  key           : {obj.get('key')}")
print(f"  ensemble_name : {obj.get('ensemble_name')}")
print(f"  candidate     : {obj.get('candidate')}")
print(f"  mode          : {obj.get('mode')}")


# =============================================================================
# 2. FETCH DTB3 FROM FRED
# =============================================================================

test_index = pd.to_datetime(obj["test_index"])

start_date = test_index.min().date()
end_date = test_index.max().date()

print(f"\nDTB3 date range: {start_date} → {end_date}")

def fetch_dtb3_from_fred(start, end):
    try:
        from pandas_datareader import data as pdr
    except ImportError:
        raise ImportError(
            "pandas_datareader yüklü değil. Önce şunu çalıştır:\n"
            "!pip install pandas_datareader"
        )

    dtb3 = pdr.DataReader(
        "DTB3",
        "fred",
        start=start,
        end=end
    )

    dtb3 = dtb3.rename(columns={"DTB3": "DTB3"})
    dtb3.index = pd.to_datetime(dtb3.index)
    dtb3 = dtb3.sort_index()

    return dtb3


dtb3_daily = fetch_dtb3_from_fred(start_date, end_date)

dtb3_path = os.path.join(REPORT_DIR, "fred_DTB3_daily.csv")
dtb3_daily.to_csv(dtb3_path, encoding="utf-8-sig")

print(f"DTB3 saved → {dtb3_path}")


# =============================================================================
# 3. METRIC HELPERS
# =============================================================================

def clean_arr(x):
    return np.nan_to_num(
        np.asarray(x, dtype=float),
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )


def cumulative_return(ret):
    r = clean_arr(ret)

    if len(r) == 0:
        return 0.0

    return float(np.prod(1.0 + r) - 1.0)


def annual_return(ret, bar_ann=BAR_ANN):
    r = clean_arr(ret)

    if len(r) == 0:
        return 0.0

    cum_ret = cumulative_return(r)
    years = len(r) / bar_ann

    if years <= 0 or cum_ret <= -0.999:
        return 0.0

    return float((1.0 + cum_ret) ** (1.0 / years) - 1.0)


def sharpe_rets(ret, bar_ann=BAR_ANN):
    r = clean_arr(ret)

    if len(r) <= 2:
        return 0.0

    sd = np.nanstd(r, ddof=1)

    if sd <= 1e-12:
        return 0.0

    return float(np.nanmean(r) / sd * np.sqrt(bar_ann))


def sortino_rets(ret, target=0.0, bar_ann=BAR_ANN):
    """
    Annualized Sortino ratio.
    Downside deviation uses only returns below target.
    """

    r = clean_arr(ret)

    if len(r) <= 2:
        return 0.0

    downside = np.minimum(r - target, 0.0)
    downside_dev = np.sqrt(np.nanmean(downside ** 2))

    if downside_dev <= 1e-12:
        return 0.0

    return float(np.nanmean(r - target) / downside_dev * np.sqrt(bar_ann))


def equity_curve(ret):
    r = clean_arr(ret)
    return np.cumprod(1.0 + r)


def drawdown_curve(ret):
    eq = equity_curve(ret)

    if len(eq) == 0:
        return np.array([])

    peak = np.maximum.accumulate(eq)
    dd = eq / (peak + 1e-12) - 1.0

    return dd


def max_drawdown(ret):
    dd = drawdown_curve(ret)

    if len(dd) == 0:
        return 0.0

    return float(np.min(dd))


def calmar_ratio(ret, bar_ann=BAR_ANN):
    """
    Calmar = Annual Return / abs(Max Drawdown)
    """

    ann = annual_return(ret, bar_ann=bar_ann)
    mdd = abs(max_drawdown(ret))

    if mdd <= 1e-12:
        return 0.0

    return float(ann / mdd)


def get_rf_bar_series(index, dtb3_daily, bar_ann=BAR_ANN):
    index = pd.to_datetime(index)

    rf = (
        dtb3_daily
        .reindex(index)
        .ffill()
        .bfill()
    )

    rf_annual_pct = rf["DTB3"].astype(float).fillna(0.0)

    # DTB3 annual percentage:
    # Example: 5.25 means annual 5.25%
    rf_bar = (rf_annual_pct / 100.0) / bar_ann
    rf_bar.name = "rf_bar_DTB3"

    return rf_bar


def sharpe_rets_rf_DTB3(ret, index, dtb3_daily, bar_ann=BAR_ANN):
    r = clean_arr(ret)
    index = pd.to_datetime(index)

    if len(r) <= 2:
        return 0.0

    rf_bar = get_rf_bar_series(
        index=index,
        dtb3_daily=dtb3_daily,
        bar_ann=bar_ann
    ).values

    n = min(len(r), len(rf_bar))
    excess = r[:n] - rf_bar[:n]

    sd = np.nanstd(excess, ddof=1)

    if sd <= 1e-12:
        return 0.0

    return float(np.nanmean(excess) / sd * np.sqrt(bar_ann))


def summarize_metrics(index, ret, sig, dtb3_daily):
    index = pd.to_datetime(index)
    ret = clean_arr(ret)
    sig = clean_arr(sig)

    out = {
        "Pool": POOL_NAME,
        "HMM": "FULL_CORE",
        "Mode": "B",
        "Sharpe (rf=0)": sharpe_rets(ret),
        "Sharpe (rf=DTB3)": sharpe_rets_rf_DTB3(
            ret=ret,
            index=index,
            dtb3_daily=dtb3_daily
        ),
        "Sortino": sortino_rets(ret),
        "Calmar": calmar_ratio(ret),
        "Ann. Return": annual_return(ret),
        "Cum. Return": cumulative_return(ret),
        "Max DD": max_drawdown(ret),
        "Activity": float(np.mean(sig != 0)) if len(sig) else 0.0,
        "Trades": int(np.sum(np.abs(np.diff(sig, prepend=0.0)) > 0)) if len(sig) else 0,
        "N Obs": int(len(ret)),
    }

    return out


# =============================================================================
# 4. EXTRACT TEST SERIES
# =============================================================================

test_index = pd.to_datetime(obj["test_index"])
test_ret = clean_arr(obj["test_ret"])
test_signal = clean_arr(obj["test_signal"])

test_prob = None
if obj.get("test_prob") is not None:
    test_prob = clean_arr(obj["test_prob"])

print("\nTest data:")
print(f"  observations : {len(test_index)}")
print(f"  start        : {test_index.min()}")
print(f"  end          : {test_index.max()}")


# =============================================================================
# 5. OVERALL METRIC TABLE
# =============================================================================

metrics = summarize_metrics(
    index=test_index,
    ret=test_ret,
    sig=test_signal,
    dtb3_daily=dtb3_daily
)

metrics_raw = pd.DataFrame([metrics])

metrics_raw_path = os.path.join(REPORT_DIR, "BEST_CONFIG_metrics_raw.csv")
metrics_raw.to_csv(metrics_raw_path, index=False, encoding="utf-8-sig")


def fmt_num(x):
    if pd.isna(x):
        return ""
    return f"{x:.3f}"


def fmt_pct(x):
    if pd.isna(x):
        return ""
    return f"{x:.2%}"


metrics_fmt = metrics_raw.copy()

for c in ["Sharpe (rf=0)", "Sharpe (rf=DTB3)", "Sortino", "Calmar"]:
    metrics_fmt[c] = metrics_fmt[c].map(fmt_num)

for c in ["Ann. Return", "Cum. Return", "Max DD", "Activity"]:
    metrics_fmt[c] = metrics_fmt[c].map(fmt_pct)

metrics_fmt["Trades"] = metrics_fmt["Trades"].astype(int).astype(str)
metrics_fmt["N Obs"] = metrics_fmt["N Obs"].astype(int).astype(str)

metrics_fmt_path = os.path.join(REPORT_DIR, "BEST_CONFIG_metrics_formatted.csv")
metrics_fmt.to_csv(metrics_fmt_path, index=False, encoding="utf-8-sig")

print("\n" + "═" * 150)
print("BEST CONFIG FULL METRIC TABLE — TEST PERIOD")
print("═" * 150)
print(metrics_fmt.to_string(index=False))
print("═" * 150)


# =============================================================================
# 6. BAR-LEVEL RETURN / EQUITY / DRAWDOWN DATA
# =============================================================================

bar_df = pd.DataFrame(index=test_index)

bar_df["ret"] = test_ret
bar_df["signal"] = test_signal

if test_prob is not None:
    bar_df["prob"] = test_prob

bar_df["equity"] = equity_curve(bar_df["ret"].values)
bar_df["drawdown"] = drawdown_curve(bar_df["ret"].values)

bar_df["rf_bar_DTB3"] = get_rf_bar_series(
    index=bar_df.index,
    dtb3_daily=dtb3_daily,
    bar_ann=BAR_ANN
).values

bar_df["excess_ret_DTB3"] = bar_df["ret"] - bar_df["rf_bar_DTB3"]
bar_df["excess_equity_DTB3"] = equity_curve(bar_df["excess_ret_DTB3"].values)
bar_df["excess_drawdown_DTB3"] = drawdown_curve(bar_df["excess_ret_DTB3"].values)

bar_path = os.path.join(REPORT_DIR, "BEST_CONFIG_bar_returns_equity_drawdown.csv")
bar_df.to_csv(bar_path, encoding="utf-8-sig")

print(f"\nBar-level data saved → {bar_path}")


# =============================================================================
# 7. YEARLY PERFORMANCE BREAKDOWN
# =============================================================================

yearly_rows = []

for year, g in bar_df.groupby(bar_df.index.year):

    y_index = g.index
    y_ret = g["ret"].values
    y_sig = g["signal"].values

    y_metrics = {
        "Year": int(year),
        "Sharpe (rf=0)": sharpe_rets(y_ret),
        "Sharpe (rf=DTB3)": sharpe_rets_rf_DTB3(
            ret=y_ret,
            index=y_index,
            dtb3_daily=dtb3_daily
        ),
        "Sortino": sortino_rets(y_ret),
        "Calmar": calmar_ratio(y_ret),
        "Ann. Return": annual_return(y_ret),
        "Cum. Return": cumulative_return(y_ret),
        "Max DD": max_drawdown(y_ret),
        "Activity": float(np.mean(y_sig != 0)) if len(y_sig) else 0.0,
        "Trades": int(np.sum(np.abs(np.diff(y_sig, prepend=0.0)) > 0)) if len(y_sig) else 0,
        "Long Ratio": float(np.mean(y_sig == 1.0)) if len(y_sig) else 0.0,
        "Short Ratio": float(np.mean(y_sig == -1.0)) if len(y_sig) else 0.0,
        "Flat Ratio": float(np.mean(y_sig == 0.0)) if len(y_sig) else 0.0,
        "N Obs": int(len(y_ret)),
    }

    yearly_rows.append(y_metrics)

yearly_raw = pd.DataFrame(yearly_rows)

yearly_raw_path = os.path.join(REPORT_DIR, "BEST_CONFIG_yearly_breakdown_raw.csv")
yearly_raw.to_csv(yearly_raw_path, index=False, encoding="utf-8-sig")

yearly_fmt = yearly_raw.copy()

for c in ["Sharpe (rf=0)", "Sharpe (rf=DTB3)", "Sortino", "Calmar"]:
    yearly_fmt[c] = yearly_fmt[c].map(fmt_num)

for c in [
    "Ann. Return",
    "Cum. Return",
    "Max DD",
    "Activity",
    "Long Ratio",
    "Short Ratio",
    "Flat Ratio",
]:
    yearly_fmt[c] = yearly_fmt[c].map(fmt_pct)

yearly_fmt["Trades"] = yearly_fmt["Trades"].astype(int).astype(str)
yearly_fmt["N Obs"] = yearly_fmt["N Obs"].astype(int).astype(str)

yearly_fmt_path = os.path.join(REPORT_DIR, "BEST_CONFIG_yearly_breakdown_formatted.csv")
yearly_fmt.to_csv(yearly_fmt_path, index=False, encoding="utf-8-sig")

print("\n" + "═" * 150)
print("YEARLY PERFORMANCE BREAKDOWN — TEST PERIOD")
print("═" * 150)
print(yearly_fmt.to_string(index=False))
print("═" * 150)


# =============================================================================
# 8. PLOTS — EQUITY CURVE AND DRAWDOWN CURVE
# =============================================================================

plt.figure(figsize=(13, 5))
plt.plot(bar_df.index, bar_df["equity"], label="Strategy Equity")
plt.title("MAIN_14 × FULL_CORE × Mode B — Equity Curve")
plt.xlabel("Date")
plt.ylabel("Equity")
plt.legend()
plt.tight_layout()

equity_plot_path = os.path.join(REPORT_DIR, "BEST_CONFIG_equity_curve.png")
plt.savefig(equity_plot_path, dpi=150)
plt.close()


plt.figure(figsize=(13, 5))
plt.plot(bar_df.index, bar_df["drawdown"], label="Drawdown")
plt.title("MAIN_14 × FULL_CORE × Mode B — Drawdown Curve")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.legend()
plt.tight_layout()

drawdown_plot_path = os.path.join(REPORT_DIR, "BEST_CONFIG_drawdown_curve.png")
plt.savefig(drawdown_plot_path, dpi=150)
plt.close()


fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

axes[0].plot(bar_df.index, bar_df["equity"], label="Strategy Equity")
axes[0].set_title("MAIN_14 × FULL_CORE × Mode B — Equity Curve")
axes[0].set_ylabel("Equity")
axes[0].legend()

axes[1].plot(bar_df.index, bar_df["drawdown"], label="Drawdown")
axes[1].set_title("MAIN_14 × FULL_CORE × Mode B — Drawdown Curve")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Drawdown")
axes[1].legend()

plt.tight_layout()

combined_plot_path = os.path.join(REPORT_DIR, "BEST_CONFIG_equity_and_drawdown.png")
plt.savefig(combined_plot_path, dpi=150)
plt.close()


# =============================================================================
# 9. FINAL PRINT
# =============================================================================

print("\nFILES SAVED")
print("═" * 150)
print(f"Metrics raw CSV                 → {metrics_raw_path}")
print(f"Metrics formatted CSV           → {metrics_fmt_path}")
print(f"Yearly raw CSV                  → {yearly_raw_path}")
print(f"Yearly formatted CSV            → {yearly_fmt_path}")
print(f"Bar returns/equity/drawdown CSV → {bar_path}")
print(f"Equity curve PNG                → {equity_plot_path}")
print(f"Drawdown curve PNG              → {drawdown_plot_path}")
print(f"Combined equity/drawdown PNG    → {combined_plot_path}")
print(f"FRED DTB3 CSV                   → {dtb3_path}")
print("═" * 150)

print("\nDONE ✅")

══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
BEST CONFIG REPORT
MAIN_14 × FULL_CORE × MODE B
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Target key : MAIN_14__HMM_FULL_CORE_K5__B_STATE_WEIGHTS_STATE_THRESHOLD
BAR_ANN    : 1512
Eval store : multi_ens_fixed_hmm_outputs/eval_output_store.pkl

Selected object:
  key           : MAIN_14__HMM_FULL_CORE_K5__B_STATE_WEIGHTS_STATE_THRESHOLD
  ensemble_name : MAIN_14
  candidate     : HMM_FULL_CORE_K5
  mode          : B_STATE_WEIGHTS_STATE_THRESHOLD

DTB3 date range: 2021-05-04 → 2025-12-31
DTB3 saved → best_config_MAIN14_FULL_MODEB_report/fred_DTB3_daily.csv

Test data:
  observations : 7523
  start        : 2021-05-04 04:00:00
  end          : 2025-12-31 16:00:00

══════════════════════════════════════════════════════════════════════

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# POOL CONFIG SUMMARY TABLE
#
# Output:
# Pool | Best Sharpe | Best Config | Avg Sharpe (all modes) | Most Consistent Mode
#
# Reads:
#   multi_ens_fixed_hmm_outputs/eval_output_store.pkl
#
# Uses TEST returns only.
# ═══════════════════════════════════════════════════════════════════════════════

import os
import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# =============================================================================
# 0. SETTINGS
# =============================================================================

BASE_DIR = "multi_ens_fixed_hmm_outputs"
REPORT_DIR = "pool_config_summary_report"

EVAL_STORE_PATH = os.path.join(BASE_DIR, "eval_output_store.pkl")

os.makedirs(REPORT_DIR, exist_ok=True)

BAR_ANN = globals().get("BAR_ANN", 6 * 252)

POOL_ORDER = ["MAIN_14", "THEORY_11", "ENSEMBLE_B"]

print("═" * 130)
print("POOL CONFIG SUMMARY TABLE")
print("═" * 130)
print(f"Eval store: {EVAL_STORE_PATH}")
print(f"BAR_ANN   : {BAR_ANN}")


# =============================================================================
# 1. LOAD EVAL STORE
# =============================================================================

if not os.path.exists(EVAL_STORE_PATH):
    raise FileNotFoundError(f"eval_output_store.pkl bulunamadı: {EVAL_STORE_PATH}")

with open(EVAL_STORE_PATH, "rb") as f:
    eval_store = pickle.load(f)

print(f"Loaded eval store keys: {len(eval_store)}")


# =============================================================================
# 2. METRIC HELPERS
# =============================================================================

def clean_arr(x):
    return np.nan_to_num(
        np.asarray(x, dtype=float),
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )


def sharpe_rets(ret, bar_ann=BAR_ANN):
    r = clean_arr(ret)

    if len(r) <= 2:
        return 0.0

    sd = np.nanstd(r, ddof=1)

    if sd <= 1e-12:
        return 0.0

    return float(np.nanmean(r) / sd * np.sqrt(bar_ann))


def cumulative_return(ret):
    r = clean_arr(ret)

    if len(r) == 0:
        return 0.0

    return float(np.prod(1.0 + r) - 1.0)


def max_drawdown(ret):
    r = clean_arr(ret)

    if len(r) == 0:
        return 0.0

    equity = np.cumprod(1.0 + r)
    peak = np.maximum.accumulate(equity)
    dd = equity / (peak + 1e-12) - 1.0

    return float(np.min(dd))


def annual_return(ret, bar_ann=BAR_ANN):
    r = clean_arr(ret)

    if len(r) == 0:
        return 0.0

    cum_ret = cumulative_return(r)
    years = len(r) / bar_ann

    if years <= 0 or cum_ret <= -0.999:
        return 0.0

    return float((1.0 + cum_ret) ** (1.0 / years) - 1.0)


# =============================================================================
# 3. LABEL HELPERS
# =============================================================================

def parse_hmm_label(candidate):
    """
    Converts candidate name to short HMM label.
    """

    if candidate == "C_REGIME_FREE":
        return "—"

    if candidate == "HMM_CLASSIC_LITERATURE_CORE":
        return "CLASSIC"

    if candidate == "HMM_FULL_CORE_K5":
        return "FULL"

    if "CLASSIC" in str(candidate).upper():
        return "CLASSIC"

    if "FULL" in str(candidate).upper():
        return "FULL"

    return str(candidate)


def parse_mode_label(mode):
    """
    Converts full mode name to A/B/C/D.
    """

    mode = str(mode)

    if mode == "C_REGIME_FREE":
        return "C"

    if mode.startswith("A_"):
        return "A"

    if mode.startswith("B_"):
        return "B"

    if mode.startswith("D_"):
        return "D"

    return mode


def config_label(hmm_label, mode_label):
    return f"{hmm_label} × {mode_label}"


# =============================================================================
# 4. BUILD LONG MODE RESULT TABLE FROM EVAL STORE
# =============================================================================

long_rows = []

for key, obj in eval_store.items():

    ensemble_name = obj.get("ensemble_name")
    candidate = obj.get("candidate")
    mode = obj.get("mode")

    if ensemble_name not in POOL_ORDER:
        continue

    if obj.get("test_ret") is None:
        continue

    test_ret = clean_arr(obj["test_ret"])

    if len(test_ret) == 0:
        continue

    test_signal = clean_arr(obj["test_signal"]) if obj.get("test_signal") is not None else np.zeros(len(test_ret))

    hmm_short = parse_hmm_label(candidate)
    mode_short = parse_mode_label(mode)
    cfg = config_label(hmm_short, mode_short)

    sh = sharpe_rets(test_ret)

    long_rows.append({
        "Pool": ensemble_name,
        "HMM": hmm_short,
        "Mode": mode_short,
        "Config": cfg,
        "Sharpe": sh,
        "Ann. Return": annual_return(test_ret),
        "Cum. Return": cumulative_return(test_ret),
        "Max DD": max_drawdown(test_ret),
        "Activity": float(np.mean(test_signal != 0)) if len(test_signal) else 0.0,
        "Trades": int(np.sum(np.abs(np.diff(test_signal, prepend=0.0)) > 0)) if len(test_signal) else 0,
        "key": key,
    })

long_df = pd.DataFrame(long_rows)

if long_df.empty:
    raise ValueError("eval_store içinden uygun test sonucu çıkarılamadı.")

# İstenen mod/config sırası için yardımcı kolon
mode_order_map = {"A": 1, "B": 2, "D": 3, "C": 4}
hmm_order_map = {"CLASSIC": 1, "FULL": 2, "—": 3}

long_df["Pool_Order"] = long_df["Pool"].map({p: i for i, p in enumerate(POOL_ORDER)})
long_df["HMM_Order"] = long_df["HMM"].map(hmm_order_map).fillna(99)
long_df["Mode_Order"] = long_df["Mode"].map(mode_order_map).fillna(99)

long_df = (
    long_df
    .sort_values(["Pool_Order", "HMM_Order", "Mode_Order"])
    .reset_index(drop=True)
)


# =============================================================================
# 5. BUILD SUMMARY TABLE
# =============================================================================

summary_rows = []

for pool in POOL_ORDER:

    pool_df = long_df[long_df["Pool"] == pool].copy()

    if pool_df.empty:
        summary_rows.append({
            "Pool": pool,
            "Best Sharpe": np.nan,
            "Best Config": "",
            "Avg Sharpe (all modes)": np.nan,
            "Most Consistent Mode": "",
        })
        continue

    # -------------------------------------------------------------------------
    # Best Sharpe and Best Config
    # -------------------------------------------------------------------------

    best_row = (
        pool_df
        .sort_values("Sharpe", ascending=False)
        .iloc[0]
    )

    best_sharpe = best_row["Sharpe"]
    best_config = best_row["Config"]

    # -------------------------------------------------------------------------
    # Average Sharpe across all modes/configs
    # -------------------------------------------------------------------------

    avg_sharpe = float(pool_df["Sharpe"].mean())

    # -------------------------------------------------------------------------
    # Most Consistent Mode
    # -------------------------------------------------------------------------
    # Mode consistency rule:
    #   For each mode A/B/D/C, take average Sharpe across available configs.
    #   Pick the mode with highest average Sharpe.
    #
    # Example:
    #   A has CLASSIC A + FULL A
    #   B has CLASSIC B + FULL B
    #   D has CLASSIC D + FULL D
    #   C has only regime-free C
    #
    # This gives mode-level consistency, not single-run best.
    # -------------------------------------------------------------------------

    mode_summary = (
        pool_df
        .groupby("Mode", as_index=False)
        .agg(
            avg_mode_sharpe=("Sharpe", "mean"),
            std_mode_sharpe=("Sharpe", "std"),
            n_configs=("Sharpe", "count"),
            min_mode_sharpe=("Sharpe", "min"),
            max_mode_sharpe=("Sharpe", "max"),
        )
    )

    # Daha dengeli olsun diye:
    # önce avg_mode_sharpe yüksek,
    # sonra std düşük,
    # sonra n_configs yüksek.
    mode_summary["std_mode_sharpe"] = mode_summary["std_mode_sharpe"].fillna(0.0)

    mode_summary = mode_summary.sort_values(
        ["avg_mode_sharpe", "std_mode_sharpe", "n_configs"],
        ascending=[False, True, False]
    )

    most_consistent_mode = mode_summary.iloc[0]["Mode"]

    summary_rows.append({
        "Pool": pool,
        "Best Sharpe": best_sharpe,
        "Best Config": best_config,
        "Avg Sharpe (all modes)": avg_sharpe,
        "Most Consistent Mode": most_consistent_mode,
    })

summary_raw = pd.DataFrame(summary_rows)


# =============================================================================
# 6. SAVE RAW OUTPUTS
# =============================================================================

long_raw_path = os.path.join(REPORT_DIR, "POOL_CONFIG_all_mode_results_raw.csv")
summary_raw_path = os.path.join(REPORT_DIR, "POOL_CONFIG_summary_raw.csv")

long_df.to_csv(
    long_raw_path,
    index=False,
    encoding="utf-8-sig"
)

summary_raw.to_csv(
    summary_raw_path,
    index=False,
    encoding="utf-8-sig"
)


# =============================================================================
# 7. FORMATTED SUMMARY
# =============================================================================

def fmt_num(x):
    if pd.isna(x):
        return ""
    return f"{x:.3f}"


summary_fmt = summary_raw.copy()

summary_fmt["Best Sharpe"] = summary_fmt["Best Sharpe"].map(fmt_num)
summary_fmt["Avg Sharpe (all modes)"] = summary_fmt["Avg Sharpe (all modes)"].map(fmt_num)

summary_fmt_path = os.path.join(REPORT_DIR, "POOL_CONFIG_summary_formatted.csv")

summary_fmt.to_csv(
    summary_fmt_path,
    index=False,
    encoding="utf-8-sig"
)


# =============================================================================
# 8. OPTIONAL: MODE CONSISTENCY DETAIL TABLE
# =============================================================================

mode_consistency_rows = []

for pool in POOL_ORDER:

    pool_df = long_df[long_df["Pool"] == pool].copy()

    if pool_df.empty:
        continue

    tmp = (
        pool_df
        .groupby("Mode", as_index=False)
        .agg(
            Avg_Sharpe=("Sharpe", "mean"),
            Std_Sharpe=("Sharpe", "std"),
            Min_Sharpe=("Sharpe", "min"),
            Max_Sharpe=("Sharpe", "max"),
            N_Configs=("Sharpe", "count"),
        )
    )

    tmp["Pool"] = pool
    tmp["Std_Sharpe"] = tmp["Std_Sharpe"].fillna(0.0)

    mode_consistency_rows.extend(tmp.to_dict("records"))

mode_consistency_df = pd.DataFrame(mode_consistency_rows)

mode_consistency_path = os.path.join(REPORT_DIR, "POOL_CONFIG_mode_consistency_detail.csv")

mode_consistency_df.to_csv(
    mode_consistency_path,
    index=False,
    encoding="utf-8-sig"
)


# =============================================================================
# 9. PRINT FINAL TABLES
# =============================================================================

print("\n" + "═" * 130)
print("POOL CONFIG SUMMARY")
print("═" * 130)
print(summary_fmt.to_string(index=False))
print("═" * 130)

print("\nALL MODE RESULTS USED")
print("═" * 130)

print_cols = [
    "Pool",
    "HMM",
    "Mode",
    "Config",
    "Sharpe",
    "Ann. Return",
    "Cum. Return",
    "Max DD",
    "Activity",
    "Trades",
]

long_print = long_df[print_cols].copy()

for c in ["Sharpe"]:
    long_print[c] = long_print[c].map(fmt_num)

for c in ["Ann. Return", "Cum. Return", "Max DD", "Activity"]:
    long_print[c] = long_print[c].map(lambda x: "" if pd.isna(x) else f"{x:.2%}")

long_print["Trades"] = long_print["Trades"].astype(int).astype(str)

print(long_print.to_string(index=False))

print("\nMODE CONSISTENCY DETAIL")
print("═" * 130)

mode_consistency_print = mode_consistency_df.copy()

for c in ["Avg_Sharpe", "Std_Sharpe", "Min_Sharpe", "Max_Sharpe"]:
    mode_consistency_print[c] = mode_consistency_print[c].map(fmt_num)

print(
    mode_consistency_print[
        ["Pool", "Mode", "Avg_Sharpe", "Std_Sharpe", "Min_Sharpe", "Max_Sharpe", "N_Configs"]
    ].to_string(index=False)
)

print("\nFILES")
print("═" * 130)
print(f"All mode raw results        → {long_raw_path}")
print(f"Summary raw                 → {summary_raw_path}")
print(f"Summary formatted           → {summary_fmt_path}")
print(f"Mode consistency detail     → {mode_consistency_path}")
print("═" * 130)

print("\nDONE ✅")

══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
POOL CONFIG SUMMARY TABLE
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Eval store: multi_ens_fixed_hmm_outputs/eval_output_store.pkl
BAR_ANN   : 1512
Loaded eval store keys: 21

══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
POOL CONFIG SUMMARY
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
      Pool Best Sharpe Best Config Avg Sharpe (all modes) Most Consistent Mode
   MAIN_14       2.943    FULL × B                  2.252                    B
 THEORY_11       2.400    FULL × B                  1.902                    B
ENSEMBLE_B       2.174 CLASSIC × A                  1.725                    A
═══════

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# MAIN_14 + THEORY_11 × FULL_CORE × MODE B
# STATE-BASED EXPERT CHANNEL WEIGHT TABLES
#
# Outputs:
#   best_modeB_state_weights_report/
#       MAIN_14_FULL_MODEB_state_expert_weights_raw.csv
#       MAIN_14_FULL_MODEB_state_expert_weights_formatted.csv
#       MAIN_14_FULL_MODEB_state_expert_weights_heatmap.png
#       THEORY_11_FULL_MODEB_state_expert_weights_raw.csv
#       THEORY_11_FULL_MODEB_state_expert_weights_formatted.csv
#       THEORY_11_FULL_MODEB_state_expert_weights_heatmap.png
#       COMBINED_FULL_MODEB_state_expert_weights_raw.csv
#       COMBINED_FULL_MODEB_state_expert_weights_formatted.csv
# ═══════════════════════════════════════════════════════════════════════════════

import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:
    sns = None


# =============================================================================
# 0. SETTINGS
# =============================================================================

BASE_DIR = "multi_ens_fixed_hmm_outputs"
REPORT_DIR = "best_modeB_state_weights_report"

EVAL_STORE_PATH = os.path.join(BASE_DIR, "eval_output_store.pkl")

os.makedirs(REPORT_DIR, exist_ok=True)

TARGET_CONFIGS = {
    "MAIN_14": {
        "key": "MAIN_14__HMM_FULL_CORE_K5__B_STATE_WEIGHTS_STATE_THRESHOLD",
        "label": "MAIN_14 × FULL_CORE × Mode B",
    },
    "THEORY_11": {
        "key": "THEORY_11__HMM_FULL_CORE_K5__B_STATE_WEIGHTS_STATE_THRESHOLD",
        "label": "THEORY_11 × FULL_CORE × Mode B",
    },
}

print("═" * 150)
print("MAIN_14 + THEORY_11 × FULL_CORE × MODE B — STATE EXPERT WEIGHTS")
print("═" * 150)
print(f"Eval store: {EVAL_STORE_PATH}")


# =============================================================================
# 1. ENSEMBLE DEFINITIONS
# =============================================================================

MAIN_14_ENSEMBLE = {
    "REAL_RATE":        "T01_REAL_RATE_SELECTED_3_growth_realrate_channel__LGBM",
    "MACRO_SURPRISE":   "MACRO_KS_SELECTED_1_infl_energy_real_yield_spx__XGB",
    "CURVE_LIQ_POLICY": "RF_MACRO_KS_01_curve_liq_realrate_policy__LGBM",
    "LIQUIDITY":        "LIQUIDITY_HGBM_TEST_SELECTED_03_macro_liquidity_rates_risk__HGBM",
    "INFL_POLICY":      "INFL_POLICY_RATES_SELECTED_3_real_yield_momentum__XGB",
    "GK_GREEKS":        "GK_RF_SELECTED_01_gamma_vega_vrp_carry_relD1__XGB",
    "SAFE_HAVEN":       "BOP_SAFEHAVEN_SELECTED_1_chf_jpy_sovereign_risk__LGBM",
    "SOVEREIGN":        "SOVEREIGN_SELECTED_2_it_de_rates_risk__LGBM",
    "CREDIT":           "CREDIT_SOV_RATES_SELECTED_02_credit_spread_risk_compact__LGBM",
    "GROWTH_POLICY":    "POLICY_GROWTH_RATES_SELECTED_01_growth_policy_curve__LGBM",
    "ENERGY":           "ENERGY_GOLD_RISK_SELECTED_01_energy_gold_oil_risk__LGBM",
    "PPP":              "T04_PPP_SELECTED_4_DXY1_clean_inflation_m2_dxy_mom63__XGB",
    "UIP_CARRY":        "TAYLOR_UIP_XGB_TEST_AUG01_policy_carry_volrisk_score_gap__XGB",
    "TAYLOR":           "TAYLOR_GBM_VAL_SELECTED_01_growth_inflation_policy_rates__XGB",
}

THEORY_11_ENSEMBLE = {
    "REAL_RATE":   "T01_REAL_RATE_SELECTED_3_growth_realrate_channel__LGBM",
    "LIQUIDITY":   "LIQUIDITY_PRESSURE_SELECTED_1_money_netliq_risk__LGBM",
    "INFL_POLICY": "INFL_POLICY_RATES_SELECTED_3_real_yield_momentum__XGB",
    "GK_OPTIONS":  "GK_RF_SELECTED_03_compact_gamma_vrp_carry__LGBM",
    "SAFE_HAVEN":  "BOP_SAFEHAVEN_SELECTED_1_chf_jpy_sovereign_risk__LGBM",
    "SOVEREIGN":   "SOVEREIGN_SELECTED_2_it_de_rates_risk__LGBM",
    "CREDIT":      "CREDIT_SOV_RATES_SELECTED_02_credit_spread_risk_compact__LGBM",
    "ENERGY":      "ENERGY_TOT_SELECTED_1_brent_gas_eu_pressure__LGBM",
    "UIP_CARRY":   "UIP_HGBM_TEST_AUG01_policy_carry_risk_gap_vol__XGB",
    "TAYLOR_LIKE": "TAYLOR_XGB_FULL_MACRO_PURE_ROBUST_01_growth_policy_score_rates__XGB",
    "CURVE":       "CURVE_GROWTH_RATES_SELECTED_1_policy_curve_rates__LGBM",
}

ENSEMBLE_MAP = {
    "MAIN_14": MAIN_14_ENSEMBLE,
    "THEORY_11": THEORY_11_ENSEMBLE,
}


# =============================================================================
# 2. PRETTY NAMES AND ORDERS
# =============================================================================

CHANNEL_PRETTY = {
    "REAL_RATE": "Real Rate",
    "MACRO_SURPRISE": "Macro Surprise",
    "CURVE_LIQ_POLICY": "Curve-Liquidity-Policy",
    "LIQUIDITY": "Liquidity",
    "INFL_POLICY": "Inflation-Policy",
    "GK_GREEKS": "Options / Garman-Kohlhagen",
    "GK_OPTIONS": "Options / Garman-Kohlhagen",
    "SAFE_HAVEN": "Safe Haven",
    "SOVEREIGN": "Sovereign Risk",
    "CREDIT": "Credit Risk",
    "GROWTH_POLICY": "Growth-Policy",
    "ENERGY": "Energy",
    "PPP": "PPP",
    "UIP_CARRY": "UIP / Carry",
    "TAYLOR": "Taylor Rule",
    "TAYLOR_LIKE": "Taylor Rule",
    "CURVE": "Curve / Rates",
}

CHANNEL_ORDER = {
    "MAIN_14": [
        "REAL_RATE",
        "MACRO_SURPRISE",
        "CURVE_LIQ_POLICY",
        "LIQUIDITY",
        "INFL_POLICY",
        "GK_GREEKS",
        "SAFE_HAVEN",
        "SOVEREIGN",
        "CREDIT",
        "GROWTH_POLICY",
        "ENERGY",
        "PPP",
        "UIP_CARRY",
        "TAYLOR",
    ],
    "THEORY_11": [
        "REAL_RATE",
        "LIQUIDITY",
        "INFL_POLICY",
        "GK_OPTIONS",
        "SAFE_HAVEN",
        "SOVEREIGN",
        "CREDIT",
        "ENERGY",
        "UIP_CARRY",
        "TAYLOR_LIKE",
        "CURVE",
    ],
}


# =============================================================================
# 3. LOAD EVAL STORE
# =============================================================================

if not os.path.exists(EVAL_STORE_PATH):
    raise FileNotFoundError(f"eval_output_store.pkl bulunamadı: {EVAL_STORE_PATH}")

with open(EVAL_STORE_PATH, "rb") as f:
    eval_store = pickle.load(f)

print(f"Loaded eval store keys: {len(eval_store)}")


# =============================================================================
# 4. CORE FUNCTION
# =============================================================================

def extract_state_weights_for_config(pool_name, target_key, ensemble_dict, channel_order):
    if target_key not in eval_store:
        print(f"\n[WARN] Target key bulunamadı: {target_key}")
        print(f"{pool_name} için benzer keyler:")

        for k in eval_store.keys():
            if pool_name in k and ("FULL" in k or "B_STATE" in k):
                print("  ", k)

        raise KeyError(f"Target key eval_store içinde yok: {target_key}")

    obj = eval_store[target_key]
    extra_info = obj.get("extra_info", {})
    state_solutions = extra_info.get("state_solutions", None)

    if state_solutions is None:
        raise ValueError(
            f"{pool_name} için extra_info['state_solutions'] bulunamadı. "
            "Bu tablo için Mode B çıktısı gerekiyor."
        )

    state_ids = sorted([int(s) for s in state_solutions.keys()])
    state_cols = [f"S{s} Weight" for s in state_ids]

    print("\n" + "═" * 150)
    print(f"{pool_name} — FULL_CORE × MODE B")
    print("═" * 150)
    print(f"Key          : {target_key}")
    print(f"States found : {state_ids}")

    rows = []

    for channel in channel_order:
        model_id = ensemble_dict[channel]
        pretty_name = CHANNEL_PRETTY.get(channel, channel)

        row = {
            "Pool": pool_name,
            "Expert Channel": pretty_name,
            "Channel Code": channel,
            "Model": model_id,
        }

        weights_for_avg = []

        for s in state_ids:
            sol_s = state_solutions.get(s, None)

            if sol_s is None:
                sol_s = state_solutions.get(str(s), None)

            if sol_s is None:
                w = np.nan
            else:
                weights_dict = sol_s.get("weights", {})
                w = weights_dict.get(model_id, 0.0)

            row[f"S{s} Weight"] = w
            weights_for_avg.append(w)

        row["Avg Weight"] = float(np.nanmean(weights_for_avg)) if len(weights_for_avg) else np.nan
        row["Max State Weight"] = float(np.nanmax(weights_for_avg)) if len(weights_for_avg) else np.nan

        if len(weights_for_avg) and not np.all(pd.isna(weights_for_avg)):
            row["Dominant State"] = f"S{state_ids[int(np.nanargmax(weights_for_avg))]}"
        else:
            row["Dominant State"] = ""

        rows.append(row)

    weights_raw = pd.DataFrame(rows)

    state_sum_rows = []

    for s in state_ids:
        col = f"S{s} Weight"

        state_sum_rows.append({
            "Pool": pool_name,
            "State": f"S{s}",
            "Weight Sum": weights_raw[col].sum(skipna=True),
            "Max Channel": weights_raw.loc[weights_raw[col].idxmax(), "Expert Channel"],
            "Max Weight": weights_raw[col].max(),
            "Min Weight": weights_raw[col].min(),
        })

    state_sum_df = pd.DataFrame(state_sum_rows)

    return weights_raw, state_sum_df, state_cols


# =============================================================================
# 5. RUN BOTH CONFIGS
# =============================================================================

combined_weights = []
combined_state_sums = []

for pool_name, cfg in TARGET_CONFIGS.items():

    target_key = cfg["key"]
    label = cfg["label"]

    ensemble_dict = ENSEMBLE_MAP[pool_name]
    channel_order = CHANNEL_ORDER[pool_name]

    weights_raw, state_sum_df, state_cols = extract_state_weights_for_config(
        pool_name=pool_name,
        target_key=target_key,
        ensemble_dict=ensemble_dict,
        channel_order=channel_order,
    )

    combined_weights.append(weights_raw)
    combined_state_sums.append(state_sum_df)

    # -------------------------------------------------------------------------
    # Save raw
    # -------------------------------------------------------------------------

    safe_name = pool_name.replace("/", "_")

    raw_path = os.path.join(
        REPORT_DIR,
        f"{safe_name}_FULL_MODEB_state_expert_weights_raw.csv"
    )

    state_sum_path = os.path.join(
        REPORT_DIR,
        f"{safe_name}_FULL_MODEB_state_weight_sums.csv"
    )

    weights_raw.to_csv(
        raw_path,
        index=False,
        encoding="utf-8-sig"
    )

    state_sum_df.to_csv(
        state_sum_path,
        index=False,
        encoding="utf-8-sig"
    )

    # -------------------------------------------------------------------------
    # Formatted
    # -------------------------------------------------------------------------

    weights_fmt = weights_raw.copy()

    weight_cols = state_cols + ["Avg Weight", "Max State Weight"]

    for c in weight_cols:
        if c in weights_fmt.columns:
            weights_fmt[c] = weights_fmt[c].map(
                lambda x: "" if pd.isna(x) else f"{x:.3f}"
            )

    formatted_path = os.path.join(
        REPORT_DIR,
        f"{safe_name}_FULL_MODEB_state_expert_weights_formatted.csv"
    )

    weights_fmt.to_csv(
        formatted_path,
        index=False,
        encoding="utf-8-sig"
    )

    # -------------------------------------------------------------------------
    # Print table
    # -------------------------------------------------------------------------

    print("\n" + "─" * 150)
    print(f"{label} — EXPERT CHANNEL WEIGHTS BY STATE")
    print("─" * 150)

    print_cols = [
        "Expert Channel",
    ] + state_cols + [
        "Avg Weight",
        "Dominant State",
    ]

    print(weights_fmt[print_cols].to_string(index=False))

    print("\nSTATE WEIGHT SUM CHECK")
    state_sum_print = state_sum_df.copy()

    for c in ["Weight Sum", "Max Weight", "Min Weight"]:
        state_sum_print[c] = state_sum_print[c].map(lambda x: f"{x:.4f}")

    print(state_sum_print.to_string(index=False))

    # -------------------------------------------------------------------------
    # Heatmap
    # -------------------------------------------------------------------------

    heatmap_df = weights_raw.set_index("Expert Channel")[state_cols].copy()

    heatmap_path = os.path.join(
        REPORT_DIR,
        f"{safe_name}_FULL_MODEB_state_expert_weights_heatmap.png"
    )

    if sns is not None:
        plt.figure(figsize=(10, max(6, len(heatmap_df) * 0.45)))

        sns.heatmap(
            heatmap_df,
            annot=True,
            fmt=".3f",
            linewidths=0.5,
            cbar_kws={"label": "Weight"}
        )

        plt.title(f"{label} — Expert Weights by State")
        plt.xlabel("HMM State")
        plt.ylabel("Expert Channel")
        plt.tight_layout()
        plt.savefig(heatmap_path, dpi=150, bbox_inches="tight")
        plt.close()

    else:
        plt.figure(figsize=(10, max(6, len(heatmap_df) * 0.45)))

        plt.imshow(heatmap_df.values, aspect="auto")
        plt.colorbar(label="Weight")
        plt.xticks(range(len(heatmap_df.columns)), heatmap_df.columns, rotation=45)
        plt.yticks(range(len(heatmap_df.index)), heatmap_df.index)

        for i in range(heatmap_df.shape[0]):
            for j in range(heatmap_df.shape[1]):
                plt.text(
                    j,
                    i,
                    f"{heatmap_df.values[i, j]:.3f}",
                    ha="center",
                    va="center",
                    fontsize=8
                )

        plt.title(f"{label} — Expert Weights by State")
        plt.xlabel("HMM State")
        plt.ylabel("Expert Channel")
        plt.tight_layout()
        plt.savefig(heatmap_path, dpi=150, bbox_inches="tight")
        plt.close()

    # -------------------------------------------------------------------------
    # Top experts per state
    # -------------------------------------------------------------------------

    top_rows = []

    for s_col in state_cols:
        tmp = (
            weights_raw[["Pool", "Expert Channel", "Channel Code", "Model", s_col]]
            .sort_values(s_col, ascending=False)
            .head(5)
            .copy()
        )

        tmp.insert(1, "State", s_col.replace(" Weight", ""))
        tmp = tmp.rename(columns={s_col: "Weight"})

        top_rows.extend(tmp.to_dict("records"))

    top_experts_df = pd.DataFrame(top_rows)

    top_experts_path = os.path.join(
        REPORT_DIR,
        f"{safe_name}_FULL_MODEB_top5_experts_per_state.csv"
    )

    top_experts_df.to_csv(
        top_experts_path,
        index=False,
        encoding="utf-8-sig"
    )

    top_print = top_experts_df.copy()
    top_print["Weight"] = top_print["Weight"].map(lambda x: f"{x:.3f}")

    print("\nTOP 5 EXPERT CHANNELS PER STATE")
    print(top_print[["Pool", "State", "Expert Channel", "Weight"]].to_string(index=False))

    print("\nFILES SAVED")
    print(f"Raw weights CSV       → {raw_path}")
    print(f"Formatted weights CSV → {formatted_path}")
    print(f"State sum CSV         → {state_sum_path}")
    print(f"Top 5 CSV             → {top_experts_path}")
    print(f"Heatmap PNG           → {heatmap_path}")


# =============================================================================
# 6. COMBINED OUTPUT
# =============================================================================

combined_weights_df = pd.concat(combined_weights, axis=0, ignore_index=True)
combined_state_sums_df = pd.concat(combined_state_sums, axis=0, ignore_index=True)

combined_raw_path = os.path.join(
    REPORT_DIR,
    "COMBINED_FULL_MODEB_state_expert_weights_raw.csv"
)

combined_state_sum_path = os.path.join(
    REPORT_DIR,
    "COMBINED_FULL_MODEB_state_weight_sums.csv"
)

combined_weights_df.to_csv(
    combined_raw_path,
    index=False,
    encoding="utf-8-sig"
)

combined_state_sums_df.to_csv(
    combined_state_sum_path,
    index=False,
    encoding="utf-8-sig"
)

combined_fmt = combined_weights_df.copy()

weight_cols_all = [
    c for c in combined_fmt.columns
    if c.startswith("S") and c.endswith("Weight")
] + [
    "Avg Weight",
    "Max State Weight",
]

for c in weight_cols_all:
    if c in combined_fmt.columns:
        combined_fmt[c] = combined_fmt[c].map(
            lambda x: "" if pd.isna(x) else f"{x:.3f}"
        )

combined_fmt_path = os.path.join(
    REPORT_DIR,
    "COMBINED_FULL_MODEB_state_expert_weights_formatted.csv"
)

combined_fmt.to_csv(
    combined_fmt_path,
    index=False,
    encoding="utf-8-sig"
)


# =============================================================================
# 7. FINAL PRINT
# =============================================================================

print("\n" + "═" * 150)
print("COMBINED MAIN_14 + THEORY_11 — FULL_CORE × MODE B STATE WEIGHTS")
print("═" * 150)

combined_print_cols = [
    "Pool",
    "Expert Channel",
] + [
    c for c in combined_fmt.columns
    if c.startswith("S") and c.endswith("Weight")
] + [
    "Avg Weight",
    "Dominant State",
]

print(combined_fmt[combined_print_cols].to_string(index=False))

print("\nCOMBINED FILES")
print("═" * 150)
print(f"Combined raw weights CSV       → {combined_raw_path}")
print(f"Combined formatted weights CSV → {combined_fmt_path}")
print(f"Combined state sum CSV         → {combined_state_sum_path}")
print(f"Report folder                  → {REPORT_DIR}/")
print("═" * 150)

print("\nDONE ✅")

══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
MAIN_14 + THEORY_11 × FULL_CORE × MODE B — STATE EXPERT WEIGHTS
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Eval store: multi_ens_fixed_hmm_outputs/eval_output_store.pkl
Loaded eval store keys: 21

══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
MAIN_14 — FULL_CORE × MODE B
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Key          : MAIN_14__HMM_FULL_CORE_K5__B_STATE_WEIGHTS_STATE_THRESHOLD
States found : [0, 1, 2, 3, 4]

───────────────────────────────────────────────────────────────────────────────────────────────────────────

In [ ]:
# =============================================================================
# 9. FINAL THESIS TABLE — POOLS + BUY & HOLD
# =============================================================================

def fmt_pct(x):
    return f"{x:.2%}" if pd.notna(x) else ""


def fmt_num(x):
    return f"{x:.3f}" if pd.notna(x) else ""


def get_buy_hold_test_returns(reference_index):
    """
    Buy & Hold benchmark:
    Same test period as ensemble outputs.
    Uses fwd_ret directly without transaction cost.
    Signal is always 1.
    """

    reference_index = pd.to_datetime(reference_index)

    if "test_df" in globals() and "fwd_ret" in test_df.columns:
        bh_ret = (
            test_df
            .reindex(reference_index)["fwd_ret"]
            .fillna(0.0)
            .astype(float)
            .values
        )

    elif "master_df" in globals() and "fwd_ret" in master_df.columns:
        bh_ret = (
            master_df
            .reindex(reference_index)["fwd_ret"]
            .fillna(0.0)
            .astype(float)
            .values
        )

    else:
        raise ValueError(
            "Buy & Hold için test_df['fwd_ret'] veya master_df['fwd_ret'] bulunamadı."
        )

    bh_signal = np.ones(len(bh_ret))

    return bh_ret, bh_signal


# -------------------------------------------------------------------------
# Ensemble TEST rows
# -------------------------------------------------------------------------

pool_order = ["MAIN_14", "THEORY_11", "ENSEMBLE_B"]

test_summary_only = (
    combined_summary_df
    .query("split == 'TEST'")
    .copy()
)

table_rows = []

for pool in pool_order:
    row_df = test_summary_only[test_summary_only["ensemble_name"] == pool]

    if row_df.empty:
        print(f"[WARN] {pool} TEST sonucu bulunamadı.")
        continue

    r = row_df.iloc[0]

    table_rows.append({
        "Pool": pool,
        "Sharpe (rf=0)": r["sharpe"],
        "Sharpe (rf=DTB3)": r["sharpe_rf_DTB3"],
        "Ann. Return": r["ann_ret"],
        "Cum. Return": r["cum_ret"],
        "Max DD": r["max_dd"],
        "Activity": r["activity"],
        "Trades": int(r["trades"]),
    })


# -------------------------------------------------------------------------
# Buy & Hold row
# -------------------------------------------------------------------------

# herhangi bir ensemble'ın test indexini referans al
first_key = regime_free_keys[0]
first_obj = eval_store[first_key]
benchmark_test_index = pd.to_datetime(first_obj["test_index"])

bh_ret, bh_signal = get_buy_hold_test_returns(benchmark_test_index)

bh_summary = summarize(
    ret=bh_ret,
    sig=bh_signal,
    index=benchmark_test_index,
    dtb3_daily=dtb3_daily
)

table_rows.append({
    "Pool": "Buy & Hold",
    "Sharpe (rf=0)": bh_summary["sharpe"],
    "Sharpe (rf=DTB3)": bh_summary["sharpe_rf_DTB3"],
    "Ann. Return": bh_summary["ann_ret"],
    "Cum. Return": bh_summary["cum_ret"],
    "Max DD": bh_summary["max_dd"],
    "Activity": 1.0,
    "Trades": 1,
})


# -------------------------------------------------------------------------
# Raw table
# -------------------------------------------------------------------------

thesis_table_raw = pd.DataFrame(table_rows)

thesis_table_raw_path = os.path.join(
    REPORT_DIR,
    "FINAL_POOL_TABLE_raw.csv"
)

thesis_table_raw.to_csv(
    thesis_table_raw_path,
    index=False,
    encoding="utf-8-sig"
)


# -------------------------------------------------------------------------
# Formatted thesis table
# -------------------------------------------------------------------------

thesis_table = thesis_table_raw.copy()

thesis_table["Sharpe (rf=0)"] = thesis_table["Sharpe (rf=0)"].map(fmt_num)
thesis_table["Sharpe (rf=DTB3)"] = thesis_table["Sharpe (rf=DTB3)"].map(fmt_num)
thesis_table["Ann. Return"] = thesis_table["Ann. Return"].map(fmt_pct)
thesis_table["Cum. Return"] = thesis_table["Cum. Return"].map(fmt_pct)
thesis_table["Max DD"] = thesis_table["Max DD"].map(fmt_pct)
thesis_table["Activity"] = thesis_table["Activity"].map(fmt_pct)
thesis_table["Trades"] = thesis_table["Trades"].astype(str)

thesis_table_path = os.path.join(
    REPORT_DIR,
    "FINAL_POOL_TABLE_formatted.csv"
)

thesis_table.to_csv(
    thesis_table_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "═" * 150)
print("FINAL POOL TABLE — TEST PERIOD")
print("═" * 150)
print(thesis_table.to_string(index=False))
print("═" * 150)

print(f"Raw final table       → {thesis_table_raw_path}")
print(f"Formatted final table → {thesis_table_path}")


══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
FINAL POOL TABLE — TEST PERIOD
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
      Pool Sharpe (rf=0) Sharpe (rf=DTB3) Ann. Return Cum. Return  Max DD Activity Trades
   MAIN_14         2.119            1.383      10.35%      63.24%  -4.65%   35.20%    881
 THEORY_11         1.875            1.214      10.17%      61.91%  -4.88%   49.01%    939
ENSEMBLE_B         1.680            1.039       9.33%      55.89% -10.47%   46.90%    987
Buy & Hold        -0.061           -0.521      -0.74%      -3.61% -22.36%  100.00%      1
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Raw final table       → all_ensembles_regime_free_report/FINAL_PO

In [ ]:
# -------------------------------------------------------------------------
# Random Strategy Benchmark row
# -------------------------------------------------------------------------

def get_reference_test_returns(reference_index):
    """
    Random benchmark için aynı test dönemindeki fwd_ret serisini alır.
    """

    reference_index = pd.to_datetime(reference_index)

    if "test_df" in globals() and "fwd_ret" in test_df.columns:
        base_ret = (
            test_df
            .reindex(reference_index)["fwd_ret"]
            .fillna(0.0)
            .astype(float)
            .values
        )

    elif "master_df" in globals() and "fwd_ret" in master_df.columns:
        base_ret = (
            master_df
            .reindex(reference_index)["fwd_ret"]
            .fillna(0.0)
            .astype(float)
            .values
        )

    else:
        raise ValueError(
            "Random benchmark için test_df['fwd_ret'] veya master_df['fwd_ret'] bulunamadı."
        )

    return base_ret


def generate_random_signal(
    n,
    target_activity=0.50,
    p_change=0.20,
    seed=42
):
    """
    Random trading signal üretir.

    Signal değerleri:
        +1 = long
         0 = flat
        -1 = short

    target_activity:
        Pozisyonda kalma oranı. Örn 0.50 ise yaklaşık %50 aktif.

    p_change:
        Her barda pozisyon değiştirme olasılığı.
        Çok yüksek olursa aşırı trade üretir.
        Çok düşük olursa daha smooth random strateji olur.
    """

    rng = np.random.default_rng(seed)

    p_flat = 1.0 - target_activity
    p_side = target_activity / 2.0

    choices = np.array([-1.0, 0.0, 1.0])
    probs = np.array([p_side, p_flat, p_side])

    sig = np.zeros(n)

    # İlk pozisyon
    sig[0] = rng.choice(choices, p=probs)

    for i in range(1, n):
        if rng.random() < p_change:
            sig[i] = rng.choice(choices, p=probs)
        else:
            sig[i] = sig[i - 1]

    return sig


def simulate_random_strategy(
    fwd_ret,
    n_sims=1000,
    target_activity=0.50,
    p_change=0.20,
    seed=42
):
    """
    1000 random strateji simüle eder.
    Her strateji için return ve signal üretir.
    """

    fwd_ret = np.nan_to_num(
        np.asarray(fwd_ret, dtype=float),
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    n = len(fwd_ret)

    tc = globals().get("TC_PER_SIDE", 0.00005)

    sim_rows = []
    sim_store = {}

    for sim_id in range(n_sims):
        sim_seed = seed + sim_id

        sig = generate_random_signal(
            n=n,
            target_activity=target_activity,
            p_change=p_change,
            seed=sim_seed
        )

        turnover = np.abs(np.diff(sig, prepend=0.0))

        ret = sig * fwd_ret - tc * turnover

        sim_store[sim_id] = {
            "ret": ret,
            "signal": sig,
        }

        s = summarize(
            ret=ret,
            sig=sig,
            index=benchmark_test_index,
            dtb3_daily=dtb3_daily
        )

        sim_rows.append({
            "sim_id": sim_id,
            "sharpe": s["sharpe"],
            "sharpe_rf_DTB3": s["sharpe_rf_DTB3"],
            "ann_ret": s["ann_ret"],
            "cum_ret": s["cum_ret"],
            "max_dd": s["max_dd"],
            "activity": s["activity"],
            "trades": s["trades"],
        })

    sim_df = pd.DataFrame(sim_rows)

    return sim_df, sim_store


# Herhangi bir ensemble'ın test indexini referans al
first_key = regime_free_keys[0]
first_obj = eval_store[first_key]
benchmark_test_index = pd.to_datetime(first_obj["test_index"])

base_fwd_ret = get_reference_test_returns(benchmark_test_index)

# Ensemble ortalama activity'sine yakın random benchmark yapmak daha adil olur
ensemble_avg_activity = np.mean([
    row["Activity"]
    for row in table_rows
    if row["Pool"] in pool_order
])

random_sim_df, random_sim_store = simulate_random_strategy(
    fwd_ret=base_fwd_ret,
    n_sims=1000,
    target_activity=float(ensemble_avg_activity),
    p_change=0.20,
    seed=42
)

# Median Sharpe'a en yakın random simülasyonu seç
median_sharpe = random_sim_df["sharpe"].median()

median_sim_id = (
    random_sim_df
    .assign(abs_diff=lambda x: np.abs(x["sharpe"] - median_sharpe))
    .sort_values("abs_diff")
    .iloc[0]["sim_id"]
)

median_sim_id = int(median_sim_id)

random_ret = random_sim_store[median_sim_id]["ret"]
random_signal = random_sim_store[median_sim_id]["signal"]

random_summary = summarize(
    ret=random_ret,
    sig=random_signal,
    index=benchmark_test_index,
    dtb3_daily=dtb3_daily
)

table_rows.append({
    "Pool": "Random Strategy",
    "Sharpe (rf=0)": random_summary["sharpe"],
    "Sharpe (rf=DTB3)": random_summary["sharpe_rf_DTB3"],
    "Ann. Return": random_summary["ann_ret"],
    "Cum. Return": random_summary["cum_ret"],
    "Max DD": random_summary["max_dd"],
    "Activity": random_summary["activity"],
    "Trades": int(random_summary["trades"]),
})


# Random simulation distribution kaydet
random_sim_path = os.path.join(
    REPORT_DIR,
    "RANDOM_STRATEGY_1000_simulations.csv"
)

random_sim_df.to_csv(
    random_sim_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Random strategy simulations → {random_sim_path}")
print(f"Selected random median sim_id → {median_sim_id}")
print(f"Random target activity → {ensemble_avg_activity:.2%}")

Random strategy simulations → best_modeB_state_weights_report/RANDOM_STRATEGY_1000_simulations.csv
Selected random median sim_id → 732
Random target activity → 43.70%


In [ ]:
# -------------------------------------------------------------------------
# Random Walk Price Benchmark row
# -------------------------------------------------------------------------

def get_reference_test_returns(reference_index):
    """
    Random walk benchmark için aynı test dönemindeki gerçek fwd_ret serisini alır.
    Strateji getirisi yine gerçek EUR/USD fwd_ret üzerinden hesaplanır.
    Sadece sinyal random-walk price process üzerinden üretilir.
    """

    reference_index = pd.to_datetime(reference_index)

    if "test_df" in globals() and "fwd_ret" in test_df.columns:
        base_ret = (
            test_df
            .reindex(reference_index)["fwd_ret"]
            .fillna(0.0)
            .astype(float)
            .values
        )

    elif "master_df" in globals() and "fwd_ret" in master_df.columns:
        base_ret = (
            master_df
            .reindex(reference_index)["fwd_ret"]
            .fillna(0.0)
            .astype(float)
            .values
        )

    else:
        raise ValueError(
            "Random walk benchmark için test_df['fwd_ret'] veya master_df['fwd_ret'] bulunamadı."
        )

    return base_ret


def get_train_return_distribution():
    """
    Random walk şok dağılımını train döneminden tahmin eder.
    Öncelik train_df['fwd_ret']; yoksa master_df üzerinden train_df indexi.
    """

    if "train_df" in globals() and "fwd_ret" in train_df.columns:
        train_ret = (
            train_df["fwd_ret"]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
            .astype(float)
            .values
        )

    elif (
        "master_df" in globals()
        and "train_df" in globals()
        and "fwd_ret" in master_df.columns
    ):
        train_ret = (
            master_df
            .reindex(train_df.index)["fwd_ret"]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
            .astype(float)
            .values
        )

    else:
        raise ValueError(
            "Random walk şok dağılımı için train_df['fwd_ret'] veya master_df['fwd_ret'] bulunamadı."
        )

    mu = float(np.nanmean(train_ret))
    sigma = float(np.nanstd(train_ret, ddof=1))

    if sigma <= 1e-12:
        raise ValueError("Train return standard deviation çok düşük. Random walk üretilemedi.")

    return mu, sigma


def generate_random_walk_price_signal(
    n,
    mu,
    sigma,
    threshold_mult=0.25,
    seed=42,
    initial_price=1.0
):
    """
    Random walk price path üretir ve bu path'in return yönünden sinyal çıkarır.

    Random walk:
        log_price_t = log_price_{t-1} + epsilon_t
        epsilon_t ~ N(mu, sigma)

    Signal:
        rw_ret > +threshold  → long
        rw_ret < -threshold  → short
        otherwise            → flat

    threshold_mult:
        threshold = threshold_mult * sigma
        Daha yüksek değer daha az işlem üretir.
    """

    rng = np.random.default_rng(seed)

    eps = rng.normal(
        loc=mu,
        scale=sigma,
        size=n
    )

    log_price = np.zeros(n)
    log_price[0] = np.log(initial_price)

    for t in range(1, n):
        log_price[t] = log_price[t - 1] + eps[t]

    rw_price = np.exp(log_price)

    rw_ret = np.zeros(n)
    rw_ret[1:] = np.diff(log_price)

    threshold = threshold_mult * sigma

    signal = np.where(
        rw_ret > threshold,
        1.0,
        np.where(rw_ret < -threshold, -1.0, 0.0)
    )

    return rw_price, rw_ret, signal


def simulate_random_walk_strategy(
    real_fwd_ret,
    mu,
    sigma,
    n_sims=1000,
    threshold_mult=0.25,
    seed=42
):
    """
    Birden fazla random walk price path üretir.
    Her path'ten sinyal çıkarır.
    Gerçek fwd_ret üzerinde strateji getirisi hesaplar.
    Median Sharpe random walk benchmark olarak seçilir.
    """

    real_fwd_ret = np.nan_to_num(
        np.asarray(real_fwd_ret, dtype=float),
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    n = len(real_fwd_ret)
    tc = globals().get("TC_PER_SIDE", 0.00005)

    sim_rows = []
    sim_store = {}

    for sim_id in range(n_sims):
        sim_seed = seed + sim_id

        rw_price, rw_ret, signal = generate_random_walk_price_signal(
            n=n,
            mu=mu,
            sigma=sigma,
            threshold_mult=threshold_mult,
            seed=sim_seed,
            initial_price=1.0
        )

        turnover = np.abs(np.diff(signal, prepend=0.0))

        strategy_ret = signal * real_fwd_ret - tc * turnover

        s = summarize(
            ret=strategy_ret,
            sig=signal,
            index=benchmark_test_index,
            dtb3_daily=dtb3_daily
        )

        sim_rows.append({
            "sim_id": sim_id,
            "sharpe": s["sharpe"],
            "sharpe_rf_DTB3": s["sharpe_rf_DTB3"],
            "ann_ret": s["ann_ret"],
            "cum_ret": s["cum_ret"],
            "max_dd": s["max_dd"],
            "activity": s["activity"],
            "trades": s["trades"],
            "threshold_mult": threshold_mult,
        })

        sim_store[sim_id] = {
            "rw_price": rw_price,
            "rw_ret": rw_ret,
            "signal": signal,
            "strategy_ret": strategy_ret,
        }

    sim_df = pd.DataFrame(sim_rows)

    return sim_df, sim_store


# Herhangi bir ensemble'ın test indexini referans al
first_key = regime_free_keys[0]
first_obj = eval_store[first_key]
benchmark_test_index = pd.to_datetime(first_obj["test_index"])

# Gerçek test fwd_ret
real_test_fwd_ret = get_reference_test_returns(benchmark_test_index)

# Train döneminden random walk shock dağılımı
rw_mu, rw_sigma = get_train_return_distribution()

print("\nRandom Walk Shock Distribution")
print(f"mu    : {rw_mu:.8f}")
print(f"sigma : {rw_sigma:.8f}")

# Random walk simülasyonları
rw_sim_df, rw_sim_store = simulate_random_walk_strategy(
    real_fwd_ret=real_test_fwd_ret,
    mu=rw_mu,
    sigma=rw_sigma,
    n_sims=1000,
    threshold_mult=0.25,
    seed=42
)

# Median Sharpe'a en yakın random walk path'i seç
median_sharpe = rw_sim_df["sharpe"].median()

median_sim_id = (
    rw_sim_df
    .assign(abs_diff=lambda x: np.abs(x["sharpe"] - median_sharpe))
    .sort_values("abs_diff")
    .iloc[0]["sim_id"]
)

median_sim_id = int(median_sim_id)

rw_selected = rw_sim_store[median_sim_id]

rw_strategy_ret = rw_selected["strategy_ret"]
rw_signal = rw_selected["signal"]
rw_price = rw_selected["rw_price"]
rw_path_ret = rw_selected["rw_ret"]

rw_summary = summarize(
    ret=rw_strategy_ret,
    sig=rw_signal,
    index=benchmark_test_index,
    dtb3_daily=dtb3_daily
)

table_rows.append({
    "Pool": "Random Walk",
    "Sharpe (rf=0)": rw_summary["sharpe"],
    "Sharpe (rf=DTB3)": rw_summary["sharpe_rf_DTB3"],
    "Ann. Return": rw_summary["ann_ret"],
    "Cum. Return": rw_summary["cum_ret"],
    "Max DD": rw_summary["max_dd"],
    "Activity": rw_summary["activity"],
    "Trades": int(rw_summary["trades"]),
})


# -------------------------------------------------------------------------
# Save random walk benchmark details
# -------------------------------------------------------------------------

rw_sim_path = os.path.join(
    REPORT_DIR,
    "RANDOM_WALK_1000_simulations.csv"
)

rw_sim_df.to_csv(
    rw_sim_path,
    index=False,
    encoding="utf-8-sig"
)

rw_detail = pd.DataFrame(index=benchmark_test_index)
rw_detail["real_fwd_ret"] = real_test_fwd_ret
rw_detail["rw_price"] = rw_price
rw_detail["rw_log_return"] = rw_path_ret
rw_detail["rw_signal"] = rw_signal
rw_detail["rw_strategy_ret"] = rw_strategy_ret
rw_detail["rw_strategy_equity"] = np.cumprod(1.0 + rw_strategy_ret)

rw_detail_path = os.path.join(
    REPORT_DIR,
    "RANDOM_WALK_selected_path_detail.csv"
)

rw_detail.to_csv(
    rw_detail_path,
    encoding="utf-8-sig"
)

print("\nRandom Walk Benchmark")
print(f"Selected median sim_id : {median_sim_id}")
print(f"Median Sharpe          : {median_sharpe:.4f}")
print(f"Selected Sharpe        : {rw_summary['sharpe']:.4f}")
print(f"Selected RF Sharpe     : {rw_summary['sharpe_rf_DTB3']:.4f}")
print(f"Ann. Return            : {rw_summary['ann_ret']:.2%}")
print(f"Cum. Return            : {rw_summary['cum_ret']:.2%}")
print(f"Max DD                 : {rw_summary['max_dd']:.2%}")
print(f"Activity               : {rw_summary['activity']:.2%}")
print(f"Trades                 : {rw_summary['trades']}")

print(f"\nRandom walk simulations → {rw_sim_path}")
print(f"Selected path detail    → {rw_detail_path}")


Random Walk Shock Distribution
mu    : -0.00000545
sigma : 0.00242284

Random Walk Benchmark
Selected median sim_id : 857
Median Sharpe          : -1.1051
Selected Sharpe        : -1.1055
Selected RF Sharpe     : -1.6207
Ann. Return            : -7.37%
Cum. Return            : -31.68%
Max DD                 : -31.86%
Activity               : 80.02%
Trades                 : 4835

Random walk simulations → best_modeB_state_weights_report/RANDOM_WALK_1000_simulations.csv
Selected path detail    → best_modeB_state_weights_report/RANDOM_WALK_selected_path_detail.csv


In [ ]:
# -------------------------------------------------------------------------
# Raw table
# -------------------------------------------------------------------------

thesis_table_raw = pd.DataFrame(table_rows)

thesis_table_raw_path = os.path.join(
    REPORT_DIR,
    "FINAL_POOL_TABLE_raw.csv"
)

thesis_table_raw.to_csv(
    thesis_table_raw_path,
    index=False,
    encoding="utf-8-sig"
)


# -------------------------------------------------------------------------
# Formatted thesis table
# -------------------------------------------------------------------------

thesis_table = thesis_table_raw.copy()

thesis_table["Sharpe (rf=0)"] = thesis_table["Sharpe (rf=0)"].map(fmt_num)
thesis_table["Sharpe (rf=DTB3)"] = thesis_table["Sharpe (rf=DTB3)"].map(fmt_num)
thesis_table["Ann. Return"] = thesis_table["Ann. Return"].map(fmt_pct)
thesis_table["Cum. Return"] = thesis_table["Cum. Return"].map(fmt_pct)
thesis_table["Max DD"] = thesis_table["Max DD"].map(fmt_pct)
thesis_table["Activity"] = thesis_table["Activity"].map(fmt_pct)
thesis_table["Trades"] = thesis_table["Trades"].astype(str)

thesis_table_path = os.path.join(
    REPORT_DIR,
    "FINAL_POOL_TABLE_formatted.csv"
)

thesis_table.to_csv(
    thesis_table_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "═" * 150)
print("FINAL POOL TABLE — TEST PERIOD")
print("═" * 150)
print(thesis_table.to_string(index=False))
print("═" * 150)

print(f"Raw final table       → {thesis_table_raw_path}")
print(f"Formatted final table → {thesis_table_path}")


══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
FINAL POOL TABLE — TEST PERIOD
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
           Pool Sharpe (rf=0) Sharpe (rf=DTB3) Ann. Return Cum. Return  Max DD Activity Trades
        MAIN_14         2.119            1.383      10.35%      63.24%  -4.65%   35.20%    881
      THEORY_11         1.875            1.214      10.17%      61.91%  -4.88%   49.01%    939
     ENSEMBLE_B         1.680            1.039       9.33%      55.89% -10.47%   46.90%    987
     Buy & Hold        -0.061           -0.521      -0.74%      -3.61% -22.36%  100.00%      1
Random Strategy        -0.202           -0.948      -1.04%      -5.05% -10.58%   43.21%    903
════════════════════════════════════════════════════════════════════════════════════════════════

In [ ]:
# =============================================================================
# TWO CORE HMM REPORTS — CLEAN FULL VERSION
#   1) CLASSIC / LITERATURE CORE
#   2) HMM_FULL_CORE
#
# Outputs for each selected HMM:
#   - Yearly state distribution table
#   - Yearly dominant state table
#   - State feature means RAW
#   - State feature means Z-SCORE
#   - CSV exports
#
# Requires:
#   - HMM_OBJECTS
#   - master_df
# =============================================================================

import os
import numpy as np
import pandas as pd

# =============================================================================
# 0. SAFETY CHECKS + OUTPUT DIR
# =============================================================================

if "HMM_OBJECTS" not in globals():
    raise ValueError("❌ HMM_OBJECTS bulunamadı. Önce HMM pipeline'ı çalıştır.")

if "master_df" not in globals():
    raise ValueError("❌ master_df bulunamadı.")

if not isinstance(HMM_OBJECTS, dict) or len(HMM_OBJECTS) == 0:
    raise ValueError("❌ HMM_OBJECTS boş.")

if "SAVE_DIR" in globals():
    REPORT_DIR = os.path.join(SAVE_DIR, "two_core_hmm_reports_clean")
else:
    REPORT_DIR = "two_core_hmm_reports_clean"

os.makedirs(REPORT_DIR, exist_ok=True)

master_df = master_df.copy()
master_df.index = pd.to_datetime(master_df.index)
master_df = master_df.sort_index()

print("═" * 160)
print("TWO CORE HMM REPORTS — YEARLY STATE DISTRIBUTION + STATE FEATURE MEANS")
print("═" * 160)
print(f"Report directory: {REPORT_DIR}")

print("\nAvailable HMMs:")
for h in HMM_OBJECTS.keys():
    print("  -", h)


# =============================================================================
# 1. SELECT TWO CORE HMMs
# =============================================================================
# Otomatik seçim:
#   - CLASSIC / LITERATURE içeren ilk HMM
#   - HMM_FULL_CORE içeren ilk HMM
#
# Eğer otomatik yanlış seçerse aşağıdaki selected_hmms listesini elle değiştir.

available_hmms = list(HMM_OBJECTS.keys())

classic_candidates = [
    h for h in available_hmms
    if (
        "CLASSIC" in str(h).upper()
        or "LITERATURE" in str(h).upper()
        or "LIT" in str(h).upper()
    )
]

full_core_candidates = [
    h for h in available_hmms
    if "HMM_FULL_CORE" in str(h)
]

selected_hmms = []

if len(classic_candidates) > 0:
    selected_hmms.append(classic_candidates[0])

if len(full_core_candidates) > 0:
    selected_hmms.append(full_core_candidates[0])

# Duplicate önle
selected_hmms = list(dict.fromkeys(selected_hmms))

if len(selected_hmms) == 0:
    raise ValueError(
        "❌ CLASSIC/LITERATURE veya HMM_FULL_CORE içeren HMM bulunamadı. "
        "Available HMMs listesinden isimleri elle selected_hmms içine yaz."
    )

print("\nSelected HMMs:")
for h in selected_hmms:
    print("  -", h)

# -------------------------------------------------------------------------
# ELLE SEÇMEK İSTERSEN BURAYI AÇ:
# -------------------------------------------------------------------------
# selected_hmms = [
#     "HMM_CLASSIC_LITERATURE_CORE_K5",
#     "HMM_FULL_CORE_K5"
# ]


# =============================================================================
# 2. HELPER FUNCTIONS
# =============================================================================

def safe_filename(name):
    return (
        str(name)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(" ", "_")
        .replace(":", "_")
        .replace("|", "_")
        .replace("__", "_")
    )


def entropy_from_probs(probs):
    p = np.asarray(probs, dtype=float)
    p = p[np.isfinite(p)]
    p = p[p > 0]

    if len(p) <= 1:
        return 0.0

    ent = -np.sum(p * np.log(p))
    max_ent = np.log(len(probs))

    if max_ent <= 0:
        return 0.0

    return float(ent / max_ent)


def get_hmm_state_series(hmm_objects, hmm_name):
    hmm_obj = hmm_objects[hmm_name]

    if "state_series" not in hmm_obj:
        raise ValueError(f"❌ {hmm_name} içinde 'state_series' yok.")

    state_ser = hmm_obj["state_series"].copy()
    state_ser.index = pd.to_datetime(state_ser.index)
    state_ser = state_ser.sort_index()
    state_ser = state_ser.dropna().astype(int)

    return state_ser


def get_hmm_features(hmm_objects, hmm_name, master_df):
    hmm_obj = hmm_objects[hmm_name]

    features = hmm_obj.get("features", [])

    if features is None:
        features = []

    features = [
        f for f in features
        if f in master_df.columns
    ]

    if len(features) == 0:
        print(f"⚠ {hmm_name}: HMM_OBJECTS içinde usable features bulunamadı.")
        print("  State feature means kısmı atlanacak.")

    return features


def make_report_frame(master_df, state_ser):
    common_index = master_df.index.intersection(state_ser.index)

    if len(common_index) == 0:
        raise ValueError("❌ master_df index ile state_series index kesişmiyor.")

    report_df = master_df.reindex(common_index).copy()
    report_df["state"] = state_ser.reindex(common_index).ffill().bfill().astype(int)
    report_df["year"] = report_df.index.year

    return report_df


def make_yearly_state_distribution(report_df, hmm_name, expected_states=None):
    yearly_counts = (
        report_df
        .groupby(["year", "state"])
        .size()
        .rename("count")
        .reset_index()
    )

    yearly_total = (
        report_df
        .groupby("year")
        .size()
        .rename("year_total")
        .reset_index()
    )

    yearly_counts = yearly_counts.merge(
        yearly_total,
        on="year",
        how="left"
    )

    yearly_counts["pct"] = yearly_counts["count"] / yearly_counts["year_total"]
    yearly_counts["hmm_name"] = hmm_name

    yearly_counts = yearly_counts[
        ["hmm_name", "year", "state", "count", "year_total", "pct"]
    ]

    yearly_wide = (
        yearly_counts
        .pivot_table(
            index="year",
            columns="state",
            values="pct",
            fill_value=0.0
        )
        .sort_index()
    )

    if expected_states is None:
        expected_states = sorted(yearly_counts["state"].dropna().astype(int).unique())

    for s in expected_states:
        if s not in yearly_wide.columns:
            yearly_wide[s] = 0.0

    yearly_wide = yearly_wide[expected_states]
    yearly_wide.columns = [f"S{int(c)}" for c in yearly_wide.columns]

    return yearly_counts, yearly_wide


def make_yearly_dominant_state(yearly_wide, hmm_name):
    rows = []

    for year, row in yearly_wide.iterrows():
        dominant_state_label = row.idxmax()
        dominant_state = int(str(dominant_state_label).replace("S", ""))
        dominant_pct = float(row.max())

        rows.append({
            "hmm_name": hmm_name,
            "year": int(year),
            "dominant_state": dominant_state,
            "dominant_state_label": dominant_state_label,
            "dominant_pct": dominant_pct,
            "state_entropy_norm": entropy_from_probs(row.values.astype(float)),
            "is_one_state_dominant_gt_70": bool(dominant_pct >= 0.70),
            "is_one_state_dominant_gt_85": bool(dominant_pct >= 0.85),
        })

    return pd.DataFrame(rows)


def make_state_feature_means(report_df, features, hmm_name):
    if len(features) == 0:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    feat_df = (
        report_df[features + ["state"]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if feat_df.empty:
        print(f"⚠ {hmm_name}: feature mean hesaplanamadı, feat_df boş.")
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    state_means_raw = feat_df.groupby("state")[features].mean()

    global_mean = feat_df[features].mean()
    global_std = feat_df[features].std().replace(0, np.nan)

    state_means_z = (state_means_raw - global_mean) / global_std
    state_means_z = state_means_z.replace([np.inf, -np.inf], np.nan)

    raw_rows = []
    z_rows = []

    for state_id in state_means_raw.index:
        for feat in state_means_raw.columns:
            raw_rows.append({
                "hmm_name": hmm_name,
                "state": int(state_id),
                "feature": feat,
                "mean_raw": float(state_means_raw.loc[state_id, feat])
            })

            z_rows.append({
                "hmm_name": hmm_name,
                "state": int(state_id),
                "feature": feat,
                "mean_z": float(state_means_z.loc[state_id, feat])
            })

    state_means_raw_long = pd.DataFrame(raw_rows)
    state_means_z_long = pd.DataFrame(z_rows)

    return state_means_raw, state_means_z, state_means_raw_long, state_means_z_long


def print_pct_table(yearly_wide):
    display = yearly_wide.copy()

    for col in display.columns:
        display[col] = display[col].map(lambda x: f"{x:.1%}")

    print(display.to_string())

    return display


def print_dominant_table(yearly_dominant_state_df):
    display = yearly_dominant_state_df.copy()

    display["dominant_pct"] = display["dominant_pct"].map(lambda x: f"{x:.1%}")
    display["state_entropy_norm"] = display["state_entropy_norm"].map(lambda x: f"{x:.3f}")

    cols = [
        "year",
        "dominant_state_label",
        "dominant_pct",
        "state_entropy_norm",
        "is_one_state_dominant_gt_70",
        "is_one_state_dominant_gt_85",
    ]

    print(display[cols].to_string(index=False))

    return display


# =============================================================================
# 3. MAIN LOOP — BOTH CORE HMMs
# =============================================================================

all_yearly_dist_long_list = []
all_dominant_list = []
all_state_means_raw_long_list = []
all_state_means_z_long_list = []       # ✅ BUG FIX: this must be list, not dict

all_yearly_dist_wide = {}
all_yearly_display = {}
all_state_means_raw_wide = {}
all_state_means_z_wide = {}

for hmm_name in selected_hmms:

    print("\n" + "█" * 160)
    print(f"HMM REPORT: {hmm_name}")
    print("█" * 160)

    # -------------------------------------------------------------------------
    # 3.1 STATE SERIES + FEATURES
    # -------------------------------------------------------------------------

    state_ser = get_hmm_state_series(
        hmm_objects=HMM_OBJECTS,
        hmm_name=hmm_name
    )

    features = get_hmm_features(
        hmm_objects=HMM_OBJECTS,
        hmm_name=hmm_name,
        master_df=master_df
    )

    print("\nHMM FEATURES")
    print("─" * 120)
    if len(features) > 0:
        for f in features:
            print(f"  - {f}")
    else:
        print("  No usable features found.")

    report_df = make_report_frame(
        master_df=master_df,
        state_ser=state_ser
    )

    # -------------------------------------------------------------------------
    # 3.2 YEARLY STATE DISTRIBUTION
    # -------------------------------------------------------------------------

    yearly_dist_long, yearly_wide = make_yearly_state_distribution(
        report_df=report_df,
        hmm_name=hmm_name,
        expected_states=[0, 1, 2, 3, 4]
    )

    print("\nYEARLY STATE DISTRIBUTION")
    print("─" * 120)

    yearly_display = print_pct_table(yearly_wide)

    all_yearly_dist_long_list.append(yearly_dist_long)
    all_yearly_dist_wide[hmm_name] = yearly_wide
    all_yearly_display[hmm_name] = yearly_display

    # -------------------------------------------------------------------------
    # 3.3 YEARLY DOMINANT STATE
    # -------------------------------------------------------------------------

    yearly_dom = make_yearly_dominant_state(
        yearly_wide=yearly_wide,
        hmm_name=hmm_name
    )

    print("\nYEARLY DOMINANT STATE")
    print("─" * 120)

    dominant_display = print_dominant_table(yearly_dom)

    all_dominant_list.append(yearly_dom)

    # -------------------------------------------------------------------------
    # 3.4 STATE FEATURE MEANS
    # -------------------------------------------------------------------------

    state_means_raw, state_means_z, raw_long, z_long = make_state_feature_means(
        report_df=report_df,
        features=features,
        hmm_name=hmm_name
    )

    if not state_means_raw.empty:
        print("\nSTATE FEATURE MEANS — RAW")
        print("─" * 120)
        print(state_means_raw.round(6).to_string())

        print("\nSTATE FEATURE MEANS — Z-SCORE")
        print("─" * 120)
        print(state_means_z.round(3).to_string())

        all_state_means_raw_wide[hmm_name] = state_means_raw
        all_state_means_z_wide[hmm_name] = state_means_z

        all_state_means_raw_long_list.append(raw_long)
        all_state_means_z_long_list.append(z_long)

    else:
        print("\nSTATE FEATURE MEANS")
        print("─" * 120)
        print("Skipped because no valid feature mean table was produced.")

    # -------------------------------------------------------------------------
    # 3.5 SAVE INDIVIDUAL HMM OUTPUTS
    # -------------------------------------------------------------------------

    safe_name = safe_filename(hmm_name)

    hmm_dir = os.path.join(REPORT_DIR, safe_name)
    os.makedirs(hmm_dir, exist_ok=True)

    yearly_dist_long.to_csv(
        os.path.join(hmm_dir, f"{safe_name}_yearly_state_distribution_long.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    yearly_wide.to_csv(
        os.path.join(hmm_dir, f"{safe_name}_yearly_state_distribution_wide.csv"),
        encoding="utf-8-sig"
    )

    yearly_display.to_csv(
        os.path.join(hmm_dir, f"{safe_name}_yearly_state_distribution_display.csv"),
        encoding="utf-8-sig"
    )

    yearly_dom.to_csv(
        os.path.join(hmm_dir, f"{safe_name}_yearly_dominant_state.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    if not state_means_raw.empty:
        state_means_raw.to_csv(
            os.path.join(hmm_dir, f"{safe_name}_state_feature_means_raw_wide.csv"),
            encoding="utf-8-sig"
        )

        state_means_z.to_csv(
            os.path.join(hmm_dir, f"{safe_name}_state_feature_means_z_wide.csv"),
            encoding="utf-8-sig"
        )

        raw_long.to_csv(
            os.path.join(hmm_dir, f"{safe_name}_state_feature_means_raw_long.csv"),
            index=False,
            encoding="utf-8-sig"
        )

        z_long.to_csv(
            os.path.join(hmm_dir, f"{safe_name}_state_feature_means_z_long.csv"),
            index=False,
            encoding="utf-8-sig"
        )

    print("\nSaved individual HMM reports to:")
    print(f"  {hmm_dir}")


# =============================================================================
# 4. COMBINED OUTPUT OBJECTS
# =============================================================================

if len(all_yearly_dist_long_list) > 0:
    yearly_state_distribution_df = pd.concat(
        all_yearly_dist_long_list,
        ignore_index=True
    )
else:
    yearly_state_distribution_df = pd.DataFrame()

if len(all_dominant_list) > 0:
    yearly_dominant_state_df = pd.concat(
        all_dominant_list,
        ignore_index=True
    )
else:
    yearly_dominant_state_df = pd.DataFrame()

if len(all_state_means_raw_long_list) > 0:
    state_means_raw_long_df = pd.concat(
        all_state_means_raw_long_list,
        ignore_index=True
    )
else:
    state_means_raw_long_df = pd.DataFrame()

if len(all_state_means_z_long_list) > 0:
    state_means_z_long_df = pd.concat(
        all_state_means_z_long_list,
        ignore_index=True
    )
else:
    state_means_z_long_df = pd.DataFrame()


# =============================================================================
# 5. COMBINED SAVE
# =============================================================================

combined_yearly_path = os.path.join(
    REPORT_DIR,
    "combined_yearly_state_distribution_long.csv"
)

combined_dominant_path = os.path.join(
    REPORT_DIR,
    "combined_yearly_dominant_state.csv"
)

combined_raw_means_path = os.path.join(
    REPORT_DIR,
    "combined_state_feature_means_raw_long.csv"
)

combined_z_means_path = os.path.join(
    REPORT_DIR,
    "combined_state_feature_means_z_long.csv"
)

yearly_state_distribution_df.to_csv(
    combined_yearly_path,
    index=False,
    encoding="utf-8-sig"
)

yearly_dominant_state_df.to_csv(
    combined_dominant_path,
    index=False,
    encoding="utf-8-sig"
)

state_means_raw_long_df.to_csv(
    combined_raw_means_path,
    index=False,
    encoding="utf-8-sig"
)

state_means_z_long_df.to_csv(
    combined_z_means_path,
    index=False,
    encoding="utf-8-sig"
)


# =============================================================================
# 6. FINAL SUMMARY
# =============================================================================

print("\n" + "═" * 160)
print("DONE — TWO CORE HMM REPORTS CREATED SUCCESSFULLY")
print("═" * 160)

print("\nHMMs reported:")
for h in selected_hmms:
    print(f"  - {h}")

print("\nGlobal objects created:")
print("  - yearly_state_distribution_df")
print("  - yearly_dominant_state_df")
print("  - state_means_raw_long_df")
print("  - state_means_z_long_df")
print("  - all_yearly_dist_wide")
print("  - all_yearly_display")
print("  - all_state_means_raw_wide")
print("  - all_state_means_z_wide")

print("\nCombined files:")
print(f"  - {combined_yearly_path}")
print(f"  - {combined_dominant_path}")
print(f"  - {combined_raw_means_path}")
print(f"  - {combined_z_means_path}")

print("═" * 160)

════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
TWO CORE HMM REPORTS — YEARLY STATE DISTRIBUTION + STATE FEATURE MEANS
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Report directory: multi_ens_fixed_hmm_outputs/two_core_hmm_reports_clean

Available HMMs:
  - HMM_CLASSIC_LITERATURE_CORE
  - HMM_FULL_CORE_K5

Selected HMMs:
  - HMM_CLASSIC_LITERATURE_CORE
  - HMM_FULL_CORE_K5

████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
HMM REPORT: HMM_CLASSIC_LITERATURE_CORE
████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████

HMM FEATURES
──────────────────

In [ ]:
# =============================================================================
# HMM_CLASSIC_LITERATURE_CORE — YEARLY STATE DISTRIBUTION PNG
# =============================================================================
# Output:
#   - HMM_CLASSIC_LITERATURE_CORE_yearly_state_distribution.png
#   - HMM_CLASSIC_LITERATURE_CORE_yearly_state_distribution.pdf
#   - HMM_CLASSIC_LITERATURE_CORE_yearly_state_distribution_wide.csv
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------------------------------------------------------
# 0. Output directory
# -----------------------------------------------------------------------------

if "SAVE_DIR" in globals():
    FIG_DIR = os.path.join(SAVE_DIR, "hmm_classic_literature_core_figures")
elif "REPORT_DIR" in globals():
    FIG_DIR = os.path.join(REPORT_DIR, "hmm_classic_literature_core_figures")
else:
    FIG_DIR = "hmm_classic_literature_core_figures"

os.makedirs(FIG_DIR, exist_ok=True)

# -----------------------------------------------------------------------------
# 1. Helpers
# -----------------------------------------------------------------------------

def safe_filename(name):
    return (
        str(name)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(" ", "_")
        .replace(":", "_")
        .replace("|", "_")
    )


def find_classic_literature_hmm_name():
    """
    HMM_OBJECTS içinden HMM_CLASSIC_LITERATURE_CORE benzeri ismi bulur.
    """

    if "HMM_OBJECTS" not in globals():
        raise ValueError("❌ HMM_OBJECTS bulunamadı.")

    available_hmms = list(HMM_OBJECTS.keys())

    exact_candidates = [
        h for h in available_hmms
        if str(h) == "HMM_CLASSIC_LITERATURE_CORE"
    ]

    if len(exact_candidates) > 0:
        return exact_candidates[0]

    loose_candidates = [
        h for h in available_hmms
        if (
            "CLASSIC" in str(h).upper()
            and "LITERATURE" in str(h).upper()
            and "CORE" in str(h).upper()
        )
    ]

    if len(loose_candidates) > 0:
        return loose_candidates[0]

    classic_candidates = [
        h for h in available_hmms
        if (
            "CLASSIC" in str(h).upper()
            or "LITERATURE" in str(h).upper()
        )
    ]

    if len(classic_candidates) > 0:
        return classic_candidates[0]

    raise ValueError(
        "❌ HMM_CLASSIC_LITERATURE_CORE benzeri HMM bulunamadı. "
        "Available HMMs listesinden ismi elle gir."
    )


def make_yearly_distribution_from_hmm_objects(hmm_name, expected_states=[0, 1, 2, 3, 4]):
    """
    yearly_state_distribution_df yoksa HMM_OBJECTS + master_df üzerinden yeniden üretir.
    """

    if "HMM_OBJECTS" not in globals():
        raise ValueError("❌ HMM_OBJECTS bulunamadı.")

    if "master_df" not in globals():
        raise ValueError("❌ master_df bulunamadı.")

    hmm_obj = HMM_OBJECTS[hmm_name]

    if "state_series" not in hmm_obj:
        raise ValueError(f"❌ {hmm_name} içinde state_series yok.")

    state_ser = hmm_obj["state_series"].copy()
    state_ser.index = pd.to_datetime(state_ser.index)
    state_ser = state_ser.sort_index().dropna().astype(int)

    df = master_df.copy()
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    common_index = df.index.intersection(state_ser.index)

    if len(common_index) == 0:
        raise ValueError("❌ master_df ile state_series index kesişmiyor.")

    report_df = pd.DataFrame(index=common_index)
    report_df["state"] = state_ser.reindex(common_index).ffill().bfill().astype(int)
    report_df["year"] = report_df.index.year

    yearly_counts = (
        report_df
        .groupby(["year", "state"])
        .size()
        .rename("count")
        .reset_index()
    )

    yearly_total = (
        report_df
        .groupby("year")
        .size()
        .rename("year_total")
        .reset_index()
    )

    yearly_counts = yearly_counts.merge(yearly_total, on="year", how="left")
    yearly_counts["pct"] = yearly_counts["count"] / yearly_counts["year_total"]
    yearly_counts["hmm_name"] = hmm_name

    yearly_counts = yearly_counts[
        ["hmm_name", "year", "state", "count", "year_total", "pct"]
    ]

    wide = (
        yearly_counts
        .pivot_table(
            index="year",
            columns="state",
            values="pct",
            fill_value=0.0
        )
        .sort_index()
    )

    for s in expected_states:
        if s not in wide.columns:
            wide[s] = 0.0

    wide = wide[expected_states]
    wide.columns = [f"S{int(c)}" for c in wide.columns]

    return yearly_counts, wide


def make_yearly_distribution_from_existing_df(hmm_name, expected_states=[0, 1, 2, 3, 4]):
    """
    yearly_state_distribution_df varsa oradan wide table üretir.
    """

    if "yearly_state_distribution_df" not in globals():
        raise ValueError("yearly_state_distribution_df yok.")

    tmp = yearly_state_distribution_df[
        yearly_state_distribution_df["hmm_name"] == hmm_name
    ].copy()

    if tmp.empty:
        raise ValueError(f"yearly_state_distribution_df içinde {hmm_name} bulunamadı.")

    wide = (
        tmp.pivot_table(
            index="year",
            columns="state",
            values="pct",
            fill_value=0.0
        )
        .sort_index()
    )

    for s in expected_states:
        if s not in wide.columns:
            wide[s] = 0.0

    wide = wide[expected_states]
    wide.columns = [f"S{int(c)}" for c in wide.columns]

    return tmp, wide


# -----------------------------------------------------------------------------
# 2. Plot function
# -----------------------------------------------------------------------------

def plot_classic_literature_core_png(
    hmm_name=None,
    expected_states=[0, 1, 2, 3, 4],
    figure_no="Figure X",
    title_name="HMM_CLASSIC_LITERATURE_CORE",
    save_dir=FIG_DIR,
):
    """
    Creates thesis-ready stacked bar PNG/PDF for HMM_CLASSIC_LITERATURE_CORE.
    """

    if hmm_name is None:
        hmm_name = find_classic_literature_hmm_name()

    print("\nSelected HMM:")
    print(hmm_name)

    # Önce mevcut yearly_state_distribution_df kullanmayı dene.
    # Yoksa HMM_OBJECTS üzerinden yeniden üret.
    try:
        yearly_long, yearly_wide = make_yearly_distribution_from_existing_df(
            hmm_name=hmm_name,
            expected_states=expected_states
        )
        print("Using existing yearly_state_distribution_df.")
    except Exception as e:
        print(f"Existing yearly_state_distribution_df kullanılamadı: {e}")
        print("Rebuilding yearly distribution from HMM_OBJECTS...")
        yearly_long, yearly_wide = make_yearly_distribution_from_hmm_objects(
            hmm_name=hmm_name,
            expected_states=expected_states
        )

    # Print normal table
    display = yearly_wide.copy()
    for col in display.columns:
        display[col] = display[col].map(lambda x: f"{x:.1%}")

    print("\nYEARLY STATE DISTRIBUTION")
    print("─" * 120)
    print(display.to_string())

    # Plot
    wide_pct = yearly_wide * 100.0

    years = wide_pct.index.astype(int).tolist()
    x = np.arange(len(years))

    fig, ax = plt.subplots(figsize=(12.5, 6.2))

    bottom = np.zeros(len(wide_pct))

    for col in wide_pct.columns:
        ax.bar(
            x,
            wide_pct[col].values,
            bottom=bottom,
            width=0.72,
            label=col,
            edgecolor="white",
            linewidth=0.4
        )
        bottom += wide_pct[col].values

    ax.set_title(
        f"{figure_no}. Yearly State Distribution of {title_name}",
        fontsize=14,
        fontweight="bold",
        pad=14
    )

    ax.set_xlabel("Year", fontsize=11)
    ax.set_ylabel("Share of Observations (%)", fontsize=11)

    ax.set_ylim(0, 100)
    ax.set_yticks(np.arange(0, 101, 20))

    ax.set_xticks(x)
    ax.set_xticklabels(years, rotation=45, ha="right", fontsize=8)

    ax.grid(axis="y", linestyle="--", alpha=0.35)
    ax.set_axisbelow(True)

    ax.legend(
        title="State",
        loc="upper center",
        bbox_to_anchor=(0.5, -0.16),
        ncol=len(wide_pct.columns),
        frameon=True,
        fontsize=9,
        title_fontsize=9
    )

    fig.text(
        0.5,
        0.035,
        "Note: Each bar shows the within-year percentage of observations assigned to states S0–S4.",
        ha="center",
        va="center",
        fontsize=8
    )

    plt.subplots_adjust(
        left=0.08,
        right=0.98,
        top=0.88,
        bottom=0.25
    )

    safe_name = safe_filename(hmm_name)

    png_path = os.path.join(
        save_dir,
        f"{safe_name}_yearly_state_distribution.png"
    )

    pdf_path = os.path.join(
        save_dir,
        f"{safe_name}_yearly_state_distribution.pdf"
    )

    csv_path = os.path.join(
        save_dir,
        f"{safe_name}_yearly_state_distribution_wide.csv"
    )

    yearly_wide.to_csv(csv_path, encoding="utf-8-sig")

    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.show()

    print("\n✅ HMM_CLASSIC_LITERATURE_CORE PNG created")
    print(f"PNG: {png_path}")
    print(f"PDF: {pdf_path}")
    print(f"CSV: {csv_path}")

    return yearly_wide, png_path, pdf_path, csv_path


# -----------------------------------------------------------------------------
# 3. Run
# -----------------------------------------------------------------------------

classic_yearly_wide, classic_png_path, classic_pdf_path, classic_csv_path = plot_classic_literature_core_png(
    hmm_name=None,  # Otomatik bulur
    expected_states=[0, 1, 2, 3, 4],
    figure_no="Figure X",
    title_name="HMM_CLASSIC_LITERATURE_CORE",
    save_dir=FIG_DIR
)


Selected HMM:
HMM_CLASSIC_LITERATURE_CORE
Using existing yearly_state_distribution_df.

YEARLY STATE DISTRIBUTION
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
         S0     S1     S2      S3      S4
year                                     
2004   1.7%  81.3%   0.0%   17.0%    0.0%
2005   0.0%   0.0%   0.0%    9.4%   90.6%
2006   0.0%   0.0%   0.0%    0.0%  100.0%
2007   0.0%   0.0%   0.0%    7.8%   92.2%
2008   0.0%   0.0%   4.6%   95.4%    0.0%
2009   0.0%  55.7%  12.5%   31.8%    0.0%
2010   0.0%  18.5%  81.5%    0.0%    0.0%
2011   0.0%   0.0%  74.1%   25.9%    0.0%
2012  35.3%  58.1%   0.0%    6.6%    0.0%
2013  86.4%   7.0%   3.9%    2.7%    0.0%
2014  72.9%  27.1%   0.0%    0.1%    0.0%
2015   0.0%   3.7%  15.4%   80.9%    0.0%
2016   0.0%   0.0%   0.0%  100.0%    0.0%
2017   0.0%   0.0%   0.0%  100.0%    0.0%
2018   0.0%   0.0%   0.0%    1.5%   98.5%
2019   0.0%   0.0%   0.0%    0.0%  100.0%
2020   0

In [ ]:
# =============================================================================
# FULL STATE INTERPRETATION + REGIME-FREE COMPARISON REPORT
#
# Amaç:
#   1) HMM state dağılımlarını yıl yıl göstermek
#   2) State feature ortalamalarını raw ve z-score olarak göstermek
#   3) State interpretation / label guess üretmek
#   4) HMM mode performansını state içinde ölçmek
#   5) Regime-free çıktıyı aynı HMM state'lerine projekte etmek
#   6) HMM - Regime-Free delta performansını state bazında göstermek
#
# Requires:
#   - master_df
#   - train_df, val_df, test_df
#   - HMM_OBJECTS veya *_states.csv
#   - EVAL_OUTPUT_STORE veya eval_output_store.pkl
#   - SAVE_DIR = "multi_ens_fixed_hmm_outputs"
# =============================================================================

import os
import pickle
import numpy as np
import pandas as pd

# =============================================================================
# 0. PATHS AND CHECKS
# =============================================================================

SAVE_DIR = globals().get("SAVE_DIR", "multi_ens_fixed_hmm_outputs")

REPORT_DIR = os.path.join(
    SAVE_DIR,
    "FULL_STATE_INTERPRETATION_WITH_REGIMEFREE_REPORT"
)
os.makedirs(REPORT_DIR, exist_ok=True)

EVAL_STORE_PATH = os.path.join(SAVE_DIR, "eval_output_store.pkl")

required_base = ["master_df", "train_df", "val_df", "test_df"]

for obj in required_base:
    if obj not in globals():
        raise ValueError(f"❌ {obj} bulunamadı.")

if "EVAL_OUTPUT_STORE" not in globals():
    if not os.path.exists(EVAL_STORE_PATH):
        raise ValueError(
            f"❌ EVAL_OUTPUT_STORE bulunamadı ve dosya yok:\n{EVAL_STORE_PATH}"
        )

    with open(EVAL_STORE_PATH, "rb") as f:
        EVAL_OUTPUT_STORE = pickle.load(f)

    print(f"✅ EVAL_OUTPUT_STORE yüklendi: {len(EVAL_OUTPUT_STORE)} output")

master_df = master_df.copy()
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

for df in [master_df, train_df, val_df, test_df]:
    df.index = pd.to_datetime(df.index)
    if df.index.tz is not None:
        df.index = df.index.tz_localize(None)
    df.sort_index(inplace=True)

test_index = pd.DatetimeIndex(test_df.index)
if test_index.tz is not None:
    test_index = test_index.tz_localize(None)

BAR_ANN_LOCAL = globals().get("BAR_ANN", 6 * 252)

print("═" * 180)
print("FULL STATE INTERPRETATION + REGIME-FREE COMPARISON REPORT")
print("═" * 180)
print(f"Report dir: {REPORT_DIR}")
print(f"Test period: {test_index.min()} → {test_index.max()}")
print(f"EVAL_OUTPUT_STORE outputs: {len(EVAL_OUTPUT_STORE)}")
print("═" * 180)


# =============================================================================
# 1. LOAD HMM STATE SERIES + FEATURE LISTS
# =============================================================================

STATE_SERIES_MAP = {}
HMM_FEATURE_MAP = {}
HMM_OBJECT_META = {}

# Preferred: use HMM_OBJECTS if available
if "HMM_OBJECTS" in globals() and isinstance(HMM_OBJECTS, dict) and len(HMM_OBJECTS) > 0:

    for hmm_name, hmm_obj in HMM_OBJECTS.items():

        state_ser = hmm_obj["state_series"].copy()
        state_ser.index = pd.to_datetime(state_ser.index)

        if state_ser.index.tz is not None:
            state_ser.index = state_ser.index.tz_localize(None)

        state_ser = state_ser.sort_index().dropna().astype(int)

        STATE_SERIES_MAP[hmm_name] = state_ser

        features = [
            f for f in hmm_obj.get("features", [])
            if f in master_df.columns
        ]

        HMM_FEATURE_MAP[hmm_name] = features

        HMM_OBJECT_META[hmm_name] = {
            "k": int(hmm_obj.get("k", state_ser.nunique())),
            "avg_persistence": float(hmm_obj.get("avg_persistence", np.nan)),
            "min_persistence": float(hmm_obj.get("min_persistence", np.nan)),
            "min_state_freq": float(hmm_obj.get("min_state_freq", np.nan)),
            "penalty_flag": bool(hmm_obj.get("penalty_flag", False)),
            "features": features,
        }

# Fallback: load *_states.csv
else:
    state_files = [
        f for f in os.listdir(SAVE_DIR)
        if f.endswith("_states.csv")
    ]

    if len(state_files) == 0:
        raise ValueError(
            f"❌ HMM_OBJECTS yok ve {SAVE_DIR} içinde *_states.csv bulunamadı."
        )

    for fname in state_files:
        path = os.path.join(SAVE_DIR, fname)
        hmm_name = fname.replace("_states.csv", "")

        tmp = pd.read_csv(path, index_col=0)
        tmp.index = pd.to_datetime(tmp.index)

        if tmp.index.tz is not None:
            tmp.index = tmp.index.tz_localize(None)

        if "state" not in tmp.columns:
            raise ValueError(f"❌ {fname} içinde state kolonu yok.")

        STATE_SERIES_MAP[hmm_name] = tmp["state"].sort_index().dropna().astype(int)

        # Feature list fallback: use known definitions if available
        if "HMM_FEATURE_SETS" in globals():
            if hmm_name == "HMM_FULL_CORE_K5" and "HMM_FULL_CORE" in HMM_FEATURE_SETS:
                HMM_FEATURE_MAP[hmm_name] = [
                    f for f in HMM_FEATURE_SETS["HMM_FULL_CORE"]
                    if f in master_df.columns
                ]
            elif hmm_name == "HMM_CLASSIC_LITERATURE_CORE" and "HMM_CLASSIC_LITERATURE_CORE" in HMM_FEATURE_SETS:
                HMM_FEATURE_MAP[hmm_name] = [
                    f for f in HMM_FEATURE_SETS["HMM_CLASSIC_LITERATURE_CORE"]
                    if f in master_df.columns
                ]
            else:
                HMM_FEATURE_MAP[hmm_name] = []
        else:
            HMM_FEATURE_MAP[hmm_name] = []

        HMM_OBJECT_META[hmm_name] = {
            "k": int(STATE_SERIES_MAP[hmm_name].nunique()),
            "avg_persistence": np.nan,
            "min_persistence": np.nan,
            "min_state_freq": np.nan,
            "penalty_flag": np.nan,
            "features": HMM_FEATURE_MAP[hmm_name],
        }

print("\n✅ Loaded HMM states:")
for hmm_name, s in STATE_SERIES_MAP.items():
    print(
        f"  {hmm_name:<35} len={len(s):>7} | "
        f"states={sorted(s.dropna().unique().astype(int).tolist())} | "
        f"features={len(HMM_FEATURE_MAP.get(hmm_name, []))}"
    )


# =============================================================================
# 2. HELPER FUNCTIONS
# =============================================================================

def clean_arr(x):
    return np.nan_to_num(
        np.asarray(x, dtype=float),
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

def calc_cum_ret(ret):
    r = clean_arr(ret)
    if len(r) == 0:
        return 0.0
    return float(np.prod(1.0 + r) - 1.0)

def calc_ann_ret(ret, bar_ann=BAR_ANN_LOCAL):
    r = clean_arr(ret)
    if len(r) == 0:
        return 0.0

    cum = calc_cum_ret(r)
    years = len(r) / bar_ann

    if years <= 0:
        return 0.0

    if cum <= -0.999:
        return -1.0

    return float((1.0 + cum) ** (1.0 / years) - 1.0)

def calc_sharpe(ret, bar_ann=BAR_ANN_LOCAL):
    r = clean_arr(ret)

    if len(r) <= 1:
        return 0.0

    sd = np.nanstd(r, ddof=1)

    if sd <= 1e-12:
        return 0.0

    return float(np.nanmean(r) / sd * np.sqrt(bar_ann))

def calc_max_dd(ret):
    r = clean_arr(ret)

    if len(r) == 0:
        return 0.0

    eq = np.cumprod(1.0 + r)
    peak = np.maximum.accumulate(eq)
    dd = eq / (peak + 1e-12) - 1.0

    return float(np.nanmin(dd))

def calc_sortino(ret, bar_ann=BAR_ANN_LOCAL):
    r = clean_arr(ret)

    if len(r) <= 1:
        return 0.0

    downside = r[r < 0]

    if len(downside) <= 1:
        return 0.0

    sd = np.nanstd(downside, ddof=1)

    if sd <= 1e-12:
        return 0.0

    return float(np.nanmean(r) / sd * np.sqrt(bar_ann))

def calc_calmar(ret):
    ann = calc_ann_ret(ret)
    dd = abs(calc_max_dd(ret))

    if dd <= 1e-12:
        return 0.0

    return float(ann / dd)

def trade_stats(ret, sig):
    r = clean_arr(ret)
    s = clean_arr(sig)

    active_ret = r[s != 0]
    turnover = np.abs(np.diff(s, prepend=0.0))

    return {
        "bars": int(len(r)),
        "active_bars": int(np.sum(s != 0)),
        "activity": float(np.mean(s != 0)) if len(s) else 0.0,
        "trades": int(np.sum(turnover > 0)) if len(s) else 0,
        "long_ratio": float(np.mean(s == 1)) if len(s) else 0.0,
        "short_ratio": float(np.mean(s == -1)) if len(s) else 0.0,
        "flat_ratio": float(np.mean(s == 0)) if len(s) else 0.0,
        "win_rate_active": float(np.mean(active_ret > 0)) if len(active_ret) else 0.0,
    }

def metric_pack(ret, sig):
    stats = trade_stats(ret, sig)

    return {
        "cum_ret": calc_cum_ret(ret),
        "ann_ret": calc_ann_ret(ret),
        "sharpe": calc_sharpe(ret),
        "sortino": calc_sortino(ret),
        "calmar": calc_calmar(ret),
        "max_dd": calc_max_dd(ret),
        **stats,
    }

def entropy_from_probs(probs):
    p = np.asarray(probs, dtype=float)
    p = p[np.isfinite(p)]
    p = p[p > 0]

    if len(p) <= 1:
        return 0.0

    ent = -np.sum(p * np.log(p))
    max_ent = np.log(len(p))

    if max_ent <= 0:
        return 0.0

    return float(ent / max_ent)

def concentration_score(probs):
    p = np.asarray(probs, dtype=float)
    p = p[np.isfinite(p)]

    if len(p) == 0:
        return np.nan

    return float(np.max(p))

def feature_separation_score(state_means_z):
    if state_means_z is None or state_means_z.empty:
        return np.nan

    vals = state_means_z.replace([np.inf, -np.inf], np.nan).values

    return float(np.nanmean(np.abs(vals)))

def top_state_features(state_means_z, state_id, top_n=6):
    if state_id not in state_means_z.index:
        return ""

    s = state_means_z.loc[state_id].dropna()
    s = s.reindex(s.abs().sort_values(ascending=False).index)

    return " | ".join([
        f"{feat}={val:+.2f}"
        for feat, val in s.head(top_n).items()
    ])

def state_label_guess(row):
    values = row.copy()

    def mean_of_keywords(keywords):
        cols = [
            c for c in values.index
            if any(k in str(c).lower() for k in keywords)
        ]

        if not cols:
            return np.nan

        return float(values[cols].mean())

    risk_val = mean_of_keywords(["risk", "vix", "vol", "credit", "hy"])
    liq_val = mean_of_keywords(["liq", "money", "net_liq", "m2"])
    growth_val = mean_of_keywords(["growth", "cfnai", "spx"])
    rates_val = mean_of_keywords(["rate", "yield", "spread", "curve", "policy"])
    gold_val = mean_of_keywords(["gold"])

    labels = []

    if np.isfinite(risk_val):
        if risk_val > 0.50:
            labels.append("high_risk")
        elif risk_val < -0.50:
            labels.append("low_risk")

    if np.isfinite(liq_val):
        if liq_val > 0.50:
            labels.append("liquidity_supportive")
        elif liq_val < -0.50:
            labels.append("liquidity_tight")

    if np.isfinite(growth_val):
        if growth_val > 0.50:
            labels.append("growth_positive")
        elif growth_val < -0.50:
            labels.append("growth_weak")

    if np.isfinite(rates_val):
        if rates_val > 0.50:
            labels.append("rates_high_or_curve_positive")
        elif rates_val < -0.50:
            labels.append("rates_low_or_curve_negative")

    if np.isfinite(gold_val):
        if gold_val > 0.50:
            labels.append("gold_up")
        elif gold_val < -0.50:
            labels.append("gold_down")

    if not labels:
        return "mixed_or_neutral"

    return " + ".join(labels)


# =============================================================================
# 3. STATE DISTRIBUTION + STATE MEANS + INTERPRETATION
# =============================================================================

state_distribution_rows = []
state_dominant_rows = []
state_means_raw_rows = []
state_means_z_rows = []
state_interpretation_rows = []
state_quality_rows = []
split_distribution_rows = []

split_label = pd.Series(index=master_df.index, dtype="object")
split_label.loc[split_label.index.intersection(train_df.index)] = "train"
split_label.loc[split_label.index.intersection(val_df.index)] = "val"
split_label.loc[split_label.index.intersection(test_df.index)] = "test"

STATE_MEANS_Z_WIDE = {}
STATE_MEANS_RAW_WIDE = {}
STATE_INTERPRETATION_MAP = {}

for hmm_name, state_ser in STATE_SERIES_MAP.items():

    features = [
        f for f in HMM_FEATURE_MAP.get(hmm_name, [])
        if f in master_df.columns
    ]

    if len(features) == 0:
        print(f"⚠ {hmm_name}: feature list boş, state means hesaplanmayacak.")

    common_index = master_df.index.intersection(state_ser.index)

    report_df = master_df.reindex(common_index).copy()
    report_df["state"] = state_ser.reindex(common_index).ffill().bfill().astype(int)
    report_df["year"] = report_df.index.year
    report_df["split"] = split_label.reindex(common_index)

    # -------------------------------------------------------------------------
    # 3.1 Yearly state distribution
    # -------------------------------------------------------------------------

    yearly_counts = (
        report_df
        .groupby(["year", "state"])
        .size()
        .rename("count")
        .reset_index()
    )

    yearly_total = (
        report_df
        .groupby("year")
        .size()
        .rename("year_total")
        .reset_index()
    )

    yearly_counts = yearly_counts.merge(yearly_total, on="year", how="left")
    yearly_counts["pct"] = yearly_counts["count"] / yearly_counts["year_total"]

    for _, r in yearly_counts.iterrows():
        state_distribution_rows.append({
            "candidate": hmm_name,
            "year": int(r["year"]),
            "state": f"S{int(r['state'])}",
            "state_id": int(r["state"]),
            "count": int(r["count"]),
            "year_total": int(r["year_total"]),
            "pct": float(r["pct"]),
        })

    yearly_wide = (
        yearly_counts
        .pivot_table(index="year", columns="state", values="pct", fill_value=0.0)
        .sort_index()
    )

    yearly_wide.columns = [f"S{int(c)}" for c in yearly_wide.columns]

    for year, row in yearly_wide.iterrows():
        dominant_state = row.idxmax()
        dominant_pct = float(row.max())
        ent = entropy_from_probs(row.values)

        state_dominant_rows.append({
            "candidate": hmm_name,
            "year": int(year),
            "dominant_state": dominant_state,
            "dominant_pct": dominant_pct,
            "state_entropy_norm": ent,
            "one_state_gt_70": bool(dominant_pct >= 0.70),
            "one_state_gt_85": bool(dominant_pct >= 0.85),
        })

    # -------------------------------------------------------------------------
    # 3.2 Split distribution
    # -------------------------------------------------------------------------

    for split_name, split_df in [
        ("train", train_df),
        ("val", val_df),
        ("test", test_df),
    ]:
        s_split = (
            state_ser
            .reindex(split_df.index)
            .ffill()
            .bfill()
            .dropna()
            .astype(int)
        )

        dist = s_split.value_counts(normalize=True).sort_index()
        counts = s_split.value_counts().sort_index()

        for st, p in dist.items():
            split_distribution_rows.append({
                "candidate": hmm_name,
                "split": split_name,
                "state": f"S{int(st)}",
                "state_id": int(st),
                "count": int(counts.loc[st]),
                "pct": float(p),
            })

    # -------------------------------------------------------------------------
    # 3.3 Feature means raw/z
    # -------------------------------------------------------------------------

    if len(features) > 0:

        feat_df = (
            report_df[features + ["state"]]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        if len(feat_df) > 0:

            state_means_raw = feat_df.groupby("state")[features].mean()

            global_mean = feat_df[features].mean()
            global_std = feat_df[features].std().replace(0, np.nan)

            state_means_z = (state_means_raw - global_mean) / global_std
            state_means_z = state_means_z.replace([np.inf, -np.inf], np.nan)

            state_means_raw.index = [f"S{int(i)}" for i in state_means_raw.index]
            state_means_z.index = [f"S{int(i)}" for i in state_means_z.index]

            STATE_MEANS_RAW_WIDE[hmm_name] = state_means_raw
            STATE_MEANS_Z_WIDE[hmm_name] = state_means_z

            for state_label in state_means_raw.index:
                for feat in features:
                    state_means_raw_rows.append({
                        "candidate": hmm_name,
                        "state": state_label,
                        "feature": feat,
                        "mean_raw": float(state_means_raw.loc[state_label, feat]),
                    })

                    state_means_z_rows.append({
                        "candidate": hmm_name,
                        "state": state_label,
                        "feature": feat,
                        "mean_z": float(state_means_z.loc[state_label, feat]),
                    })

            # Interpretation
            state_freq = report_df["state"].value_counts(normalize=True).sort_index()
            state_count = report_df["state"].value_counts().sort_index()

            interp_rows_for_candidate = []

            for state_label in state_means_z.index:
                state_id = int(state_label.replace("S", ""))

                top_feats = top_state_features(state_means_z, state_label, top_n=6)
                label_guess = state_label_guess(state_means_z.loc[state_label])

                freq = float(state_freq.get(state_id, np.nan))
                count = int(state_count.get(state_id, 0))

                interp_row = {
                    "candidate": hmm_name,
                    "state": state_label,
                    "state_id": state_id,
                    "freq_all_sample": freq,
                    "count_all_sample": count,
                    "label_guess": label_guess,
                    "top_features": top_feats,
                }

                state_interpretation_rows.append(interp_row)
                interp_rows_for_candidate.append(interp_row)

            STATE_INTERPRETATION_MAP[hmm_name] = pd.DataFrame(interp_rows_for_candidate)

            # Quality
            full_dist = report_df["state"].value_counts(normalize=True).sort_index()
            min_freq = float(full_dist.min())
            max_freq = float(full_dist.max())
            ent = entropy_from_probs(full_dist.values)
            conc = concentration_score(full_dist.values)
            sep = feature_separation_score(state_means_z)

            meta = HMM_OBJECT_META.get(hmm_name, {})

            warnings_list = []

            if min_freq < 0.03:
                warnings_list.append("very_small_state")

            if max_freq > 0.85:
                warnings_list.append("one_state_dominates")

            if ent < 0.45:
                warnings_list.append("low_state_entropy")

            if np.isfinite(sep) and sep < 0.25:
                warnings_list.append("weak_feature_separation")

            avg_pers = meta.get("avg_persistence", np.nan)

            if np.isfinite(avg_pers) and avg_pers < 0.45:
                warnings_list.append("low_persistence")

            if meta.get("penalty_flag", False):
                warnings_list.append("pipeline_penalty_flag")

            if len(warnings_list) == 0:
                quality_label = "GOOD_OR_ACCEPTABLE"
            elif len(warnings_list) <= 2:
                quality_label = "CHECK_BUT_USABLE"
            else:
                quality_label = "WEAK_OR_UNSTABLE"

            state_quality_rows.append({
                "candidate": hmm_name,
                "k": meta.get("k", report_df["state"].nunique()),
                "n_features": len(features),
                "features_str": ", ".join(features),
                "min_state_freq": min_freq,
                "max_state_freq": max_freq,
                "state_entropy_norm": ent,
                "state_concentration": conc,
                "feature_separation_score": sep,
                "avg_persistence": meta.get("avg_persistence", np.nan),
                "min_persistence": meta.get("min_persistence", np.nan),
                "pipeline_min_state_freq": meta.get("min_state_freq", np.nan),
                "penalty_flag": meta.get("penalty_flag", np.nan),
                "quality_label": quality_label,
                "warnings": ", ".join(warnings_list),
            })

state_distribution_yearly_df = pd.DataFrame(state_distribution_rows)
state_dominant_yearly_df = pd.DataFrame(state_dominant_rows)
state_means_raw_long_df = pd.DataFrame(state_means_raw_rows)
state_means_z_long_df = pd.DataFrame(state_means_z_rows)
state_interpretation_df = pd.DataFrame(state_interpretation_rows)
state_quality_summary_df = pd.DataFrame(state_quality_rows)
split_state_distribution_df = pd.DataFrame(split_distribution_rows)


# =============================================================================
# 4. STATE-CONDITIONAL PERFORMANCE INCLUDING REGIME-FREE
# =============================================================================

state_perf_rows = []

for output_name, obj in EVAL_OUTPUT_STORE.items():

    ensemble_name = obj.get("ensemble_name")
    original_candidate = obj.get("candidate")
    mode = obj.get("mode")

    if obj.get("test_ret") is None or obj.get("test_signal") is None:
        continue

    ret = clean_arr(obj["test_ret"])
    sig = clean_arr(obj["test_signal"])

    if obj.get("test_index") is not None:
        idx = pd.DatetimeIndex(obj["test_index"])
    else:
        idx = test_index[:len(ret)]

    if idx.tz is not None:
        idx = idx.tz_localize(None)

    n = min(len(idx), len(ret), len(sig))
    idx = idx[:n]
    ret = ret[:n]
    sig = sig[:n]

    # HMM mode: use its own candidate states
    if original_candidate not in [None, "C_REGIME_FREE", "REGIME_FREE"]:

        if original_candidate not in STATE_SERIES_MAP:
            print(f"⚠ State series bulunamadı, skip: {output_name}")
            continue

        projected_candidates = [original_candidate]
        output_type = "HMM_MODE"

    # Regime-free: project onto all HMM candidate states
    else:
        projected_candidates = list(STATE_SERIES_MAP.keys())
        output_type = "REGIME_FREE_PROJECTED"

    for projected_candidate in projected_candidates:

        state_ser = (
            STATE_SERIES_MAP[projected_candidate]
            .reindex(idx)
            .ffill()
            .bfill()
        )

        tmp = pd.DataFrame({
            "year": idx.year,
            "state_id": state_ser.values.astype(int),
            "state": [f"S{int(x)}" for x in state_ser.values.astype(int)],
            "ret": ret,
            "signal": sig,
        }, index=idx)

        if output_type == "REGIME_FREE_PROJECTED":
            projected_output_name = (
                f"{ensemble_name}__{projected_candidate}__"
                f"C_REGIME_FREE_PROJECTED_ON_STATES"
            )
            effective_mode = "C_REGIME_FREE_PROJECTED"
        else:
            projected_output_name = output_name
            effective_mode = mode

        # ALL TEST × STATE
        for state_label, g in tmp.groupby("state"):

            r = g["ret"].values
            s = g["signal"].values

            row = {
                "output_name": projected_output_name,
                "source_output_name": output_name,
                "ensemble_name": ensemble_name,
                "candidate": projected_candidate,
                "original_candidate": original_candidate,
                "mode": effective_mode,
                "output_type": output_type,
                "year": "ALL_TEST",
                "state": state_label,
            }

            row.update(metric_pack(r, s))
            state_perf_rows.append(row)

        # YEAR × STATE
        for (year, state_label), g in tmp.groupby(["year", "state"]):

            r = g["ret"].values
            s = g["signal"].values

            row = {
                "output_name": projected_output_name,
                "source_output_name": output_name,
                "ensemble_name": ensemble_name,
                "candidate": projected_candidate,
                "original_candidate": original_candidate,
                "mode": effective_mode,
                "output_type": output_type,
                "year": int(year),
                "state": state_label,
            }

            row.update(metric_pack(r, s))
            state_perf_rows.append(row)

state_conditional_with_regimefree_df = (
    pd.DataFrame(state_perf_rows)
    .sort_values(["candidate", "ensemble_name", "output_type", "output_name", "year", "state"])
    .reset_index(drop=True)
)

if state_conditional_with_regimefree_df.empty:
    raise ValueError("❌ state_conditional_with_regimefree_df boş.")


# =============================================================================
# 5. MERGE STATE MEANS / INTERPRETATION INTO PERFORMANCE TABLE
# =============================================================================

state_info_for_merge = state_interpretation_df.copy()

if not state_info_for_merge.empty:
    state_info_for_merge = state_info_for_merge[
        [
            "candidate",
            "state",
            "freq_all_sample",
            "count_all_sample",
            "label_guess",
            "top_features",
        ]
    ].drop_duplicates()

    state_conditional_with_regimefree_labeled_df = (
        state_conditional_with_regimefree_df
        .merge(
            state_info_for_merge,
            on=["candidate", "state"],
            how="left"
        )
    )
else:
    state_conditional_with_regimefree_labeled_df = state_conditional_with_regimefree_df.copy()


# =============================================================================
# 6. HMM VS REGIME-FREE SAME STATE COMPARISON
# =============================================================================

all_test_df = state_conditional_with_regimefree_labeled_df[
    state_conditional_with_regimefree_labeled_df["year"] == "ALL_TEST"
].copy()

rf_all = all_test_df[
    all_test_df["output_type"] == "REGIME_FREE_PROJECTED"
].copy()

hmm_all = all_test_df[
    all_test_df["output_type"] == "HMM_MODE"
].copy()

comparison_rows = []

for _, hmm_row in hmm_all.iterrows():

    ensemble_name = hmm_row["ensemble_name"]
    candidate = hmm_row["candidate"]
    state = hmm_row["state"]

    rf_match = rf_all[
        (rf_all["ensemble_name"] == ensemble_name)
        & (rf_all["candidate"] == candidate)
        & (rf_all["state"] == state)
    ]

    if rf_match.empty:
        continue

    rf_row = rf_match.iloc[0]

    comparison_rows.append({
        "ensemble_name": ensemble_name,
        "candidate": candidate,
        "state": state,
        "label_guess": hmm_row.get("label_guess", np.nan),
        "top_features": hmm_row.get("top_features", np.nan),
        "freq_all_sample": hmm_row.get("freq_all_sample", np.nan),
        "count_all_sample": hmm_row.get("count_all_sample", np.nan),

        "hmm_output_name": hmm_row["output_name"],
        "hmm_mode": hmm_row["mode"],
        "rf_output_name": rf_row["output_name"],

        "hmm_cum_ret": hmm_row["cum_ret"],
        "rf_cum_ret": rf_row["cum_ret"],
        "delta_cum_ret_hmm_minus_rf": hmm_row["cum_ret"] - rf_row["cum_ret"],

        "hmm_ann_ret": hmm_row["ann_ret"],
        "rf_ann_ret": rf_row["ann_ret"],
        "delta_ann_ret_hmm_minus_rf": hmm_row["ann_ret"] - rf_row["ann_ret"],

        "hmm_sharpe": hmm_row["sharpe"],
        "rf_sharpe": rf_row["sharpe"],
        "delta_sharpe_hmm_minus_rf": hmm_row["sharpe"] - rf_row["sharpe"],

        "hmm_max_dd": hmm_row["max_dd"],
        "rf_max_dd": rf_row["max_dd"],
        "delta_max_dd_hmm_minus_rf": hmm_row["max_dd"] - rf_row["max_dd"],

        "hmm_activity": hmm_row["activity"],
        "rf_activity": rf_row["activity"],
        "delta_activity_hmm_minus_rf": hmm_row["activity"] - rf_row["activity"],

        "hmm_trades": hmm_row["trades"],
        "rf_trades": rf_row["trades"],
        "delta_trades_hmm_minus_rf": hmm_row["trades"] - rf_row["trades"],
    })

hmm_vs_rf_same_state_labeled_df = (
    pd.DataFrame(comparison_rows)
    .sort_values(["delta_sharpe_hmm_minus_rf", "delta_cum_ret_hmm_minus_rf"], ascending=[False, False])
    .reset_index(drop=True)
)


# =============================================================================
# 7. YEAR × STATE COMPARISON WITH LABELS
# =============================================================================

rf_year = state_conditional_with_regimefree_labeled_df[
    (state_conditional_with_regimefree_labeled_df["output_type"] == "REGIME_FREE_PROJECTED")
    & (state_conditional_with_regimefree_labeled_df["year"] != "ALL_TEST")
].copy()

hmm_year = state_conditional_with_regimefree_labeled_df[
    (state_conditional_with_regimefree_labeled_df["output_type"] == "HMM_MODE")
    & (state_conditional_with_regimefree_labeled_df["year"] != "ALL_TEST")
].copy()

yearly_comparison_rows = []

for _, hmm_row in hmm_year.iterrows():

    ensemble_name = hmm_row["ensemble_name"]
    candidate = hmm_row["candidate"]
    state = hmm_row["state"]
    year = hmm_row["year"]

    rf_match = rf_year[
        (rf_year["ensemble_name"] == ensemble_name)
        & (rf_year["candidate"] == candidate)
        & (rf_year["state"] == state)
        & (rf_year["year"] == year)
    ]

    if rf_match.empty:
        continue

    rf_row = rf_match.iloc[0]

    yearly_comparison_rows.append({
        "ensemble_name": ensemble_name,
        "candidate": candidate,
        "year": int(year),
        "state": state,
        "label_guess": hmm_row.get("label_guess", np.nan),
        "top_features": hmm_row.get("top_features", np.nan),

        "hmm_output_name": hmm_row["output_name"],
        "hmm_mode": hmm_row["mode"],

        "hmm_cum_ret": hmm_row["cum_ret"],
        "rf_cum_ret": rf_row["cum_ret"],
        "delta_cum_ret_hmm_minus_rf": hmm_row["cum_ret"] - rf_row["cum_ret"],

        "hmm_sharpe": hmm_row["sharpe"],
        "rf_sharpe": rf_row["sharpe"],
        "delta_sharpe_hmm_minus_rf": hmm_row["sharpe"] - rf_row["sharpe"],

        "hmm_activity": hmm_row["activity"],
        "rf_activity": rf_row["activity"],
        "delta_activity_hmm_minus_rf": hmm_row["activity"] - rf_row["activity"],

        "hmm_trades": hmm_row["trades"],
        "rf_trades": rf_row["trades"],
        "delta_trades_hmm_minus_rf": hmm_row["trades"] - rf_row["trades"],
    })

hmm_vs_rf_year_state_labeled_df = (
    pd.DataFrame(yearly_comparison_rows)
    .sort_values(["year", "candidate", "ensemble_name", "state", "delta_sharpe_hmm_minus_rf"])
    .reset_index(drop=True)
)


# =============================================================================
# 8. PIVOTS
# =============================================================================

# State distribution pivot
state_distribution_pivot = (
    state_distribution_yearly_df
    .pivot_table(
        index=["candidate", "year"],
        columns="state",
        values="pct",
        fill_value=0.0
    )
    .reset_index()
)

# All-test state cum return including RF
state_cumret_all_with_rf_pivot = (
    all_test_df
    .pivot_table(
        index="output_name",
        columns="state",
        values="cum_ret",
        aggfunc="first"
    )
)

state_sharpe_all_with_rf_pivot = (
    all_test_df
    .pivot_table(
        index="output_name",
        columns="state",
        values="sharpe",
        aggfunc="first"
    )
)

state_activity_all_with_rf_pivot = (
    all_test_df
    .pivot_table(
        index="output_name",
        columns="state",
        values="activity",
        aggfunc="first"
    )
)

# Year-state delta pivots
if not hmm_vs_rf_year_state_labeled_df.empty:
    year_state_delta_cumret_pivot = (
        hmm_vs_rf_year_state_labeled_df
        .assign(year_state=lambda x: x["year"].astype(str) + "_" + x["state"])
        .pivot_table(
            index="hmm_output_name",
            columns="year_state",
            values="delta_cum_ret_hmm_minus_rf",
            aggfunc="first"
        )
    )

    year_state_delta_sharpe_pivot = (
        hmm_vs_rf_year_state_labeled_df
        .assign(year_state=lambda x: x["year"].astype(str) + "_" + x["state"])
        .pivot_table(
            index="hmm_output_name",
            columns="year_state",
            values="delta_sharpe_hmm_minus_rf",
            aggfunc="first"
        )
    )
else:
    year_state_delta_cumret_pivot = pd.DataFrame()
    year_state_delta_sharpe_pivot = pd.DataFrame()

# State mean z wide combined
state_means_z_combined_rows = []

for candidate, zdf in STATE_MEANS_Z_WIDE.items():
    tmp = zdf.copy()
    tmp.insert(0, "state", tmp.index)
    tmp.insert(0, "candidate", candidate)
    state_means_z_combined_rows.append(tmp.reset_index(drop=True))

state_means_z_wide_combined_df = (
    pd.concat(state_means_z_combined_rows, axis=0, ignore_index=True)
    if state_means_z_combined_rows
    else pd.DataFrame()
)

state_means_raw_combined_rows = []

for candidate, rdf in STATE_MEANS_RAW_WIDE.items():
    tmp = rdf.copy()
    tmp.insert(0, "state", tmp.index)
    tmp.insert(0, "candidate", candidate)
    state_means_raw_combined_rows.append(tmp.reset_index(drop=True))

state_means_raw_wide_combined_df = (
    pd.concat(state_means_raw_combined_rows, axis=0, ignore_index=True)
    if state_means_raw_combined_rows
    else pd.DataFrame()
)


# =============================================================================
# 9. SUMMARY TABLES
# =============================================================================

if not hmm_vs_rf_same_state_labeled_df.empty:

    hmm_vs_rf_mode_summary_labeled_df = (
        hmm_vs_rf_same_state_labeled_df
        .groupby(["candidate", "hmm_mode"], as_index=False)
        .agg(
            n_state_cells=("state", "count"),
            avg_delta_sharpe=("delta_sharpe_hmm_minus_rf", "mean"),
            median_delta_sharpe=("delta_sharpe_hmm_minus_rf", "median"),
            best_delta_sharpe=("delta_sharpe_hmm_minus_rf", "max"),
            worst_delta_sharpe=("delta_sharpe_hmm_minus_rf", "min"),

            avg_delta_cum_ret=("delta_cum_ret_hmm_minus_rf", "mean"),
            median_delta_cum_ret=("delta_cum_ret_hmm_minus_rf", "median"),
            best_delta_cum_ret=("delta_cum_ret_hmm_minus_rf", "max"),
            worst_delta_cum_ret=("delta_cum_ret_hmm_minus_rf", "min"),

            avg_delta_activity=("delta_activity_hmm_minus_rf", "mean"),
            avg_delta_trades=("delta_trades_hmm_minus_rf", "mean"),
        )
        .sort_values(["avg_delta_sharpe", "avg_delta_cum_ret"], ascending=[False, False])
        .reset_index(drop=True)
    )

    hmm_vs_rf_ensemble_summary_labeled_df = (
        hmm_vs_rf_same_state_labeled_df
        .groupby(["ensemble_name", "candidate", "hmm_mode"], as_index=False)
        .agg(
            n_state_cells=("state", "count"),
            avg_delta_sharpe=("delta_sharpe_hmm_minus_rf", "mean"),
            median_delta_sharpe=("delta_sharpe_hmm_minus_rf", "median"),
            best_delta_sharpe=("delta_sharpe_hmm_minus_rf", "max"),
            worst_delta_sharpe=("delta_sharpe_hmm_minus_rf", "min"),

            avg_delta_cum_ret=("delta_cum_ret_hmm_minus_rf", "mean"),
            median_delta_cum_ret=("delta_cum_ret_hmm_minus_rf", "median"),
            best_delta_cum_ret=("delta_cum_ret_hmm_minus_rf", "max"),
            worst_delta_cum_ret=("delta_cum_ret_hmm_minus_rf", "min"),

            avg_delta_activity=("delta_activity_hmm_minus_rf", "mean"),
            avg_delta_trades=("delta_trades_hmm_minus_rf", "mean"),
        )
        .sort_values(["avg_delta_sharpe", "avg_delta_cum_ret"], ascending=[False, False])
        .reset_index(drop=True)
    )

else:
    hmm_vs_rf_mode_summary_labeled_df = pd.DataFrame()
    hmm_vs_rf_ensemble_summary_labeled_df = pd.DataFrame()


# =============================================================================
# 10. SAVE CSV OUTPUTS
# =============================================================================

state_distribution_yearly_df.to_csv(
    os.path.join(REPORT_DIR, "01_state_distribution_yearly_long.csv"),
    index=False,
    encoding="utf-8-sig"
)

state_distribution_pivot.to_csv(
    os.path.join(REPORT_DIR, "02_state_distribution_yearly_pivot.csv"),
    index=False,
    encoding="utf-8-sig"
)

state_dominant_yearly_df.to_csv(
    os.path.join(REPORT_DIR, "03_state_dominant_yearly.csv"),
    index=False,
    encoding="utf-8-sig"
)

split_state_distribution_df.to_csv(
    os.path.join(REPORT_DIR, "04_split_state_distribution.csv"),
    index=False,
    encoding="utf-8-sig"
)

state_means_raw_long_df.to_csv(
    os.path.join(REPORT_DIR, "05_state_feature_means_raw_long.csv"),
    index=False,
    encoding="utf-8-sig"
)

state_means_z_long_df.to_csv(
    os.path.join(REPORT_DIR, "06_state_feature_means_z_long.csv"),
    index=False,
    encoding="utf-8-sig"
)

state_means_raw_wide_combined_df.to_csv(
    os.path.join(REPORT_DIR, "07_state_feature_means_raw_wide.csv"),
    index=False,
    encoding="utf-8-sig"
)

state_means_z_wide_combined_df.to_csv(
    os.path.join(REPORT_DIR, "08_state_feature_means_z_wide.csv"),
    index=False,
    encoding="utf-8-sig"
)

state_interpretation_df.to_csv(
    os.path.join(REPORT_DIR, "09_state_interpretation_summary.csv"),
    index=False,
    encoding="utf-8-sig"
)

state_quality_summary_df.to_csv(
    os.path.join(REPORT_DIR, "10_state_quality_summary.csv"),
    index=False,
    encoding="utf-8-sig"
)

state_conditional_with_regimefree_labeled_df.to_csv(
    os.path.join(REPORT_DIR, "11_state_conditional_performance_with_regimefree_labeled_long.csv"),
    index=False,
    encoding="utf-8-sig"
)

state_cumret_all_with_rf_pivot.to_csv(
    os.path.join(REPORT_DIR, "12_state_cumret_all_with_regimefree_pivot.csv"),
    encoding="utf-8-sig"
)

state_sharpe_all_with_rf_pivot.to_csv(
    os.path.join(REPORT_DIR, "13_state_sharpe_all_with_regimefree_pivot.csv"),
    encoding="utf-8-sig"
)

state_activity_all_with_rf_pivot.to_csv(
    os.path.join(REPORT_DIR, "14_state_activity_all_with_regimefree_pivot.csv"),
    encoding="utf-8-sig"
)

hmm_vs_rf_same_state_labeled_df.to_csv(
    os.path.join(REPORT_DIR, "15_hmm_vs_regimefree_same_state_labeled_all_test.csv"),
    index=False,
    encoding="utf-8-sig"
)

hmm_vs_rf_year_state_labeled_df.to_csv(
    os.path.join(REPORT_DIR, "16_hmm_vs_regimefree_year_state_labeled.csv"),
    index=False,
    encoding="utf-8-sig"
)

year_state_delta_cumret_pivot.to_csv(
    os.path.join(REPORT_DIR, "17_year_state_delta_cumret_hmm_minus_rf_pivot.csv"),
    encoding="utf-8-sig"
)

year_state_delta_sharpe_pivot.to_csv(
    os.path.join(REPORT_DIR, "18_year_state_delta_sharpe_hmm_minus_rf_pivot.csv"),
    encoding="utf-8-sig"
)

hmm_vs_rf_mode_summary_labeled_df.to_csv(
    os.path.join(REPORT_DIR, "19_hmm_vs_regimefree_mode_summary_labeled.csv"),
    index=False,
    encoding="utf-8-sig"
)

hmm_vs_rf_ensemble_summary_labeled_df.to_csv(
    os.path.join(REPORT_DIR, "20_hmm_vs_regimefree_ensemble_summary_labeled.csv"),
    index=False,
    encoding="utf-8-sig"
)


# =============================================================================
# 11. DISPLAY
# =============================================================================

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 360)

print("\n" + "═" * 180)
print("1) YEARLY STATE DISTRIBUTION PIVOT (%)")
print("Hangi yıl hangi state baskın?")
print("═" * 180)

display_state_dist = state_distribution_pivot.copy()

for c in display_state_dist.columns:
    if str(c).startswith("S"):
        display_state_dist[c] = display_state_dist[c] * 100

display(display_state_dist.round(2))


print("\n" + "═" * 180)
print("2) YEARLY DOMINANT STATE")
print("Yıl bazında dominant state + entropy")
print("═" * 180)

display_dom = state_dominant_yearly_df.copy()
display_dom["dominant_pct"] = display_dom["dominant_pct"] * 100
display(display_dom.round(3))


print("\n" + "═" * 180)
print("3) STATE FEATURE MEANS — Z-SCORE WIDE")
print("State'lerin makro/finansal karakteri. Pozitif/negatif ayrışma yorum için ana tablo.")
print("═" * 180)

display(state_means_z_wide_combined_df.round(3))


print("\n" + "═" * 180)
print("4) STATE INTERPRETATION SUMMARY")
print("State label guess + en ayırt edici feature'lar")
print("═" * 180)

display_interp = state_interpretation_df.copy()
if "freq_all_sample" in display_interp.columns:
    display_interp["freq_all_sample"] = display_interp["freq_all_sample"] * 100
display(display_interp.round(3))


print("\n" + "═" * 180)
print("5) STATE QUALITY SUMMARY")
print("State dağılımı, entropy, persistence ve feature ayrışması")
print("═" * 180)

display_quality = state_quality_summary_df.copy()

for c in ["min_state_freq", "max_state_freq", "state_concentration", "pipeline_min_state_freq"]:
    if c in display_quality.columns:
        display_quality[c] = display_quality[c] * 100

display(display_quality.round(3))


print("\n" + "═" * 180)
print("6) STATE CONDITIONAL CUMULATIVE RETURN INCLUDING REGIME-FREE (%)")
print("Regime-free de aynı HMM state'leri içine projekte edildi.")
print("═" * 180)

display_cum = state_cumret_all_with_rf_pivot.copy() * 100
display(display_cum.round(2))


print("\n" + "═" * 180)
print("7) STATE CONDITIONAL SHARPE INCLUDING REGIME-FREE")
print("Her state içinde HMM output ve projected regime-free Sharpe")
print("═" * 180)

display(state_sharpe_all_with_rf_pivot.round(3))


print("\n" + "═" * 180)
print("8) HMM VS REGIME-FREE SAME STATE — LABELED ALL TEST")
print("Pozitif delta: HMM aynı state içinde regime-free'den daha iyi.")
print("═" * 180)

display_comp = hmm_vs_rf_same_state_labeled_df.copy()

percent_cols = [
    "freq_all_sample",
    "hmm_cum_ret",
    "rf_cum_ret",
    "delta_cum_ret_hmm_minus_rf",
    "hmm_ann_ret",
    "rf_ann_ret",
    "delta_ann_ret_hmm_minus_rf",
    "hmm_max_dd",
    "rf_max_dd",
    "delta_max_dd_hmm_minus_rf",
    "hmm_activity",
    "rf_activity",
    "delta_activity_hmm_minus_rf",
]

for c in percent_cols:
    if c in display_comp.columns:
        display_comp[c] = display_comp[c] * 100

display_cols = [
    "ensemble_name",
    "candidate",
    "state",
    "label_guess",
    "top_features",
    "freq_all_sample",
    "hmm_mode",
    "hmm_cum_ret",
    "rf_cum_ret",
    "delta_cum_ret_hmm_minus_rf",
    "hmm_sharpe",
    "rf_sharpe",
    "delta_sharpe_hmm_minus_rf",
    "hmm_max_dd",
    "rf_max_dd",
    "hmm_activity",
    "rf_activity",
    "hmm_trades",
    "rf_trades",
]

display_cols = [c for c in display_cols if c in display_comp.columns]

display(display_comp[display_cols].round(3))


print("\n" + "═" * 180)
print("9) YEAR × STATE DELTA CUMULATIVE RETURN PIVOT (%) — HMM MINUS REGIME-FREE")
print("Pozitif değer: ilgili yıl-state hücresinde HMM regime-free'den daha iyi.")
print("═" * 180)

display_delta_ret = year_state_delta_cumret_pivot.copy() * 100
display(display_delta_ret.round(2))


print("\n" + "═" * 180)
print("10) YEAR × STATE DELTA SHARPE PIVOT — HMM MINUS REGIME-FREE")
print("Pozitif değer: ilgili yıl-state hücresinde HMM Sharpe olarak daha iyi.")
print("═" * 180)

display(year_state_delta_sharpe_pivot.round(3))


print("\n" + "═" * 180)
print("11) HMM VS REGIME-FREE MODE SUMMARY")
print("Hangi HMM mode state içinde RF'e göre daha çok değer katmış?")
print("═" * 180)

display_mode = hmm_vs_rf_mode_summary_labeled_df.copy()

for c in [
    "avg_delta_cum_ret",
    "median_delta_cum_ret",
    "best_delta_cum_ret",
    "worst_delta_cum_ret",
    "avg_delta_activity",
]:
    if c in display_mode.columns:
        display_mode[c] = display_mode[c] * 100

display(display_mode.round(4))


print("\n" + "═" * 180)
print("12) HMM VS REGIME-FREE ENSEMBLE SUMMARY")
print("Hangi ensemble + candidate + mode RF karşısında daha iyi?")
print("═" * 180)

display_ens = hmm_vs_rf_ensemble_summary_labeled_df.copy()

for c in [
    "avg_delta_cum_ret",
    "median_delta_cum_ret",
    "best_delta_cum_ret",
    "worst_delta_cum_ret",
    "avg_delta_activity",
]:
    if c in display_ens.columns:
        display_ens[c] = display_ens[c] * 100

display(display_ens.round(4))


print("\n" + "═" * 120)
print("✅ FULL STATE INTERPRETATION + REGIME-FREE COMPARISON REPORT TAMAMLANDI")
print(f"📁 Kayıt klasörü: {REPORT_DIR}/")
print("Ana dosyalar:")
print("  01_state_distribution_yearly_long.csv")
print("  02_state_distribution_yearly_pivot.csv")
print("  03_state_dominant_yearly.csv")
print("  08_state_feature_means_z_wide.csv")
print("  09_state_interpretation_summary.csv")
print("  10_state_quality_summary.csv")
print("  11_state_conditional_performance_with_regimefree_labeled_long.csv")
print("  15_hmm_vs_regimefree_same_state_labeled_all_test.csv")
print("  16_hmm_vs_regimefree_year_state_labeled.csv")
print("  17_year_state_delta_cumret_hmm_minus_rf_pivot.csv")
print("  18_year_state_delta_sharpe_hmm_minus_rf_pivot.csv")
print("═" * 120)

════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
FULL STATE INTERPRETATION + REGIME-FREE COMPARISON REPORT
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Report dir: multi_ens_fixed_hmm_outputs/FULL_STATE_INTERPRETATION_WITH_REGIMEFREE_REPORT
Test period: 2021-05-04 04:00:00 → 2025-12-31 16:00:00
EVAL_OUTPUT_STORE outputs: 30
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

✅ Loaded HMM states:
  HMM_CLASSIC_LITERATURE_CORE         len=  37612 | states=[0, 1, 2, 3, 4] | features=7
  HMM_FULL_CORE_K5                    len=  37612 | states=[0, 1, 2, 3, 4] | features=10
  HMM_FINANCIAL_CONDITIONS

state,candidate,year,S0,S1,S2,S3,S4
0,HMM_CLASSIC_LITERATURE_CORE,2004,1.72,81.28,0.00,17.00,0.00
1,HMM_CLASSIC_LITERATURE_CORE,2005,0.00,0.00,0.00,9.39,90.61
2,HMM_CLASSIC_LITERATURE_CORE,2006,0.00,0.00,0.00,0.00,100.00
3,HMM_CLASSIC_LITERATURE_CORE,2007,0.00,0.00,0.00,7.82,92.18
4,HMM_CLASSIC_LITERATURE_CORE,2008,0.00,0.00,4.65,95.35,0.00
5,HMM_CLASSIC_LITERATURE_CORE,2009,0.00,55.74,12.46,31.79,0.00
6,HMM_CLASSIC_LITERATURE_CORE,2010,0.00,18.46,81.54,0.00,0.00
7,HMM_CLASSIC_LITERATURE_CORE,2011,0.00,0.00,74.08,25.92,0.00
8,HMM_CLASSIC_LITERATURE_CORE,2012,35.28,58.10,0.00,6.62,0.00
9,HMM_CLASSIC_LITERATURE_CORE,2013,86.41,7.00,3.89,2.70,0.00



════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
2) YEARLY DOMINANT STATE
Yıl bazında dominant state + entropy
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


,candidate,year,dominant_state,dominant_pct,state_entropy_norm,one_state_gt_70,one_state_gt_85
0,HMM_CLASSIC_LITERATURE_CORE,2004,S1,81.281,0.491,True,False
1,HMM_CLASSIC_LITERATURE_CORE,2005,S4,90.606,0.449,True,True
2,HMM_CLASSIC_LITERATURE_CORE,2006,S4,100.000,0.000,True,True
3,HMM_CLASSIC_LITERATURE_CORE,2007,S4,92.176,0.396,True,True
4,HMM_CLASSIC_LITERATURE_CORE,2008,S3,95.355,0.271,True,True
5,HMM_CLASSIC_LITERATURE_CORE,2009,S1,55.745,0.864,False,False
6,HMM_CLASSIC_LITERATURE_CORE,2010,S2,81.545,0.690,True,False
7,HMM_CLASSIC_LITERATURE_CORE,2011,S2,74.084,0.825,True,False
8,HMM_CLASSIC_LITERATURE_CORE,2012,S1,58.096,0.785,False,False
9,HMM_CLASSIC_LITERATURE_CORE,2013,S0,86.407,0.387,True,True



════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
3) STATE FEATURE MEANS — Z-SCORE WIDE
State'lerin makro/finansal karakteri. Pozitif/negatif ayrışma yorum için ana tablo.
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


...


════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
11) HMM VS REGIME-FREE MODE SUMMARY
Hangi HMM mode state içinde RF'e göre daha çok değer katmış?
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


,candidate,hmm_mode,n_state_cells,avg_delta_sharpe,median_delta_sharpe,best_delta_sharpe,worst_delta_sharpe,avg_delta_cum_ret,median_delta_cum_ret,best_delta_cum_ret,worst_delta_cum_ret,avg_delta_activity,avg_delta_trades
0,HMM_CLASSIC_LITERATURE_CORE,A_STATE_WEIGHTS_GLOBAL_THRESHOLD,6,0.2708,0.1828,0.9601,-0.1150,3.8796,2.4444,14.9305,-1.5122,0.4864,-2.1667
1,HMM_CLASSIC_LITERATURE_CORE,B_STATE_WEIGHTS_STATE_THRESHOLD,6,0.1619,0.1228,0.4785,-0.0858,4.8435,1.1463,28.2577,-7.4452,0.4468,2.8333
2,HMM_FULL_CORE_K5,B_STATE_WEIGHTS_STATE_THRESHOLD,15,0.1138,0.3593,1.2929,-3.1200,3.5032,0.6395,54.4225,-19.0687,-1.0989,-14.0000
3,HMM_FULL_CORE_K5,D_POSTERIOR_WEIGHTED,15,-0.0950,0.1072,1.8495,-1.8373,0.6149,0.2061,15.6032,-12.7256,7.7543,10.2667
4,HMM_CLASSIC_LITERATURE_CORE,D_POSTERIOR_WEIGHTED,9,-0.3395,-0.0047,1.7390,-4.6410,0.0420,0.5250,7.3789,-8.1764,16.3890,-5.0000
5,HMM_FULL_CORE_K5,A_STATE_WEIGHTS_GLOBAL_THRESHOLD,15,-0.4640,-0.0288,1.0985,-3.3173,0.5337,-0.1187,6.2954,-3.0762,1.2280,11.0000
6,HMM_FINANCIAL_CONDITIONS_K5,A_STATE_WEIGHTS_GLOBAL_THRESHOLD,9,-1.9192,-0.0666,12.7966,-19.4784,1.9876,0.0957,20.3635,-6.4527,7.8673,18.2222
7,HMM_FINANCIAL_CONDITIONS_K5,D_POSTERIOR_WEIGHTED,9,-4.8556,-0.1007,0.7335,-44.6494,-0.8594,-0.1626,2.5419,-4.1621,-14.4361,-78.3333
8,HMM_FINANCIAL_CONDITIONS_K5,B_STATE_WEIGHTS_STATE_THRESHOLD,9,-5.3296,0.5334,11.0177,-48.9595,9.4101,0.1104,61.3100,-6.7743,-4.7050,4.6667



════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
12) HMM VS REGIME-FREE ENSEMBLE SUMMARY
Hangi ensemble + candidate + mode RF karşısında daha iyi?
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


,ensemble_name,candidate,hmm_mode,n_state_cells,avg_delta_sharpe,median_delta_sharpe,best_delta_sharpe,worst_delta_sharpe,avg_delta_cum_ret,median_delta_cum_ret,best_delta_cum_ret,worst_delta_cum_ret,avg_delta_activity,avg_delta_trades
0,ENSEMBLE_B,HMM_FINANCIAL_CONDITIONS_K5,B_STATE_WEIGHTS_STATE_THRESHOLD,3,4.1662,0.7477,11.0177,0.7333,3.1812,3.6552,5.7779,0.1104,-4.4992,-79.6667
1,ENSEMBLE_B,HMM_FINANCIAL_CONDITIONS_K5,A_STATE_WEIGHTS_GLOBAL_THRESHOLD,3,4.1628,0.5493,12.7966,-0.8577,4.6893,0.1572,20.3635,-6.4527,12.9557,18.3333
2,MAIN_14,HMM_FULL_CORE_K5,B_STATE_WEIGHTS_STATE_THRESHOLD,5,0.7695,1.0073,1.2929,0.1303,11.0618,0.8620,54.4225,-1.9450,9.9023,29.8000
3,THEORY_11,HMM_CLASSIC_LITERATURE_CORE,D_POSTERIOR_WEIGHTED,3,0.6487,0.2117,1.7390,-0.0047,-2.2862,0.5250,0.7928,-8.1764,-10.8047,-69.3333
4,THEORY_11,HMM_FULL_CORE_K5,B_STATE_WEIGHTS_STATE_THRESHOLD,5,0.4586,0.8506,1.1622,-1.4610,4.3272,0.7380,22.2460,-4.2230,-0.1667,-3.6000
5,MAIN_14,HMM_CLASSIC_LITERATURE_CORE,A_STATE_WEIGHTS_GLOBAL_THRESHOLD,2,0.4411,0.4411,0.9601,-0.0778,0.8562,0.8562,3.2245,-1.5122,0.0359,13.5000
6,ENSEMBLE_B,HMM_CLASSIC_LITERATURE_CORE,A_STATE_WEIGHTS_GLOBAL_THRESHOLD,2,0.4203,0.4203,0.4919,0.3486,8.2974,8.2974,14.9305,1.6642,0.9509,-40.5000
7,ENSEMBLE_B,HMM_FINANCIAL_CONDITIONS_K5,D_POSTERIOR_WEIGHTED,3,0.4156,0.6140,0.7335,-0.1007,0.9857,0.4113,2.5419,0.0040,-20.2196,-90.6667
8,MAIN_14,HMM_FULL_CORE_K5,D_POSTERIOR_WEIGHTED,5,0.4046,0.1150,1.8495,-0.5476,0.7298,0.7176,3.2244,-2.9485,-0.0078,-20.4000
9,MAIN_14,HMM_CLASSIC_LITERATURE_CORE,B_STATE_WEIGHTS_STATE_THRESHOLD,2,0.1964,0.1964,0.4785,-0.0858,14.9917,14.9917,28.2577,1.7258,16.4828,70.5000



════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
✅ FULL STATE INTERPRETATION + REGIME-FREE COMPARISON REPORT TAMAMLANDI
📁 Kayıt klasörü: multi_ens_fixed_hmm_outputs/FULL_STATE_INTERPRETATION_WITH_REGIMEFREE_REPORT/
Ana dosyalar:
  01_state_distribution_yearly_long.csv
  02_state_distribution_yearly_pivot.csv
  03_state_dominant_yearly.csv
  08_state_feature_means_z_wide.csv
  09_state_interpretation_summary.csv
  10_state_quality_summary.csv
  11_state_conditional_performance_with_regimefree_labeled_long.csv
  15_hmm_vs_regimefree_same_state_labeled_all_test.csv
  16_hmm_vs_regimefree_year_state_labeled.csv
  17_year_state_delta_cumret_hmm_minus_rf_pivot.csv
  18_year_state_delta_sharpe_hmm_minus_rf_pivot.csv
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


In [ ]:
import numpy as np
import pandas as pd
from scipy.special import logsumexp

# Causal forward-only alpha
def causal_states(model, X):
    X = np.asarray(X, dtype=float)
    log_emlik = model._compute_log_likelihood(X)
    n_obs, n_st = log_emlik.shape
    log_start = np.log(np.maximum(model.startprob_, 1e-300))
    log_trans = np.log(np.maximum(model.transmat_,  1e-300))
    log_alpha = np.zeros((n_obs, n_st))
    log_alpha[0] = log_start + log_emlik[0]
    log_alpha[0] -= logsumexp(log_alpha[0])
    for t in range(1, n_obs):
        pred = logsumexp(log_alpha[t-1][:, None] + log_trans, axis=0)
        log_alpha[t] = pred + log_emlik[t]
        log_alpha[t] -= logsumexp(log_alpha[t])
    return np.argmax(np.exp(log_alpha), axis=1)

# Viterbi
def viterbi_states(model, X):
    _, states = model.decode(np.asarray(X, dtype=float), algorithm="viterbi")
    return states

# Test — HMM_FULL_CORE_K5 üzerinde
hmm  = HMM_OBJECTS["HMM_FULL_CORE_K5"]
model = hmm["model"]

# Tüm data
from sklearn.preprocessing import RobustScaler
feats = hmm["features"]
all_index = train_df.index.union(val_df.index).union(test_df.index).sort_values()
X_all = master_df.reindex(all_index)[feats].ffill().fillna(0).values.astype(float)

scaler = RobustScaler()
scaler.fit(master_df.reindex(train_df.index)[feats].ffill().dropna().values.astype(float))
X_sc = scaler.transform(X_all)
X_sc = np.nan_to_num(X_sc)

s_causal  = causal_states(model, X_sc)
s_viterbi = viterbi_states(model, X_sc)

disagree = (s_causal != s_viterbi).mean()
print(f"Disagreement: {disagree:.4%}  ({(s_causal != s_viterbi).sum():,} / {len(s_causal):,} bar)")

# Val/test bazında ayrı
idx_val  = all_index.isin(val_df.index)
idx_test = all_index.isin(test_df.index)
print(f"Val  disagreement: {(s_causal[idx_val]  != s_viterbi[idx_val]).mean():.4%}")
print(f"Test disagreement: {(s_causal[idx_test] != s_viterbi[idx_test]).mean():.4%}")

Disagreement: 3.8419%  (1,445 / 37,612 bar)
Val  disagreement: 5.1449%
Test disagreement: 3.6953%
